# OsmoFlux — Reproducibility Notebook

**Paper:** Fresh-Dilute Cascade Architecture for Compliance-Constrained Reverse Electrodialysis of SWRO Brine  
**Author:** Rohan Agarwal, Blue Leaf Labs Independent Research, Cupertino, CA  
**ORCID:** 0009-0005-7518-0805  
**Repository:** https://github.com/blueleaflabs/osmoflux  

---

## How to reproduce

This notebook reproduces all Grand Sweep v12 model outputs and submission figures from scratch.

**Steps:**
1. Run cells 1–19 in order (model foundation: constants, physics, harness). These execute in under a second and define all required functions.
2. Run cell 20 (Grand Sweep v12 runner). This is the full compute step — expected runtime ~30–60 min on a modern laptop using `multiprocessing.Pool`.
3. Run cell 21 (Figure Generator v7). Reads the parquets written by cell 20 and writes all figures to `runs/GRAND-SWEEP/<RUN_TIMESTAMP>/figures/`.

**Canonical run timestamp:** `20260531_183521`  
**Output directory:** `runs/GRAND-SWEEP/20260531_183521/`

**SHA-256 verification** (compare your outputs against the canonical run):
```
cascade_sweep.parquet:      b041a0410cf8b840191de70a37b0667633d0592f0acfa2797ddbc1078594a641
single_stage_sweep.parquet: 15ea8ae74121d4a8b7f410f4bcae09399cf00e17c4bac35e6a5d85ed27ac1604
morris_all.parquet:         17f4a7e3af961929c6574fbb44b0b50efc11cf687ac3d36bb8b7986b06167ff7
```

**macOS:** use `shasum -a 256 <file>` — **Linux:** use `sha256sum <file>`

---

> **Note:** The Monte Carlo TEA cell is intentionally excluded from this notebook.
> MC-TEA results are not cited in the paper and do not underlie any reported claim.
> See `GRAND-SWEEP-MANIFEST.json` for full run provenance.


OsmoFlux notebook

This notebook contains multiple electrical/hydraulic abstractions of a Reverse Electrodialysis (RED) stack. All of them share the same high-level physical ingredients:
	•	A salinity gradient produces an electromotive force (EMF / “OCV”) via Nernst-type relations.
	•	Current flowing through the stack causes (i) ohmic losses and (ii) salt transport that changes concentrations along the flow direction.
	•	Power delivered to an external load depends on the operating point set by the load (or MPPT).

The models differ mainly in how they treat axial variation and electrical topology.
Additional comment

⸻

Coordinate picture and topology (what “axial” means here)
	•	Flow evolves along a single “axial” direction (call it y): inlet → outlet.
	•	The model assumes uniformity across the membrane span (x) and across the stacking direction (z) within each axial slice.
	•	Cell pairs are stacked in the z direction between one electrode pair at the ends of the stack.

Hydraulics (common assumption used here):
	•	Cell pairs are hydraulic parallel (manifolds distribute the streams across pairs).
	•	Axial slices are hydraulic series (bulk concentrations evolve slice-by-slice along the flow path).

⸻

Core input variables and conventions (shared)

Area and scale:
	•	A_MEM_M2 = active membrane area per cell pair m² / cell pair.
	•	N_CELL_PAIRS = number of cell pairs in electrical series.
	•	Total active area used for power density: A_total = A_MEM_M2 * N_CELL_PAIRS.

Flow convention:
	•	Q_H_M3_S, Q_L_M3_S are total stack volumetric flow rates m³/s, not per pair.
	•	If per-pair flow is needed later: Q_per_pair = Q_total / N_CELL_PAIRS.

Axial discretization:
	•	N_SEG = number of axial segments used to march concentrations along the flow direction.
	•	Each segment represents a control volume for advection + salt transfer updates.

Outputs (common):
	•	V_TERM_V terminal voltage at the electrodes V
	•	I_TOTAL_A total stack current A
	•	P_TOTAL_W delivered electrical power W
	•	P_DENS_W_M2 = P_TOTAL_W / (A_MEM_M2 * N_CELL_PAIRS) W/m²
	•	per_segment[...] arrays for concentrations, segment EMFs, segment resistances, etc.

⸻

Model 0D (lumped “no-axial-variation” baseline)

Purpose:
	•	Fast baseline for sanity checks and parameter sweeps.
	•	Treats the stack as a single equivalent source with one EMF and one internal resistance based on inlet (or bulk) conditions.

Electrical abstraction:
	•	A single Thevenin element: V = E_stack − I * R_int.
	•	No explicit axial concentration depletion; concentration is treated as uniform / averaged.

Implication:
	•	0D can overestimate performance when axial depletion and polarization would reduce EMF and increase effective resistance.

⸻

Model 1D axial — two electrical topologies

Both 1D models resolve bulk concentration evolution along the flow direction by splitting the flow path into N_SEG slices. The difference is how those slices are connected electrically.

A) Legacy 1D axial model (parallel-branch / segmented-electrode-equivalent)

What it assumes electrically:
	•	Axial segments are treated as electrically parallel branches that share the same V_TERM_V.
	•	Each segment carries its own branch current I_SEG[j].

This is physically consistent only for:
	•	A segmented-electrode / multi-converter design (each axial section effectively has its own electrode pair or independent electrical extraction).

Why it is kept:
	•	It matches the original spreadsheet-style implementation and provides a regression baseline for plots and outputs.

Key computational structure:
	•	Iterate on a shared terminal voltage V_TERM_V.
	•	For each segment, compute local E_seg[j] and R_seg[j], then compute I_SEG[j].
	•	Sum currents: I_TOTAL_A = Σ I_SEG[j].
	•	Update V_TERM_V by the external closure (fixed load or MPPT).

Important caution:
	•	Because the model’s electrical degrees of freedom are larger than a standard stack, it can show behaviors that are artifacts for a single-electrode device (for example, multiple segments appearing “blocked” simultaneously under a high initial V_TERM_V).

⸻

B) Standard 1D axial model (series-current / single-electrode-pair Tier A)

What it assumes electrically:
	•	There is one electrode pair, therefore one stack current I_TOTAL_A.
	•	The same I_TOTAL_A flows through every axial location.
	•	Axial segmentation is a series decomposition used for concentration bookkeeping, not electrical branching.

Electrical composition:
	•	Each segment has a local EMF and resistance determined by its local concentrations:
	•	E_seg[j] = E(C_H[j], C_L[j])
	•	R_seg[j] = R(C_H[j], C_L[j], params)
	•	Totals are series sums:
	•	E_total = Σ E_seg[j]
	•	R_int = Σ R_seg[j]
	•	Terminal voltage is then:
	•	V_TERM_V = E_total − I_TOTAL_A * R_int

Key computational structure:
	•	Solve for the single unknown I_TOTAL_A using the external electrical closure.
	•	For a trial current I, march concentrations along the axis (advection + salt transfer), compute E_total(I) and R_int(I), then evaluate the closure residual.
	•	Use a robust scalar solver (bracketing/bisection) to find the operating current.

This model is the physically correct starting point for a conventional RED stack and is the intended base for adding fouling and other real losses without changing topology.

⸻

External electrical “modes” (applies to 0D and both 1D models)

MODE = 'fixed_load'
	•	External resistance R_LOAD_OHM is specified.
	•	Operating point satisfies: V_TERM_V = I_TOTAL_A * R_LOAD_OHM.

MODE = 'max_power'
	•	Ideal maximum power transfer condition at the terminals (Thevenin): R_LOAD ≈ R_int.
	•	Implemented as an operating condition rather than a fixed R_LOAD_OHM.

MODE = 'mppt'
	•	The notebook uses an MPPT-style outer loop that updates an effective load toward the internal resistance estimate.
	•	This is a control abstraction for matching load to source over time/iterations; it does not change the underlying physical topology.

⸻

Fouling and “Tiering” strategy (how future complexity is meant to enter)

Tier A (implemented for the standard topology):
	•	1D axial, series-current, single I_TOTAL_A.
	•	Cleanest place to start adding fouling: apply multipliers to R_seg[j] and track how resistance grows axially.

Tier B/C/D (planned knobs, not required for Tier A regression):
	•	Add one effect at a time behind explicit switches:
	•	concentration polarization (CP)
	•	non-ideal activities
	•	electrode losses
	•	shunt currents / manifold leakage
	•	Cross-flow is inherently 2D in bulk fields; it should not be approximated by axial electrical parallel branches.
	•	Segmented-electrode behavior is already represented by the legacy parallel-branch model and should remain a distinct topology option.

⸻

Practical reading guide (what to run when comparing models)
	•	Use 0D for quick sanity checks and “order-of-magnitude” expectations.
	•	Use 1D series-current when the goal is realism for a standard stack and when adding fouling.
	•	Use 1D legacy parallel-branch only when intentionally exploring segmented-electrode behavior or maintaining regression against older outputs.

Both 1D models return the same output schema so plots can be produced consistently and comparisons can be made directly.

## Variables
Constants, unit conversions, and model-wide options/toggles.

In [ ]:
# ── CANARY: if you see this, you are running the correct file ──
print("CORRECT FILE LOADED — OsmoFlux modified", __import__("time").strftime("%Y-%m-%d %H:%M:%S"))
from __future__ import annotations

from dataclasses import dataclass, replace, asdict
from typing import Dict, Any, Optional, Literal, Tuple
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# Physical constants (SI)
# -------------------------
F = 96485.33212  # Faraday constant [C/mol]
R = 8.314462618  # Gas constant [J/mol/K]

# Temperature defaults
T_DEFAULT_K = 298.15  # [K] (25 C)

# -------------------------
# Unit conversions
# -------------------------
L_PER_M3 = 1000.0          # [L/m^3]
M3_PER_L = 1.0 / L_PER_M3  # [m^3/L]
PA_PER_BAR = 1e5           # [Pa/bar]

# -------------------------
# Model options / toggles
# -------------------------
@dataclass(frozen=True)
class ModelOptions:
    """Big fixed switches / constants that were previously in the sheet's 'ModelOption' tab.
    Small and stable; run-specific things in DesignInputs.
    """
    # Activity coefficient handling (starting simple)
    USE_ACTIVITY_COEFF: bool = False

    # If True, clamp concentrations to a small positive floor to avoid log/negative issues
    CLAMP_CONC_POSITIVE: bool = True
    CONC_FLOOR_MOL_L: float = 1e-9  # [mol/L]

    # Numerical knobs for axial fixed-point iteration
    MAX_ITERS: int = 200
    REL_TOL: float = 1e-7
    ABS_TOL: float = 1e-10
    DAMPING: float = 0.35  # 0<damping<=1, lower = more stable

    # Whether to store per-iteration diagnostics (slower)
    STORE_ITER_DIAGNOSTICS: bool = False


OPTIONS = ModelOptions()


## nacl_conductivity
Spreadsheet-style NaCl conductivity correlation.
The function signature is:
`KAPPA_NACL_SI(c_mol_L, T_K) -> S_per_m`


In [ ]:
# -------------------------
# NaCl conductivity (kappa)
# -------------------------

import numpy as np

# Molar mass of NaCl
MW_NACL_G_PER_MOL = 58.44277  # [g/mol]

# 25C table (Foxboro/Gilson), stored as mg/L -> conductivity at 25C
# Using the mS/cm column.
NACL_MG_L_TABLE = np.array([
    1, 3, 10, 30, 100, 300, 1000, 3000, 10000, 30000, 50000, 100000, 200000,
    250000, 300000, 357000
], dtype=float)  # [mg/L]
# Extended to NaCl saturation (~357 g/L at 25°C).
# Added points (25°C, mS/cm) from CRC Handbook / Isono (1980):
#   250 g/L → ~257 mS/cm,  300 g/L → ~285 mS/cm,  357 g/L → ~307 mS/cm
NACL_KAPPA_MS_CM_25C_TABLE = np.array([
    0.0022, 0.0065, 0.0214, 0.064, 0.21, 0.617, 1.99, 5.69, 17.6, 48.6, 78.3, 140.0, 226.0,
    257.0, 285.0, 307.0
], dtype=float)  # [mS/cm] at 25C

# Unit conversion:
# 1 mS/cm = 0.1 S/m
MS_CM_TO_S_M = 0.1

# Optional temperature correction (OFF by default to match 25C data exactly)
# set USE_TEMP_CORRECTION = False
# and adjust TEMP_COEFF_PER_K to match the sheet.
USE_TEMP_CORRECTION = False
T_REF_K = 298.15
TEMP_COEFF_PER_K = 0.0212  # crude placeholder, only used if USE_TEMP_CORRECTION = False


def _mol_L_to_mg_L_nacl(c_mol_L: float) -> float:
    """Convert NaCl concentration from mol/L -> mg/L."""
    # mol/L * g/mol = g/L; *1000 => mg/L
    return float(c_mol_L) * MW_NACL_G_PER_MOL * 1000.0


def kappa_nacl_fit_default(c_mol_L: float, T_K: float) -> float:
    """
    Conductivity κ for aqueous NaCl.

    INPUTS:
      - c_mol_L: NaCl concentration [mol/L]
      - T_K: temperature [K]

    OUTPUT:
      - κ [S/m]

    Data basis:
      - Foxboro/Gilson table values at 25C (298.15 K), from sheet.

    Implementation:
      - Convert mol/L -> mg/L
      - Interpolate κ in log-log space versus mg/L
      - Convert mS/cm -> S/m
      - Optional temperature correction (disabled by default)
    """
    c = max(float(c_mol_L), 0.0)
    T = float(T_K)

    # Convert concentration to mg/L for lookup
    mg_L = _mol_L_to_mg_L_nacl(c)
    min_mg_L, max_mg_L = NACL_TABLE_RANGE_MG_L()
    is_out_of_range = (mg_L < min_mg_L) or (mg_L > max_mg_L)



    # Clamp to table range to avoid nonsense at extreme extrapolation
    mg_L_clamped = min(max(mg_L, NACL_MG_L_TABLE[0]), NACL_MG_L_TABLE[-1])

    # Convert table κ to SI [S/m]
    kappa_S_m_table = NACL_KAPPA_MS_CM_25C_TABLE * MS_CM_TO_S_M  # [S/m] at 25C

    # Log-log interpolation across decades
    x = np.log10(NACL_MG_L_TABLE)
    y = np.log10(kappa_S_m_table)
    kappa_25C = 10 ** np.interp(np.log10(mg_L_clamped), x, y)  # [S/m] at 25C

    if USE_TEMP_CORRECTION:
        # Simple linearized correction; replace with sheet’s exact rule if needed
        kappa = kappa_25C * max(1.0 + TEMP_COEFF_PER_K * (T - T_REF_K), 0.0)
    else:
        kappa = kappa_25C

    # Safety floor to avoid divide-by-zero elsewhere
    return float(max(kappa, 1e-9))


def KAPPA_NACL_SI(c_mol_L: float, T_K: float) -> float:
    """Public wrapper: NaCl conductivity in SI units [S/m]."""
    return kappa_nacl_fit_default(c_mol_L, T_K)


# Quick sanity checks against Google sheet table (at 25C):
# We convert mg/L -> mol/L and verify κ comes back close (interpolation exact on table points)
def _mg_L_to_mol_L_nacl(mg_L: float) -> float:
    return float(mg_L) / 1000.0 / MW_NACL_G_PER_MOL  # mg/L -> g/L -> mol/L

def NACL_TABLE_RANGE_MG_L() -> tuple[float, float]:
    """Return (min_mg_L, max_mg_L) covered by the conductivity table."""
    return float(NACL_MG_L_TABLE[0]), float(NACL_MG_L_TABLE[-1])


print("Sanity check vs table points (25C):")
for mg_L, ms_cm in zip(NACL_MG_L_TABLE, NACL_KAPPA_MS_CM_25C_TABLE):
    c_mol_L = _mg_L_to_mol_L_nacl(mg_L)
    kappa_pred_S_m = KAPPA_NACL_SI(c_mol_L, 298.15)
    kappa_true_S_m = ms_cm * MS_CM_TO_S_M
    rel_err = (kappa_pred_S_m - kappa_true_S_m) / kappa_true_S_m
    print(f"mg/L={mg_L:>7.0f}  κ_true={kappa_true_S_m:>10.6g} S/m  κ_pred={kappa_pred_S_m:>10.6g} S/m  rel_err={rel_err:+.2e}")

## design_inputs (per run)
A run-specific parameter container.

This is the place to mirror sheet's **3-DesignInputs** tab variable names.

The class is immutable (`frozen=True`) so cloning is simple: `replace(params, FIELD=value)`.

In [ ]:
# -------------------------
# Run-specific inputs
# -------------------------
@dataclass(frozen=True)
class DesignInputs:
    # ---- Core thermodynamics ----
    T_K: float = T_DEFAULT_K  # temperature [K]

    # ---- Stack topology ----
    N_CELL_PAIRS: int = 10  # number of cell pairs (variable per run) [count]
    A_MEM_M2: float = 0.05  # active membrane area per cell pair [m^2] | 0.05 ≈ 22x22 cm pilot scale (REDstack BV upscaling study)
    
    # ---- Hydraulics ----
    # Volumetric flow rates for high-salinity (H) and low-salinity (L) streams
    Q_H_M3_S: float = 5.0e-6  # [m^3/s]  # TOTAL stack high-stream flow | ~1-2 cm/s at pilot scale
    Q_L_M3_S: float = 5.0e-6  # [m^3/s]  # TOTAL stack low-stream flow | ~1-2 cm/s at pilot scale

    # Inlet concentrations (NaCl), mol/L
    C_H_IN_MOL_L: float = 0.5   # [mol/L]
    C_L_IN_MOL_L: float = 0.02  # [mol/L]

    # ---- Channel geometry (used for solution resistance models) ----
    GAP_M: float = 200e-6     # intermembrane gap / spacer thickness [m]
    # model spacer porosity or tortuosity -- add here later

    # ---- Membrane / electrochemical parameters ----
    ALPHA_PERM: float = 0.975  # permselectivity factor (0..1) | Fumasep FKS-PET-130: 96–99%, midpoint 97.5%
    BETA: float = 1.0         # charge efficiency / salt transfer efficiency (0..1)
    V_ELECTRODE_PAIR_V: float = 0.1  # total electrode overpotential [V] per pair | Veerman 2010
    
    # Area-specific resistances (ASR) [ohm*m^2] (typical spreadsheet inputs)
    ASR_AEM_OHM_M2: float = 2.35e-4  # [Ω·m^2] | Fumasep FAS-PET-130: 1.7–3.0e-4, midpoint 2.35e-4
    ASR_CEM_OHM_M2: float = 3.6e-4   # [Ω·m^2] | Fumasep FKS-PET-130: 2.6–4.6e-4, midpoint 3.6e-4

    # Optional extra ASR terms (spacer, electrodes, manifolds, contacts)
    ASR_EXTRA_OHM_M2: float = 0.0   # [Ω·m^2]
    SPACER_SHADOW_SOL: float = 1.56    # spacer shadow factor for solution channels | Vermaas 2014: 1.5–1.93
    SPACER_SHADOW_MEM: float = 1.10    # spacer shadow factor for membranes | Vermaas 2014: 1.06–1.14

    # ---- External electrical closure ----
    MODE: Literal["fixed_load", "max_power", "mppt"] = "fixed_load"
    R_LOAD_OHM: float = 0.05  # [Ω] used if MODE='fixed_load' (also default MPPT init); near R_int for base-case stack

    # ---- MPPT (maximum power point tracking) knobs ----
    # MPPT is modeled as a controller that selects an effective load resistance.
    # In steady-state, the oracle optimum is near R_load ~= R_int (Thevenin match),
    # but controller limits (step size / inefficiency) can be imposed here.
    MPPT_ETA: float = 1.0  # [0..1] electrical efficiency multiplier applied to delivered power
    MPPT_ALPHA: float = 0.5  # [0..1] per-iteration fractional move toward estimated optimum
    MPPT_MAX_DELTA_R_OHM: float = 5.0  # [Ω/iter] clamp on how fast the controller can change R_load
    MPPT_N_ITERS: int = 8  # controller update iterations used in 0D; 1D updates once per fixed-point iter
    MPPT_R_INIT_OHM: Optional[float] = None  # optional initial MPPT load override [Ω]

    # ---- 1D axial discretization ----
    N_SEG: int = 40  # number of axial segments [count] | standardized across all experiments

    # ---- Fouling / degradation ----
    FOULING_MODEL: str = "none"  # none|linear|exp_inlet|exp_outlet|gaussian|logistic|covalent_ions

    # ── Concentration-Dependent Fouling (CDF) — novel spatially-resolved model ────
    # Physical basis: divalent cations (Ca²⁺, Mg²⁺) in the high-salinity feed bind
    # to fixed sulfonate groups on CEMs, reducing effective fixed-charge density and
    # raising ASR in proportion to local ion activity.
    #
    # Literature basis and novelty claim:
    #   This model is a direct spatial extension of Rijnaarts et al. (2017)
    #   Environ. Sci. Technol. 51, 13028–13035, who quantified average power-density
    #   loss from divalent cations in RED but used a spatially uniform (stack-lumped)
    #   resistance correction.  OsmoFlux extends that result by embedding the
    #   divalent resistance penalty *inside the 1D axial march*, where the fouling
    #   severity at each segment is computed from the local C_H already tracked by
    #   the hydraulic solver.  This closed-loop spatial coupling — highest penalty
    #   at the inlet where C_H is greatest, diminishing downstream as the stream is
    #   depleted — is absent from all published RED models reviewed, including
    #   Gurreri et al. (2023) Membranes 13(3) 322 and Ciofalo et al. (2019).
    #
    #   Concentration scaling exponent n≈0.5 (FOULING_N_EXP default):
    #     From Długołęcki et al. (2010) J. Membr. Sci. 346, 163–171, who showed
    #     CEM area-specific resistance scales roughly as the square root of ionic
    #     strength for the monovalent/divalent range relevant to RED feed waters.
    #
    #   Severity parameter FOULING_BETA:
    #     Gurreri 2023 measured 2–8× ASR increase for pure divalent solutions.
    #     FOULING_BETA = 0.6 at DIVALENT_FRACTION = 0.20 gives ~12% inlet penalty,
    #     consistent with ~10–20% power-density losses reported by Rijnaarts 2017
    #     for natural seawater (~15% divalent cation fraction).
    #
    # Formula (per segment j):
    #   foul_cdf(j) = FOULING_BETA × DIVALENT_FRACTION × (C_H_local[j] / C_REF)^N_EXP
    #   R_seg(j)    = R_seg_clean(j) × (1 + foul_cdf(j) + foul_phenom(j))
    #
    # Set DIVALENT_FRACTION=0 (default) or FOULING_BETA=0 to reproduce clean-membrane
    # baseline; all existing experiments are unaffected by these defaults.
    DIVALENT_FRACTION: float = 0.0    # fraction of dissolved ions that are divalent [0..1]
                                      # 0 = pure NaCl (default, no CDF penalty)
                                      # typical river water ≈ 0.05–0.15
                                      # typical seawater   ≈ 0.10–0.20
    FOULING_BETA: float = 0.0         # max fractional ASR increase at (C_H=C_REF, f_div=1)
                                      # defensible range 0.2–0.8 from Gurreri 2023
                                      # 0.0 = disabled (default, backward compatible)
    FOULING_C_REF_MOL_L: float = 0.5  # normalisation concentration [mol/L]
    FOULING_N_EXP: float = 0.5        # concentration scaling exponent (Długołęcki 2010)

    # Fixed-point initialization / stabilization (optional overrides per run)
    INIT_V_TERM_V: Optional[float] = None  # initial guess terminal voltage [V]

    # ── Plant-scale parameters (Monte Carlo TEA / what-if scenarios) ──────────
    # These describe the physical plant that the RED stack is treating brine from.
    # Default values are calibrated to the Carlsbad Poseidon SWRO plant.
    # Every field is tagged # VALIDATE — review before running Monte Carlo.
    # To model a different plant: override these in ExperimentSpec.base_overrides.
    #
    # All Monte Carlo TEA calculations read exclusively from these fields —
    # no external config dict is required.

    # ── Brine volume ───────────────────────────────────────────────────────────
    PLANT_BRINE_FLOW_M3_DAY: float = 190_000.0  # [m³/day]   # VALIDATE — Carlsbad ~190,000 m³/day discharge (Poseidon EIR Table 3-1: 54 MGD product)

    # ── Membrane economics ─────────────────────────────────────────────────────
    PLANT_CAPEX_PER_M2: float = 50.0            # [$/m²]     # VALIDATE — RED stack incl. membranes+hardware (Post 2008, Daniilidis 2014: $30–80/m²)
    PLANT_OPEX_FRAC_CAPEX_PER_YR: float = 0.03 # [frac/yr]  # VALIDATE — annual O&M as fraction of capex (typical 2–4%)
    PLANT_AMORT_YR: int = 20                    # [years]    # VALIDATE — straight-line amortisation period
    PLANT_CAPACITY_FACTOR: float = 0.913        # [0..1]     # VALIDATE — ~8000 hr/yr operational ÷ 8760 = 0.913

    # ── Electricity market ─────────────────────────────────────────────────────
    PLANT_ELEC_PRICE_USD_KWH: float = 0.22      # [$/kWh]    # VALIDATE — CA commercial/industrial rate; used as lognormal mean in MC
    PLANT_BRINE_DISPOSAL_COST_USD_M3: float = 0.15  # [$/m³] # VALIDATE — evap pond avoided-cost proxy (diffuser $0.04, deep-well $0.25)
    PLANT_DISCOUNT_RATE: float = 0.05               # [frac] # VALIDATE — real discount rate for NPV (infrastructure range 3-8%)


def clone_params(p: DesignInputs, **updates) -> DesignInputs:
    """Convenience wrapper for sweeps. Equivalent to dataclasses.replace."""
    return replace(p, **updates)


# Display the exact fields (so we can align with the sheet)
print("DesignInputs fields:\n - " + "\n - ".join(DesignInputs.__dataclass_fields__.keys()))


In [ ]:
# -------------------------
# BASELINE RESET (single source of truth)
# -------------------------
# This cell is the ONLY place where baseline inputs are constructed.
# All legacy and harness sweeps must reference these objects ONLY.
# Do not mutate these objects in-place.

GLOBAL_MODE = "fixed_load"

base_ref = DesignInputs(
    # --- Legacy-parity defaults (do not change without updating parity expectations) ---
    N_CELL_PAIRS=10,
    R_LOAD_OHM=1.0,
    A_MEM_M2=0.0100,          # per cell-pair membrane area (legacy convention)
)

# Sweep grids (legacy parity)
R_LOAD_vals_legacy = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0]
N_vals_legacy = [5, 10, 20, 40]

# Second canonical sweep (area vs pairs), if needed
A_MEM_vals_legacy = [0.0025, 0.005, 0.01, 0.02, 0.04]

print("[baseline] GLOBAL_MODE =", GLOBAL_MODE)
print("[baseline] base_ref:", base_ref)


## built_in_functions
Helper functions shared by both models: Nernst potential, resistance building blocks, conversions, sanity checks.

In [ ]:
# -------------------------
# Shared helpers
# -------------------------

def _clamp_conc(c_mol_L: float) -> float:
    if OPTIONS.CLAMP_CONC_POSITIVE:
        return max(float(c_mol_L), OPTIONS.CONC_FLOOR_MOL_L)
    return float(c_mol_L)



# Robinson & Stokes (1959) tabulated data for NaCl γ± at 25°C
_RS_CONC = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5,
            0.7, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0]
_RS_GAMMA = [0.965, 0.952, 0.929, 0.903, 0.872, 0.822, 0.778, 0.735, 0.710, 0.681,
             0.667, 0.657, 0.657, 0.668, 0.714, 0.783, 0.874, 0.986]

def _nacl_mean_activity_coefficient(c_mol_L: float) -> float:
    """Mean ionic activity coefficient γ± for NaCl at 25°C.
    
    Log-linear interpolation on Robinson & Stokes (1959) tabulated data.
    Valid 0.001–6.0 mol/L.  Returns γ± (dimensionless).
    Extrapolates flat beyond table bounds.
    """
    c = max(float(c_mol_L), _RS_CONC[0])
    if c >= _RS_CONC[-1]:
        return _RS_GAMMA[-1]
    # Log-linear interpolation (linear in log(c))
    lc = math.log(c)
    for i in range(len(_RS_CONC) - 1):
        lc0 = math.log(_RS_CONC[i])
        lc1 = math.log(_RS_CONC[i + 1])
        if lc <= lc1:
            t = (lc - lc0) / (lc1 - lc0)
            return float(_RS_GAMMA[i] + t * (_RS_GAMMA[i + 1] - _RS_GAMMA[i]))
    return _RS_GAMMA[-1]


def NERNST_OCV_CELLPAIR_V(C_H_MOL_L: float, C_L_MOL_L: float, T_K: float, ALPHA_PERM: float) -> float:
    """Open-circuit voltage for a single cell pair [V].
    
    Typical spreadsheet form (ideal, monovalent):
      E_cell = 2 * α * (R*T/F) * ln( a_H / a_L )
    where α is permselectivity (0..1). Here a≈c if activity coefficients are ignored.
    """
    cH = _clamp_conc(C_H_MOL_L)
    cL = _clamp_conc(C_L_MOL_L)
    # Use activities a = γ± · c (Pitzer correction for non-ideal solutions)
    gH = _nacl_mean_activity_coefficient(cH)
    gL = _nacl_mean_activity_coefficient(cL)
    return float(2.0 * ALPHA_PERM * (R * T_K / F) * math.log((gH * cH) / (gL * cL)))


def ASR_SOLUTION_OHM_M2(C_MOL_L: float, T_K: float, GAP_M: float) -> float:
    """Solution area-specific resistance [Ω·m^2] for one channel gap.
    
    Simple model:
      ASR_sol = GAP / κ
    where κ is conductivity [S/m].
    
    If the sheet uses a more detailed spacer model, replace this exactly.
    """
    kappa = KAPPA_NACL_SI(C_MOL_L, T_K)  # [S/m]
    return float(GAP_M / kappa)  # [Ω·m^2]


def ASR_CELLPAIR_OHM_M2(p: DesignInputs, C_H_MOL_L: float, C_L_MOL_L: float) -> float:
    """Total area-specific resistance per cell pair [Ω·m^2]."""
    asr_sol_H = ASR_SOLUTION_OHM_M2(C_H_MOL_L, p.T_K, p.GAP_M)
    asr_sol_L = ASR_SOLUTION_OHM_M2(C_L_MOL_L, p.T_K, p.GAP_M)
    # Spacer shadow effect (Vermaas et al. 2014, ES&T)
    beta_sol = getattr(p, "SPACER_SHADOW_SOL", 1.0)
    beta_mem = getattr(p, "SPACER_SHADOW_MEM", 1.0)
    return float(beta_mem * (p.ASR_AEM_OHM_M2 + p.ASR_CEM_OHM_M2) + beta_sol * (asr_sol_H + asr_sol_L) + p.ASR_EXTRA_OHM_M2)


def R_STACK_INTERNAL_OHM(p: DesignInputs, C_H_MOL_L: float, C_L_MOL_L: float) -> float:
    """Internal resistance of the whole stack [Ω] at given concentrations."""
    asr = ASR_CELLPAIR_OHM_M2(p, C_H_MOL_L, C_L_MOL_L)   # [Ω·m^2] per cell pair
    r_cellpair = asr / p.A_MEM_M2                        # [Ω] per cell pair
    return float(p.N_CELL_PAIRS * r_cellpair)            # series


def E_STACK_OCV_V(p: DesignInputs, C_H_MOL_L: float, C_L_MOL_L: float) -> float:
    """Stack OCV [V] = N_CELL_PAIRS * E_cellpair."""
    e_cell = NERNST_OCV_CELLPAIR_V(C_H_MOL_L, C_L_MOL_L, p.T_K, p.ALPHA_PERM)
    return float(p.N_CELL_PAIRS * e_cell)


def salt_transfer_mol_per_s(I_A: float, N_CELL_PAIRS: int, BETA: float) -> float:
    """Faradaic NaCl transfer rate [mol/s] from H->L implied by current.
    
    For monovalent ions, charge flow I corresponds to mol/s = I/F.
    In a RED stack, each cell pair contributes; with N in series, current is common,
    so total mol/s transfer scales with N (spreadsheet convention often uses N).
    
    BETA is a catch-all for charge efficiency / leakage effects (0..1).
    """
    return float(BETA * I_A * N_CELL_PAIRS / F)


def apply_salt_balance(
    C_H_IN_MOL_L: float, C_L_IN_MOL_L: float,
    Q_H_M3_S: float, Q_L_M3_S: float,
    n_dot_mol_s: float
) -> Tuple[float, float]:
    """Update outlet concentrations from a segment given salt transfer n_dot [mol/s].
    
    Convention: +n_dot means NaCl moves from high-salinity stream to low-salinity stream.
    
    Mass balance:
      ΔC_H = -n_dot / Q_H   (mol/m^3) then convert to mol/L
      ΔC_L = +n_dot / Q_L
    """
    # mol/m^3
    dC_H_mol_m3 = -n_dot_mol_s / max(Q_H_M3_S, 1e-30)
    dC_L_mol_m3 = +n_dot_mol_s / max(Q_L_M3_S, 1e-30)

    # convert to mol/L
    dC_H_mol_L = dC_H_mol_m3 / L_PER_M3
    dC_L_mol_L = dC_L_mol_m3 / L_PER_M3

    C_H_OUT = C_H_IN_MOL_L + dC_H_mol_L
    C_L_OUT = C_L_IN_MOL_L + dC_L_mol_L

    # Clamp to non-negative if requested
    if OPTIONS.CLAMP_CONC_POSITIVE:
        C_H_OUT = max(C_H_OUT, OPTIONS.CONC_FLOOR_MOL_L)
        C_L_OUT = max(C_L_OUT, OPTIONS.CONC_FLOOR_MOL_L)

    return float(C_H_OUT), float(C_L_OUT)


def sanity_check_inputs(p: DesignInputs) -> None:
    assert p.N_CELL_PAIRS >= 1
    assert p.N_SEG >= 1
    assert p.A_MEM_M2 > 0
    assert p.Q_H_M3_S > 0 and p.Q_L_M3_S > 0
    assert p.C_H_IN_MOL_L > 0 and p.C_L_IN_MOL_L > 0
    assert 0 <= p.ALPHA_PERM <= 1.0
    assert 0 <= p.BETA <= 1.0
    assert p.GAP_M > 0


# -------------------------
# MPPT helpers (electrical closure)
# -------------------------

def _clamp_positive_ohms(r_ohm: float, floor_ohm: float = 1e-6) -> float:
    """Clamp a resistance to a small positive value to avoid division-by-zero."""
    return float(max(float(r_ohm), float(floor_ohm)))

def _mppt_update_r_load(r_load_ohm: float, r_opt_ohm: float, alpha: float, max_delta_ohm: float) -> float:
    """Single MPPT controller update step toward an estimated optimal load (controller-limited)."""
    r_load = _clamp_positive_ohms(r_load_ohm)
    r_opt = _clamp_positive_ohms(r_opt_ohm)
    a = float(min(max(alpha, 0.0), 1.0))
    max_d = float(max(max_delta_ohm, 0.0))

    # Move fractionally toward r_opt with an absolute delta clamp (models finite controller bandwidth).
    r_target = (1.0 - a) * r_load + a * r_opt
    delta = r_target - r_load
    if max_d > 0.0:
        delta = float(np.clip(delta, -max_d, +max_d))
    return _clamp_positive_ohms(r_load + delta)

def _mppt_init_r_load(p: DesignInputs) -> float:
    """Initialize MPPT load resistance from overrides (falls back to R_LOAD_OHM)."""
    r0 = p.MPPT_R_INIT_OHM if p.MPPT_R_INIT_OHM is not None else p.R_LOAD_OHM
    return _clamp_positive_ohms(float(r0))


## 5) model functions
Two pure functions:
- `model_0d(params)`
- `model_1d_axial(params, mode=...)`

Return type: structured dict with both scalar outputs and per-segment arrays.
Axial model uses a fixed-point iterative loop with damping (no circular refs).

In [ ]:
# -------------------------
# 0D model
# -------------------------
def model_0d(p: DesignInputs, mode: Optional[Literal["fixed_load", "max_power", "mppt"]] = None) -> Dict[str, Any]:
    sanity_check_inputs(p)
    mode = mode or p.MODE

    # Treat as single lump with inlet concentrations
    C_H = p.C_H_IN_MOL_L
    C_L = p.C_L_IN_MOL_L

    E_ocv = E_STACK_OCV_V(p, C_H, C_L)         # [V]
    # Subtract electrode overpotential (Veerman 2010)
    _v_elec_0d = getattr(p, "V_ELECTRODE_PAIR_V", 0.0)
    E_ocv = max(E_ocv - _v_elec_0d, 0.0)
    R_int = R_STACK_INTERNAL_OHM(p, C_H, C_L)  # [Ω]

    if mode == "fixed_load":
        R_load = float(p.R_LOAD_OHM)
        I = E_ocv / (R_int + R_load)  # [A]
        V_term = I * R_load           # [V]
    elif mode == "max_power":
        # Thevenin: max power when R_load = R_int (oracle impedance match)
        R_load = R_int
        I = E_ocv / (R_int + R_load)
        V_term = I * R_load
    elif mode == "mppt":
        # Controller-limited MPPT: iteratively move an effective load toward the estimated optimum.
        # In this steady-state 0D setting, the target optimum is the Thevenin match R_opt ~= R_int.
        R_opt = R_int
        R_load = _mppt_init_r_load(p)
        for _ in range(int(max(p.MPPT_N_ITERS, 0))):
            R_load = _mppt_update_r_load(R_load, R_opt, p.MPPT_ALPHA, p.MPPT_MAX_DELTA_R_OHM)
        I = E_ocv / (R_int + R_load)
        V_term = I * R_load
    else:
        raise ValueError(f"Unknown mode: {mode}")

    P_raw = V_term * I  # [W] (ideal electrical power into the selected load)
    eta = float(p.MPPT_ETA) if mode == "mppt" else 1.0
    P = float(P_raw * eta)  # [W] delivered power (after MPPT efficiency)

    n_dot = salt_transfer_mol_per_s(I, p.N_CELL_PAIRS, p.BETA)  # [mol/s]
    C_H_OUT, C_L_OUT = apply_salt_balance(C_H, C_L, p.Q_H_M3_S, p.Q_L_M3_S, n_dot)

    return {
        "mode": mode,
        "E_OCV_V": E_ocv,
        "R_INT_OHM": R_int,
        "R_LOAD_OHM": R_load,
        "V_TERM_V": V_term,
        "I_A": I,
        "P_W": P,
        "P_RAW_W": float(P_raw),
        "MPPT_ETA": float(eta),
        "C_H_OUT_MOL_L": C_H_OUT,
        "C_L_OUT_MOL_L": C_L_OUT,
        "n_dot_salt_mol_s": n_dot,
    }


# -------------------------
# 1D axial model
# -------------------------

# -------------------------
# Fouling / degradation models (axial)
# -------------------------
def _x_seg(j: int, N: int) -> float:
    """Normalized axial coordinate in [0,1] at segment center."""
    return (j + 0.5) / float(max(N, 1))

def fouling_none(x: float) -> float:
    """No added resistance."""
    return 0.0

def fouling_linear(x: float) -> float:
    """Linear fouling ramp along flow (quick first test)."""
    beta = 0.30  # max fractional resistance increase at outlet (x=1)
    return beta * x

def fouling_exp_inlet(x: float) -> float:
    """Inlet-peaked fouling (decays downstream)."""
    beta = 0.40
    k = 4.0
    return beta * float(np.exp(-k * x))

def fouling_exp_outlet(x: float) -> float:
    """Outlet-peaked fouling (grows downstream)."""
    beta = 0.40
    k = 4.0
    num = float(np.exp(k * x) - 1.0)
    den = float(np.exp(k) - 1.0)
    return beta * (num / max(den, 1e-30))

def fouling_gaussian_hotspot(x: float) -> float:
    """Localized hotspot (e.g., precipitation zone)."""
    beta = 0.60
    x0 = 0.65
    sigma = 0.10
    return beta * float(np.exp(-0.5 * ((x - x0) / max(sigma, 1e-30)) ** 2))

def fouling_logistic(x: float) -> float:
    """Saturating fouling (growth then plateau)."""
    beta = 0.50
    k = 10.0
    x0 = 0.35
    return beta * float(1.0 / (1.0 + np.exp(-k * (x - x0))))

def fouling_covalent_ions(x: float) -> float:
    """Shell: covalent-ion / complexing species that disproportionately suppresses power.
    Placeholder: steep early resistance penalty.
    """
    # TODO: replace with a mechanistic mapping from (divalent/covalent fraction, pH, local conc) -> penalty
    beta = 1.00
    x0 = 0.20
    k = 18.0
    return beta * float(1.0 / (1.0 + np.exp(-k * (x - x0))))

FOULING_FUNCS = {
    "none": fouling_none,
    "linear": fouling_linear,
    "exp_inlet": fouling_exp_inlet,
    "exp_outlet": fouling_exp_outlet,
    "gaussian": fouling_gaussian_hotspot,
    "logistic": fouling_logistic,
    "covalent_ions": fouling_covalent_ions,
}


# ── Concentration-Dependent Fouling (CDF) — mechanistic model ────────────────
def fouling_cdf_penalty(C_H_local: float, p: 'DesignInputs') -> float:
    """Fractional ASR increase from divalent-ion binding at one axial segment.

    Model:
        foul_cdf = FOULING_BETA × DIVALENT_FRACTION × (C_H_local / C_REF)^N_EXP

    Returns 0.0 when FOULING_BETA=0 or DIVALENT_FRACTION=0 (both default),
    so existing experiments with no CDF parameters set are completely unaffected.

    Literature:
        Scaling exponent N_EXP≈0.5 from Długołęcki et al. (2010) J. Membr. Sci.
        346, 163–171 (square-root concentration dependence of CEM ASR).
        FOULING_BETA calibrated against Gurreri et al. (2023) Membranes 13, 322
        and Rijnaarts et al. (2017) ES&T 51, 13028.
    """
    f_div = float(getattr(p, "DIVALENT_FRACTION",  0.0))
    beta  = float(getattr(p, "FOULING_BETA",        0.0))
    c_ref = float(getattr(p, "FOULING_C_REF_MOL_L", 0.5))
    n_exp = float(getattr(p, "FOULING_N_EXP",        0.5))

    if beta <= 0.0 or f_div <= 0.0:
        return 0.0

    c_ratio = max(C_H_local, 0.0) / max(c_ref, 1e-12)
    return float(beta * f_div * (c_ratio ** n_exp))


def segment_resistance_ohm(
    p: 'DesignInputs',
    j: int,
    N: int,
    R_seg_clean: float,
    C_H_local: float = 0.0,
) -> 'tuple[float, float, float]':
    """Return (R_seg_fouled, R_seg_clean, foul_frac_total) for axial segment j.

    Total fouling fraction = CDF term (mechanistic) + phenomenological term (shape).

    Backward compatibility: callers that only use the first return value are
    unaffected; the extra two values are new diagnostic outputs.
    """
    # ── CDF term: concentration-driven, inlet-heavy ───────────────────────
    cdf = fouling_cdf_penalty(C_H_local, p)

    # ── Phenomenological term: legacy shape overlay ───────────────────────
    key  = getattr(p, "FOULING_MODEL", "none") or "none"
    f    = FOULING_FUNCS.get(str(key), fouling_none)
    x    = _x_seg(j, N)
    phenom = float(f(x))

    foul_frac = max(cdf + phenom, 0.0)
    return float(R_seg_clean) * (1.0 + foul_frac), float(R_seg_clean), foul_frac


def _axial_branch_params(p: DesignInputs, j: int, C_H: float, C_L: float) -> Tuple[float, float]:
    """Return (E_seg, R_seg) for axial segment j using local concentrations.

    Topology (matches sheet):
      - cell pairs: electrical series -> stack-level E_stack, R_stack
      - axial segments: electrical parallel -> per-branch:
            E_seg = E_stack / N_SEG
            R_seg_clean = R_stack * N_SEG
      - fouling modifies segment resistance only (via segment_resistance_ohm)
    """
    N = int(p.N_SEG)

    # MPPT state (effective load resistance). Initialized once per model evaluation.
    E_stack = float(E_STACK_OCV_V(p, C_H, C_L))         # [V]
    R_stack = float(R_STACK_INTERNAL_OHM(p, C_H, C_L))  # [Ω]

    E_seg = E_stack / float(max(N, 1))
    R_seg_clean = R_stack * float(max(N, 1))
    R_seg, _, _ = segment_resistance_ohm(p, j=j, N=N, R_seg_clean=R_seg_clean, C_H_local=C_H)
    return float(E_seg), float(R_seg)

def _axial_segment_params_series_current(p: DesignInputs, j: int, C_H: float, C_L: float) -> Tuple[float, float]:
    """Return (E_seg, R_seg) for axial segment j using local concentrations.

    Topology (standard single-electrode 1D axial model):
      - cell pairs: electrical series -> stack-level E_stack, R_stack
      - axial segments: electrical series decomposition (bookkeeping), so per-segment contributions sum:
            E_seg = E_stack / N_SEG
            R_seg_clean = R_stack / N_SEG
      - fouling modifies segment resistance only (via segment_resistance_ohm)
    """
    N = int(p.N_SEG)

    E_stack = float(E_STACK_OCV_V(p, C_H, C_L))         # [V]
    R_stack = float(R_STACK_INTERNAL_OHM(p, C_H, C_L))  # [Ω]

    E_seg = E_stack / float(max(N, 1))
    R_seg_clean = R_stack / float(max(N, 1))
    R_seg, _, _ = segment_resistance_ohm(p, j=j, N=N, R_seg_clean=R_seg_clean, C_H_local=C_H)
    return float(E_seg), float(R_seg)




def model_1d_axial(
    p: DesignInputs,
    mode: Optional[Literal["fixed_load", "max_power", "mppt"]] = None,
) -> Dict[str, Any]:
    """1D axial RED model.

    Hydraulics: segments in series (outlet of j feeds inlet of j+1).
    Electrical: segments in parallel (common terminal voltage V_term; branch currents distribute).

    Electrical closure:
      - fixed_load: V_term = I_total * R_LOAD_OHM
      - max_power: choose equivalent Thevenin load -> V_term = V_oc_eq/2 (oracle match)
      - mppt: controller-limited effective R_load updated toward R_int_eq, then V_term = I_total * R_load
    """
    sanity_check_inputs(p)
    mode = mode or p.MODE
    N = int(p.N_SEG)

    # Arrays per segment (0..N-1)
    C_H_IN = np.zeros(N, dtype=float)
    C_L_IN = np.zeros(N, dtype=float)
    C_H_OUT = np.zeros(N, dtype=float)
    C_L_OUT = np.zeros(N, dtype=float)

    E_seg = np.zeros(N, dtype=float)      # [V]
    R_seg = np.zeros(N, dtype=float)      # [Ω]
    I_seg = np.zeros(N, dtype=float)      # [A]
    n_dot_seg = np.zeros(N, dtype=float)  # [mol/s]

    # Diagnostics / feasibility flags
    min_mg_L, max_mg_L = NACL_TABLE_RANGE_MG_L()
    any_kappa_oob = False
    any_conc_clamped = False
    any_inversion = False

    # Initialize hydraulic inlets
    C_H_IN[0] = p.C_H_IN_MOL_L
    C_L_IN[0] = p.C_L_IN_MOL_L

    # MPPT state (effective load resistance). Initialized once per model evaluation.
    R_load_mppt = _mppt_init_r_load(p) if mode == "mppt" else None

    # Initial guess for terminal voltage
    if p.INIT_V_TERM_V is not None:
        V_term = float(p.INIT_V_TERM_V)
    else:
        V_term = 0.5 * E_STACK_OCV_V(p, p.C_H_IN_MOL_L, p.C_L_IN_MOL_L)

    iter_diag = [] if OPTIONS.STORE_ITER_DIAGNOSTICS else None
    converged = False

    for it in range(OPTIONS.MAX_ITERS):
        V_old = V_term

        # Step 1: march hydraulically, compute local branch params and branch currents
        for j in range(N):
            if j > 0:
                C_H_IN[j] = C_H_OUT[j - 1]
                C_L_IN[j] = C_L_OUT[j - 1]

            # Track concentration validity vs kappa table (mg/L)
            mgL_H = _mol_L_to_mg_L_nacl(C_H_IN[j])
            mgL_L = _mol_L_to_mg_L_nacl(C_L_IN[j])
            if (mgL_H < min_mg_L) or (mgL_H > max_mg_L) or (mgL_L < min_mg_L) or (mgL_L > max_mg_L):
                any_kappa_oob = True

            # Local segment electrochem based on inlet concentrations (sheet often uses inlet or avg; adjust if needed)
            E_seg[j], R_seg[j] = _axial_branch_params(p, j, C_H_IN[j], C_L_IN[j])

            # Branch current given common V_term; clamp to non-negative (no reversal in this model)
            I_seg[j] = max((E_seg[j] - V_term) / max(R_seg[j], 1e-30), 0.0)

            # Salt transfer and outlet concentrations for this segment
            n_dot_seg[j] = salt_transfer_mol_per_s(I_seg[j], p.N_CELL_PAIRS, p.BETA)
            C_H_OUT[j], C_L_OUT[j] = apply_salt_balance(
                C_H_IN[j], C_L_IN[j], p.Q_H_M3_S, p.Q_L_M3_S, n_dot_seg[j]
            )

            # Clamp/inversion diagnostics
            if (C_H_OUT[j] <= OPTIONS.CONC_FLOOR_MOL_L + 1e-30) or (C_L_OUT[j] <= OPTIONS.CONC_FLOOR_MOL_L + 1e-30):
                any_conc_clamped = True
            if C_L_OUT[j] > C_H_OUT[j]:
                any_inversion = True

        # Step 2: electrical closure update V_term
        if mode == "fixed_load":
            I_total = float(np.sum(I_seg))
            V_new = I_total * float(p.R_LOAD_OHM)
        elif mode == "max_power":
            # Equivalent Thevenin from parallel branches (given local E/R)
            G = np.sum(1.0 / np.maximum(R_seg, 1e-30))  # [1/Ω]
            V_oc_eq = float(np.sum(E_seg / np.maximum(R_seg, 1e-30)) / max(G, 1e-30))
            V_new = 0.5 * V_oc_eq  # max-power choice (oracle impedance match)
        elif mode == "mppt":
            # Controller-limited MPPT: update an effective load toward estimated optimum.
            # For the parallel-branch equivalent, R_opt ~= R_int_eq = 1/G.
            I_total = float(np.sum(I_seg))
            G = np.sum(1.0 / np.maximum(R_seg, 1e-30))  # [1/Ω]
            R_int_eq = float(1.0 / max(G, 1e-30))
            assert R_load_mppt is not None
            R_load_mppt = _mppt_update_r_load(R_load_mppt, R_int_eq, p.MPPT_ALPHA, p.MPPT_MAX_DELTA_R_OHM)
            V_new = I_total * float(R_load_mppt)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        # Damped fixed-point update
        V_term = (1.0 - OPTIONS.DAMPING) * V_old + OPTIONS.DAMPING * V_new

        # Hard physics clamp: V_term cannot exceed the Thevenin OCV of the parallel array.
        # Compute V_oc_eq fresh from the current E_seg / R_seg (concentrations may have changed).
        _G_clamp = np.sum(1.0 / np.maximum(R_seg, 1e-30))
        _V_oc_clamp = float(np.sum(E_seg / np.maximum(R_seg, 1e-30)) / max(_G_clamp, 1e-30))
        V_term = float(np.clip(V_term, 0.0, _V_oc_clamp))

        # Convergence check
        abs_err = abs(V_term - V_old)
        rel_err = abs_err / max(abs(V_old), 1e-12)

        if OPTIONS.STORE_ITER_DIAGNOSTICS:
            iter_diag.append({"iter": it, "V_term": V_term, "abs_err": abs_err, "rel_err": rel_err})

        if abs_err < OPTIONS.ABS_TOL or rel_err < OPTIONS.REL_TOL:
            converged = True
            break

    # Final totals
    I_total = float(np.sum(I_seg))
    P_raw_total = float(V_term * I_total)
    eta = float(p.MPPT_ETA) if mode == "mppt" else 1.0
    P_total = float(P_raw_total * eta)

    # Equivalent internal resistance seen at terminals (parallel combination of R_seg)
    G_tot = float(np.sum(1.0 / np.maximum(R_seg, 1e-30)))
    R_int_eq = float(1.0 / max(G_tot, 1e-30))
    # Open-circuit equivalent voltage at terminals (weighted)
    V_oc_eq = float(np.sum(E_seg / np.maximum(R_seg, 1e-30)) / max(G_tot, 1e-30))

    if mode == "fixed_load":
        R_load = float(p.R_LOAD_OHM)
    elif mode == "mppt":
        R_load = float(R_load_mppt) if R_load_mppt is not None else float(p.R_LOAD_OHM)
    else:
        R_load = R_int_eq

    active_mask = I_seg > 0.0
    frac_active = float(np.mean(active_mask))
    n_active = int(np.sum(active_mask))

    if I_total <= 0.0:
        regime = "blocked"
    elif frac_active < 0.99:
        regime = "partial"
    else:
        regime = "conducting"

    return {
        "mode": mode,
        "converged": converged,
        "iters_used": (it + 1),
        "any_kappa_oob": any_kappa_oob,
        "any_conc_clamped": any_conc_clamped,
        "any_inversion": any_inversion,
        "frac_active_segments": frac_active,
        "n_active_segments": n_active,
        "regime": regime,
        "V_TERM_V": float(V_term),
        "I_TOTAL_A": I_total,
        "P_TOTAL_W": P_total,
        "P_RAW_TOTAL_W": float(P_raw_total),
        "MPPT_ETA": float(eta),
        "R_INT_EQ_OHM": R_int_eq,
        "V_OC_EQ_V": V_oc_eq,
        "R_LOAD_OHM": R_load,
        "per_segment": {
            "C_H_IN_MOL_L": C_H_IN,
            "C_L_IN_MOL_L": C_L_IN,
            "C_H_OUT_MOL_L": C_H_OUT,
            "C_L_OUT_MOL_L": C_L_OUT,
            "E_SEG_V": E_seg,
            "R_SEG_OHM": R_seg,
            "I_SEG_A": I_seg,
            "n_dot_seg_mol_s": n_dot_seg,
        },
        "iter_diagnostics": (pd.DataFrame(iter_diag) if OPTIONS.STORE_ITER_DIAGNOSTICS else None),
    }



# -------------------------
# 1D axial model (STANDARD topology): series-current, single electrode pair
# -------------------------
def _march_axial_given_current(
    p: DesignInputs,
    I: float,
) -> Dict[str, Any]:
    """Hydraulic march with *uniform* axial current I (single-electrode series-current convention).

    Returns totals (E_total, R_total) plus per-segment diagnostics.
    """
    N = int(p.N_SEG)

    C_H_IN = np.zeros(N, dtype=float)
    C_L_IN = np.zeros(N, dtype=float)
    C_H_OUT = np.zeros(N, dtype=float)
    C_L_OUT = np.zeros(N, dtype=float)

    E_seg           = np.zeros(N, dtype=float)  # [V] local EMF contribution
    R_seg           = np.zeros(N, dtype=float)  # [Ω] local fouled resistance
    R_seg_clean_arr = np.zeros(N, dtype=float)  # [Ω] clean resistance (no fouling)
    foul_frac_seg   = np.zeros(N, dtype=float)  # [-] total fouling fraction per segment
    I_seg = np.full(N, float(I), dtype=float)   # [A] uniform by topology
    n_dot_seg = np.zeros(N, dtype=float)         # [mol/s]

    # Diagnostics / feasibility flags
    min_mg_L, max_mg_L = NACL_TABLE_RANGE_MG_L()
    any_kappa_oob = False
    any_conc_clamped = False
    any_inversion = False

    # Initialize hydraulic inlets
    C_H_IN[0] = p.C_H_IN_MOL_L
    C_L_IN[0] = p.C_L_IN_MOL_L

    for j in range(N):
        if j > 0:
            C_H_IN[j] = C_H_OUT[j - 1]
            C_L_IN[j] = C_L_OUT[j - 1]

        # conductivity table safety check (same convention as legacy)
        mgL_H = _mol_L_to_mg_L_nacl(C_H_IN[j])
        mgL_L = _mol_L_to_mg_L_nacl(C_L_IN[j])
        if (mgL_H < min_mg_L) or (mgL_H > max_mg_L) or (mgL_L < min_mg_L) or (mgL_L > max_mg_L):
            any_kappa_oob = True

        # Local electrical params computed from *local* bulk concentrations
        E_seg[j], R_seg[j] = _axial_segment_params_series_current(p, j, C_H_IN[j], C_L_IN[j])

        # Capture clean resistance and fouling fraction for diagnostics.
        # We call segment_resistance_ohm directly here (cheap: pure arithmetic).
        _E_tmp   = float(E_STACK_OCV_V(p, C_H_IN[j], C_L_IN[j]))
        _R_stack = float(R_STACK_INTERNAL_OHM(p, C_H_IN[j], C_L_IN[j]))
        _R_clean = _R_stack / float(max(N, 1))
        _, R_seg_clean_arr[j], foul_frac_seg[j] = segment_resistance_ohm(
            p, j=j, N=N, R_seg_clean=_R_clean, C_H_local=C_H_IN[j]
        )

        # Salt transfer uses the same stack current through this slice
        n_dot_seg[j] = salt_transfer_mol_per_s(float(I), p.N_CELL_PAIRS, p.BETA) / N

        # Update bulk concentrations by advection + salt flux
        C_H_OUT[j], C_L_OUT[j] = apply_salt_balance(
            C_H_IN[j], C_L_IN[j], p.Q_H_M3_S, p.Q_L_M3_S, n_dot_seg[j]
        )

        if (C_H_OUT[j] <= OPTIONS.CONC_FLOOR_MOL_L + 1e-30) or (C_L_OUT[j] <= OPTIONS.CONC_FLOOR_MOL_L + 1e-30):
            any_conc_clamped = True
        if C_L_OUT[j] > C_H_OUT[j]:
            any_inversion = True

    E_total = float(np.sum(E_seg))
    R_total = float(np.sum(R_seg))
    # Subtract electrode overpotential (Veerman 2010)
    _v_elec_march = getattr(p, "V_ELECTRODE_PAIR_V", 0.0)
    E_total = max(E_total - _v_elec_march, 0.0)

    R_clean_total = float(np.sum(R_seg_clean_arr))
    R_foul_total  = float(np.sum(R_seg - R_seg_clean_arr))

    return {
        "E_total":        E_total,
        "R_total":        R_total,
        "R_clean_total":  R_clean_total,
        "R_foul_total":   R_foul_total,
        "any_kappa_oob":  any_kappa_oob,
        "any_conc_clamped": any_conc_clamped,
        "any_inversion":  any_inversion,
        "per_segment": {
            "C_H_IN_MOL_L":    C_H_IN,
            "C_L_IN_MOL_L":    C_L_IN,
            "C_H_OUT_MOL_L":   C_H_OUT,
            "C_L_OUT_MOL_L":   C_L_OUT,
            "E_SEG_V":         E_seg,
            "R_SEG_CLEAN_OHM": R_seg_clean_arr,
            "FOUL_FRAC_SEG":   foul_frac_seg,
            "R_SEG_OHM":       R_seg,
            "I_SEG_A":         I_seg,
            "n_dot_seg_mol_s": n_dot_seg,
        },
    }


def model_1d_axial_series_current(
    p: DesignInputs,
    mode: Optional[Literal["fixed_load", "max_power", "mppt"]] = None,
) -> Dict[str, Any]:
    """1D axial RED model for a standard stack (single electrode pair).

    Hydraulics: segments in series (outlet of j feeds inlet of j+1).
    Electrical: *series-current* (one stack current I flows through every axial location and every cell pair).

    Electrical closure:
      - fixed_load: V_term(I) = I * R_LOAD_OHM
      - max_power: oracle impedance match (R_load = R_int_eq) => V_term = V_oc_eq/2
      - mppt: controller-limited effective R_load updated toward R_int_eq, then V_term = I * R_load
    """
    sanity_check_inputs(p)
    mode = mode or p.MODE

    # MPPT outer loop: update an effective load based on estimated internal resistance.
    if mode == "mppt":
        R_load = _mppt_init_r_load(p)
        diag_steps = []
        res_last = None
        for k in range(int(max(p.MPPT_N_ITERS, 0))):
            res_last = model_1d_axial_series_current(
                replace(p, R_LOAD_OHM=float(R_load)),
                mode="fixed_load",
            )
            R_int = float(res_last["R_INT_EQ_OHM"])
            R_load = _mppt_update_r_load(R_load, R_int, p.MPPT_ALPHA, p.MPPT_MAX_DELTA_R_OHM)
            diag_steps.append({"mppt_iter": k, "R_eff": float(R_load), "R_int": float(R_int), "P_RAW_TOTAL_W": float(res_last["P_RAW_TOTAL_W"])})
        if res_last is None:
            res_last = model_1d_axial_series_current(p, mode="fixed_load")
        res_last["mode"] = "mppt"
        res_last["R_LOAD_OHM"] = float(R_load)
        res_last["mppt_diagnostics"] = pd.DataFrame(diag_steps)
        res_last["MPPT_ETA"] = float(p.MPPT_ETA)
        res_last["P_TOTAL_W"] = float(res_last["P_RAW_TOTAL_W"] * float(p.MPPT_ETA))
        return res_last

    # Define scalar equation in I. Note: E_total and R_total depend on I via concentration evolution.
    def _f(I: float) -> Tuple[float, Dict[str, Any]]:
        march = _march_axial_given_current(p, I)
        E_total = float(march["E_total"])
        R_int = float(march["R_total"])
        V_term = float(E_total - I * R_int)

        if mode == "fixed_load":
            return float(V_term - I * float(p.R_LOAD_OHM)), march
        elif mode == "max_power":
            # For Thevenin source: optimum at R_load = R_int => V_term = I*R_int.
            # So: E_total - I*R_int - I*R_int = 0
            return float(E_total - 2.0 * I * R_int), march
        else:
            raise ValueError(mode)

    # Bracket the root on I >= 0 using a robust expansion strategy.
    f0, march0 = _f(0.0)

    # If f0 <= 0, the model implies no available EMF (should not happen unless E_total is zero).
    if f0 <= 0.0:
        march_star = march0
        I_star = 0.0
        converged = True
        it_used = 0
    else:
        # Start with a heuristic upper bound based on the I=0 Thevenin estimate.
        E0 = float(march0["E_total"])
        R0 = float(march0["R_total"])
        if mode == "fixed_load":
            I_hi = float(E0 / max(R0 + float(p.R_LOAD_OHM), 1e-12)) * 4.0
        else:
            I_hi = float(E0 / max(2.0 * R0, 1e-12)) * 4.0
        I_hi = max(I_hi, 1e-12)

        f_hi, march_hi = _f(I_hi)
        expand = 0
        while (f_hi > 0.0) and (expand < 40):
            I_hi *= 2.0
            f_hi, march_hi = _f(I_hi)
            expand += 1

        if f_hi > 0.0:
            # No sign change found: fall back to I=0 (treat as blocked under current assumptions).
            march_star = march0
            I_star = 0.0
            converged = False
            it_used = 0
        else:
            # Bisection (monotonicity is expected in practice; bisection is robust regardless).
            I_lo = 0.0
            f_lo = f0
            march_star = march0
            I_star = 0.0
            converged = False
            it_used = 0

            for it in range(int(max(OPTIONS.MAX_ITERS, 1))):
                I_mid = 0.5 * (I_lo + I_hi)
                f_mid, march_mid = _f(I_mid)

                march_star = march_mid
                I_star = float(I_mid)
                it_used = it + 1

                if abs(f_mid) < OPTIONS.ABS_TOL:
                    converged = True
                    break

                if f_mid > 0.0:
                    I_lo, f_lo = I_mid, f_mid
                else:
                    I_hi = I_mid

                if abs(I_hi - I_lo) < OPTIONS.ABS_TOL:
                    converged = True
                    break

    # Final outputs (match legacy schema keys as much as possible)
    E_total = float(march_star["E_total"])
    R_int = float(march_star["R_total"])
    V_term = float(E_total - I_star * R_int)
    V_term = max(V_term, 0.0)

    if mode == "fixed_load":
        R_load = float(p.R_LOAD_OHM)
    else:
        R_load = float(R_int)

    P_raw = float(V_term * I_star)

    # ── Fouling summary (stack level) ────────────────────────────────────────
    R_clean_total = float(march_star.get("R_clean_total", R_int))
    R_foul_total  = float(march_star.get("R_foul_total",  0.0))
    # Counterfactual clean power: same I_star but through R_clean_total only
    V_term_clean          = max(E_total - I_star * R_clean_total, 0.0)
    P_clean               = float(V_term_clean * I_star)
    foul_power_penalty_w  = float(max(P_clean - P_raw, 0.0))
    foul_power_penalty_frac = (
        foul_power_penalty_w / max(P_clean, 1e-30) if P_clean > 0.0 else 0.0
    )

    return {
        "mode": mode,
        "converged": bool(converged),
        "iters_used": int(it_used),
        "any_kappa_oob": bool(march_star["any_kappa_oob"]),
        "any_conc_clamped": bool(march_star["any_conc_clamped"]),
        "any_inversion": bool(march_star["any_inversion"]),
        "regime": ("blocked" if I_star <= 0.0 else "conducting"),
        "V_TERM_V": float(V_term),
        "I_TOTAL_A": float(I_star),
        "P_RAW_TOTAL_W": float(P_raw),
        "P_TOTAL_W": float(P_raw),  # MPPT wrapper overwrites with eta
        "MPPT_ETA": 1.0,
        "R_INT_EQ_OHM": float(R_int),
        "V_OC_EQ_V": float(E_total),
        "R_LOAD_OHM": float(R_load),
        # ── Fouling diagnostics ───────────────────────────────────────────
        "R_CLEAN_TOTAL_OHM":       R_clean_total,
        "R_FOUL_TOTAL_OHM":        R_foul_total,
        "FOUL_POWER_PENALTY_W":    foul_power_penalty_w,
        "FOUL_POWER_PENALTY_FRAC": foul_power_penalty_frac,
        # ─────────────────────────────────────────────────────────────────
        "per_segment": march_star["per_segment"],
    }


## 6) regime map builder
Sweep 1–2 variables across ranges and collect outputs into a tidy DataFrame.
Includes quick plotting helpers for heatmaps/contours suitable for an infographic.

In [ ]:
# -------------------------
# Regime map builder
# -------------------------
@dataclass(frozen=True)
class SweepSpec:
    # 1D sweep: var1_values; 2D sweep: mesh(var1_values, var2_values)
    var1: str
    var1_values: np.ndarray
    var2: Optional[str] = None
    var2_values: Optional[np.ndarray] = None


def run_one(p: DesignInputs, which: Literal["0d", "1d", "1d_series"] = "1d", mode: Optional[str] = None) -> Dict[str, Any]:
    if which == "0d":
        return model_0d(p, mode=mode)
    if which == "1d":
        return model_1d_axial(p, mode=mode)
    if which == "1d_series":
        return model_1d_axial_series_current(p, mode=mode)
    raise ValueError(which)


def run_regime_map(
    fixed_params: DesignInputs,
    sweep_spec: SweepSpec,
    outputs_to_collect: Tuple[str, ...] = ("P_TOTAL_W", "V_TERM_V", "I_TOTAL_A"),
    which: Literal["0d", "1d", "1d_series"] = "1d",
    mode: Optional[Literal["fixed_load", "max_power", "mppt"]] = None,
) -> pd.DataFrame:
    """Return tidy DF: one row per run.

    Adds convenience columns:
      - A_TOTAL_MEM_M2: total active membrane area in the stack = A_MEM_M2 * N_CELL_PAIRS
      - P_DENS_W_M2: power density based on total membrane area
      - regime / feasibility flags (for 1D runs)
    """
    rows = []

    v1 = sweep_spec.var1
    v1_vals = np.array(sweep_spec.var1_values, dtype=float)

    def _append_row(p: DesignInputs, res: Dict[str, Any], base_row: Dict[str, Any]) -> None:
        A_total = float(p.A_MEM_M2 * p.N_CELL_PAIRS)
        base_row["A_TOTAL_MEM_M2"] = A_total

        # Normalize power to membrane area (use 1D or 0D keys)
        # 1d_series returns P_TOTAL_W; 0d returns P_W. Check P_TOTAL_W first.
        if which == "1d":
            P = float(res.get("P_TOTAL_W", np.nan))
        elif which == "0d":
            P = float(res.get("P_W", res.get("P_TOTAL_W", np.nan)))
        else:  # 1d_series
            P = float(res.get("P_TOTAL_W", res.get("P_W", np.nan)))
        base_row["P_DENS_W_M2"] = (P / A_total) if (A_total > 0 and np.isfinite(P)) else np.nan

        # Diagnostics if present
        for dk in [
            "converged", "iters_used",
            "any_kappa_oob", "any_conc_clamped", "any_inversion",
            "frac_active_segments", "n_active_segments", "regime",
        ]:
            if dk in res:
                base_row[dk] = res[dk]

        # Numeric regime code for easy plotting
        reg = base_row.get("regime", None)
        if reg == "blocked":
            base_row["REGIME_CODE"] = 0
        elif reg == "partial":
            base_row["REGIME_CODE"] = 1
        elif reg == "conducting":
            base_row["REGIME_CODE"] = 2
        else:
            base_row["REGIME_CODE"] = np.nan

        rows.append(base_row)

    if sweep_spec.var2 is None:
        for x in v1_vals:
            p = clone_params(fixed_params, **{v1: type(getattr(fixed_params, v1))(x)})
            res = run_one(p, which=which, mode=mode)
            row = {v1: x, "which": which, "mode": mode or p.MODE}

            for k in outputs_to_collect:
                row[k] = res.get(k, np.nan)

            if which == "0d":
                row.setdefault("I_TOTAL_A", res.get("I_A", np.nan))
                row.setdefault("P_TOTAL_W", res.get("P_W", np.nan))
                row.setdefault("V_TERM_V", res.get("V_TERM_V", np.nan))

            _append_row(p, res, row)
    else:
        v2 = sweep_spec.var2
        v2_vals = np.array(sweep_spec.var2_values, dtype=float)

        for x in v1_vals:
            for y in v2_vals:
                p = clone_params(
                    fixed_params,
                    **{
                        v1: type(getattr(fixed_params, v1))(x),
                        v2: type(getattr(fixed_params, v2))(y),
                    },
                )
                res = run_one(p, which=which, mode=mode)
                row = {v1: x, v2: y, "which": which, "mode": mode or p.MODE}

                for k in outputs_to_collect:
                    row[k] = res.get(k, np.nan)

                if which == "0d":
                    row.setdefault("I_TOTAL_A", res.get("I_A", np.nan))
                    row.setdefault("P_TOTAL_W", res.get("P_W", np.nan))
                    row.setdefault("V_TERM_V", res.get("V_TERM_V", np.nan))

                _append_row(p, res, row)

    return pd.DataFrame(rows)


def heatmap_from_df(df: pd.DataFrame, x: str, y: str, z: str, title: str = "") -> None:
    """Simple heatmap from tidy DF without seaborn."""
    if df.empty or z not in df.columns:
        print(f"[heatmap_from_df] skipping '{title}': empty df or missing column '{z}'")
        return
    # Use pivot_table (aggfunc='mean') to gracefully handle non-rectangular or
    # duplicate-keyed grids that would raise in df.pivot().
    pivot = df.pivot_table(index=y, columns=x, values=z, aggfunc="mean")
    X = pivot.columns.values
    Y = pivot.index.values
    Z = pivot.values

    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.imshow(Z, origin="lower", aspect="auto")
    ax.set_xticks(np.arange(len(X)))
    ax.set_xticklabels([f"{v:g}" for v in X], rotation=45, ha="right")
    ax.set_yticks(np.arange(len(Y)))
    ax.set_yticklabels([f"{v:g}" for v in Y])
    ax.set_xlabel(_pretty(x), fontsize=11)
    ax.set_ylabel(_pretty(y), fontsize=11)
    # Wrap title at ~60 chars so it fits on two lines without shrinking the axes
    import textwrap as _tw
    wrapped = "\n".join(_tw.wrap(title or f"{z} heatmap", width=60))
    ax.set_title(wrapped, fontsize=11, pad=10)
    fig.colorbar(im, ax=ax, label=_pretty(z))
    fig.tight_layout()
    plt.show()


def contour_from_df(df: pd.DataFrame, x: str, y: str, z: str, title: str = "") -> None:
    pivot = df.pivot(index=y, columns=x, values=z)
    X = pivot.columns.values
    Y = pivot.index.values
    Z = pivot.values
    XX, YY = np.meshgrid(X, Y)

    plt.figure()
    plt.contourf(XX, YY, Z, levels=20)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(title or f"{z} contour")
    plt.colorbar(label=z)
    plt.tight_layout()
    plt.show()


## Reproducible experiment harness (machine + harness)

This section standardizes how we define and run experiments:

- The **machine** is the physics engine (`model_0d`, `model_1d_axial`, `model_1d_axial_series_current`) and stays “pure”.
- The **harness** defines knobs, validates them, applies sweeps, runs named experiments, saves run bundles, and emits standardized plots.

Everything below this section (starting at **Validation harness**) is preserved as legacy / scratch, but the intent is that new work runs through `run_experiment(...)`.


In [ ]:
# -------------------------
# Harness core: knobs, validation, sweeps, runs, standard plots
# -------------------------

from dataclasses import dataclass
import json, os, time
from pathlib import Path
import shutil
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# A small set of knobs that should be explicitly reviewed for every new experiment family.
# These are "required" in the sense that they should never accidentally drift unnoticed.
# Defaults exist in DesignInputs, but we still validate for sanity/physics.
REQUIRED_KNOBS = {
    "N_CELL_PAIRS",
    "A_MEM_M2",
    "Q_H_M3_S",
    "Q_L_M3_S",
    "C_H_IN_MOL_L",
    "C_L_IN_MOL_L",
    "GAP_M",
    "ALPHA_PERM",
    "ASR_AEM_OHM_M2",
    "ASR_CEM_OHM_M2",
    "MODE",
}

def make_base_knobs(overrides: Dict[str, Any]) -> Dict[str, Any]:
    """Return a full knob dict using DesignInputs defaults + overrides.
    This is intentionally dict-based so we can JSON-save configs and keep backward compatibility.
    """
    base = {k: f.default for k, f in DesignInputs.__dataclass_fields__.items()}
    base.update(dict(overrides or {}))
    return base

def _require(cond: bool, msg: str) -> None:
    if not cond:
        raise ValueError(msg)

def validate_knobs(knobs: Dict[str, Any]) -> None:
    """Lightweight validation for missing knobs + non-physical values + cross-field constraints."""
    missing = [k for k in REQUIRED_KNOBS if k not in knobs]
    _require(len(missing) == 0, f"Missing required knobs: {missing}")

    # Non-physical checks (keep conservative: fail on clearly impossible values)
    _require(int(knobs["N_CELL_PAIRS"]) > 0, "N_CELL_PAIRS must be > 0")
    _require(float(knobs["A_MEM_M2"]) > 0.0, "A_MEM_M2 must be > 0")
    _require(float(knobs["Q_H_M3_S"]) > 0.0, "Q_H_M3_S must be > 0")
    _require(float(knobs["Q_L_M3_S"]) > 0.0, "Q_L_M3_S must be > 0")
    _require(float(knobs["GAP_M"]) > 0.0, "GAP_M must be > 0")
    _require(0.0 <= float(knobs["ALPHA_PERM"]) <= 1.0, "ALPHA_PERM must be in [0,1]")
    _require(float(knobs["ASR_AEM_OHM_M2"]) >= 0.0, "ASR_AEM_OHM_M2 must be >= 0")
    _require(float(knobs["ASR_CEM_OHM_M2"]) >= 0.0, "ASR_CEM_OHM_M2 must be >= 0")
    _require(float(knobs["ASR_EXTRA_OHM_M2"]) >= 0.0, "ASR_EXTRA_OHM_M2 must be >= 0")

    # Concentrations (allow equality; we may study edge cases)
    _require(float(knobs["C_H_IN_MOL_L"]) >= 0.0, "C_H_IN_MOL_L must be >= 0")
    _require(float(knobs["C_L_IN_MOL_L"]) >= 0.0, "C_L_IN_MOL_L must be >= 0")

    # Mode-specific constraints
    mode = knobs.get("MODE", "fixed_load")
    _require(mode in ("fixed_load", "max_power", "mppt"), f"MODE must be fixed_load|max_power|mppt, got {mode}")
    if mode == "fixed_load":
        _require("R_LOAD_OHM" in knobs and knobs["R_LOAD_OHM"] is not None, "MODE=fixed_load requires R_LOAD_OHM")
        _require(float(knobs["R_LOAD_OHM"]) > 0.0, "R_LOAD_OHM must be > 0")
    if mode == "mppt":
        # MPPT knobs have defaults; keep a minimal sanity check
        _require(float(knobs.get("MPPT_ETA", 1.0)) > 0.0, "MPPT_ETA must be > 0")
        _require(int(knobs.get("MPPT_N_ITERS", 1)) >= 1, "MPPT_N_ITERS must be >= 1")

    # Discretization
    _require(int(knobs.get("N_SEG", 1)) >= 1, "N_SEG must be >= 1")

def knobs_to_design_inputs(knobs: Dict[str, Any]) -> DesignInputs:
    """Convert knob dict into the frozen DesignInputs dataclass, preserving naming conventions."""
    # Filter to known fields only (helps forward-compat with extra knobs)
    fields = DesignInputs.__dataclass_fields__.keys()
    filt = {k: knobs[k] for k in knobs.keys() if k in fields}
    return DesignInputs(**filt)

# --- Sweep adapter ---
@dataclass(frozen=True)
class Sweep1D:
    var: str
    values: np.ndarray

@dataclass(frozen=True)
class Sweep2D:
    var1: str
    var1_values: np.ndarray
    var2: str
    var2_values: np.ndarray

@dataclass(frozen=True)
class Sweep3D:
    var1: str
    var1_values: np.ndarray
    var2: str
    var2_values: np.ndarray
    var3: str
    var3_values: np.ndarray


@dataclass(frozen=True)
class SweepMorris:
    """Morris one-at-a-time elementary effects screening.

    Parameters
    ----------
    params : dict
        {param_name: (lo, hi)} — the parameter ranges to screen.
    r : int
        Number of Morris trajectories (default 10).
    p : int
        Number of grid levels (default 4).
    seed : int
        Random seed for trajectory generation.
    """
    params: Dict[str, tuple]   # {name: (lo, hi)}
    r:      int = 10
    p:      int = 4
    seed:   int = 42

    @property
    def var_names(self) -> List[str]:
        return list(self.params.keys())

    def generate_trajectories(self, base_knobs: dict) -> List[Dict[str, Any]]:
        """Generate Morris OAT trajectory points.

        Returns a list of knob dicts.  Each trajectory has (k+1) points
        where k = len(params).  Adjacent points differ in exactly one
        parameter by delta = p / (2*(p-1)).  Metadata fields _morris_traj
        and _morris_step are attached to each knob dict for post-processing.
        """
        rng = np.random.RandomState(self.seed)
        names = self.var_names
        k = len(names)
        delta = self.p / (2.0 * (self.p - 1))

        all_knobs = []
        for traj_idx in range(self.r):
            # Random base point on the grid
            x = np.zeros(k)
            for j in range(k):
                x[j] = rng.choice(np.linspace(0, 1 - delta, self.p))

            # Random permutation of parameter indices
            order = rng.permutation(k)

            # Build trajectory: k+1 points
            traj_points = [x.copy()]
            for step, j in enumerate(order):
                x_new = x.copy()
                if rng.random() < 0.5:
                    x_new[j] = x[j] + delta
                else:
                    x_new[j] = x[j] - delta
                # Clamp to [0, 1]
                x_new[j] = np.clip(x_new[j], 0.0, 1.0)
                traj_points.append(x_new.copy())
                x = x_new

            # Convert normalized points to knob dicts
            for pt_idx, xpt in enumerate(traj_points):
                knobs = dict(base_knobs)
                for j, name in enumerate(names):
                    lo, hi = self.params[name]
                    knobs[name] = float(lo + xpt[j] * (hi - lo))
                knobs["_morris_traj"] = traj_idx
                knobs["_morris_step"] = pt_idx
                # Track which param changed (step 0 = base, no change)
                if pt_idx > 0:
                    knobs["_morris_changed_param"] = names[order[pt_idx - 1]]
                else:
                    knobs["_morris_changed_param"] = "__base__"
                all_knobs.append(knobs)

        return all_knobs


def apply_sweep(base_knobs: Dict[str, Any], sweep: Optional[Union[Sweep1D, Sweep2D, "Sweep3D"]]) -> List[Dict[str, Any]]:
    """Return list of knob dicts: one per run. Does not mutate base_knobs."""
    if sweep is None:
        return [dict(base_knobs)]

    # 1D sweep
    if isinstance(sweep, Sweep1D):
        out: List[Dict[str, Any]] = []
        for v in list(sweep.values):
            k = dict(base_knobs)
            k[sweep.var] = float(v)
            out.append(k)
        return out

    # 2D sweep
    if isinstance(sweep, Sweep2D):
        out2: List[Dict[str, Any]] = []
        for x in list(sweep.var1_values):
            for y in list(sweep.var2_values):
                k = dict(base_knobs)
                k[sweep.var1] = float(x)
                k[sweep.var2] = float(y)
                out2.append(k)
        return out2

    # 3D sweep
    if isinstance(sweep, Sweep3D):
        out3: List[Dict[str, Any]] = []
        for x in list(sweep.var1_values):
            for y in list(sweep.var2_values):
                for z in list(sweep.var3_values):
                    k = dict(base_knobs)
                    k[sweep.var1] = float(x)
                    k[sweep.var2] = float(y)
                    k[sweep.var3] = float(z)
                    out3.append(k)
        return out3

    # Morris OAT screening
    if isinstance(sweep, SweepMorris):
        return sweep.generate_trajectories(base_knobs)

    raise TypeError(f"Unknown sweep type: {type(sweep)}")

# --- Run result schema helpers ---
def _jsonable(x: Any) -> Any:
    if isinstance(x, (np.floating, np.integer)):
        return x.item()
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, dict):
        return {k: _jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_jsonable(v) for v in x]
    return x

def _now_tag() -> str:
    return time.strftime("%Y%m%d_%H%M%S")

def save_run_bundle(bundle: Dict[str, Any], out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / "run_bundle.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(_jsonable(bundle), f, indent=2, sort_keys=True)
    return path

def make_run_bundle(experiment_id: str, knobs: Dict[str, Any], res: Dict[str, Any], which: str, mode: str) -> Dict[str, Any]:
    """Normalize run output into a stable schema."""
    # Canonical curves (best effort; keys vary by model)
    I = res.get("I_TOTAL_A", res.get("I_A"))
    V = res.get("V_TERM_V", res.get("V_V"))
    P = res.get("P_TOTAL_W", res.get("P_W"))
    bundle = {
        "experiment_id": experiment_id,
        "which": which,
        "mode": mode,
        "knobs_used": dict(knobs),
        "results_raw": dict(res),
        "curves": {
            "I_A": I,
            "V_V": V,
            "P_W": P,
        },
        "summary": {
            "P_TOTAL_W": res.get("P_TOTAL_W", res.get("P_W")),
            "V_TERM_V": res.get("V_TERM_V", res.get("V_V")),
            "I_TOTAL_A": res.get("I_TOTAL_A", res.get("I_A")),
            "P_DENS_W_M2": res.get("P_DENS_W_M2"),  # present in regime map rows; may be missing here
        },
        "meta": {
            "timestamp": _now_tag(),
        }
    }
    return bundle


# --- NEW: aggregated sweep bundle saver (primary artifact) ---
from dataclasses import asdict  # add alongside existing dataclass import

def save_sweep_table_parquet(df: pd.DataFrame, run_dir: Path, fname: str = "sweep_table.parquet") -> str:
    """Save the sweep table (tidy DF) as parquet. Returns filename (relative to run_dir)."""
    run_dir.mkdir(parents=True, exist_ok=True)
    path = run_dir / fname
    try:
        df.to_parquet(path, index=False)
    except Exception as e:
        raise RuntimeError(
            "Failed to write parquet. If pyarrow/fastparquet is missing, install one of them."
        ) from e
    return fname


def _regime_append_row_legacy_compatible(
    p: DesignInputs,
    res: Dict[str, Any],
    row: Dict[str, Any],
    which: str,
) -> Dict[str, Any]:
    """Match the legacy regime-map row semantics (including derived columns)."""
    A_total = float(p.A_MEM_M2 * p.N_CELL_PAIRS)
    row["A_TOTAL_MEM_M2"] = A_total

    # Normalize power to membrane area (legacy semantics)
    # Both 1d and 1d_series use P_TOTAL_W as the canonical key.
    # P_W is the 0d key; fall back to it only as a last resort.
    if which == "1d":
        P = float(res.get("P_TOTAL_W", np.nan))
    else:
        P = float(res.get("P_TOTAL_W", res.get("P_W", np.nan)))
    row["P_DENS_W_M2"] = (P / A_total) if (A_total > 0 and np.isfinite(P)) else np.nan

    # Diagnostics if present
    for dk in [
        "converged", "iters_used",
        "any_kappa_oob", "any_conc_clamped", "any_inversion",
        "frac_active_segments", "n_active_segments", "regime",
        "R_INT_EQ_OHM", "V_OC_EQ_V",
    ]:
        if dk in res:
            row[dk] = res[dk]

    # Numeric regime code for easy plotting
    reg = row.get("regime", None)
    if reg == "blocked":
        row["REGIME_CODE"] = 0
    elif reg == "partial":
        row["REGIME_CODE"] = 1
    elif reg == "conducting":
        row["REGIME_CODE"] = 2
    else:
        row["REGIME_CODE"] = np.nan

    return row


def save_sweep_bundle(
    experiment_id: str,
    run_dir: Path,
    spec: "ExperimentSpec",
    *,
    artifacts: Dict[str, str],
    points_summary: List[Dict[str, Any]],
) -> Path:
    """Save ONE aggregated envelope for an experiment run.

    Primary artifact:
        runs/<experiment_id>/<run_tag>/run_bundle.json

    The plot-ready sweep table is stored in parquet (see artifacts pointers).
    """
    run_dir.mkdir(parents=True, exist_ok=True)

    spec_dict = {k: v for k, v in asdict(spec).items() if k != 'post_analysis'}

    import datetime as _dt
    agg = {
        "experiment_id": experiment_id,
        "run_datetime_iso": _dt.datetime.now().isoformat(timespec="seconds"),
        "run_tag": run_dir.name,
        "description": spec.description,
        "spec": spec_dict,
        "base_overrides": {k: _jsonable(v) for k, v in spec.base_overrides.items()},
        "base_knobs_full": {k: _jsonable(v) for k, v in make_base_knobs(spec.base_overrides).items()},
        "artifacts": dict(artifacts),
        "points_summary": list(points_summary),
    }

    path = run_dir / "run_bundle.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(_jsonable(agg), f, indent=2, sort_keys=True)
    return path



def make_standard_plots(bundle: Dict[str, Any], title_prefix: str = "") -> None:
    """Standard plot set for a single run bundle. Reads bundle only; does not execute the model."""
    I = bundle["curves"].get("I_A", None)
    V = bundle["curves"].get("V_V", None)
    P = bundle["curves"].get("P_W", None)

    # If curves are scalars, skip line plots (still print summary)
    if isinstance(I, (int, float, np.floating, np.integer)) or I is None:
        print(f"[plots] {bundle['experiment_id']} produced scalar outputs; skipping line plots.")
        return

    I = np.asarray(I, dtype=float)
    if V is not None:
        V = np.asarray(V, dtype=float)
    if P is not None:
        P = np.asarray(P, dtype=float)

    ttl = f"{title_prefix}{bundle['experiment_id']} ({bundle['which']}, mode={bundle['mode']})"

    if V is not None:
        plt.figure()
        plt.plot(I, V)
        plt.xlabel("Current I [A]")
        plt.ylabel("Terminal voltage V [V]")
        plt.title(ttl + " — V–I")
        plt.grid(True)
        plt.show()

    if P is not None:
        plt.figure()
        plt.plot(I, P)
        plt.xlabel("Current I [A]")
        plt.ylabel("Power P [W]")
        plt.title(ttl + " — P–I")
        plt.grid(True)
        plt.show()


# -------------------------
# Disk-first plotting (reads saved artifacts only)
# -------------------------

def load_run_from_disk(run_dir: Path) -> Tuple[Dict[str, Any], pd.DataFrame]:
    """Load (envelope JSON, sweep table DF) from a run directory."""
    bundle_path = run_dir / "run_bundle.json"
    with open(bundle_path, "r", encoding="utf-8") as f:
        env = json.load(f)
    artifacts = env.get("artifacts", {})
    sweep_fname = artifacts.get("sweep_table_parquet", None)
    if sweep_fname is None:
        raise KeyError("run_bundle.json is missing artifacts['sweep_table_parquet']")
    df = pd.read_parquet(run_dir / sweep_fname)
    return env, df


from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_LABELS = {
    "R_LOAD_OHM": "Load resistance R_load [Ω]",
    "N_CELL_PAIRS": "Cell pairs N",
    "A_MEM_M2": "Membrane area per pair A_mem [m²]",
    "GAP_M": "Spacer gap [m]",
    "C_H_IN_MOL_L": "High stream inlet C_H,in [mol/L]",
    "C_L_IN_MOL_L": "Low stream inlet C_L,in [mol/L]",
    "P_TOTAL_W": "Total power P [W]",
    "P_MAX_W": "Max power P_max [W]",
    "P_DENS_W_M2": "Power density P/A [W/m²]",
    "V_TERM_V": "Terminal voltage V [V]",
    "I_TOTAL_A": "Current I [A]",
    "REGIME_CODE": "Regime code",
    "converged": "Converged (0/1)",
    "any_kappa_oob": "Any κ out-of-bounds (0/1)",
    "any_conc_clamped": "Any concentration clamped (0/1)",
    "any_inversion": "Any inversion (0/1)",
    "R_INT_EQ_OHM": "Internal resistance R_int [Ω]",
    "V_OC_EQ_V": "Open-circuit voltage V_oc [V]",
    "A_TOTAL_MEM_M2": "Total membrane area A_total [m²]",
    # ── CDF fouling diagnostics ──────────────────────────────────────────────
    "DIVALENT_FRACTION":        "Divalent ion fraction f_div [–]",
    "FOULING_BETA":             "Fouling severity β [–]",
    "R_CLEAN_TOTAL_OHM":        "Clean internal resistance R_clean [Ω]",
    "R_FOUL_TOTAL_OHM":         "Resistance added by fouling ΔR_foul [Ω]",
    "FOUL_POWER_PENALTY_W":     "Power lost to fouling ΔP_foul [W]",
    "FOUL_POWER_PENALTY_FRAC":  "Fractional power loss to fouling [–]",
    "R_SEG_CLEAN_OHM":          "Segment clean resistance [Ω]",
    "FOUL_FRAC_SEG":            "Segment fouling fraction [–]",
}

def _pretty(name: str) -> str:
    return _LABELS.get(name, name)

def _get_sweep_axes_from_env(env: dict, df: pd.DataFrame):
    """Return (var1, var2, var3) from the saved spec. Missing axes are None."""
    spec = env.get("spec") or {}
    sweep = spec.get("sweep")
    if not isinstance(sweep, dict):
        return (None, None, None)

    if "var" in sweep:
        x = sweep["var"]
        return (x, None, None) if x in df.columns else (None, None, None)

    if "var3" in sweep and "var1" in sweep and "var2" in sweep:
        x, y, z = sweep["var1"], sweep["var2"], sweep["var3"]
        ok = all(v in df.columns for v in [x, y, z])
        return (x, y, z) if ok else (None, None, None)

    if "var1" in sweep and "var2" in sweep:
        x, y = sweep["var1"], sweep["var2"]
        return (x, y, None) if (x in df.columns and y in df.columns) else (None, None, None)

    return (None, None, None)

def _filter_quality(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "converged" in out.columns:
        out = out[out["converged"] == True]
    if "any_kappa_oob" in out.columns:
        out = out[out["any_kappa_oob"] == False]
    return out

def _first_present(df: pd.DataFrame, names):
    for n in names:
        if n in df.columns:
            return n
    return None

def _lineplot(df: pd.DataFrame, x: str, y: str, title: str, group: str | None = None, max_groups: int = 6) -> None:
    import textwrap as _tw
    fig, ax = plt.subplots(figsize=(10, 6))
    if group is None or group not in df.columns or df[group].nunique() <= 1:
        d = df.sort_values(x)
        ax.plot(d[x].astype(float).values, d[y].astype(float).values)
    else:
        levels = sorted(df[group].dropna().unique())
        if len(levels) > max_groups:
            idxs = np.linspace(0, len(levels)-1, max_groups).round().astype(int)
            levels = [levels[i] for i in idxs]
        for gv in levels:
            d = df[df[group] == gv].sort_values(x)
            ax.plot(d[x].astype(float).values, d[y].astype(float).values, label=f"{_pretty(group)}={gv}")
        ax.legend(fontsize=9)
    ax.set_xlabel(_pretty(x), fontsize=11)
    ax.set_ylabel(_pretty(y), fontsize=11)
    ax.set_title("\n".join(_tw.wrap(title, width=70)), fontsize=11, pad=10)
    ax.grid(True)
    fig.tight_layout()
    plt.show()


def _plot_3d_geometry_sweep(
    df_raw: pd.DataFrame,
    df_ok: pd.DataFrame,
    var_salinity: str,
    var_ncp: str,
    var_area: str,
    P: Optional[str],
    Pd: Optional[str],
    prefix: str,
) -> None:
    """
    3D sweep visualisation: C_H_IN_MOL_L × N_CELL_PAIRS × A_MEM_M2 (max_power mode).

    Produces:
      A. Per-salinity geometry-plane heatmaps:
         1. P_DENS [W/m²]       — power density on N × A plane for each C_H
         2. R_INT_EQ_OHM [Ω]    — impedance-matched internal resistance
         3. V_OC_EQ_V [V]       — open-circuit voltage
         4. I_TOTAL_A [A]        — operating current

      B. Envelope line plots (best geometry at each C_H):
         5. Best P_DENS vs C_H
         6. R_INT at best geometry vs C_H
         7. V_OC at best geometry vs C_H
         8. I at best geometry vs C_H
         9. Optimal N_CELL_PAIRS vs C_H  (argmax P_DENS)
        10. Optimal A_MEM_M2 vs C_H     (argmax P_DENS)

      C. Slice line plots — P_DENS, R_INT, V_OC, I vs C_H:
        11-14. Sliced by N_CELL_PAIRS levels (fixed representative A_MEM)
        15-18. Sliced by A_MEM_M2 levels (fixed representative N_CELL_PAIRS)
    """
    metric = Pd if Pd and Pd in df_ok.columns else P
    if metric is None:
        print(f"[3D plot] {prefix}: no power metric available")
        return

    salinity_levels = sorted(df_ok[var_salinity].dropna().unique())
    R_col = "R_INT_EQ_OHM" if "R_INT_EQ_OHM" in df_ok.columns else None
    V_col = "V_OC_EQ_V"    if "V_OC_EQ_V"    in df_ok.columns else None
    I_col = "I_TOTAL_A"    if "I_TOTAL_A"    in df_ok.columns else None

    # ── A. Per-salinity heatmaps on geometry plane ──────────────────────────
    heatmap_metrics = [
        (metric,  "power density"),
        (R_col,   "internal resistance"),
        (V_col,   "open-circuit voltage"),
        (I_col,   "current"),
    ]
    for c_h in salinity_levels:
        slice_df = df_ok[df_ok[var_salinity] == c_h]
        if slice_df.empty:
            continue
        for col, desc in heatmap_metrics:
            if col is None or col not in slice_df.columns:
                continue
            try:
                heatmap_from_df(
                    slice_df,
                    x=var_ncp,
                    y=var_area,
                    z=col,
                    title=(
                        f"{prefix}\n"
                        f"{_pretty(col)} ({desc}) — geometry plane  |  "
                        f"C_H = {c_h:.2g} mol/L"
                    ),
                )
            except Exception as _e:
                print(f"[3D plot] {prefix}: heatmap '{col}' at C_H={c_h} failed: {_e}")

    # ── Build envelope (best-P_DENS geometry at each C_H) ───────────────────
    best_rows = []
    for c_h in salinity_levels:
        slice_df = df_ok[df_ok[var_salinity] == c_h].copy()
        if slice_df.empty:
            continue
        idx_best = slice_df[metric].astype(float).idxmax()
        row = slice_df.loc[idx_best].copy()
        best_rows.append(row)

    if not best_rows:
        return
    env_df = pd.DataFrame(best_rows).sort_values(var_salinity)
    sal_vals = env_df[var_salinity].astype(float).values

    # ── B. Envelope line plots ───────────────────────────────────────────────
    # B1: Best P_DENS vs C_H
    import textwrap as _tw

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(sal_vals, env_df[metric].astype(float).values, marker="o")
    ax.set_xlabel(_pretty(var_salinity), fontsize=11)
    ax.set_ylabel(_pretty(metric), fontsize=11)
    ax.set_title("\n".join(_tw.wrap(f"{prefix} — best {_pretty(metric)} vs {_pretty(var_salinity)} (max over all geometries)", width=70)), fontsize=11, pad=10)
    ax.grid(True)
    fig.tight_layout()

    plt.show()

    # B2: R_INT at best geometry vs C_H
    if R_col and R_col in env_df.columns:
        import textwrap as _tw

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(sal_vals, env_df[R_col].astype(float).values, marker="o", color="tab:orange")
        ax.set_xlabel(_pretty(var_salinity), fontsize=11)
        ax.set_ylabel(_pretty(R_col), fontsize=11)
        ax.set_title("\n".join(_tw.wrap(f"{prefix} — {_pretty(R_col)} at best geometry vs {_pretty(var_salinity)}", width=70)), fontsize=11, pad=10)
        ax.grid(True)
        fig.tight_layout()

        plt.show()

    # B3: V_OC at best geometry vs C_H
    if V_col and V_col in env_df.columns:
        import textwrap as _tw

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(sal_vals, env_df[V_col].astype(float).values, marker="o", color="tab:green")
        ax.set_xlabel(_pretty(var_salinity), fontsize=11)
        ax.set_ylabel(_pretty(V_col), fontsize=11)
        ax.set_title("\n".join(_tw.wrap(f"{prefix} — {_pretty(V_col)} at best geometry vs {_pretty(var_salinity)}", width=70)), fontsize=11, pad=10)
        ax.grid(True)
        fig.tight_layout()

        plt.show()

    # B4: I at best geometry vs C_H
    if I_col and I_col in env_df.columns:
        import textwrap as _tw

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(sal_vals, env_df[I_col].astype(float).values, marker="o", color="tab:red")
        ax.set_xlabel(_pretty(var_salinity), fontsize=11)
        ax.set_ylabel(_pretty(I_col), fontsize=11)
        ax.set_title("\n".join(_tw.wrap(f"{prefix} — {_pretty(I_col)} at best geometry vs {_pretty(var_salinity)}", width=70)), fontsize=11, pad=10)
        ax.grid(True)
        fig.tight_layout()

        plt.show()

    # B5-6: Optimal geometry axes vs C_H
    import textwrap as _tw

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(sal_vals, env_df[var_ncp].astype(float).values, marker="o", color="tab:purple")
    ax.set_xlabel(_pretty(var_salinity), fontsize=11)
    ax.set_ylabel(_pretty(var_ncp), fontsize=11)
    ax.set_title("\n".join(_tw.wrap(f"{prefix} — optimal {_pretty(var_ncp)} vs {_pretty(var_salinity)} (argmax {_pretty(metric)})", width=70)), fontsize=11, pad=10)
    ax.grid(True)
    fig.tight_layout()

    plt.show()

    import textwrap as _tw


    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(sal_vals, env_df[var_area].astype(float).values, marker="o", color="tab:brown")
    ax.set_xlabel(_pretty(var_salinity), fontsize=11)
    ax.set_ylabel(_pretty(var_area), fontsize=11)
    ax.set_title("\n".join(_tw.wrap(f"{prefix} — optimal {_pretty(var_area)} vs {_pretty(var_salinity)} (argmax {_pretty(metric)})", width=70)), fontsize=11, pad=10)
    ax.grid(True)
    fig.tight_layout()

    plt.show()

    # ── C. Slice line plots ──────────────────────────────────────────────────
    # Helper: pick up to max_lines representative levels from a column
    def _pick_levels(col: str, max_lines: int = 6) -> list:
        levels = sorted(df_ok[col].dropna().unique())
        if len(levels) <= max_lines:
            return levels
        idxs = np.linspace(0, len(levels) - 1, max_lines).round().astype(int)
        return [levels[i] for i in idxs]

    slice_metrics = [
        (metric, "tab:blue"),
        (R_col,  "tab:orange"),
        (V_col,  "tab:green"),
        (I_col,  "tab:red"),
    ]

    # C1: Sliced by N_CELL_PAIRS — each curve is one N level, x = C_H
    ncp_levels = _pick_levels(var_ncp)
    for col, color in slice_metrics:
        if col is None or col not in df_ok.columns:
            continue
        import textwrap as _tw

        fig, ax = plt.subplots(figsize=(10, 6))
        cmap = plt.get_cmap("tab10")
        for i, ncp_val in enumerate(ncp_levels):
            sub = df_ok[df_ok[var_ncp] == ncp_val].sort_values(var_salinity)
            if sub.empty:
                continue
            ax.plot(
                sub[var_salinity].astype(float).values,
                sub[col].astype(float).values,
                marker="o", color=cmap(i),
                label=f"N={int(ncp_val)}",
            )
        ax.set_xlabel(_pretty(var_salinity), fontsize=11)
        ax.set_ylabel(_pretty(col), fontsize=11)
        ax.set_title("\n".join(_tw.wrap(f"{prefix} — {_pretty(col)} vs {_pretty(var_salinity)} (slices by {_pretty(var_ncp)})", width=70)), fontsize=11, pad=10)
        ax.legend(fontsize=8)
        ax.grid(True)
        fig.tight_layout()

        plt.show()

    # C2: Sliced by A_MEM_M2 — each curve is one A level, x = C_H
    area_levels = _pick_levels(var_area)
    for col, color in slice_metrics:
        if col is None or col not in df_ok.columns:
            continue
        import textwrap as _tw

        fig, ax = plt.subplots(figsize=(10, 6))
        cmap = plt.get_cmap("tab10")
        for i, a_val in enumerate(area_levels):
            sub = df_ok[df_ok[var_area] == a_val].sort_values(var_salinity)
            if sub.empty:
                continue
            ax.plot(
                sub[var_salinity].astype(float).values,
                sub[col].astype(float).values,
                marker="o", color=cmap(i),
                label=f"A={a_val:.4g} m²",
            )
        ax.set_xlabel(_pretty(var_salinity), fontsize=11)
        ax.set_ylabel(_pretty(col), fontsize=11)
        ax.set_title("\n".join(_tw.wrap(f"{prefix} — {_pretty(col)} vs {_pretty(var_salinity)} (slices by {_pretty(var_area)})", width=70)), fontsize=11, pad=10)
        ax.legend(fontsize=8)
        ax.grid(True)
        fig.tight_layout()

        plt.show()


def plot_run_from_disk(run_dir: Path) -> None:
    env, df = load_run_from_disk(run_dir)

    exp_id = env.get("experiment_id", "")
    run_tag = env.get("run_tag", "")
    prefix = f"{exp_id} [{run_tag}]".strip()

    x, y, z3 = _get_sweep_axes_from_env(env, df)
    if x is None:
        print(f"[plot_run_from_disk] {prefix}: no sweep axes detected in run_bundle.json spec.")
        return

    # Choose canonical power metric
    P = _first_present(df, ["P_TOTAL_W", "P_MAX_W"])
    Pd = _first_present(df, ["P_DENS_W_M2"])
    V = _first_present(df, ["V_TERM_V"])
    I = _first_present(df, ["I_TOTAL_A"])

    # If quality filtering wipes everything, fall back to unfiltered (but print it)
    df_ok = _filter_quality(df)
    if len(df_ok) == 0:
        print(f"[plot_run_from_disk] {prefix}: quality filter removed all rows; plotting unfiltered.")
        df_ok = df.copy()

    # -------------------------
    # 3D sweep: sliced 2D heatmaps (var1=C_H slices, var2×var3 geometry plane)
    # Convention: var1=C_H_IN_MOL_L, var2=N_CELL_PAIRS, var3=A_MEM_M2
    # -------------------------
    if z3 is not None:
        _plot_3d_geometry_sweep(df, df_ok, x, y, z3, P, Pd, prefix)
        return

    # -------------------------
    # 1D sweep: line plots
    # -------------------------
    if y is None:
        for metric in [P, Pd, V, I, "REGIME_CODE"]:
            if metric and metric in df_ok.columns:
                _lineplot(df_ok, x=x, y=metric, title=f"{prefix} — {_pretty(metric)} vs {_pretty(x)}")
        return

    # -------------------------
    # 2D sweep: heatmaps for common metrics
    # -------------------------
    # physics heatmaps
    for metric in [P, Pd, V, I, "REGIME_CODE"]:
        if metric and metric in df_ok.columns:
            try:
                heatmap_from_df(
                    df_ok,
                    x=x,
                    y=y,
                    z=metric,
                    title=f"{prefix} — {_pretty(metric)} heatmap",
                )
            except Exception as _e:
                print(f"[plot_run_from_disk] {prefix}: heatmap '{metric}' failed: {_e}")

    # diagnostic heatmaps (unfiltered)
    for metric in ["converged", "any_kappa_oob", "any_conc_clamped", "any_inversion"]:
        if metric in df.columns:
            try:
                heatmap_from_df(
                    df,
                    x=x,
                    y=y,
                    z=metric,
                    title=f"{prefix} — {_pretty(metric)} heatmap",
                )
            except Exception as _e:
                print(f"[plot_run_from_disk] {prefix}: heatmap '{metric}' failed: {_e}")

    # -------------------------
    # Special-case: load sweep present => ridge/optimum plots + slices
    # Works for SYNOPSYS-BASELINE-RUN (C_H_IN_MOL_L × R_LOAD_OHM)
    # -------------------------
    axes = {x, y}
    if "R_LOAD_OHM" in axes and P is not None:
        load_axis = "R_LOAD_OHM"
        other_axis = (y if x == load_axis else x)

        # slices: P vs load for a few other-axis levels
        try:
            _lineplot(
                df_ok,
                x=load_axis,
                y=P,
                title=f"{prefix} — {_pretty(P)} vs {_pretty(load_axis)} (slices by {_pretty(other_axis)})",
                group=other_axis,
                max_groups=6,
            )
        except Exception as _e:
            print(f"[plot_run_from_disk] {prefix}: slice lineplot failed: {_e}")

        # ridge: best power + optimal load vs other axis
        d = df_ok[[other_axis, load_axis, P] + ([V] if V else []) + ([I] if I else [])].copy()
        d[P] = d[P].astype(float)
        d[load_axis] = d[load_axis].astype(float)

        # idx of best power per other-axis value
        idx = d.groupby(other_axis)[P].idxmax()
        ridge = d.loc[idx].sort_values(other_axis)

        # P_best vs other axis
        import textwrap as _tw

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(ridge[other_axis].astype(float).values, ridge[P].astype(float).values)
        ax.set_xlabel(_pretty(other_axis), fontsize=11)
        ax.set_ylabel(_pretty(P), fontsize=11)
        ax.set_title("\n".join(_tw.wrap(f"{prefix} — best {_pretty(P)} vs {_pretty(other_axis)} (max over R_load)", width=70)), fontsize=11, pad=10)
        ax.grid(True)
        fig.tight_layout()

        plt.show()

        # R_opt vs other axis
        import textwrap as _tw

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(ridge[other_axis].astype(float).values, ridge[load_axis].astype(float).values)
        ax.set_xlabel(_pretty(other_axis), fontsize=11)
        ax.set_ylabel(_pretty(load_axis), fontsize=11)
        ax.set_title("\n".join(_tw.wrap(f"{prefix} — optimal {_pretty(load_axis)} vs {_pretty(other_axis)} (argmax power)", width=70)), fontsize=11, pad=10)
        ax.grid(True)
        fig.tight_layout()

        plt.show()

        # optional V/I at optimum if present
        if V and V in ridge.columns:
            import textwrap as _tw

            fig, ax = plt.subplots(figsize=(10, 6))
            ax.plot(ridge[other_axis].astype(float).values, ridge[V].astype(float).values)
            ax.set_xlabel(_pretty(other_axis), fontsize=11)
            ax.set_ylabel(_pretty(V), fontsize=11)
            ax.set_title("\n".join(_tw.wrap(f"{prefix} — {_pretty(V)} at optimal load vs {_pretty(other_axis)}", width=70)), fontsize=11, pad=10)
            ax.grid(True)
            fig.tight_layout()

            plt.show()

        if I and I in ridge.columns:
            import textwrap as _tw

            fig, ax = plt.subplots(figsize=(10, 6))
            ax.plot(ridge[other_axis].astype(float).values, ridge[I].astype(float).values)
            ax.set_xlabel(_pretty(other_axis), fontsize=11)
            ax.set_ylabel(_pretty(I), fontsize=11)
            ax.set_title("\n".join(_tw.wrap(f"{prefix} — {_pretty(I)} at optimal load vs {_pretty(other_axis)}", width=70)), fontsize=11, pad=10)
            ax.grid(True)
            fig.tight_layout()

            plt.show()
            
                    
        # --- Experiment orchestration ---
@dataclass
class ExperimentSpec:
    experiment_id: str
    description: str
    base_overrides: Dict[str, Any]
    which: Literal["0d", "1d", "1d_series"] = "1d"
    mode: Optional[Literal["fixed_load", "max_power", "mppt"]] = None
    sweep: Optional[Union[Sweep1D, Sweep2D, Sweep3D]] = None
    outputs_to_collect: Tuple[str, ...] = ("P_TOTAL_W", "V_TERM_V", "I_TOTAL_A")
    baseline_id: Optional[str] = None
    # Display metadata — used by plotters and post-analysis hooks for titles/colours.
    # plot_label    : human-readable name shown in figure titles and PDF headers.
    # plot_color    : hex colour for primary series in figures.
    # plot_seawater_mol_l : ambient seawater concentration (mol/L) for harm-reduction
    #                 calculations; defaults to 0.600 (35 ppt, coastal CA).
    plot_label: Optional[str] = None
    plot_color: Optional[str] = None
    plot_seawater_mol_l: float = 0.600
    # Optional post-analysis hook — called after sweep_table.parquet is written.
    # Signature: post_analysis(run_dir: Path, df: pd.DataFrame, spec: ExperimentSpec) -> None
    # Use to append derived parquets and figures into run_dir.
    # Any *.png written into run_dir is automatically picked up by generate_experiment_pdf.
    post_analysis: Optional[Any] = None  # Callable[[Path, pd.DataFrame, ExperimentSpec], None]

    # ── Monte Carlo Techno-Economic Analysis (optional, default off) ──────────
    # When run_monte_carlo=True, run_experiment calls run_mc_tea() after post_analysis.
    # mc_config keys (all optional, see _MC_DEFAULTS in MC TEA cell):
    #   n_samples, seed, deployment_N, deployment_A,
    #   elec_price_mean, elec_price_std_frac,
    #   fouling_beta_lo, fouling_beta_hi
    # Additional axes can be added to _MC_AXIS_MAP in the MC TEA cell.
    run_monte_carlo: bool = False
    mc_config: Optional[Dict[str, Any]] = None



# ─────────────────────────────────────────────────────────────────────────────
# PDF report generator — produces a self-contained experiment report
# Uses: matplotlib (figures as PNG bytes) + reportlab (PDF layout)
# ─────────────────────────────────────────────────────────────────────────────

def generate_experiment_pdf(run_dir: Path, *, extra_png_dirs: list = None) -> Path:
    """
    Generate a PDF report for an experiment run.

    Layout:
      Page 1   : Cover — experiment ID, datetime, description, all parameters
      Page 2+  : Figures, one per page, with a uniform label FIG-<experiment_id>-<NNN>
                 and caption derived from the figure title.

    The PDF is saved as:
        <run_dir>/<run_tag>_<experiment_id>.pdf
    and the path is returned.
    """
    from reportlab.lib.pagesizes import letter
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, PageBreak, Image as RLImage, Table, TableStyle
    )
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.lib import colors
    import io
    import textwrap

    # ── Load bundle ──────────────────────────────────────────────────────────
    bundle_path = run_dir / "run_bundle.json"
    with open(bundle_path, "r", encoding="utf-8") as f:
        bundle = json.load(f)

    exp_id      = bundle.get("experiment_id", "UNKNOWN")
    run_tag     = bundle.get("run_tag", run_dir.name)
    run_dt      = bundle.get("run_datetime_iso", "unknown")
    description = bundle.get("description", "")
    base_over   = bundle.get("base_overrides", {})
    base_full   = bundle.get("base_knobs_full", {})
    spec_dict   = bundle.get("spec", {})

    # ── Capture all matplotlib figures currently open ────────────────────────
    # We re-run plot_run_from_disk into a non-interactive backend to collect figures.
    import matplotlib
    prev_backend = matplotlib.get_backend()
    matplotlib.use("Agg")
    import matplotlib.pyplot as _plt2

    _plt2.close("all")
    _captured_figs: list = []

    # Monkey-patch plt.show() to capture figures instead of displaying
    import matplotlib.pyplot as _plt_orig
    _orig_show = _plt_orig.show

    def _capture_show():
        # Capture only the current (most recently created) figure, then close it.
        # Using get_fignums() and appending all open figures on every show() call
        # produces O(n^2) duplicates — figure k gets captured n-k+1 times.
        fig_num = _plt2.gcf().number
        _captured_figs.append(_plt2.figure(fig_num))
        _plt2.close(fig_num)  # close immediately so it never gets re-captured

    _plt_orig.show = _capture_show

    try:
        plot_run_from_disk(run_dir)
    finally:
        _plt_orig.show = _orig_show
        matplotlib.use(prev_backend)

    # Convert figures to PNG bytes
    fig_images: list = []  # list of (png_bytes, title_str)
    for fig in _captured_figs:
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
        buf.seek(0)
        suptitle = fig._suptitle.get_text() if fig._suptitle else ""
        # Prefer axes title as caption source
        axes_titles = [ax.get_title() for ax in fig.get_axes() if ax.get_title()]
        caption = suptitle or (axes_titles[0] if axes_titles else "")
        fig_images.append((buf.read(), caption))
        _plt2.close(fig)

    # ── Build PDF ─────────────────────────────────────────────────────────────
    pdf_name = f"{run_tag}_{exp_id}.pdf"
    pdf_path = run_dir / pdf_name

    doc = SimpleDocTemplate(
        str(pdf_path),
        pagesize=letter,
        leftMargin=0.75 * inch,
        rightMargin=0.75 * inch,
        topMargin=0.75 * inch,
        bottomMargin=0.75 * inch,
    )
    W, H = letter
    usable_w = W - 1.5 * inch

    styles = getSampleStyleSheet()
    s_title   = ParagraphStyle("rpt_title",   parent=styles["Title"],   fontSize=18, spaceAfter=6)
    s_h1      = ParagraphStyle("rpt_h1",      parent=styles["Heading1"], fontSize=13, spaceAfter=4, spaceBefore=10)
    s_h2      = ParagraphStyle("rpt_h2",      parent=styles["Heading2"], fontSize=11, spaceAfter=3, spaceBefore=6)
    s_body    = ParagraphStyle("rpt_body",    parent=styles["Normal"],   fontSize=9,  spaceAfter=3, leading=13)
    s_mono    = ParagraphStyle("rpt_mono",    parent=styles["Code"],     fontSize=8,  spaceAfter=2, leading=11,
                                fontName="Courier", textColor=colors.HexColor("#222222"))
    s_caption = ParagraphStyle("rpt_caption", parent=styles["Normal"],   fontSize=8,  textColor=colors.HexColor("#444444"),
                                spaceAfter=6, italic=True)
    s_fig_lbl = ParagraphStyle("rpt_fig_lbl", parent=styles["Normal"],   fontSize=9, fontName="Helvetica-Bold",
                                spaceAfter=2, textColor=colors.HexColor("#003366"))

    story = []

    # ── Page 1: Cover ─────────────────────────────────────────────────────────
    story.append(Paragraph("OsmoFlux Experiment Report", s_title))
    story.append(Spacer(1, 0.1 * inch))

    # Header table: ID / datetime / tag
    header_data = [
        ["Experiment ID:",  exp_id],
        ["Run tag:",        run_tag],
        ["Run datetime:",   run_dt],
    ]
    tbl = Table(header_data, colWidths=[1.8*inch, usable_w - 1.8*inch])
    tbl.setStyle(TableStyle([
        ("FONTNAME",  (0,0),(-1,-1), "Helvetica"),
        ("FONTNAME",  (0,0),(0,-1),  "Helvetica-Bold"),
        ("FONTSIZE",  (0,0),(-1,-1), 9),
        ("BOTTOMPADDING", (0,0),(-1,-1), 3),
        ("TOPPADDING",    (0,0),(-1,-1), 3),
        ("ROWBACKGROUNDS", (0,0),(-1,-1), [colors.HexColor("#EEF3FA"), colors.white]),
        ("BOX",       (0,0),(-1,-1), 0.5, colors.HexColor("#AAAAAA")),
        ("INNERGRID",  (0,0),(-1,-1), 0.25, colors.HexColor("#CCCCCC")),
    ]))
    story.append(tbl)
    story.append(Spacer(1, 0.15 * inch))

    # Description
    if description:
        story.append(Paragraph("Description", s_h1))
        story.append(Paragraph(description.replace("\n", " "), s_body))
        story.append(Spacer(1, 0.1 * inch))

    # Base overrides
    story.append(Paragraph("Base Overrides", s_h1))
    for k, v in sorted(base_over.items()):
        story.append(Paragraph(f"<b>{k}</b>: {v}", s_body))
    story.append(Spacer(1, 0.1 * inch))

    # Sweep definition
    sweep_dict = spec_dict.get("sweep") or {}
    if sweep_dict:
        story.append(Paragraph("Sweep Definition", s_h1))
        sweep_type = "Sweep3D" if "var3" in sweep_dict else ("Sweep2D" if "var2" in sweep_dict else "Sweep1D")
        story.append(Paragraph(f"Type: <b>{sweep_type}</b>", s_body))
        for vk in ["var", "var1", "var2", "var3"]:
            if vk in sweep_dict:
                val_key = "values" if vk == "var" else f"{vk}_values"
                vals = sweep_dict.get(val_key, [])
                vals_str = ", ".join(f"{v:g}" for v in vals) if vals else "—"
                story.append(Paragraph(f"<b>{sweep_dict[vk]}</b>: [{vals_str}]", s_mono))
        story.append(Spacer(1, 0.1 * inch))

    # outputs_to_collect
    out_cols = spec_dict.get("outputs_to_collect", [])
    if out_cols:
        story.append(Paragraph("Outputs Collected", s_h1))
        story.append(Paragraph(", ".join(str(c) for c in out_cols), s_body))
        story.append(Spacer(1, 0.1 * inch))

    # Full base knobs table
    story.append(Paragraph("Full Parameter Set (base knobs)", s_h1))
    story.append(Paragraph("All DesignInputs fields at time of run (defaults + overrides applied):", s_body))
    param_rows = [["Parameter", "Value"]]
    skip_keys = {"INIT_V_TERM_V"}  # internal solver init — not user-facing
    for k, v in sorted(base_full.items()):
        if k in skip_keys:
            continue
        param_rows.append([k, str(v)])
    param_tbl = Table(param_rows, colWidths=[2.5*inch, usable_w - 2.5*inch])
    param_tbl.setStyle(TableStyle([
        ("FONTNAME",  (0,0),(-1,0),  "Helvetica-Bold"),
        ("FONTNAME",  (0,1),(-1,-1), "Courier"),
        ("FONTSIZE",  (0,0),(-1,-1), 8),
        ("BACKGROUND",(0,0),(-1,0),  colors.HexColor("#003366")),
        ("TEXTCOLOR", (0,0),(-1,0),  colors.white),
        ("ROWBACKGROUNDS", (0,1),(-1,-1), [colors.HexColor("#F5F8FF"), colors.white]),
        ("BOTTOMPADDING", (0,0),(-1,-1), 2),
        ("TOPPADDING",    (0,0),(-1,-1), 2),
        ("BOX",       (0,0),(-1,-1), 0.5, colors.HexColor("#AAAAAA")),
        ("INNERGRID",  (0,0),(-1,-1), 0.25, colors.HexColor("#CCCCCC")),
    ]))
    story.append(param_tbl)

    # ── Table of Contents ─────────────────────────────────────────────────────
    # Build a logical ToC that collapses repeated-series entries into one line.
    # e.g. six heatmaps "... | C_H = 0.5/1.0/.../3.0 mol/L" become one entry:
    #   "Geometry heatmap — <metric>  ×6 salinity levels (C_H = 0.5 … 3.0 mol/L)"
    #
    # Strategy:
    #   1. Strip the "[run_tag]" prefix that every caption shares.
    #   2. Detect the varying token (anything matching "C_H = <number>") and
    #      group consecutive captions that differ only in that token.
    #   3. Each group becomes one ToC row: description + "×N <dimension>" note.
    # ──────────────────────────────────────────────────────────────────────────
    import re as _re

    def _toc_entries(fig_images_in, exp_id_in):
        """
        Build a logical ToC by grouping figures that represent the same plot type
        repeated over a sweep dimension (e.g. one heatmap per salinity level).

        Groups figures by (section, template) where the template is the caption
        with the varying dimension token replaced by "…". Non-adjacent figures
        with the same template (e.g. metric A at C_H=0.5, 1.0, …) are merged
        into one row with their figure numbers listed compactly.

        Returns a list of dicts with keys:
          label_str    – compact fig-number list, e.g. "001, 005, 009, 013, 017, 021"
          description  – template description with "…" for the varying token
          dimension_note – "×6 salinity levels (C_H,in = 0.5, 1.0, … mol/L)"
          count        – number of instances
        """
        from collections import OrderedDict as _OD

        # Patterns for stripping prefix and detecting the varying dimension token
        _PREFIX = _re.compile(r'^[^\[]+\[[^\]]+\]\s*')   # "EXP [tag] "
        _DIM    = _re.compile(r'\|\s*C_H\s*=\s*([\d.]+)\s*mol/L')

        def _normalise(cap):
            c = cap.replace("\n", " ").strip()
            return _PREFIX.sub('', c).strip().lstrip('—').strip()

        def _template(norm):
            return _DIM.sub('| C_H = … mol/L', norm)

        def _section(norm):
            if _DIM.search(norm):
                return 'A'           # per-salinity heatmaps
            elif 'slices by' in norm:
                return 'C'           # slice line plots
            else:
                return 'B'           # envelope / argmax

        # Section labels for the description column header
        _SEC_LABELS = {
            'A': 'Geometry heatmap (per salinity)',
            'B': 'Envelope / argmax vs salinity',
            'C': 'Slice plot vs salinity',
        }

        groups = _OD()
        for fig_num, (_, raw_cap) in enumerate(fig_images_in, start=1):
            norm = _normalise(raw_cap)
            sec  = _section(norm)
            tmpl = _template(norm)
            key  = (sec, tmpl)
            if key not in groups:
                groups[key] = {
                    'section': sec,
                    'template': tmpl,
                    'fig_nums': [],
                    'dim_vals': [],
                }
            groups[key]['fig_nums'].append(fig_num)
            # Extract C_H value if present
            m = _DIM.search(norm)
            if m:
                groups[key]['dim_vals'].append(m.group(1))

        entries = []
        for (sec, tmpl), g in groups.items():
            figs   = g['fig_nums']
            count  = len(figs)
            prefix = f"FIG-{exp_id_in}-"

            # Compact label: if all consecutive → "001–006", else "001, 005, 009, …"
            # Use short numeric labels only — experiment ID shown in report header
            if count == 1:
                lbl = f"{figs[0]:03d}"
            elif figs == list(range(figs[0], figs[0] + count)):
                lbl = f"{figs[0]:03d} – {figs[-1]:03d}"
            else:
                if len(figs) > 4:
                    lbl = f"{figs[0]:03d}, {figs[1]:03d}, … {figs[-1]:03d}"
                else:
                    lbl = ", ".join(f"{n:03d}" for n in figs)

            # Description: clean up the template a little
            desc = tmpl

            # Dimension note
            if g['dim_vals']:
                vals_str = ', '.join(g['dim_vals'])
                dim_note = f"x{count} salinity levels (C_H,in = {vals_str} mol/L)"
            elif count > 1:
                dim_note = f"x{count} instances"
            else:
                dim_note = ""

            entries.append({
                'section':        sec,
                'section_label':  _SEC_LABELS.get(sec, sec),
                'label_str':      lbl,
                'description':    desc,
                'dimension_note': dim_note,
                'count':          count,
            })

        return entries

    toc_entries = _toc_entries(fig_images, exp_id)

    # Render ToC page
    story.append(PageBreak())
    story.append(Paragraph("Table of Contents — Figures", s_h1))
    story.append(Spacer(1, 0.08 * inch))
    story.append(Paragraph(
        f"This report contains <b>{len(fig_images)}</b> figures across "
        f"<b>{len(toc_entries)}</b> logical plot types. "
        "Repeated series (e.g. one heatmap per salinity level) are collapsed into one row.",
        s_body,
    ))
    story.append(Spacer(1, 0.12 * inch))

    s_toc_sec  = ParagraphStyle("toc_sec",  parent=styles["Normal"],
                                 fontSize=8, fontName="Helvetica-Bold",
                                 textColor=colors.white, leading=11)

    # Build table rows, inserting a section-header row when the section changes
    # Paragraph styles for table cells — enables word-wrap inside ReportLab Table
    s_cell_hdr  = ParagraphStyle("cell_hdr",  parent=styles["Normal"], fontSize=8,
                                  fontName="Helvetica-Bold", textColor=colors.white, leading=11)
    s_cell_sec  = ParagraphStyle("cell_sec",  parent=styles["Normal"], fontSize=8,
                                  fontName="Helvetica-Bold", textColor=colors.white, leading=11)
    s_cell_lbl  = ParagraphStyle("cell_lbl",  parent=styles["Normal"], fontSize=7.5,
                                  fontName="Courier", leading=11)
    s_cell_desc = ParagraphStyle("cell_desc", parent=styles["Normal"], fontSize=7.5,
                                  fontName="Helvetica", leading=11)
    s_cell_dim  = ParagraphStyle("cell_dim",  parent=styles["Normal"], fontSize=7.5,
                                  fontName="Helvetica-Oblique",
                                  textColor=colors.HexColor("#444444"), leading=11)

    toc_rows   = [[Paragraph("Fig #", s_cell_hdr),
                   Paragraph("Plot description", s_cell_hdr),
                   Paragraph("Sweep instances", s_cell_hdr)]]
    row_styles = []   # (row_index, background_color) for section headers
    current_sec = None
    for e in toc_entries:
        # Section divider row
        if e['section'] != current_sec:
            current_sec = e['section']
            sec_row_idx = len(toc_rows)
            toc_rows.append([Paragraph(e['section_label'], s_cell_sec), '', ''])
            row_styles.append(('SECTION', sec_row_idx))

        toc_rows.append([
            Paragraph(e['label_str'],      s_cell_lbl),
            Paragraph(e['description'],    s_cell_desc),
            Paragraph(e['dimension_note'], s_cell_dim),
        ])

    col_w = [0.55*inch, 4.3*inch, 2.05*inch]
    toc_tbl = Table(toc_rows, colWidths=col_w, repeatRows=1)

    # Base style — font/size handled by Paragraph styles; only layout/color here
    base_style = [
        ("BACKGROUND",    (0,0),(-1,0),  colors.HexColor("#003366")),
        ("ROWBACKGROUNDS",(0,1),(-1,-1), [colors.HexColor("#F5F8FF"), colors.white]),
        ("VALIGN",        (0,0),(-1,-1), "TOP"),
        ("BOTTOMPADDING", (0,0),(-1,-1), 4),
        ("TOPPADDING",    (0,0),(-1,-1), 4),
        ("LEFTPADDING",   (0,0),(-1,-1), 4),
        ("RIGHTPADDING",  (0,0),(-1,-1), 4),
        ("BOX",           (0,0),(-1,-1), 0.5, colors.HexColor("#AAAAAA")),
        ("INNERGRID",     (0,0),(-1,-1), 0.25, colors.HexColor("#CCCCCC")),
    ]
    # Section header rows: dark teal, bold, span all columns
    for kind, ridx in row_styles:
        base_style += [
            ("BACKGROUND",  (0,ridx),(-1,ridx), colors.HexColor("#1A6B72")),
            ("TEXTCOLOR",   (0,ridx),(-1,ridx), colors.white),
            ("FONTNAME",    (0,ridx),(-1,ridx), "Helvetica-Bold"),
            ("FONTSIZE",    (0,ridx),(-1,ridx), 8),
            ("SPAN",        (0,ridx),(-1,ridx)),
        ]

    toc_tbl.setStyle(TableStyle(base_style))
    story.append(toc_tbl)

    # ── Figures ───────────────────────────────────────────────────────────────
    for fig_num, (png_bytes, raw_caption) in enumerate(fig_images, start=1):
        story.append(PageBreak())
        fig_label = f"FIG-{exp_id}-{fig_num:03d}"

        # Figure label (bold, blue)
        story.append(Paragraph(fig_label, s_fig_lbl))

        # Caption (wrapped, smaller)
        caption_clean = raw_caption.replace("\n", " ").strip()
        if caption_clean:
            story.append(Paragraph(caption_clean, s_caption))

        # Image — scale to fit usable width
        from reportlab.lib.utils import ImageReader
        buf = io.BytesIO(png_bytes)
        ir = ImageReader(buf)
        img_w, img_h = ir.getSize()
        scale = min(usable_w / img_w, (H - 2.5 * inch) / img_h)
        rli = RLImage(io.BytesIO(png_bytes), width=img_w * scale, height=img_h * scale)
        story.append(rli)

    # ── Helper: render a list of PNG paths into the story ───────────────────────
    def _add_png_section(section_title, png_paths, label_prefix="SUPP"):
        if not png_paths:
            return
        # Suppress PIL decompression bomb check — our figures can be large
        from PIL import Image as _PILImage
        _PILImage.MAX_IMAGE_PIXELS = None
        story.append(PageBreak())
        story.append(Paragraph(section_title, s_h1))
        story.append(Spacer(1, 0.08 * inch))
        for png_path in png_paths:
            from reportlab.lib.utils import ImageReader as _IR2
            with open(png_path, "rb") as _pf:
                _bytes = _pf.read()
            _ir2 = _IR2(io.BytesIO(_bytes))
            _iw, _ih = _ir2.getSize()
            _sc = min(usable_w / _iw, (H - 2.5 * inch) / _ih)
            story.append(PageBreak())
            story.append(Paragraph(f"{label_prefix}: {png_path.name}", s_fig_lbl))
            story.append(RLImage(io.BytesIO(_bytes), width=_iw * _sc, height=_ih * _sc))

    # ── Monte Carlo TEA figures (mc_fig_*.png in run_dir) ────────────────────
    mc_pngs = sorted(run_dir.glob("mc_fig_*.png"))
    _add_png_section("Monte Carlo Techno-Economic Analysis", mc_pngs, label_prefix="MC-FIG")

    # ── Supplementary figures — other *.png from post_analysis hooks ─────────
    supp_pngs = [p for p in sorted(run_dir.glob("*.png"))
                 if not p.name.startswith("mc_fig_")]
    _add_png_section("Supplementary Figures (post-analysis)", supp_pngs, label_prefix="SUPP")

    # ── Extra PNG directories (e.g. linked parent/child experiment run_dirs) ──
    for extra_dir in (extra_png_dirs or []):
        extra_dir = Path(extra_dir)
        extra_pngs = sorted(extra_dir.glob("*.png"))
        dir_label = extra_dir.parent.name  # experiment_id
        _add_png_section(f"Linked Experiment: {dir_label}", extra_pngs, label_prefix=dir_label)

    # ── Build ─────────────────────────────────────────────────────────────────
    doc.build(story)
    print(f"[PDF] Report saved: {pdf_path}")
    return pdf_path


def run_experiment(
    spec: ExperimentSpec,
    *,
    make_plots: bool = True,
    save: bool = True,
    runs_root: str = "runs",
    save_points: bool = False,          # legacy/debug only (default off)
    cleanup_points: bool = False,       # only relevant if save_points=True
) -> List[Dict[str, Any]]:
    """Run an ExperimentSpec. Returns a list of per-point bundles (in-memory only)."""
    base_knobs = make_base_knobs(spec.base_overrides)
    validate_knobs(base_knobs)

    run_knobs = apply_sweep(base_knobs, spec.sweep)

    bundles: List[Dict[str, Any]] = []
    rows: List[Dict[str, Any]] = []
    run_dir = Path(runs_root) / spec.experiment_id / _now_tag()

    # Identify sweep columns (legacy-compatible)
    sweep_cols: List[str] = []
    if isinstance(spec.sweep, Sweep1D):
        sweep_cols = [spec.sweep.var]
    elif isinstance(spec.sweep, Sweep2D):
        sweep_cols = [spec.sweep.var1, spec.sweep.var2]
    elif isinstance(spec.sweep, Sweep3D):
        sweep_cols = [spec.sweep.var1, spec.sweep.var2, spec.sweep.var3]
    elif isinstance(spec.sweep, SweepMorris):
        sweep_cols = spec.sweep.var_names + ["_morris_traj", "_morris_step", "_morris_changed_param"]

    for i, k in enumerate(run_knobs):
        validate_knobs(k)
        p = knobs_to_design_inputs(k)

        # NOTE: do not touch physics/model code
        res = run_one(p, which=spec.which, mode=spec.mode)

        bundle = make_run_bundle(
            experiment_id=spec.experiment_id,
            knobs=k,
            res=res,
            which=spec.which,
            mode=(spec.mode or k.get("MODE", "fixed_load")),
        )
        bundles.append(bundle)

        # Build a plot-ready, legacy-compatible row for the sweep table
        row: Dict[str, Any] = {"which": spec.which, "mode": spec.mode or p.MODE}
        for c in sweep_cols:
            val = k.get(c, np.nan)
            try:
                row[c] = float(val)
            except (ValueError, TypeError):
                row[c] = val  # Morris metadata (string columns)

        for outk in spec.outputs_to_collect:
            row[outk] = res.get(outk, np.nan)

        # Legacy 0d normalization (kept consistent with run_regime_map)
        if spec.which == "0d":
            row.setdefault("I_TOTAL_A", res.get("I_A", np.nan))
            row.setdefault("P_TOTAL_W", res.get("P_W", np.nan))
            row.setdefault("V_TERM_V", res.get("V_TERM_V", np.nan))

        row = _regime_append_row_legacy_compatible(p, res, row, spec.which)
        rows.append(row)

        # Optional per-point bundles (legacy/debug)
        if save and save_points:
            pt_dir = run_dir / f"pt_{i:03d}"
            save_run_bundle(bundle, pt_dir)

    # Save artifacts (parquet table + json envelope)
    agg_path = None
    if save:
        df = pd.DataFrame(rows)
        sweep_fname = save_sweep_table_parquet(df, run_dir, fname="sweep_table.parquet")

        # Lightweight point summaries for quick inspection (keep small)
        points_summary = df[sweep_cols + list(spec.outputs_to_collect)].to_dict(orient="records") if len(sweep_cols) else df[list(spec.outputs_to_collect)].to_dict(orient="records")

        agg_path = save_sweep_bundle(
            spec.experiment_id,
            run_dir,
            spec,
            artifacts={"sweep_table_parquet": sweep_fname},
            points_summary=points_summary,
        )

    # Option (b): remove redundant per-point folders after agg exists
    if save and save_points and cleanup_points and agg_path is not None and agg_path.exists():
        for d in sorted(run_dir.glob("pt_*")):
            if d.is_dir():
                shutil.rmtree(d, ignore_errors=True)

    # Post-analysis hook — runs after parquet is written, before plotting/PDF
    if save and spec.post_analysis is not None:
        try:
            _pa_df = pd.read_parquet(run_dir / "sweep_table.parquet")
            spec.post_analysis(run_dir, _pa_df, spec)
        except Exception as _pa_err:
            print(f"[post_analysis] Warning: hook failed (non-fatal): {_pa_err}")
            import traceback; traceback.print_exc()

    # Monte Carlo TEA hook — runs after post_analysis so deployment point parquets exist
    if save and spec.run_monte_carlo:
        try:
            run_mc_tea(run_dir, spec)
        except Exception as _mc_err:
            print(f"[mc_tea] Warning: Monte Carlo TEA failed (non-fatal): {_mc_err}")
            import traceback; traceback.print_exc()

    # Disk-first plotting (reads saved artifacts only)
    # NOTE: plot_run_from_disk is called INSIDE generate_experiment_pdf with
    # plt.show() monkey-patched to capture figures. We do NOT call it here
    # directly — that was causing 1D sweeps to render inline in the notebook
    # (plt.show() fires live before the PDF generator intercepts it), while
    # 2D/3D heatmap paths happened to avoid this because their figures were
    # created inside the PDF generator's capture context. One code path for all.
    #
    # MC-only experiments (sweep=None, run_monte_carlo=True): run_mc_tea() calls
    # generate_experiment_pdf directly with extra_png_dirs linking the parent brine
    # run. Skipping the call here avoids a redundant second PDF and a matplotlib
    # backend conflict from switching Agg→prev→Agg in quick succession.
    _mc_only = spec.run_monte_carlo and (spec.sweep is None)
    if make_plots and save and not _mc_only:
        try:
            generate_experiment_pdf(run_dir)
        except Exception as _pdf_err:
            print(f"[PDF] Warning: PDF generation failed (non-fatal): {_pdf_err}")

    print(
        f"[run_experiment] {spec.experiment_id}: {len(bundles)} run(s). "
        f"Saved under: {run_dir if save else '(not saved)'}"
    )
    return bundles


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Brine disposal post-analysis hook
#
# make_brine_post_analysis() returns a callable assigned to ExperimentSpec.post_analysis.
# run_experiment calls it automatically after sweep_table.parquet is written:
#     spec.post_analysis(run_dir, sweep_df, spec)
#
# Plant context is read directly from the ExperimentSpec:
#   spec.base_overrides["C_H_IN_MOL_L"]  — brine inlet concentration
#   spec.plot_label                       — figure title / PDF header
#   spec.plot_color                       — primary series colour
#   spec.plot_seawater_mol_l              — ambient seawater (default 0.600 mol/L)
#
# The hook writes into run_dir alongside sweep_table.parquet:
#   harm_sweep.parquet    — sweep enriched with c_h_out, c_l_out, harm_redux
#   cascade.parquet       — cascade (fresh L/stage) results
#   single_equiv.parquet  — single extended-stack equivalent
#   optimal.parquet       — threshold-constrained optimal N per env. threshold
#   brine_fig_*.png       — 6 figures (auto-included in experiment PDF)
#
# To add a new plant: add an ExperimentSpec to the registry with the
# appropriate base_overrides, plot_label, plot_color, and plot_seawater_mol_l.
# ─────────────────────────────────────────────────────────────────────────────
from pathlib import Path as _Path
import pandas as _pd
import numpy as _np
import warnings as _warnings

_BD_SW = 0.600  # default ambient seawater (mol/L)

def _bh_run_point(c_h_in, N, A, c_l_in=0.02, n_seg=20):
    p = DesignInputs(
        C_H_IN_MOL_L=float(c_h_in), C_L_IN_MOL_L=float(c_l_in),
        N_CELL_PAIRS=int(N), A_MEM_M2=float(A),
        MODE='max_power', N_SEG=int(n_seg), GAP_M=0.001,
        DIVALENT_FRACTION=0.0, FOULING_BETA=0.0,
    )
    r = model_1d_axial_series_current(p, mode='max_power')
    seg = r['per_segment']
    c_h_out = float(seg['C_H_OUT_MOL_L'][-1])
    c_l_out = float(seg['C_L_OUT_MOL_L'][-1])
    p_total = float(r['P_TOTAL_W'])
    return dict(c_h_out=c_h_out, c_l_out=c_l_out,
                p_total=p_total, p_dens=p_total / (int(N) * float(A)),
                inv=bool(r['any_inversion']), converged=bool(r['converged']))

def _bh_harm_redux(c_h_in, c_h_out, c_l_out, sw=_BD_SW):
    baseline = max(c_h_in - sw, 0.0)
    if baseline == 0.0:
        return 0.0
    worst = max(max(c_h_out - sw, 0.0), max(c_l_out - sw, 0.0))
    return (1.0 - worst / baseline) * 100.0

def _bh_harm_redux_thresh(c_h_in, thresh, sw=_BD_SW):
    baseline = max(c_h_in - sw, 0.0)
    if baseline == 0.0:
        return 100.0
    return (1.0 - max(thresh - sw, 0.0) / baseline) * 100.0

def _bh_cascade(c_h_in, n_stages, N_per=40, A=0.01, c_l_fresh=0.02, sw=_BD_SW):
    c_h, total_p, worst_l, any_inv = c_h_in, 0.0, c_l_fresh, False
    for _ in range(n_stages):
        if c_h <= c_l_fresh + 0.01:
            break
        r = _bh_run_point(c_h, N_per, A, c_l_in=c_l_fresh)
        c_h = r['c_h_out']; total_p += r['p_total']
        worst_l = max(worst_l, r['c_l_out']); any_inv = any_inv or r['inv']
    return dict(c_h_out=c_h, worst_l_out=worst_l, p_total=total_p, any_inv=any_inv)


def make_brine_post_analysis(*, n_cascade_stages=8, cascade_N_per_stage=40,
                              cascade_A=0.01, cascade_c_l_fresh=0.02,
                              thresholds=None):
    """
    Factory: returns a post_analysis callable for brine disposal ExperimentSpecs.
    Assign to ExperimentSpec.post_analysis.  The callable reads all plant context
    (c_h_in, label, colour, seawater) directly from the ExperimentSpec it is
    called with — no separate plant config dict needed.

    Example:
        ExperimentSpec(
            ...,
            base_overrides={"C_H_IN_MOL_L": 1.15, ...},
            plot_label="Carlsbad (Poseidon SWRO, 1.15 mol/L)",
            plot_color="#D95319",
            plot_seawater_mol_l=0.600,
            post_analysis=make_brine_post_analysis(),
        )
    """
    import matplotlib
    from matplotlib.lines import Line2D

    _thresholds = thresholds or [
        ('35 ppt (strict)',        0.600),
        ('37 ppt (CA Ocean Plan)', 0.634),
        ('40 ppt (lenient)',       0.685),
    ]
    _thresh_colors = ['#D95319', '#EDB120', '#77AC30']

    def _post(run_dir, sweep_df, spec):
        # Read plant context from spec — no external plant_config dict required
        c_h_in = float(spec.base_overrides.get('C_H_IN_MOL_L', 1.0))
        sw     = float(spec.plot_seawater_mol_l)
        label  = str(spec.plot_label or f'C_H={c_h_in} mol/L')
        color  = str(spec.plot_color or '#0072BD')
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        _warnings.filterwarnings('ignore')
        run_dir = _Path(run_dir)

        N_col = 'N_CELL_PAIRS' if 'N_CELL_PAIRS' in sweep_df.columns else None
        A_col = 'A_MEM_M2'     if 'A_MEM_M2'     in sweep_df.columns else None
        if N_col is None or A_col is None:
            print(f'[brine_post] WARNING: N_CELL_PAIRS or A_MEM_M2 not in sweep — skipping.')
            return
        A_vals  = sorted(sweep_df[A_col].dropna().unique().tolist())
        A_fixed = min(A_vals, key=lambda a: abs(a - 0.01))

        # ── 1. Harm sweep ───────────────────────────────────────────────────
        print('[brine_post] Computing harm metric ...')
        harm_rows = []
        for _, row in sweep_df.iterrows():
            N = int(row[N_col]); A = float(row[A_col])
            r = _bh_run_point(c_h_in, N, A)
            base = {c: row[c] for c in sweep_df.columns}
            base.update(c_h_out=r['c_h_out'], c_l_out=r['c_l_out'],
                        p_dens=r['p_dens'], inv=r['inv'],
                        harm_redux=_bh_harm_redux(c_h_in, r['c_h_out'], r['c_l_out'], sw))
            harm_rows.append(base)
        harm_df = _pd.DataFrame(harm_rows)
        harm_df.to_parquet(run_dir / 'harm_sweep.parquet', index=False)
        print(f'[brine_post]   harm_sweep.parquet ({len(harm_df)} rows)')

        # ── 2. Cascade ──────────────────────────────────────────────────────
        print('[brine_post] Cascade sweep ...')
        casc_rows, sing_rows = [], []
        for s in range(1, n_cascade_stages + 1):
            cr = _bh_cascade(c_h_in, s, N_per=cascade_N_per_stage,
                             A=cascade_A, c_l_fresh=cascade_c_l_fresh, sw=sw)
            sr = _bh_run_point(c_h_in, cascade_N_per_stage * s, cascade_A, c_l_in=cascade_c_l_fresh)
            casc_rows.append(dict(stages=s, c_h_out=cr['c_h_out'],
                                  worst_l_out=cr['worst_l_out'],
                                  p_total_mW=cr['p_total'] * 1000,
                                  harm_redux=_bh_harm_redux(c_h_in, cr['c_h_out'], cr['worst_l_out'], sw),
                                  any_inv=cr['any_inv']))
            sing_rows.append(dict(stages=s, N_eq=cascade_N_per_stage * s,
                                  c_h_out=sr['c_h_out'], c_l_out=sr['c_l_out'],
                                  p_total_mW=sr['p_total'] * 1000,
                                  harm_redux=_bh_harm_redux(c_h_in, sr['c_h_out'], sr['c_l_out'], sw),
                                  inv=sr['inv']))
        casc_df = _pd.DataFrame(casc_rows)
        sing_df = _pd.DataFrame(sing_rows)
        casc_df.to_parquet(run_dir / 'cascade.parquet', index=False)
        sing_df.to_parquet(run_dir / 'single_equiv.parquet', index=False)
        print('[brine_post]   cascade.parquet, single_equiv.parquet')

        # ── 3. Optimal configs ──────────────────────────────────────────────
        sub_fa = harm_df[harm_df[A_col].round(6) == round(A_fixed, 6)].copy()
        opt_rows = []
        for tlabel, thresh in _thresholds:
            valid = sub_fa[(sub_fa['c_l_out'] <= thresh) & (~sub_fa['inv'])]
            flag  = 'ok' if not valid.empty else 'threshold_not_met'
            if valid.empty:
                valid = sub_fa[~sub_fa['inv']]
            if valid.empty:
                continue
            best = valid.loc[valid['harm_redux'].idxmax()]
            opt_rows.append(dict(threshold_label=tlabel, threshold_mol_L=thresh,
                                 N_opt=int(best[N_col]), harm_redux=float(best['harm_redux']),
                                 p_dens=float(best['p_dens']), c_l_out=float(best['c_l_out']),
                                 c_h_out=float(best['c_h_out']), flag=flag))
        opt_df = _pd.DataFrame(opt_rows)
        opt_df.to_parquet(run_dir / 'optimal.parquet', index=False)
        print('[brine_post]   optimal.parquet')
        if not opt_df.empty:
            print(opt_df[['threshold_label','N_opt','harm_redux','p_dens','c_l_out']].to_string(index=False))

        # ── 4. Figures — written to run_dir, picked up by PDF ───────────────
        print('[brine_post] Generating figures ...')

        def _save(fig, name):
            fig.savefig(run_dir / name, dpi=150, bbox_inches='tight')
            plt.close(fig)
            print(f'[brine_post]   {name}')

        pd_col = 'P_DENS_W_M2' if 'P_DENS_W_M2' in harm_df.columns else 'p_dens'
        sub1d  = harm_df[harm_df[A_col].round(6) == round(A_fixed, 6)].sort_values(N_col)
        Ns     = sub1d[N_col].values.astype(int)
        hr     = sub1d['harm_redux'].values
        pden   = sub1d[pd_col].values
        inv    = sub1d['inv'].values.astype(bool)

        # Fig A: Regime map
        fig, ax = plt.subplots(figsize=(10, 6))
        ax2 = ax.twinx()
        ax.plot(Ns, hr, '-o', color=color, lw=2.5, ms=7, zorder=5)
        ax2.plot(Ns, pden, '--s', color=color, lw=2, ms=6, alpha=0.55)
        if inv.any():
            ax.scatter(Ns[inv], hr[inv], marker='x', s=140, color='navy', lw=2.5, zorder=6)
        for (tl, thresh), tc in zip(_thresholds, _thresh_colors):
            redux = _bh_harm_redux_thresh(c_h_in, thresh, sw)
            ax.axhline(redux, color=tc, lw=1.8, ls='--', alpha=0.85)
            ax.text(max(Ns) * 0.88, redux + 1.5, tl.split('(')[0].strip(), fontsize=8, color=tc)
        ax.set_xlabel('N_CELL_PAIRS', fontsize=11)
        ax.set_ylabel('Brine harm reduction (%)', fontsize=11)
        ax2.set_ylabel('Power density (W/m\u00b2)', fontsize=10, color=color)
        ax.set_title(f'{label}\nRegime Map: Harm Reduction vs N_CELL_PAIRS  (A_MEM={A_fixed:.4g} m\u00b2/pair)',
                     fontsize=10, fontweight='bold')
        ax.set_ylim(-5, 110); ax.set_xscale('log')
        ax.set_xticks(Ns); ax.set_xticklabels([str(n) for n in Ns], fontsize=9)
        ax.grid(True, alpha=0.25)
        for lbl in ax2.get_yticklabels(): lbl.set_color(color)
        ax.legend(handles=[
            Line2D([0],[0], color=color, lw=2.5, marker='o', label='Harm reduction % (left)'),
            Line2D([0],[0], color=color, lw=2, marker='s', ls='--', alpha=0.6, label='Power density W/m\u00b2 (right)'),
            Line2D([0],[0], color='navy', lw=0, marker='x', ms=9, label='Conc. inversion'),
        ] + [Line2D([0],[0], color=tc, lw=1.8, ls='--', label=tl.split('(')[0].strip())
             for (tl,_), tc in zip(_thresholds, _thresh_colors)],
            fontsize=8, loc='upper left', framealpha=0.92)
        fig.tight_layout()
        _save(fig, 'brine_fig_A_regime_map.png')

        # Heatmap helper
        def _hmap(val_col, cmap, vmin, vmax, title, cbar_label, fname,
                  mark_opt=False, thresh_contours=False):
            n_list = sorted(harm_df[N_col].unique())
            a_list = sorted(harm_df[A_col].unique())
            fig, ax = plt.subplots(figsize=(max(8, len(a_list)*1.4), max(6, len(n_list)*0.9)))
            pivot  = harm_df.pivot_table(index=N_col, columns=A_col, values=val_col)
            harm_p = harm_df.pivot_table(index=N_col, columns=A_col, values='harm_redux')
            im = ax.imshow(pivot.values, aspect='auto', origin='lower',
                           vmin=vmin, vmax=vmax, cmap=cmap,
                           extent=[-0.5, len(a_list)-0.5, -0.5, len(n_list)-0.5])
            if thresh_contours:
                for (tl, thresh), tc in zip(_thresholds, _thresh_colors):
                    rt = _bh_harm_redux_thresh(c_h_in, thresh, sw)
                    try:
                        cs = ax.contour(harm_p.values, levels=[rt], colors=[tc], linewidths=2,
                                        extent=[-0.5, len(a_list)-0.5, -0.5, len(n_list)-0.5])
                        ax.clabel(cs, fmt=tl.split('(')[0].strip(), fontsize=7)
                    except Exception:
                        pass
            if mark_opt:
                oi = _np.unravel_index(_np.nanargmax(pivot.values), pivot.values.shape)
                ax.plot(oi[1], oi[0], '*', ms=15, color='white',
                        markeredgecolor='black', markeredgewidth=0.7, zorder=10)
            for i_n, N in enumerate(n_list):
                for i_a, A in enumerate(a_list):
                    row = harm_df[(harm_df[N_col] == N) & (harm_df[A_col].round(6) == round(A, 6))]
                    if row.empty:
                        continue
                    val    = row[val_col].values[0]
                    is_inv = row['inv'].values[0]
                    pfx    = 'X\n' if is_inv else ''
                    txt    = f'{pfx}{val:.0f}' if 'harm' in val_col else f'{pfx}{val:.2f}'
                    light  = val < (vmax or 100) * 0.35 or val > (vmax or 100) * 0.8
                    ax.text(i_a, i_n, txt, ha='center', va='center',
                            fontsize=6, color='white' if light else 'black', alpha=0.85)
            ax.set_xticks(range(len(a_list)))
            ax.set_xticklabels([str(a) for a in a_list], rotation=45, fontsize=8)
            ax.set_yticks(range(len(n_list)))
            ax.set_yticklabels([str(n) for n in n_list], fontsize=9)
            ax.set_xlabel('A_MEM (m\u00b2/pair)', fontsize=10)
            ax.set_ylabel('N_CELL_PAIRS', fontsize=10)
            ax.set_title(f'{label}\n{title}', fontsize=10, fontweight='bold')
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label(cbar_label, fontsize=9)
            fig.tight_layout()
            _save(fig, fname)

        _hmap('harm_redux', 'RdYlGn', 0, 100,
              'Harm Reduction (%)  N x A  (X=inversion, contours=thresholds)',
              'Harm reduction (%)', 'brine_fig_B_harm_heatmap.png', thresh_contours=True)
        _hmap(pd_col, 'plasma', 0, float(harm_df[pd_col].max()),
              'Power Density (W/m\u00b2)  N x A  (* = peak, X = inversion)',
              'Power density (W/m\u00b2)', 'brine_fig_C_power_heatmap.png', mark_opt=True)

        # Fig D: Cascade vs single-stack
        stgs   = casc_df['stages'].values
        c_harm = casc_df['harm_redux'].values; c_pow = casc_df['p_total_mW'].values
        s_harm = sing_df['harm_redux'].values; s_pow = sing_df['p_total_mW'].values
        s_inv  = sing_df['inv'].values.astype(bool)
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle(f'{label}  |  Cascade vs Single Extended Stack  (N={cascade_N_per_stage}/stage, A={cascade_A} m\u00b2)',
                     fontsize=10, fontweight='bold')
        ax = axes[0]
        ax.plot(stgs, c_harm, '-o', color='#D95319', lw=2.5, ms=8, label='Cascade (fresh L/stage)')
        ax.plot(stgs, s_harm, '--s', color='#0072BD', lw=2, ms=7,
                label=f'Single stack (N=stages x {cascade_N_per_stage})')
        for (tl, thresh), tc in zip(_thresholds, _thresh_colors):
            redux = _bh_harm_redux_thresh(c_h_in, thresh, sw)
            ax.axhline(redux, color=tc, lw=1.5, ls=':', alpha=0.85)
            ax.text(max(stgs)*1.01, redux+1, tl.split('(')[0].strip(), fontsize=8, color=tc)
        if s_inv.any():
            ax.scatter(stgs[s_inv], s_harm[s_inv], marker='x', s=140, color='navy', lw=2.5, zorder=6)
        ax.set_xlabel('Cascade stages', fontsize=11)
        ax.set_ylabel('Harm reduction (%)', fontsize=11)
        ax.set_title('Harm Reduction', fontsize=11)
        ax.legend(fontsize=9); ax.grid(True, alpha=0.25); ax.set_ylim(-5, 110)
        ax = axes[1]
        ax.plot(stgs, c_pow, '-o', color='#D95319', lw=2.5, ms=8, label='Cascade cumulative')
        ax.plot(stgs, s_pow, '--s', color='#0072BD', lw=2, ms=7, label='Single stack')
        if s_inv.any():
            ax.scatter(stgs[s_inv], s_pow[s_inv], marker='x', s=140, color='navy', lw=2.5, zorder=6)
        for sv, pv in zip(stgs, c_pow):
            ax.annotate(f'{pv:.1f}', (sv, pv), xytext=(4, 4), textcoords='offset points', fontsize=7.5)
        ax.set_xlabel('Cascade stages', fontsize=11)
        ax.set_ylabel('Total power (mW)', fontsize=11)
        ax.set_title('Power Recovery', fontsize=11)
        ax.legend(fontsize=9); ax.grid(True, alpha=0.25)
        fig.tight_layout()
        _save(fig, 'brine_fig_D_cascade.png')

        # Fig E: Optimal configs bar chart
        if not opt_df.empty:
            fig, ax = plt.subplots(figsize=(8, 6))
            xp = _np.arange(len(opt_df)); w = 0.38
            harms = opt_df['harm_redux'].values; pds = opt_df['p_dens'].values
            ax.bar(xp-w/2, harms, w, color=_thresh_colors[:len(opt_df)],
                   alpha=0.88, edgecolor='white', lw=1.2)
            ax2 = ax.twinx()
            ax2.bar(xp+w/2, pds, w, color=_thresh_colors[:len(opt_df)],
                    alpha=0.38, edgecolor='white', lw=1.2, hatch='//')
            for xi, ro in enumerate(opt_df.itertuples()):
                ax.text(xi-w/2, ro.harm_redux+1.5, f'N={ro.N_opt}',
                        ha='center', fontsize=9, fontweight='bold')
                ax2.text(xi+w/2, ro.p_dens+0.015, f'{ro.p_dens:.2f}',
                         ha='center', fontsize=8)
            ax.set_xticks(xp)
            ax.set_xticklabels(opt_df['threshold_label'].values, fontsize=9)
            ax.set_xlabel('Environmental threshold', fontsize=10)
            ax.set_ylabel('Harm reduction (%)', fontsize=10)
            ax2.set_ylabel('Power density (W/m\u00b2)', fontsize=10)
            ax.set_ylim(0, 118); ax.grid(axis='y', alpha=0.25)
            ax.set_title(f'{label}\nOptimal Config per Threshold  (A={A_fixed:.4g} m\u00b2/pair)',
                         fontsize=10, fontweight='bold')
            ax.legend(handles=[
                Line2D([0],[0], color='gray', lw=8, alpha=0.88, label='Harm reduction % (left)'),
                Line2D([0],[0], color='gray', lw=8, alpha=0.38, label='Power density W/m\u00b2 (right)'),
            ], fontsize=8, loc='upper left', framealpha=0.9)
            fig.tight_layout()
            _save(fig, 'brine_fig_E_optimal.png')

        # Fig F: Economics
        ELEC = 0.22; CAPEX = 50.0; AMORT = 20; HRS = 8000; Q = 1e-5
        brine_costs = [
            ('Diffuser ($0.04/m3)',   0.04, '#0072BD'),
            ('Evap ponds ($0.15/m3)', 0.15, '#77AC30'),
            ('Deep-well ($0.25/m3)',  0.25, '#A2142F'),
        ]
        sub_e = harm_df[harm_df[A_col].round(6) == round(A_fixed, 6)].sort_values(N_col)
        Ns_e  = sub_e[N_col].values.astype(int)
        inv_e = sub_e['inv'].values.astype(bool)
        pde   = sub_e[pd_col].values
        hre   = sub_e['harm_redux'].values
        net   = pde * HRS * ELEC / 1000 - CAPEX / AMORT
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(Ns_e[~inv_e], net[~inv_e], '-o', color=color, lw=2.5, ms=7,
                zorder=5, label='Net energy (rev-capex)')
        if inv_e.any():
            ax.scatter(Ns_e[inv_e], net[inv_e], marker='x', s=140, color='navy', lw=2.5, zorder=6)
        for Nv, v in zip(Ns_e[~inv_e], net[~inv_e]):
            ax.annotate(f'{v:.1f}', (Nv, v), xytext=(3, 5), textcoords='offset points', fontsize=8)
        ax2 = ax.twinx()
        bvol = Q * 3600 * HRS / (Ns_e * A_fixed)
        for blbl, bcost, bc in brine_costs:
            sav = (hre / 100) * bvol * bcost
            ax2.plot(Ns_e[~inv_e], sav[~inv_e], '--', color=bc, lw=1.8, label=blbl)
            if inv_e.any():
                ax2.scatter(Ns_e[inv_e], sav[inv_e], marker='x', s=80, color=bc, lw=1.5)
        ax.axhline(0, color='k', lw=0.8, ls=':')
        ax.set_xscale('log')
        ax.set_xticks(Ns_e); ax.set_xticklabels([str(n) for n in Ns_e], fontsize=8)
        ax.set_xlabel('N_CELL_PAIRS', fontsize=10)
        ax.set_ylabel('Net energy value ($/m\u00b2/yr)', fontsize=10)
        ax2.set_ylabel('Brine disposal savings ($/m\u00b2/yr)', fontsize=10)
        ax.grid(True, alpha=0.25)
        ax.set_title(
            f'{label}\nAnnual Economics per m\u00b2  (${ELEC}/kWh | ${CAPEX}/m\u00b2 | {AMORT}-yr amort.)',
            fontsize=10, fontweight='bold')
        ax.legend(handles=[
            Line2D([0],[0], color=color, lw=2.5, marker='o', label='Net energy rev-capex (left)'),
            Line2D([0],[0], color='navy', lw=0, marker='x', ms=8, label='Inversion'),
        ] + [Line2D([0],[0], color=bc, lw=1.8, ls='--', label=bl) for bl, _, bc in brine_costs],
            fontsize=8, loc='upper right', framealpha=0.9)
        fig.tight_layout()
        _save(fig, 'brine_fig_F_economics.png')
        print(f'[brine_post] Complete. All artifacts in {run_dir}')

    return _post


print('Brine hook loaded: make_brine_post_analysis.')
print('Usage: set plot_label, plot_color, plot_seawater_mol_l on ExperimentSpec;')
print('       assign post_analysis=make_brine_post_analysis().')


# ─────────────────────────────────────────────────────────────────────────────
# Fouling-aware post-analysis hook
#
# make_fouling_post_analysis() mirrors make_brine_post_analysis() in structure.
# Assign to ExperimentSpec.post_analysis on the BRINE-*-FOULING experiments.
#
# The hook expects a Sweep2D(DIVALENT_FRACTION × N_CELL_PAIRS) table and:
#   1. Loads the companion clean sweep parquet (from baseline_id run_dir) so
#      every fouled point can be compared to its clean counterpart.
#   2. Produces four publication-quality figures written to run_dir:
#      fouling_fig_A_penalty_heatmap.png  — FOUL_POWER_PENALTY_FRAC heatmap
#      fouling_fig_B_abs_loss_heatmap.png — FOUL_POWER_PENALTY_W heatmap
#      fouling_fig_C_clean_vs_fouled.png  — line plot at realistic div fraction
#      fouling_fig_D_economics_shift.png  — how fouling moves the break-even
#   3. Writes fouling_summary.parquet with per-(N, f_div) penalty stats.
#
# Plant context is read from the ExperimentSpec (same pattern as brine hook):
#   spec.base_overrides["C_H_IN_MOL_L"]  — brine salinity
#   spec.plot_label                       — figure titles
#   spec.plot_color                       — primary series colour
# ─────────────────────────────────────────────────────────────────────────────

_REALISTIC_DIV_FRAC = 0.20   # seawater divalent fraction (Rijnaarts 2017 basis)
_ELEC_PRICE         = 0.22   # $/kWh (CA commercial rate)
_CAPEX_PER_M2       = 50.0   # $/m² membrane (representative)
_AMORT_YR           = 20     # amortisation period


def make_fouling_post_analysis(*, realistic_div_frac=_REALISTIC_DIV_FRAC):
    """
    Factory: returns a post_analysis callable for BRINE-*-FOULING ExperimentSpecs.

    Expects sweep_df to contain columns from a Sweep2D over:
        var1 = DIVALENT_FRACTION
        var2 = N_CELL_PAIRS
    and outputs:
        FOUL_POWER_PENALTY_FRAC, FOUL_POWER_PENALTY_W, P_TOTAL_W, P_DENS_W_M2,
        R_CLEAN_TOTAL_OHM, R_FOUL_TOTAL_OHM

    Assign via:
        ExperimentSpec(..., post_analysis=make_fouling_post_analysis())
    """
    import matplotlib
    from matplotlib.lines import Line2D
    import warnings as _w

    def _post(run_dir, sweep_df, spec):
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        _w.filterwarnings('ignore')
        run_dir = _Path(run_dir)

        c_h_in = float(spec.base_overrides.get('C_H_IN_MOL_L', 1.0))
        label  = str(spec.plot_label or f'C_H={c_h_in} mol/L')
        color  = str(spec.plot_color or '#0072BD')

        # ── Validate sweep columns ──────────────────────────────────────────
        needed = {'DIVALENT_FRACTION', 'N_CELL_PAIRS',
                  'FOUL_POWER_PENALTY_FRAC', 'FOUL_POWER_PENALTY_W',
                  'P_TOTAL_W', 'P_DENS_W_M2'}
        missing = needed - set(sweep_df.columns)
        if missing:
            print(f'[fouling_post] WARNING: missing columns {missing} — skipping.')
            return

        df = sweep_df.copy()
        df['DIVALENT_FRACTION'] = df['DIVALENT_FRACTION'].astype(float).round(4)
        df['N_CELL_PAIRS']      = df['N_CELL_PAIRS'].astype(int)

        divs = sorted(df['DIVALENT_FRACTION'].unique())
        Ns   = sorted(df['N_CELL_PAIRS'].unique())

        # ── Try to load companion clean sweep (DIVALENT_FRACTION=0 rows) ───
        # The DIVALENT_FRACTION=0 rows inside the fouling sweep ARE the clean
        # baseline (FOULING_BETA active but f_div=0 ⟹ zero CDF penalty, only
        # exp_outlet biofouling applies). We separate them explicitly.
        clean_df = df[df['DIVALENT_FRACTION'] == 0.0].set_index('N_CELL_PAIRS')

        # ── Write summary parquet ───────────────────────────────────────────
        df.to_parquet(run_dir / 'fouling_summary.parquet', index=False)
        print(f'[fouling_post]   fouling_summary.parquet ({len(df)} rows)')

        def _save(fig, name):
            fig.savefig(run_dir / name, dpi=150, bbox_inches='tight')
            plt.close(fig)
            print(f'[fouling_post]   {name}')

        # ── Fig A: FOUL_POWER_PENALTY_FRAC heatmap (div × N) ───────────────
        fig, ax = plt.subplots(figsize=(max(8, len(divs)*1.6), max(5, len(Ns)*0.9)))
        pivot = df.pivot_table(index='N_CELL_PAIRS', columns='DIVALENT_FRACTION',
                               values='FOUL_POWER_PENALTY_FRAC')
        pivot_pct = pivot * 100.0
        vmax = float(_np.nanmax(pivot_pct.values))
        im = ax.imshow(pivot_pct.values, aspect='auto', origin='lower',
                       vmin=0, vmax=min(vmax, 35), cmap='YlOrRd',
                       extent=[-0.5, len(divs)-0.5, -0.5, len(Ns)-0.5])
        for i_n, N in enumerate(Ns):
            for i_d, d in enumerate(divs):
                row = df[(df['N_CELL_PAIRS']==N) & (df['DIVALENT_FRACTION']==d)]
                if row.empty: continue
                val = float(row['FOUL_POWER_PENALTY_FRAC'].values[0]) * 100
                ax.text(i_d, i_n, f'{val:.1f}%', ha='center', va='center',
                        fontsize=8, color='black' if val < 20 else 'white')
        ax.set_xticks(range(len(divs)))
        ax.set_xticklabels([f'{d:.0%}' for d in divs], fontsize=9)
        ax.set_yticks(range(len(Ns)))
        ax.set_yticklabels([str(N) for N in Ns], fontsize=9)
        ax.set_xlabel('Divalent ion fraction', fontsize=10)
        ax.set_ylabel('N_CELL_PAIRS', fontsize=10)
        ax.set_title(f'{label}\nFouling Power Penalty (% of clean power lost)',
                     fontsize=10, fontweight='bold')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label(
            'Power penalty (%)', fontsize=9)
        fig.tight_layout()
        _save(fig, 'fouling_fig_A_penalty_heatmap.png')

        # ── Fig B: FOUL_POWER_PENALTY_W heatmap (absolute watts lost) ──────
        fig, ax = plt.subplots(figsize=(max(8, len(divs)*1.6), max(5, len(Ns)*0.9)))
        pivot_w = df.pivot_table(index='N_CELL_PAIRS', columns='DIVALENT_FRACTION',
                                 values='FOUL_POWER_PENALTY_W') * 1000  # → mW
        vmax_w  = float(_np.nanmax(pivot_w.values))
        im2 = ax.imshow(pivot_w.values, aspect='auto', origin='lower',
                        vmin=0, vmax=vmax_w, cmap='plasma',
                        extent=[-0.5, len(divs)-0.5, -0.5, len(Ns)-0.5])
        for i_n, N in enumerate(Ns):
            for i_d, d in enumerate(divs):
                row = df[(df['N_CELL_PAIRS']==N) & (df['DIVALENT_FRACTION']==d)]
                if row.empty: continue
                val = float(row['FOUL_POWER_PENALTY_W'].values[0]) * 1000
                ax.text(i_d, i_n, f'{val:.1f}', ha='center', va='center',
                        fontsize=8, color='white' if val > vmax_w*0.5 else 'black')
        ax.set_xticks(range(len(divs)))
        ax.set_xticklabels([f'{d:.0%}' for d in divs], fontsize=9)
        ax.set_yticks(range(len(Ns)))
        ax.set_yticklabels([str(N) for N in Ns], fontsize=9)
        ax.set_xlabel('Divalent ion fraction', fontsize=10)
        ax.set_ylabel('N_CELL_PAIRS', fontsize=10)
        ax.set_title(f'{label}\nAbsolute Fouling Power Loss (mW)',
                     fontsize=10, fontweight='bold')
        plt.colorbar(im2, ax=ax, fraction=0.046, pad=0.04).set_label(
            'Power lost (mW)', fontsize=9)
        fig.tight_layout()
        _save(fig, 'fouling_fig_B_abs_loss_heatmap.png')

        # ── Fig C: Clean vs fouled line plot at realistic divalent fraction ─
        # Pick the div fraction closest to realistic_div_frac
        rdiv = min(divs, key=lambda d: abs(d - realistic_div_frac))
        sub_r = df[df['DIVALENT_FRACTION'] == rdiv].sort_values('N_CELL_PAIRS')
        sub_0 = df[df['DIVALENT_FRACTION'] == 0.0].sort_values('N_CELL_PAIRS')

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle(
            f'{label}  |  Clean vs Fouled  (f_div={rdiv:.0%}, combined CDF + biofouling)',
            fontsize=10, fontweight='bold')

        ax = axes[0]
        ax.plot(sub_0['N_CELL_PAIRS'], sub_0['P_DENS_W_M2'],
                '-o', color='#0072BD', lw=2.5, ms=7, label='Clean (f_div=0)')
        ax.plot(sub_r['N_CELL_PAIRS'], sub_r['P_DENS_W_M2'],
                '--s', color=color, lw=2.5, ms=7, label=f'Fouled (f_div={rdiv:.0%})')
        # Shade the gap
        Ns_shared = sorted(set(sub_0['N_CELL_PAIRS']) & set(sub_r['N_CELL_PAIRS']))
        pd0 = sub_0.set_index('N_CELL_PAIRS').reindex(Ns_shared)['P_DENS_W_M2'].values
        pdr = sub_r.set_index('N_CELL_PAIRS').reindex(Ns_shared)['P_DENS_W_M2'].values
        ax.fill_between(Ns_shared, pdr, pd0, alpha=0.15, color=color, label='Penalty band')
        ax.set_xscale('log')
        ax.set_xticks(Ns); ax.set_xticklabels([str(n) for n in Ns], fontsize=9)
        ax.set_xlabel('N_CELL_PAIRS', fontsize=10)
        ax.set_ylabel('Power density (W/m²)', fontsize=10)
        ax.set_title('Power Density', fontsize=11)
        ax.legend(fontsize=9); ax.grid(True, alpha=0.25)

        ax = axes[1]
        pct_loss = sub_r['FOUL_POWER_PENALTY_FRAC'].values * 100
        Ns_r     = sub_r['N_CELL_PAIRS'].values
        ax.bar(range(len(Ns_r)), pct_loss, color=color, alpha=0.8, edgecolor='white')
        ax.set_xticks(range(len(Ns_r)))
        ax.set_xticklabels([str(n) for n in Ns_r], fontsize=9)
        ax.set_xlabel('N_CELL_PAIRS', fontsize=10)
        ax.set_ylabel('Power penalty (%)', fontsize=10)
        ax.set_title(f'% Power Lost at f_div={rdiv:.0%}', fontsize=11)
        for i, v in enumerate(pct_loss):
            ax.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=9)
        ax.set_ylim(0, max(pct_loss) * 1.25)
        ax.grid(axis='y', alpha=0.25)
        fig.tight_layout()
        _save(fig, 'fouling_fig_C_clean_vs_fouled.png')

        # ── Fig D: Economics — how fouling shifts break-even ────────────────
        HRS = 8000   # annual operating hours
        sub_0s = sub_0.set_index('N_CELL_PAIRS')
        sub_rs = sub_r.set_index('N_CELL_PAIRS')
        Ns_ec  = sorted(set(sub_0s.index) & set(sub_rs.index))

        net_clean  = _np.array([sub_0s.loc[N, 'P_DENS_W_M2'] * HRS * _ELEC_PRICE / 1000
                                 - _CAPEX_PER_M2 / _AMORT_YR for N in Ns_ec])
        net_fouled = _np.array([sub_rs.loc[N, 'P_DENS_W_M2'] * HRS * _ELEC_PRICE / 1000
                                 - _CAPEX_PER_M2 / _AMORT_YR for N in Ns_ec])

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(Ns_ec, net_clean,  '-o', color='#0072BD', lw=2.5, ms=7, label='Clean')
        ax.plot(Ns_ec, net_fouled, '--s', color=color,    lw=2.5, ms=7,
                label=f'Fouled (f_div={rdiv:.0%})')
        ax.fill_between(Ns_ec, net_fouled, net_clean,
                        alpha=0.15, color=color, label='Revenue lost to fouling')
        ax.axhline(0, color='k', lw=0.9, ls=':')
        for i, (Nv, cv, fv) in enumerate(zip(Ns_ec, net_clean, net_fouled)):
            ax.annotate(f'{cv:.2f}', (Nv, cv), xytext=(4, 5),
                        textcoords='offset points', fontsize=7.5, color='#0072BD')
            ax.annotate(f'{fv:.2f}', (Nv, fv), xytext=(4, -10),
                        textcoords='offset points', fontsize=7.5, color=color)
        ax.set_xscale('log')
        ax.set_xticks(Ns_ec); ax.set_xticklabels([str(n) for n in Ns_ec], fontsize=9)
        ax.set_xlabel('N_CELL_PAIRS', fontsize=10)
        ax.set_ylabel('Net annual value ($/m²/yr)', fontsize=10)
        ax.set_title(
            f'{label}\nEconomic Impact of Fouling  (${_ELEC_PRICE}/kWh | '
            f'${_CAPEX_PER_M2}/m² | {_AMORT_YR}-yr amort.)',
            fontsize=10, fontweight='bold')
        ax.legend(fontsize=9); ax.grid(True, alpha=0.25)
        fig.tight_layout()
        _save(fig, 'fouling_fig_D_economics_shift.png')

        print(f'[fouling_post] Complete. All artifacts in {run_dir}')

    return _post


print('Fouling hook loaded: make_fouling_post_analysis.')
print('Usage: assign post_analysis=make_fouling_post_analysis() to BRINE-*-FOULING specs.')


# ─────────────────────────────────────────────────────────────────────────────
# Prototype validation post-analysis hook
#
# make_prototype_post_analysis() returns a callable for PROTOTYPE-VALIDATION.
# Sweep2D: C_H_IN_MOL_L × N_CELL_PAIRS (3 × 2 = 6 runs)
#
# Figures produced (auto-included in PDF):
#   proto_fig_A_summary_grid.png   — 2×3 panel: V_oc, R_int, P_dens across
#                                    the sweep grid with annotated values
#   proto_fig_B_iv_envelopes.png   — I–V / P–V envelopes per salinity level,
#                                    useful for choosing R_LOAD on the bench
#
# To overlay physical measurements from the MM420, pass meas_dict:
#   meas_dict = {
#       (5,  0.60): {"V_oc_V": 0.38, "R_int_ohm": 4.2},
#       (10, 0.60): {"V_oc_V": 0.71, "R_int_ohm": 2.1},
#   }
# Any key not present is silently skipped (partial measurements are fine).
# ─────────────────────────────────────────────────────────────────────────────

def make_prototype_post_analysis(*, meas_dict=None):
    """
    Factory for PROTOTYPE-VALIDATION post_analysis hook.

    meas_dict: optional dict keyed by (N_CELL_PAIRS: int, C_H_IN_MOL_L: float)
               containing measured values from the MM420. Example:
                 {(5, 0.60): {"V_oc_V": 0.38, "R_int_ohm": 4.2}}
               Overlaid as filled diamond markers on the bar charts.
               Partial dictionaries (some keys missing) are fine.
    """
    import matplotlib
    from matplotlib.lines import Line2D
    import matplotlib.gridspec as gridspec
    import warnings as _w

    _meas = meas_dict or {}

    def _post(run_dir, sweep_df, spec):
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        _w.filterwarnings('ignore')
        run_dir = _Path(run_dir)

        label = str(spec.plot_label or 'Prototype')
        color = str(spec.plot_color or '#7E2F8E')

        needed = {'C_H_IN_MOL_L', 'N_CELL_PAIRS', 'V_OC_EQ_V', 'R_INT_EQ_OHM',
                  'P_DENS_W_M2', 'P_TOTAL_W'}
        missing = needed - set(sweep_df.columns)
        if missing:
            print(f'[proto_post] WARNING: missing columns {missing} — skipping.')
            return

        df = sweep_df.copy()
        df['C_H_IN_MOL_L'] = df['C_H_IN_MOL_L'].round(3)
        df['N_CELL_PAIRS']  = df['N_CELL_PAIRS'].astype(int)

        C_H_vals = sorted(df['C_H_IN_MOL_L'].unique())
        N_vals   = sorted(df['N_CELL_PAIRS'].unique())
        colors_ch = ['#0072BD', '#77AC30', '#D95319'][:len(C_H_vals)]

        def _save(fig, name):
            fig.savefig(run_dir / name, dpi=150, bbox_inches='tight')
            plt.close(fig)
            print(f'[proto_post]   {name}')

        # ── Fig A: Summary grid — V_oc, R_int, P_dens ──────────────────────
        metrics = [
            ('V_OC_EQ_V',    'Open-circuit voltage (V)',  'V_oc_V',    False),
            ('R_INT_EQ_OHM', 'Internal resistance (Ω)',   'R_int_ohm', True),
            ('P_DENS_W_M2',  'Power density (W/m²)',       None,        False),
        ]
        fig = plt.figure(figsize=(5 * len(metrics), 4.5 * len(N_vals)))
        fig.suptitle(
            f'{label}  |  Model predictions vs MM420 measurements  |  Fumasep FKS/FAS-PET-130  |  A_MEM=0.0025 m²  |  C_L=0.017 mol/L',
            fontsize=9, fontweight='bold'
        )
        gs = gridspec.GridSpec(len(N_vals), len(metrics), figure=fig,
                               hspace=0.50, wspace=0.40)
        x_pos = _np.arange(len(C_H_vals))

        for col, (metric, ylabel, meas_key, log_scale) in enumerate(metrics):
            for row, N in enumerate(N_vals):
                ax = fig.add_subplot(gs[row, col])
                sub = df[df['N_CELL_PAIRS'] == N].sort_values('C_H_IN_MOL_L')

                vals = []
                for ch in C_H_vals:
                    r = sub[sub['C_H_IN_MOL_L'] == ch]
                    vals.append(float(r[metric].values[0]) if not r.empty else _np.nan)

                ax.bar(x_pos, vals, color=colors_ch, alpha=0.78,
                       edgecolor='white', linewidth=1.2)
                for xi, v in enumerate(vals):
                    if not _np.isnan(v):
                        fmt = f'{v:.3f}' if 'V_OC' in metric else (f'{v:.2f}' if 'R_INT' in metric else f'{v:.4f}')
                        ax.text(xi, v * (1.12 if log_scale else 1.04), fmt,
                                ha='center', fontsize=8.5)

                # Measured overlays
                if meas_key:
                    for xi, ch in enumerate(C_H_vals):
                        key = (N, round(ch, 3))
                        if key in _meas and meas_key in _meas[key]:
                            mv = _meas[key][meas_key]
                            if mv is not None:
                                ax.plot(xi, mv, 'D', ms=10, color='black',
                                        markeredgecolor='white', markeredgewidth=0.8,
                                        zorder=6)
                                ax.annotate(f'meas={mv:.3f}', (xi, mv),
                                            xytext=(0, 10), textcoords='offset points',
                                            ha='center', fontsize=7.5, color='black',
                                            fontweight='bold')

                ax.set_xticks(x_pos)
                ax.set_xticklabels([f'{ch} mol/L' for ch in C_H_vals], fontsize=8.5)
                ax.set_ylabel(ylabel, fontsize=9)
                ax.set_title(f'N = {N} cell pairs — {metric.split("_")[0]}',
                             fontsize=9, fontweight='bold')
                ax.grid(axis='y', alpha=0.25)
                if log_scale:
                    ax.set_yscale('log')

        legend_handles = [
            Line2D([0],[0], color=c, lw=9, alpha=0.78, label=f'C_H = {ch} mol/L')
            for c, ch in zip(colors_ch, C_H_vals)
        ]
        if _meas:
            legend_handles.append(
                Line2D([0],[0], marker='D', color='black', lw=0, ms=9,
                       label='MM420 measurement')
            )
        fig.legend(handles=legend_handles, loc='lower center',
                   ncol=len(legend_handles), fontsize=9, framealpha=0.92,
                   bbox_to_anchor=(0.5, 0.01))
        fig.tight_layout(rect=[0, 0.06, 1, 1])
        _save(fig, 'proto_fig_A_summary_grid.png')

        # ── Fig B: I–V / P–V envelopes ─────────────────────────────────────
        ls_by_N = {5: '-', 10: '--', 7: '-.', 20: ':'}
        fig, axes = plt.subplots(1, len(C_H_vals),
                                 figsize=(5.5 * len(C_H_vals), 5), sharey=False)
        if len(C_H_vals) == 1:
            axes = [axes]
        fig.suptitle(
            f'{label}  |  I-V / P-V envelopes (Thevenin model)  |  dots = max-power point  |  set R_ext ~ R_int for max transfer',
            fontsize=9, fontweight='bold'
        )

        for ax, ch, c_ch in zip(axes, C_H_vals, colors_ch):
            ax2 = ax.twinx()
            for N in N_vals:
                row = df[(df['C_H_IN_MOL_L'] == ch) & (df['N_CELL_PAIRS'] == N)]
                if row.empty:
                    continue
                v_oc  = float(row['V_OC_EQ_V'].values[0])
                r_int = float(row['R_INT_EQ_OHM'].values[0])
                i_sc  = v_oc / r_int
                i_arr = _np.linspace(0, i_sc * 0.999, 150)
                v_arr = v_oc - i_arr * r_int
                p_arr = v_arr * i_arr * 1000   # mW
                ls    = ls_by_N.get(N, ':')

                ax.plot(i_arr * 1000, v_arr, ls=ls, color=c_ch, lw=2.5,
                        label=f'N={N}  V_oc={v_oc:.3f}V  R_int={r_int:.2f}Ω')
                ax2.plot(i_arr * 1000, p_arr, ls=ls, color=c_ch, lw=1.8, alpha=0.50)

                # Max-power point
                i_mp = i_sc / 2
                v_mp = v_oc / 2
                p_mp = (v_oc ** 2) / (4 * r_int) * 1000
                ax.plot(i_mp * 1000, v_mp, 'o', ms=8, color=c_ch, zorder=5,
                        markeredgecolor='white', markeredgewidth=0.8)
                ax2.plot(i_mp * 1000, p_mp, 'o', ms=6, color=c_ch, alpha=0.55, zorder=5)
                ax.annotate(f'R_opt≈{r_int:.1f}Ω', (i_mp * 1000, v_mp),
                            xytext=(5, 5), textcoords='offset points',
                            fontsize=7.5, color=c_ch)

            ax.set_xlabel('Current (mA)', fontsize=10)
            ax.set_ylabel('Terminal voltage (V)', fontsize=10)
            ax2.set_ylabel('Power (mW)', fontsize=9, color='gray')
            for tl in ax2.get_yticklabels():
                tl.set_color('gray')
            ax.set_title(f'C_H = {ch} mol/L', fontsize=10, fontweight='bold')
            ax.legend(fontsize=8, loc='upper right', framealpha=0.9)
            ax.grid(True, alpha=0.25)
            ax.set_xlim(left=0)
            ax.set_ylim(bottom=0)

        fig.tight_layout()
        _save(fig, 'proto_fig_B_iv_envelopes.png')

        # ── Console summary ─────────────────────────────────────────────────
        print(f'[proto_post] Prediction summary (compare to MM420):')
        print(f'  {"N":>4}  {"C_H":>6}  {"V_oc (V)":>10}  {"R_int (Ω)":>11}  {"P_dens (W/m²)":>14}')
        for _, row in df.sort_values(['N_CELL_PAIRS','C_H_IN_MOL_L']).iterrows():
            N   = int(row['N_CELL_PAIRS'])
            ch  = float(row['C_H_IN_MOL_L'])
            voc = float(row['V_OC_EQ_V'])
            rin = float(row['R_INT_EQ_OHM'])
            pd  = float(row['P_DENS_W_M2'])
            mstr = ''
            key = (N, round(ch, 3))
            if key in _meas:
                m = _meas[key]
                mv = m.get('V_oc_V'); mr = m.get('R_int_ohm')
                if mv: mstr += f'  V_oc err={abs(voc-mv)/mv*100:.1f}%'
                if mr: mstr += f'  R_int err={abs(rin-mr)/mr*100:.1f}%'
            print(f'  {N:>4}  {ch:>6.2f}  {voc:>10.4f}  {rin:>11.4f}  {pd:>14.6f}{mstr}')
        print(f'[proto_post] Complete. Figures in {run_dir}')

    return _post


print('Prototype hook loaded: make_prototype_post_analysis().')
print('Usage: make_prototype_post_analysis(meas_dict={(N, C_H): {"V_oc_V": x, "R_int_ohm": y}})')


# ─────────────────────────────────────────────────────────────────────────────
# Uncertainty Quantification post-analysis hook
#
# make_uq_post_analysis() returns a callable for UQ-* experiment specs.
# Each UQ experiment is a Sweep1D over one uncertain parameter while all
# others are held at their best-estimate defaults.
#
# After all three UQ sweeps run, call build_uq_tornado() to assemble the
# combined tornado chart from the three individual run directories.
#
# Figures produced per experiment:
#   uq_fig_A_band_plot.png    — output metric vs swept parameter with
#                               ±band shading between min/max and the
#                               best-estimate (centre) value marked
#   uq_fig_B_sensitivity.png  — bar chart: % change in output per unit
#                               fractional change in parameter (elasticity)
#
# Combined figure (call separately after all three sweeps complete):
#   build_uq_tornado(run_dirs, output_path) → tornado_plot.png
# ─────────────────────────────────────────────────────────────────────────────

def make_uq_post_analysis(*, param_label, param_best, output_metrics=None):
    """
    Factory for UQ Sweep1D post_analysis hooks.

    param_label : human-readable parameter name, e.g. 'ASR_CEM (Ω·m²)'
    param_best  : best-estimate (centre) value of the swept parameter
    output_metrics : list of output column names to plot (default: standard set)
    """
    import matplotlib
    from matplotlib.lines import Line2D
    import warnings as _w

    _outputs = output_metrics or [
        'P_DENS_W_M2', 'P_TOTAL_W', 'V_OC_EQ_V', 'R_INT_EQ_OHM',
    ]
    _pretty_map = {
        'P_DENS_W_M2':           'Power density (W/m²)',
        'P_TOTAL_W':             'Total power (W)',
        'V_OC_EQ_V':             'Open-circuit voltage (V)',
        'R_INT_EQ_OHM':          'Internal resistance (Ω)',
        'FOUL_POWER_PENALTY_FRAC': 'Fouling power penalty (fraction)',
        'FOUL_POWER_PENALTY_W':  'Fouling power loss (W)',
    }

    def _post(run_dir, sweep_df, spec):
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        _w.filterwarnings('ignore')
        run_dir = _Path(run_dir)

        label = str(spec.plot_label or 'UQ sweep')
        color = str(spec.plot_color or '#0072BD')

        # Identify the sweep axis (the one column that varies)
        sweep_spec = getattr(spec, 'sweep', None)
        if sweep_spec is None or not hasattr(sweep_spec, 'var'):
            print('[uq_post] WARNING: no Sweep1D detected — skipping.')
            return
        param_col = sweep_spec.var
        if param_col not in sweep_df.columns:
            print(f'[uq_post] WARNING: {param_col} not in sweep_df — skipping.')
            return

        df = sweep_df.copy().sort_values(param_col)
        x  = df[param_col].values

        # Find best-estimate row (closest to param_best)
        best_idx = int(_np.argmin(_np.abs(x - param_best)))

        present_outputs = [m for m in _outputs if m in df.columns]
        if not present_outputs:
            print('[uq_post] WARNING: none of the requested output metrics found.')
            return

        # ── Fig A: Band plot — one panel per output metric ──────────────────
        ncols = min(2, len(present_outputs))
        nrows = (len(present_outputs) + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(6.5 * ncols, 4.5 * nrows),
                                 squeeze=False)
        fig.suptitle(
            f'{label}  |  Uncertainty band: {param_label}',
            fontsize=11, fontweight='bold'
        )

        for ax_flat, metric in zip(axes.flat, present_outputs):
            y = df[metric].values
            y_best = float(y[best_idx])
            y_min  = float(y.min())
            y_max  = float(y.max())
            pct_range = (y_max - y_min) / abs(y_best) * 100 if y_best != 0 else 0

            ax_flat.plot(x, y, '-o', color=color, lw=2.2, ms=6,
                         markeredgecolor='white', markeredgewidth=0.8, zorder=3)
            ax_flat.axhline(y_best, color='gray', lw=1.2, ls='--', alpha=0.7,
                            label=f'Best estimate: {y_best:.4g}')
            ax_flat.axvline(param_best, color='gray', lw=1.2, ls=':', alpha=0.7)
            ax_flat.fill_between(x, y_min, y_max, color=color, alpha=0.10,
                                 label=f'Full range: {pct_range:.1f}%')
            # Mark best-estimate point
            ax_flat.plot(x[best_idx], y_best, 'D', ms=10, color=color,
                         markeredgecolor='white', markeredgewidth=1.2, zorder=5,
                         label=f'Centre ({param_best:.4g})')

            ax_flat.set_xlabel(param_label, fontsize=10)
            ax_flat.set_ylabel(_pretty_map.get(metric, metric), fontsize=10)
            ax_flat.set_title(
                f'{_pretty_map.get(metric, metric)}  |  range [{y_min:.4g}, {y_max:.4g}]  ({pct_range:.1f}% spread)',
                fontsize=9
            )
            ax_flat.legend(fontsize=8, framealpha=0.9)
            ax_flat.grid(True, alpha=0.25)

        # Hide any unused axes
        for ax_flat in axes.flat[len(present_outputs):]:
            ax_flat.set_visible(False)

        fig.tight_layout(rect=[0, 0, 1, 0.94])
        out_A = run_dir / 'uq_fig_A_band_plot.png'
        fig.savefig(out_A, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'[uq_post]   uq_fig_A_band_plot.png')

        # ── Fig B: Elasticity bar chart ──────────────────────────────────────
        # Elasticity = (Δy/y_best) / (Δx/x_best) evaluated across the full range
        fig2, axes2 = plt.subplots(1, len(present_outputs),
                                   figsize=(4.5 * len(present_outputs), 4.5),
                                   squeeze=False)
        fig2.suptitle(
            f'{label}  |  Output sensitivity to {param_label}',
            fontsize=11, fontweight='bold'
        )

        for ax2, metric in zip(axes2.flat, present_outputs):
            y    = df[metric].values
            y_best = float(y[best_idx])
            x_best = float(param_best)

            elasticities = []
            for xi, yi in zip(x, y):
                if x_best == 0 or y_best == 0:
                    elasticities.append(0.0)
                else:
                    elasticities.append(((yi - y_best) / y_best) /
                                        ((xi - x_best) / x_best)
                                        if xi != x_best else 0.0)

            bar_colors = ['#D95319' if e < 0 else '#0072BD' for e in elasticities]
            ax2.bar(range(len(x)), elasticities, color=bar_colors,
                    edgecolor='white', linewidth=0.8, alpha=0.85)
            ax2.axhline(0, color='black', lw=0.8)
            ax2.set_xticks(range(len(x)))
            ax2.set_xticklabels([f'{xi:.4g}' for xi in x], fontsize=8, rotation=30)
            ax2.set_xlabel(param_label, fontsize=9)
            ax2.set_ylabel('Elasticity (Δoutput%  /  Δparam%)', fontsize=9)
            ax2.set_title(_pretty_map.get(metric, metric), fontsize=9, fontweight='bold')
            ax2.grid(axis='y', alpha=0.25)

        fig2.tight_layout(rect=[0, 0, 1, 0.92])
        out_B = run_dir / 'uq_fig_B_sensitivity.png'
        fig2.savefig(out_B, dpi=150, bbox_inches='tight')
        plt.close(fig2)
        print(f'[uq_post]   uq_fig_B_sensitivity.png')

        # ── Console summary ──────────────────────────────────────────────────
        print(f'[uq_post] {param_label} sensitivity summary:')
        print(f'  {"param value":>14}' + ''.join(f'  {m[:14]:>14}' for m in present_outputs))
        for _, row in df.iterrows():
            vals = ''.join(f'  {float(row[m]):>14.5g}' for m in present_outputs)
            marker = ' ← best estimate' if abs(float(row[param_col]) - param_best) < 1e-10 else ''
            print(f'  {float(row[param_col]):>14.5g}{vals}{marker}')

        # Save a machine-readable summary for the tornado builder
        uq_summary = {
            'param_col':   param_col,
            'param_label': param_label,
            'param_best':  param_best,
            'param_values': x.tolist(),
            'outputs': {
                m: {
                    'values':     df[m].values.tolist(),
                    'best':       float(df[m].values[best_idx]),
                    'min':        float(df[m].values.min()),
                    'max':        float(df[m].values.max()),
                    'pct_range':  float((df[m].values.max() - df[m].values.min())
                                        / abs(float(df[m].values[best_idx])) * 100)
                                  if float(df[m].values[best_idx]) != 0 else 0.0,
                }
                for m in present_outputs
            }
        }
        import json as _json
        summary_path = run_dir / 'uq_summary.json'
        with open(summary_path, 'w') as _f:
            _json.dump(uq_summary, _f, indent=2)
        print(f'[uq_post]   uq_summary.json (use with build_uq_tornado)')
        print(f'[uq_post] Complete.')

    return _post


def build_uq_tornado(run_dirs, output_path=None, metric='P_DENS_W_M2'):
    """
    Build a tornado chart from multiple completed UQ sweep run directories.

    run_dirs    : list of Path-like strings pointing to completed UQ run dirs
                  each must contain a uq_summary.json written by make_uq_post_analysis
    output_path : where to save the PNG (default: alongside first run_dir)
    metric      : which output metric to feature in the tornado (default P_DENS_W_M2)

    Returns the path to the saved figure.

    Usage (after all UQ sweeps have run):
        build_uq_tornado([
            'runs/UQ-ASR-CEM/...',
            'runs/UQ-FOULING-BETA/...',
            'runs/UQ-ALPHA-PERM/...',
        ])
    """
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import json as _json
    _warnings.filterwarnings('ignore')

    summaries = []
    for rd in run_dirs:
        p = _Path(rd) / 'uq_summary.json'
        if not p.exists():
            print(f'[tornado] WARNING: {p} not found — skipping.')
            continue
        with open(p) as f:
            summaries.append(_json.load(f))

    if not summaries:
        print('[tornado] No summaries found.')
        return None

    fig, ax = plt.subplots(figsize=(9, 1.5 + 1.2 * len(summaries)))
    fig.suptitle(
        f'Tornado chart: sensitivity of {metric} to uncertain parameters  |  bars show output range across literature range',
        fontsize=11, fontweight='bold'
    )

    bars_data = []
    for s in summaries:
        if metric not in s['outputs']:
            print(f'[tornado] metric {metric} not in {s["param_label"]} summary — skipping.')
            continue
        o = s['outputs'][metric]
        bars_data.append({
            'label':     s['param_label'],
            'best':      o['best'],
            'low':       o['min'],
            'high':      o['max'],
            'pct_range': o['pct_range'],
        })

    if not bars_data:
        print('[tornado] No plottable data.')
        plt.close(fig)
        return None

    # Sort by absolute range (widest bar at top)
    bars_data.sort(key=lambda b: abs(b['high'] - b['low']), reverse=True)

    y_positions = range(len(bars_data))
    best_ref = bars_data[0]['best']  # use first (widest) as reference

    for yi, b in zip(y_positions, bars_data):
        low_delta  = b['low']  - b['best']
        high_delta = b['high'] - b['best']
        ax.barh(yi, low_delta,  left=b['best'], color='#D95319', alpha=0.80,
                edgecolor='white', height=0.55)
        ax.barh(yi, high_delta, left=b['best'], color='#0072BD', alpha=0.80,
                edgecolor='white', height=0.55)
        ax.text(b['low']  - abs(best_ref) * 0.005, yi, f'{b["low"]:.4g}',
                va='center', ha='right', fontsize=8.5, color='#D95319')
        ax.text(b['high'] + abs(best_ref) * 0.005, yi, f'{b["high"]:.4g}',
                va='center', ha='left',  fontsize=8.5, color='#0072BD')
        ax.text(b['best'], yi + 0.32, f'±{b["pct_range"]:.1f}%',
                va='bottom', ha='center', fontsize=8, color='black')

    ax.set_yticks(list(y_positions))
    ax.set_yticklabels([b['label'] for b in bars_data], fontsize=10)
    ax.axvline(best_ref, color='black', lw=1.5, ls='--', alpha=0.6,
               label=f'Best estimate: {best_ref:.4g}')
    ax.set_xlabel(f'{metric} ({_pretty_map_tornado.get(metric, metric)})', fontsize=10)
    ax.legend(fontsize=9, loc='lower right')
    ax.grid(axis='x', alpha=0.25)
    ax.invert_yaxis()
    fig.tight_layout(rect=[0, 0, 1, 0.90])

    if output_path is None:
        output_path = _Path(run_dirs[0]).parent / 'uq_tornado_plot.png'
    else:
        output_path = _Path(output_path)
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'[tornado] Saved: {output_path}')
    return output_path


_pretty_map_tornado = {
    'P_DENS_W_M2':             'W/m²',
    'P_TOTAL_W':               'W',
    'V_OC_EQ_V':               'V',
    'R_INT_EQ_OHM':            'Ω',
    'FOUL_POWER_PENALTY_FRAC': 'fraction',
}

print('UQ hooks loaded: make_uq_post_analysis(), build_uq_tornado().')


# ── Morris screening post-analysis hook ──────────────────────────────────────
#
# Computes Morris elementary effects (μ*, σ) from the sweep_table produced by
# a SweepMorris run.  Generates two publication-quality figures:
#   A: μ* bar chart (parameter importance ranking)
#   B: μ* vs σ scatter (distinguishing linear vs nonlinear/interactive effects)
# ─────────────────────────────────────────────────────────────────────────────

def make_morris_post_analysis(*, output_metrics=None):
    """Factory for Morris screening post_analysis hooks.

    output_metrics : list of output column names to analyze
                     (default: P_DENS_W_M2, FOUL_POWER_PENALTY_FRAC)
    """
    import matplotlib
    import warnings as _w

    _outputs = output_metrics or ['P_DENS_W_M2', 'FOUL_POWER_PENALTY_FRAC']
    _pretty = {
        'P_DENS_W_M2':              'Power density  [W/m²]',
        'P_TOTAL_W':                'Total power  [W]',
        'V_OC_EQ_V':                'Open-circuit voltage  [V]',
        'R_INT_EQ_OHM':             'Internal resistance  [Ω]',
        'FOUL_POWER_PENALTY_FRAC':  'Fouling penalty  [fraction]',
        'V_TERM_V':                 'Terminal voltage  [V]',
        'I_TOTAL_A':                'Total current  [A]',
    }

    def _post(run_dir, sweep_df, spec):
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        _w.filterwarnings('ignore')
        run_dir = _Path(run_dir)

        sweep = getattr(spec, 'sweep', None)
        if sweep is None or not hasattr(sweep, 'params'):
            print('[morris] WARNING: no SweepMorris detected — skipping.')
            return

        param_names = sweep.var_names
        k = len(param_names)
        r = sweep.r

        # ── Compute elementary effects ───────────────────────────────────────
        # For each trajectory (r), there are k+1 points.
        # The elementary effect of param j in trajectory t is:
        #   EE_j = (y[step where j changed] - y[previous step]) / delta_j
        # where delta_j is the normalized step in [0,1] space.

        results = {}  # {metric: {param: [list of EE values]}}
        for metric in _outputs:
            if metric not in sweep_df.columns:
                continue
            ee_dict = {p: [] for p in param_names}

            for t in range(r):
                traj = sweep_df[sweep_df['_morris_traj'] == t].sort_values('_morris_step')
                if len(traj) < k + 1:
                    continue

                y_vals = traj[metric].values
                changed = traj['_morris_changed_param'].values

                for step_idx in range(1, len(traj)):
                    param = changed[step_idx]
                    if param == '__base__' or param not in param_names:
                        continue

                    # Compute normalized delta for this parameter
                    lo, hi = sweep.params[param]
                    x_before = traj.iloc[step_idx - 1][param]
                    x_after  = traj.iloc[step_idx][param]
                    dx_norm  = (x_after - x_before) / max(hi - lo, 1e-30)

                    if abs(dx_norm) < 1e-12:
                        continue

                    ee = (y_vals[step_idx] - y_vals[step_idx - 1]) / dx_norm
                    ee_dict[param].append(ee)

            results[metric] = ee_dict

        # ── Build summary DataFrame ──────────────────────────────────────────
        summary_rows = []
        for metric, ee_dict in results.items():
            for param, ee_list in ee_dict.items():
                if not ee_list:
                    continue
                arr = np.array(ee_list)
                summary_rows.append({
                    'metric':   metric,
                    'param':    param,
                    'mu_star':  float(np.mean(np.abs(arr))),
                    'mu':       float(np.mean(arr)),
                    'sigma':    float(np.std(arr, ddof=1)) if len(arr) > 1 else 0.0,
                    'n_ee':     len(arr),
                })

        if not summary_rows:
            print('[morris] WARNING: no elementary effects computed.')
            return

        morris_df = pd.DataFrame(summary_rows)
        morris_df.to_parquet(run_dir / 'morris_screening.parquet', index=False)
        print(f'[morris]   morris_screening.parquet ({len(morris_df)} rows)')

        # ── Figure A: μ* bar chart (one panel per metric) ────────────────────
        metrics_available = [m for m in _outputs if m in results and results[m]]
        n_metrics = len(metrics_available)
        if n_metrics == 0:
            return

        fig, axes = plt.subplots(1, n_metrics, figsize=(5.5 * n_metrics, 5),
                                 constrained_layout=True, squeeze=False)
        axes = axes[0]

        for ax, metric in zip(axes, metrics_available):
            sub = morris_df[morris_df['metric'] == metric].sort_values('mu_star', ascending=True)
            colors = ['#D95319' if v > sub['mu_star'].median() else '#0072BD'
                       for v in sub['mu_star']]
            ax.barh(sub['param'], sub['mu_star'], color=colors,
                    edgecolor='white', linewidth=0.5)
            ax.set_xlabel('μ*  (mean absolute elementary effect)')
            ax.set_title(_pretty.get(metric, metric), fontsize=10)
            # Annotate values
            for i, (_, row) in enumerate(sub.iterrows()):
                ax.text(row['mu_star'] + sub['mu_star'].max() * 0.02, i,
                        f"{row['mu_star']:.4f}", va='center', fontsize=8)

        fig.suptitle(
            'Morris Screening: Parameter Importance Ranking\n'
            'Higher μ* = stronger influence on output  |  '
            f'r = {r} trajectories, k = {k} parameters',
            fontsize=11,
        )

        fig_path = run_dir / 'morris_fig_A_importance.png'
        fig.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'[morris]   {fig_path.name}')

        # ── Figure B: μ* vs σ scatter (one panel per metric) ─────────────────
        fig, axes = plt.subplots(1, n_metrics, figsize=(5.5 * n_metrics, 5),
                                 constrained_layout=True, squeeze=False)
        axes = axes[0]

        for ax, metric in zip(axes, metrics_available):
            sub = morris_df[morris_df['metric'] == metric]
            ax.scatter(sub['mu_star'], sub['sigma'], s=80, c='#0072BD',
                       edgecolors='white', linewidths=0.8, zorder=3)
            for _, row in sub.iterrows():
                ax.annotate(row['param'], (row['mu_star'], row['sigma']),
                            textcoords='offset points', xytext=(6, 4),
                            fontsize=7.5)
            # Reference line: σ = μ* (above = nonlinear/interactive)
            max_val = max(sub['mu_star'].max(), sub['sigma'].max()) * 1.1
            ax.plot([0, max_val], [0, max_val], '--', color='gray', lw=0.8,
                    label='σ = μ* (nonlinear/interactive)')
            ax.set_xlabel('μ*  (importance)')
            ax.set_ylabel('σ  (nonlinearity / interactions)')
            ax.set_title(_pretty.get(metric, metric), fontsize=10)
            ax.legend(fontsize=7, loc='upper left')
            ax.set_xlim(left=0)
            ax.set_ylim(bottom=0)

        fig.suptitle(
            'Morris Screening: Importance vs Nonlinearity\n'
            'Below diagonal = mainly linear effect  |  '
            'Above diagonal = nonlinear or interactive',
            fontsize=11,
        )

        fig_path = run_dir / 'morris_fig_B_scatter.png'
        fig.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'[morris]   {fig_path.name}')

        # Console summary
        print('[morris] Parameter importance ranking (P_DENS_W_M2):')
        if 'P_DENS_W_M2' in results:
            sub = morris_df[morris_df['metric'] == 'P_DENS_W_M2'].sort_values('mu_star', ascending=False)
            for _, row in sub.iterrows():
                marker = '  ← dominant' if row['mu_star'] == sub['mu_star'].max() else ''
                print(f"    {row['param']:25s}  μ*={row['mu_star']:.4f}  σ={row['sigma']:.4f}{marker}")

        print('[morris] Complete.')

    return _post


print('Morris hook loaded: make_morris_post_analysis().')


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# Monte Carlo Techno-Economic Analysis (TEA) Engine
#
# run_mc_tea(run_dir, spec) is called automatically by run_experiment() when
# spec.run_monte_carlo = True.  It:
#
#   1. Resolves the deployment point (N, A) from harm_sweep.parquet if available,
#      or falls back to mc_config["deployment_N"] / ["deployment_A"].
#
#   2. Samples two stochastic axes using spec.mc_config:
#        - ELECTRICITY_PRICE  : lognormal  (mean = PLANT_ELEC_PRICE_USD_KWH,
#                                           relative σ = elec_price_std_frac)
#        - FOULING_BETA       : uniform    (lo = fouling_beta_lo,
#                                           hi = fouling_beta_hi)
#
#   3. For each sample: re-runs the 1D-series model at the deployment point
#      with the sampled FOULING_BETA, then computes TEA metrics:
#        - ANNUAL_ENERGY_KWH     : P_total × capacity_factor × 8760 h/yr
#        - ANNUAL_REVENUE_USD    : energy × elec_price
#        - ANNUAL_BRINE_SAVINGS  : avoided brine disposal cost
#        - ANNUAL_OPEX_USD       : opex_frac × total_capex
#        - ANNUAL_AMORT_USD      : total_capex / amort_yr
#        - NET_ANNUAL_USD        : revenue + savings - opex - amort
#        - NPV_USD               : NPV over amort period at zero discount (simple)
#        - LCOE_USD_KWH          : (amort + opex) / annual_energy
#        - PAYBACK_YR            : total_capex / (revenue + savings - opex)
#
#   4. Writes mc_tea.parquet + 3 publication-quality PNG figures into run_dir:
#        mc_fig_A_npv_distribution.png   — NPV histogram with percentile bands
#        mc_fig_B_sensitivity_spider.png — tornado / one-at-a-time sensitivity
#        mc_fig_C_joint_scatter.png      — 2D scatter: elec_price vs FOULING_BETA,
#                                          coloured by NPV
#
# Architecture:
#   - Additional MC axes can be added to mc_config without touching engine core:
#     add *_lo/*_hi (uniform) or *_mean/*_std_frac (lognormal) pairs and a
#     mapping to DesignInputs field name in _MC_AXIS_MAP below.
#   - All plant parameters are read from spec.base_overrides overlaid on
#     DesignInputs defaults — no external config dict needed.
# ─────────────────────────────────────────────────────────────────────────────

from pathlib import Path as _Path
import warnings as _w
import numpy as _np
import pandas as _pd

# ── Mapping: mc_config axis name → DesignInputs field ──────────────────────
# To add a new sweep axis: add a row here + corresponding lo/hi or mean/std_frac
# keys to mc_config.  The engine will sample and pass the value to the model.
_MC_AXIS_MAP: dict = {
    # axis_name         : (DesignInputs_field, distribution)
    # distribution: 'lognormal', 'uniform', or 'triangular'
    # Always-active axes (sampled in both legacy and loss-model modes):
    "elec_price"        : ("PLANT_ELEC_PRICE_USD_KWH",         "lognormal"),
    "fouling_beta"      : ("FOULING_BETA",                      "uniform"),
    # MTFC axes (only sampled when enable_loss_model=True; else held at base_knobs):
    "brine_disp_cost"   : ("PLANT_BRINE_DISPOSAL_COST_USD_M3",  "uniform"),
    "capex_per_m2"      : ("PLANT_CAPEX_PER_M2",                "triangular"),
    "discount_rate"     : ("PLANT_DISCOUNT_RATE",                "uniform"),
    "capacity_factor"   : ("PLANT_CAPACITY_FACTOR",              "uniform"),
}

_MC_DEFAULTS: dict = {
    "n_samples"          : 5000,
    "seed"               : 42,
    "deployment_N"       : None,   # resolved from harm_sweep if None
    "deployment_A"       : None,   # resolved from harm_sweep if None
    "elec_price_mean"    : None,   # falls back to PLANT_ELEC_PRICE_USD_KWH
    "elec_price_std_frac": 0.30,   # 30% relative σ — CA wholesale price volatility
    "fouling_beta_lo"    : 0.2,    # Gurreri 2023 lower bound
    "fouling_beta_hi"    : 0.8,    # Gurreri 2023 upper bound
    # ── Backward-compatibility switches ──────────────────────────────
    # When both are default (legacy/False), engine is identical to original 2-axis version.
    "npv_mode"              : "legacy",     # "legacy" | "discounted"
    "enable_loss_model"     : False,        # gate for new axes + compliance + loss
    # ── New economic axes (only sampled when enable_loss_model=True) ──
    "brine_disp_cost_lo"    : 0.04,         # coastal diffuser $/m³
    "brine_disp_cost_hi"    : 0.25,         # deep-well injection $/m³
    "capex_per_m2_min"      : 30.0,         # triangular min $/m²
    "capex_per_m2_mode"     : 50.0,         # triangular mode $/m²
    "capex_per_m2_max"      : 100.0,        # triangular max $/m²
    "discount_rate_lo"      : 0.03,         # infrastructure lower bound
    "discount_rate_hi"      : 0.08,         # infrastructure upper bound
    # ── Capacity factor + membrane lifetime (sampled when enable_loss_model=True) ──
    "capacity_factor_lo"    : 0.80,         # minimum uptime
    "capacity_factor_hi"    : 0.95,         # maximum uptime
    "membrane_lifetime_lo"  : 5.0,          # years — aggressive fouling regime
    "membrane_lifetime_hi"  : 15.0,         # years — well-managed operation
    # ── Loss model parameters (only used when enable_loss_model=True) ─
    "harm_redux_req"        : 95.0,         # % harm reduction for compliance
    "sev_day_usd_min"       : 5_000.0,      # triangular: daily NC cost
    "sev_day_usd_mode"      : 15_000.0,
    "sev_day_usd_max"       : 50_000.0,
    "p_enforce"             : 0.10,         # P(major enforcement | NC)
    "retrofit_cost_usd_min" : 5_000_000.0,  # triangular: enforcement cost
    "retrofit_cost_usd_mode": 10_000_000.0,
    "retrofit_cost_usd_max" : 20_000_000.0,
}


def _mc_resolve_deployment(run_dir: _Path, cfg: dict, spec) -> tuple:
    """
    Resolve (N, A) deployment point.
    Priority: cfg override > harm_sweep optimal (N=80 at A=0.01 from BRINE-CARLSBAD) > fallback.
    """
    if cfg.get("deployment_N") is not None and cfg.get("deployment_A") is not None:
        return int(cfg["deployment_N"]), float(cfg["deployment_A"])

    # Try to read from harm_sweep.parquet (written by brine post-analysis hook)
    hp = run_dir / "harm_sweep.parquet"
    if hp.exists():
        try:
            harm_df = _pd.read_parquet(hp)
            # Filter out inversions, find highest harm_redux row
            valid = harm_df[~harm_df.get("inv", _pd.Series(False, index=harm_df.index))]
            if len(valid) == 0:
                valid = harm_df
            # Use the row with highest harm_redux as the dual-objective optimum
            best = valid.loc[valid["harm_redux"].idxmax()]
            N = int(best.get("N_CELL_PAIRS", 80))
            A = float(best.get("A_MEM_M2", 0.05))
            # If harm_sweep selected a smaller A_MEM than the spec's target,
            # use the spec's A_MEM (pilot-scale floor) for TEA deployment
            A_spec = float(spec.base_overrides.get("A_MEM_M2", 0.05))
            if A < A_spec:
                print(f"[mc_tea] harm_sweep optimal A={A:.4g} < spec A={A_spec:.4g}; using spec A for TEA")
                A = A_spec
            print(f"[mc_tea] Deployment point from harm_sweep: N={N}, A={A:.4g} m² "
                  f"(harm_redux={best['harm_redux']:.1f}%)")
            return N, A
        except Exception as e:
            print(f"[mc_tea] Warning: could not read harm_sweep.parquet ({e}); using cfg fallback")

    # Final fallback: use N_CELL_PAIRS / A_MEM_M2 from base_overrides
    base_di_fields = DesignInputs.__dataclass_fields__
    N_fb = int(spec.base_overrides.get("N_CELL_PAIRS",
               base_di_fields["N_CELL_PAIRS"].default))
    A_fb = float(spec.base_overrides.get("A_MEM_M2",
                 base_di_fields["A_MEM_M2"].default))
    print(f"[mc_tea] Deployment point from base_overrides fallback: N={N_fb}, A={A_fb:.4g} m²")
    return N_fb, A_fb


def _mc_base_params(spec) -> dict:
    """Build a knob dict from DesignInputs defaults + spec.base_overrides."""
    base = {k: f.default for k, f in DesignInputs.__dataclass_fields__.items()}
    base.update(spec.base_overrides or {})
    return base


def _mc_run_one(base_knobs: dict, N: int, A: float, fouling_beta: float) -> dict:
    """Run model at (N, A, fouling_beta); return power + concentration outputs.

    Only overrides N_CELL_PAIRS, A_MEM_M2, FOULING_BETA, and MODE.
    All other knobs (DIVALENT_FRACTION, FOULING_MODEL, etc.) come from
    base_knobs via the ExperimentSpec. No setdefault() calls.
    """
    overrides = dict(base_knobs)
    overrides["N_CELL_PAIRS"] = N
    overrides["A_MEM_M2"]     = A
    overrides["FOULING_BETA"] = fouling_beta
    overrides["MODE"]         = "max_power"
    p = knobs_to_design_inputs(overrides)
    res = model_1d_axial_series_current(p, mode="max_power")
    seg = res["per_segment"]
    return dict(
        P_TOTAL_W  = float(res.get("P_TOTAL_W", 0.0)),
        C_H_OUT    = float(seg["C_H_OUT_MOL_L"][-1]),
        C_L_OUT    = float(seg["C_L_OUT_MOL_L"][-1]),
        INV        = bool(res["any_inversion"]),
    )


def _mc_tea_metrics(p_total_w: float, elec_price: float,
                    brine_disp_cost: float, capex_per_m2: float,
                    discount_rate: float, npv_mode: str,
                    base_knobs: dict, N: int, A: float,
                    capacity_factor: float = None,
                    membrane_lifetime_yr: float = None) -> dict:
    """
    Compute all TEA metrics for one MC sample.
    capacity_factor and membrane_lifetime_yr are sampled externally when
    enable_loss_model=True; otherwise held at base_knobs values.
    npv_mode: "legacy" = net_annual*amort_yr; "discounted" = annuity - capex.
    """
    brine_flow_m3d   = float(base_knobs.get("PLANT_BRINE_FLOW_M3_DAY",    190_000.0))
    opex_frac        = float(base_knobs.get("PLANT_OPEX_FRAC_CAPEX_PER_YR", 0.03))
    amort_yr         = int(  base_knobs.get("PLANT_AMORT_YR",               20))
    cap_factor       = capacity_factor if capacity_factor is not None else \
                       float(base_knobs.get("PLANT_CAPACITY_FACTOR", 0.913))

    total_mem_area_m2 = N * A            # total membrane area deployed [m²]
    hrs_per_yr        = 8760.0 * cap_factor

    # Energy
    annual_energy_kwh = p_total_w / 1000.0 * hrs_per_yr

    # Capital & operating costs
    total_capex       = total_mem_area_m2 * capex_per_m2
    annual_amort      = total_capex / amort_yr if amort_yr > 0 else 0.0
    annual_opex       = total_capex * opex_frac

    mem_lt = float(membrane_lifetime_yr) if membrane_lifetime_yr is not None else float(amort_yr)
    mem_lt = max(mem_lt, 1.0)
    annual_mem_replace = (total_mem_area_m2 * capex_per_m2 / mem_lt) if mem_lt < amort_yr else 0.0

    # ── Plant-scale deployment ─────────────────────────────────────────────
    # The 1D model runs ONE stack with flow Q_H_M3_S. A plant retrofit deploys
    # n_parallel identical stacks to treat the full brine discharge volume.
    # Energy output, capex, and costs all scale with n_parallel;
    # brine disposal savings apply to the full plant flow independently.
    q_model_m3s    = float(base_knobs.get("Q_H_M3_S", 5e-6))  # fixed: matches DesignInputs default
    q_model_m3d    = q_model_m3s * 86400               # one stack flow [m³/day]
    n_parallel     = brine_flow_m3d / q_model_m3d if q_model_m3d > 0 else 1.0

    plant_p_w      = p_total_w * n_parallel             # total plant power [W]
    plant_area_m2  = total_mem_area_m2 * n_parallel     # total membrane area [m²]
    plant_capex    = plant_area_m2 * capex_per_m2
    plant_amort    = plant_capex / amort_yr if amort_yr > 0 else 0.0
    plant_opex     = plant_capex * opex_frac
    plant_mem_replace = annual_mem_replace * n_parallel

    # Revenue: electricity from full plant power
    plant_energy_kwh  = plant_p_w / 1000.0 * hrs_per_yr
    annual_revenue    = plant_energy_kwh * elec_price

    # Brine savings: full plant disposal cost avoided (independent of stack count)
    annual_brine_vol  = brine_flow_m3d * 365.0
    annual_brine_sav  = annual_brine_vol * brine_disp_cost

    # TEA summary
    net_annual = annual_revenue + annual_brine_sav - plant_opex - plant_amort - plant_mem_replace

    # NPV: dual mode for backward compatibility
    if npv_mode == "discounted":
        if discount_rate > 0:
            annuity_factor = (1 - (1 + discount_rate)**(-amort_yr)) / discount_rate
        else:
            annuity_factor = float(amort_yr)
        npv = net_annual * annuity_factor - plant_capex
    else:
        # Legacy: npv = net_annual * amort_yr (original, no discounting)
        npv = net_annual * amort_yr

    lcoe       = (plant_amort + plant_opex) / plant_energy_kwh \
                 if plant_energy_kwh > 0 else _np.nan
    payback    = plant_capex / (annual_revenue + annual_brine_sav - plant_opex) \
                 if (annual_revenue + annual_brine_sav - plant_opex) > 0 else _np.nan

    return dict(
        P_TOTAL_W             = p_total_w,          # single stack [W]
        P_PLANT_W             = plant_p_w,          # full plant [W]
        N_PARALLEL_STACKS     = n_parallel,
        PLANT_MEM_AREA_M2     = plant_area_m2,
        PLANT_CAPEX_USD       = plant_capex,
        ANNUAL_ENERGY_KWH     = plant_energy_kwh,
        ANNUAL_REVENUE_USD    = annual_revenue,
        ANNUAL_BRINE_SAVINGS  = annual_brine_sav,
        ANNUAL_OPEX_USD       = plant_opex,
        ANNUAL_AMORT_USD      = plant_amort,
        NET_ANNUAL_USD        = net_annual,
        NPV_USD               = npv,
        LCOE_USD_KWH          = lcoe,
        PAYBACK_YR            = payback,
        DISCOUNT_RATE         = discount_rate,
        BRINE_DISP_COST_M3   = brine_disp_cost,
        CAPEX_PER_M2          = capex_per_m2,
        CAPACITY_FACTOR_USED  = cap_factor,
        MEMBRANE_LIFETIME_YR  = mem_lt,
        ANNUAL_MEM_REPLACE_USD = plant_mem_replace,
    )


def run_mc_tea(run_dir: _Path, spec) -> None:
    """
    Entry point called by run_experiment() when spec.run_monte_carlo = True.
    Generates mc_tea.parquet + 3 PNG figures in run_dir.
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    _w.filterwarnings("ignore")

    run_dir = _Path(run_dir)
    cfg = dict(_MC_DEFAULTS)
    cfg.update(spec.mc_config or {})

    n_samples  = int(cfg["n_samples"])
    seed       = int(cfg["seed"])
    rng        = _np.random.default_rng(seed)

    base_knobs = _mc_base_params(spec)
    label      = str(spec.plot_label or spec.experiment_id)
    color      = str(spec.plot_color or "#0072BD")

    # ── Resolve deployment point ───────────────────────────────────────────
    N_dep, A_dep = _mc_resolve_deployment(run_dir, cfg, spec)
    print(f"[mc_tea] Starting Monte Carlo TEA: {n_samples} samples, seed={seed}")
    print(f"[mc_tea] Deployment: N={N_dep}, A={A_dep:.4g} m² | {N_dep*A_dep:.4g} m² total")

    # ── Read switches ─────────────────────────────────────────────────────
    npv_mode    = str(cfg.get("npv_mode", "legacy"))
    enable_loss = bool(cfg.get("enable_loss_model", False))
    print(f"[mc_tea] npv_mode={npv_mode}, enable_loss_model={enable_loss}")

    # ── Sample stochastic axes ─────────────────────────────────────────────
    # Electricity price: lognormal (always active)
    ep_mean = cfg.get("elec_price_mean") or float(
        base_knobs.get("PLANT_ELEC_PRICE_USD_KWH", 0.22))
    ep_std_frac = float(cfg["elec_price_std_frac"])
    ep_sigma = _np.sqrt(_np.log(1 + ep_std_frac**2))
    ep_mu    = _np.log(ep_mean) - 0.5 * ep_sigma**2
    elec_prices = rng.lognormal(ep_mu, ep_sigma, n_samples)

    # Fouling beta: uniform (always active)
    fb_lo = float(cfg["fouling_beta_lo"])
    fb_hi = float(cfg["fouling_beta_hi"])
    fouling_betas = rng.uniform(fb_lo, fb_hi, n_samples)

    # MTFC axes: only sampled when enable_loss_model=True
    if enable_loss:
        bd_lo = float(cfg["brine_disp_cost_lo"])
        bd_hi = float(cfg["brine_disp_cost_hi"])
        brine_disp_costs = rng.uniform(bd_lo, bd_hi, n_samples)

        cx_min  = float(cfg["capex_per_m2_min"])
        cx_mode = float(cfg["capex_per_m2_mode"])
        cx_max  = float(cfg["capex_per_m2_max"])
        capex_samples = rng.triangular(cx_min, cx_mode, cx_max, n_samples)

        dr_lo = float(cfg["discount_rate_lo"])
        dr_hi = float(cfg["discount_rate_hi"])
        discount_rates = rng.uniform(dr_lo, dr_hi, n_samples)

        cf_lo = float(cfg.get("capacity_factor_lo", 0.80))
        cf_hi = float(cfg.get("capacity_factor_hi", 0.95))
        capacity_factors = rng.uniform(cf_lo, cf_hi, n_samples)

        ml_lo = float(cfg.get("membrane_lifetime_lo", 5.0))
        ml_hi = float(cfg.get("membrane_lifetime_hi", 15.0))
        membrane_lifetimes = rng.uniform(ml_lo, ml_hi, n_samples)
    else:
        brine_disp_costs   = _np.full(n_samples, float(base_knobs.get("PLANT_BRINE_DISPOSAL_COST_USD_M3", 0.15)))
        capex_samples      = _np.full(n_samples, float(base_knobs.get("PLANT_CAPEX_PER_M2", 50.0)))
        discount_rates     = _np.full(n_samples, float(base_knobs.get("PLANT_DISCOUNT_RATE", 0.05)))
        capacity_factors   = _np.full(n_samples, float(base_knobs.get("PLANT_CAPACITY_FACTOR", 0.913)))
        membrane_lifetimes = _np.full(n_samples, 10.0)

    # Pre-read harm_redux inputs (for loss model)
    c_h_in_mol = float(base_knobs.get("C_H_IN_MOL_L", 1.15))
    sw_mol     = float(getattr(spec, "plot_seawater_mol_l", 0.600) or 0.600)

    # ── Run model for each sample ──────────────────────────────────────────
    print(f"[mc_tea] Running {n_samples} model evaluations ...")
    rows = []
    n_report = max(1, n_samples // 10)
    for i in range(n_samples):
        if i > 0 and i % n_report == 0:
            print(f"[mc_tea]   {i}/{n_samples} ...")
        fb   = float(fouling_betas[i])
        ep   = float(elec_prices[i])
        bd   = float(brine_disp_costs[i])
        cx   = float(capex_samples[i])
        dr   = float(discount_rates[i])
        cf   = float(capacity_factors[i])
        ml   = float(membrane_lifetimes[i])

        model_out = _mc_run_one(base_knobs, N_dep, A_dep, fb)
        ptot = model_out["P_TOTAL_W"]

        metrics = _mc_tea_metrics(ptot, ep, bd, cx, dr, npv_mode,
                                  base_knobs, N_dep, A_dep,
                                  capacity_factor=cf, membrane_lifetime_yr=ml)
        row = dict(
            SAMPLE_IDX           = i,
            ELEC_PRICE_USD_KWH   = ep,
            FOULING_BETA         = fb,
            CAPACITY_FACTOR      = cf,
            MEMBRANE_LIFETIME_YR = ml,
        )
        row.update(metrics)

        # ── Compliance + loss model (gated) ────────────────────────────
        if enable_loss:
            c_h_out = model_out["C_H_OUT"]
            c_l_out = model_out["C_L_OUT"]
            baseline = max(c_h_in_mol - sw_mol, 0.0)
            if baseline == 0.0:
                harm_redux = 0.0
            else:
                worst = max(max(c_h_out - sw_mol, 0.0),
                            max(c_l_out - sw_mol, 0.0))
                harm_redux = (1.0 - worst / baseline) * 100.0

            harm_redux_req = float(cfg.get("harm_redux_req", 95.0))
            compliant = (harm_redux >= harm_redux_req)

            # Loss model (i.i.d. annual-state: each draw = one possible year)
            sev_day = float(rng.triangular(
                float(cfg.get("sev_day_usd_min",  5_000.0)),
                float(cfg.get("sev_day_usd_mode", 15_000.0)),
                float(cfg.get("sev_day_usd_max",  50_000.0))))

            p_enforce = float(cfg.get("p_enforce", 0.10))
            enforce_flag = (rng.random() < p_enforce) if not compliant else False

            retrofit_cost = float(rng.triangular(
                float(cfg.get("retrofit_cost_usd_min",  5_000_000.0)),
                float(cfg.get("retrofit_cost_usd_mode", 10_000_000.0)),
                float(cfg.get("retrofit_cost_usd_max",  20_000_000.0)))) \
                if enforce_flag else 0.0

            nc_days = 0.0 if compliant else 365.0
            annual_loss = nc_days * sev_day + retrofit_cost

            row.update(dict(
                HARM_REDUX_PCT    = harm_redux,
                COMPLIANT         = compliant,
                NC_DAYS           = nc_days,
                SEV_DAY_USD       = sev_day,
                ENFORCE_FLAG      = enforce_flag,
                RETROFIT_COST_USD = retrofit_cost,
                ANNUAL_LOSS_USD   = annual_loss,
            ))

        rows.append(row)

    mc_df = _pd.DataFrame(rows)
    mc_df.to_parquet(run_dir / "mc_tea.parquet", index=False)
    print(f"[mc_tea] mc_tea.parquet written ({len(mc_df)} rows)")

    npv  = mc_df["NPV_USD"].values
    lcoe = mc_df["LCOE_USD_KWH"].values
    ep_s = mc_df["ELEC_PRICE_USD_KWH"].values
    fb_s = mc_df["FOULING_BETA"].values

    def _save(fig, name, dpi=100):
        fig.savefig(run_dir / name, dpi=dpi, bbox_inches="tight")
        plt.close(fig)
        print(f"[mc_tea]   saved {name}")

    p5, p25, p50, p75, p95 = _np.percentile(npv, [5, 25, 50, 75, 95])
    pct_positive = 100 * _np.mean(npv > 0)

    # ── Fig A: NPV distribution histogram ─────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(npv / 1e6, bins=60, color=color, alpha=0.75, edgecolor="white", linewidth=0.4)
    for pct_val, pct_lbl, pct_col in [
        (p5,  "P5",  "#A2142F"),
        (p25, "P25", "#EDB120"),
        (p50, "P50", "#0072BD"),
        (p75, "P75", "#EDB120"),
        (p95, "P95", "#A2142F"),
    ]:
        ax.axvline(pct_val / 1e6, color=pct_col, lw=1.5, ls="--",
                   label=f"{pct_lbl}: ${pct_val/1e6:.2f}M")
    ax.axvline(0, color="black", lw=1.0, ls=":")
    ax.set_xlabel("NPV (M$)", fontsize=11)
    ax.set_ylabel("Sample count", fontsize=11)
    ax.set_title(
        f"{label}\nMonte Carlo TEA — NPV Distribution  "
        f"(N={N_dep}, A={A_dep:.4g} m², {n_samples:,} samples)\n"
        f"P50 = ${p50/1e6:.2f}M  |  P(NPV>0) = {pct_positive:.1f}%",
        fontsize=10, fontweight="bold"
    )
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    _save(fig, "mc_fig_A_npv_distribution.png")

    # ── Fig B: Tornado / one-at-a-time sensitivity ─────────────────────────
    # Compute NPV swing for each axis: hold other at median, sweep lo→hi
    fig, ax = plt.subplots(figsize=(8, 4 + (1.5 if enable_loss else 0)))

    # Median values for hold-all-else-constant
    ep_med = float(_np.median(ep_s))
    fb_med = float(_np.median(fb_s))
    bd_med = float(_np.median(mc_df["BRINE_DISP_COST_M3"].values))
    cx_med = float(_np.median(mc_df["CAPEX_PER_M2"].values))
    dr_med = float(_np.median(mc_df["DISCOUNT_RATE"].values))

    axes_info = [
        ("Electricity price\n($/kWh)",
         "elec_price", _np.percentile(ep_s, 10), _np.percentile(ep_s, 90)),
        ("Fouling severity\n(FOULING_BETA)",
         "fouling_beta", fb_lo, fb_hi),
    ]
    if enable_loss:
        bd_s  = mc_df["BRINE_DISP_COST_M3"].values
        cx_s  = mc_df["CAPEX_PER_M2"].values
        dr_s  = mc_df["DISCOUNT_RATE"].values
        cf_s  = mc_df["CAPACITY_FACTOR"].values if "CAPACITY_FACTOR" in mc_df.columns \
                else _np.full(len(mc_df), 0.913)
        ml_s  = mc_df["MEMBRANE_LIFETIME_YR"].values if "MEMBRANE_LIFETIME_YR" in mc_df.columns \
                else _np.full(len(mc_df), 10.0)
        axes_info.extend([
            ("Brine disposal cost\n($/m³)",
             "brine_disp_cost", _np.percentile(bd_s, 10), _np.percentile(bd_s, 90)),
            ("Capex\n($/m²)",
             "capex_per_m2", _np.percentile(cx_s, 10), _np.percentile(cx_s, 90)),
            ("Discount rate",
             "discount_rate", _np.percentile(dr_s, 10), _np.percentile(dr_s, 90)),
            ("Capacity factor",
             "capacity_factor", _np.percentile(cf_s, 10), _np.percentile(cf_s, 90)),
            ("Membrane lifetime\n(yr)",
             "membrane_lifetime_yr", _np.percentile(ml_s, 10), _np.percentile(ml_s, 90)),
        ])

    base_npv = float(_np.median(npv))
    tornado_rows = []
    for axis_label, axis_key, lo_val, hi_val in axes_info:
        # Build kwargs for lo and hi: all at median except this axis
        cf_med = float(_np.median(cf_s)) if enable_loss else 0.913
        ml_med = float(_np.median(ml_s)) if enable_loss else 10.0

        def _tornado_npv(ep, fb, bd, cx, dr, cf=None, ml=None):
            out = _mc_run_one(base_knobs, N_dep, A_dep, fb)
            return _mc_tea_metrics(
                out["P_TOTAL_W"], ep, bd, cx, dr, npv_mode,
                base_knobs, N_dep, A_dep,
                capacity_factor=cf, membrane_lifetime_yr=ml)["NPV_USD"]

        kw = dict(ep=ep_med, fb=fb_med, bd=bd_med, cx=cx_med, dr=dr_med,
                  cf=cf_med, ml=ml_med)
        kw_lo = dict(kw); kw_hi = dict(kw)
        _map = {"elec_price": "ep", "fouling_beta": "fb",
                "brine_disp_cost": "bd", "capex_per_m2": "cx", "discount_rate": "dr",
                "capacity_factor": "cf", "membrane_lifetime_yr": "ml"}
        kw_lo[_map[axis_key]] = float(lo_val)
        kw_hi[_map[axis_key]] = float(hi_val)
        npv_lo = _tornado_npv(**kw_lo)
        npv_hi = _tornado_npv(**kw_hi)
        tornado_rows.append((axis_label, npv_lo / 1e6, npv_hi / 1e6))

    # Sort by swing width
    tornado_rows.sort(key=lambda r: abs(r[2] - r[1]))
    y_pos = range(len(tornado_rows))
    base_m = base_npv / 1e6
    for yi, (lbl, lo_npv, hi_npv) in zip(y_pos, tornado_rows):
        lo_c = "#A2142F" if lo_npv < base_m else "#77AC30"
        hi_c = "#77AC30" if hi_npv > base_m else "#A2142F"
        ax.barh(yi, hi_npv - base_m, left=base_m, color=hi_c, alpha=0.8, height=0.5)
        ax.barh(yi, lo_npv - base_m, left=base_m, color=lo_c, alpha=0.8, height=0.5)
        ax.text(hi_npv + 0.02, yi, f"+${hi_npv - base_m:+.2f}M", va="center", fontsize=8)
        ax.text(lo_npv - 0.02, yi, f"${lo_npv - base_m:+.2f}M", va="center",
                ha="right", fontsize=8)
    ax.axvline(base_m, color="black", lw=1.2, ls="-")
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels([r[0] for r in tornado_rows], fontsize=9)
    ax.set_xlabel("NPV (M$)", fontsize=10)
    ax.set_title(
        f"{label}\nSensitivity Analysis (Tornado) — NPV swing at P10/P90\n"
        f"Baseline NPV = ${base_m:.2f}M",
        fontsize=10, fontweight="bold"
    )
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    _save(fig, "mc_fig_B_sensitivity_tornado.png")

    # ── Fig C: Joint scatter — elec_price vs FOULING_BETA, colored by NPV ──
    fig, ax = plt.subplots(figsize=(8, 5.5))
    sc = ax.scatter(ep_s, fb_s, c=npv / 1e6, cmap="RdYlGn",
                    s=8, alpha=0.6, linewidths=0, rasterized=True)
    cb = fig.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label("NPV (M$)", fontsize=9)
    ax.set_xlabel("Electricity price ($/kWh)", fontsize=10)
    ax.set_ylabel("Fouling severity (FOULING_BETA)", fontsize=10)
    # Mark P50 operating point
    ax.axvline(ep_mean, color="navy",    lw=1.3, ls="--", alpha=0.7, label=f"Mean price ${ep_mean:.2f}/kWh")
    ax.axhline(_np.median(fb_s), color="darkgreen", lw=1.3, ls="--", alpha=0.7,
               label=f"Median β={_np.median(fb_s):.2f}")
    ax.legend(fontsize=8, loc="upper right")
    ax.set_title(
        f"{label}\nMonte Carlo Joint Distribution — NPV colored surface\n"
        f"(N={N_dep}, A={A_dep:.4g} m², {n_samples:,} samples)",
        fontsize=10, fontweight="bold"
    )
    ax.grid(alpha=0.18)
    fig.tight_layout()
    _save(fig, "mc_fig_C_joint_scatter.png", dpi=72)  # low dpi: scatter rasterizes large

    # ── Summary printout ───────────────────────────────────────────────────
    print(f"\n[mc_tea] ─── RESULTS SUMMARY ───────────────────────────────")
    print(f"[mc_tea] Deployment: N={N_dep} cell pairs, A={A_dep:.4g} m²/pair, "
          f"{N_dep*A_dep:.3g} m² total")
    print(f"[mc_tea] NPV:  P5=${p5/1e6:.2f}M  P25=${p25/1e6:.2f}M  P50=${p50/1e6:.2f}M  "
          f"P75=${p75/1e6:.2f}M  P95=${p95/1e6:.2f}M")
    print(f"[mc_tea] P(NPV > 0) = {pct_positive:.1f}%")
    lcoe_50 = float(_np.nanmedian(lcoe))
    print(f"[mc_tea] LCOE P50 = ${lcoe_50:.4f}/kWh")
    if enable_loss:
        nc_prob = 100.0 * _np.mean(mc_df["COMPLIANT"].values == False)
        el_mean = _np.mean(mc_df["ANNUAL_LOSS_USD"].values)
        el_p90  = _np.percentile(mc_df["ANNUAL_LOSS_USD"].values, 90)
        el_p95  = _np.percentile(mc_df["ANNUAL_LOSS_USD"].values, 95)
        print(f"[mc_tea] P(non-compliant) = {nc_prob:.1f}%")
        print(f"[mc_tea] E[Annual Loss] = ${el_mean/1e6:.2f}M  |  "
              f"P90=${el_p90/1e6:.2f}M  |  P95=${el_p95/1e6:.2f}M")
    print(f"[mc_tea] ────────────────────────────────────────────────────\n")

    # ── Fig D: Annual loss distribution (loss model only) ────────────────
    if enable_loss:
        fig, ax = plt.subplots(figsize=(9, 5.5))
        loss_mit = mc_df["ANNUAL_LOSS_USD"].values / 1e6

        # Unmitigated baseline: always non-compliant (0% harm redux)
        rng_unmit = _np.random.default_rng(seed + 999)
        sev_unmit = rng_unmit.triangular(
            float(cfg.get("sev_day_usd_min",  5_000.0)),
            float(cfg.get("sev_day_usd_mode", 15_000.0)),
            float(cfg.get("sev_day_usd_max",  50_000.0)), n_samples)
        enf_unmit = rng_unmit.random(n_samples) < float(cfg.get("p_enforce", 0.10))
        ret_unmit = _np.where(enf_unmit, rng_unmit.triangular(
            float(cfg.get("retrofit_cost_usd_min",  5_000_000.0)),
            float(cfg.get("retrofit_cost_usd_mode", 10_000_000.0)),
            float(cfg.get("retrofit_cost_usd_max",  20_000_000.0)), n_samples), 0.0)
        loss_unmit = (365.0 * sev_unmit + ret_unmit) / 1e6

        hi_edge = max(_np.percentile(loss_unmit, 99),
                      _np.percentile(loss_mit, 99)) * 1.1
        bins = _np.linspace(0, hi_edge, 50)
        ax.hist(loss_unmit, bins=bins, alpha=0.5, color="#A2142F",
                edgecolor="white", linewidth=0.3,
                label=f"Unmitigated  E[L]=${_np.mean(loss_unmit):.1f}M")
        ax.hist(loss_mit, bins=bins, alpha=0.6, color="#0072BD",
                edgecolor="white", linewidth=0.3,
                label=f"Mitigated  E[L]=${_np.mean(loss_mit):.1f}M")
        for pv, pl, pc in [
            (_np.percentile(loss_mit, 90), "P90 mit", "#EDB120"),
            (_np.percentile(loss_mit, 95), "P95 mit", "#A2142F"),
        ]:
            ax.axvline(pv, color=pc, lw=1.3, ls="--", label=f"{pl}: ${pv:.1f}M")
        ax.set_xlabel("Annual Loss (M$)", fontsize=11)
        ax.set_ylabel("Sample count", fontsize=11)
        ax.set_title(
            f"{label}\nAnnual Loss Distribution — Unmitigated vs RED-Mitigated\n"
            f"(harm_redux_req={float(cfg.get('harm_redux_req', 95.0)):.0f}%, "
            f"N={N_dep}, A={A_dep:.4g} m², {n_samples:,} samples)",
            fontsize=10, fontweight="bold")
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(axis="y", alpha=0.25)
        fig.tight_layout()
        _save(fig, "mc_fig_D_annual_loss.png")

    # ── Fig E: Break-even contour (loss model only) ─────────────────────
    if enable_loss:
        fig, ax = plt.subplots(figsize=(8, 5.5))
        bd_vals = mc_df["BRINE_DISP_COST_M3"].values
        dr_vals = mc_df["DISCOUNT_RATE"].values
        npv_vals = mc_df["NPV_USD"].values / 1e6

        n_bins = 20
        bd_edges = _np.linspace(bd_vals.min(), bd_vals.max(), n_bins + 1)
        dr_edges = _np.linspace(dr_vals.min(), dr_vals.max(), n_bins + 1)
        bd_centers = 0.5 * (bd_edges[:-1] + bd_edges[1:])
        dr_centers = 0.5 * (dr_edges[:-1] + dr_edges[1:])

        npv_grid = _np.full((n_bins, n_bins), _np.nan)
        bd_idx = _np.clip(_np.digitize(bd_vals, bd_edges) - 1, 0, n_bins - 1)
        dr_idx = _np.clip(_np.digitize(dr_vals, dr_edges) - 1, 0, n_bins - 1)
        from collections import defaultdict as _ddict
        _grid_bins = _ddict(list)
        for k in range(len(npv_vals)):
            _grid_bins[(bd_idx[k], dr_idx[k])].append(npv_vals[k])
        for (bi, di), vals in _grid_bins.items():
            npv_grid[di, bi] = _np.mean(vals)

        im = ax.pcolormesh(bd_edges, dr_edges, npv_grid,
                           cmap="RdYlGn", shading="flat")
        cb = fig.colorbar(im, ax=ax, pad=0.02)
        cb.set_label("Mean NPV (M$)", fontsize=9)
        try:
            cs = ax.contour(bd_centers, dr_centers, npv_grid,
                            levels=[0], colors=["black"], linewidths=[2])
            ax.clabel(cs, fmt="NPV=0", fontsize=8)
        except Exception:
            pass
        ax.set_xlabel("Brine Disposal Cost ($/m³)", fontsize=11)
        ax.set_ylabel("Discount Rate", fontsize=11)
        ax.set_title(
            f"{label}\nBreak-Even Surface — NPV vs Economic Parameters\n"
            f"(N={N_dep}, A={A_dep:.4g} m², {n_samples:,} samples, "
            f"npv_mode={npv_mode})",
            fontsize=10, fontweight="bold")
        ax.grid(alpha=0.18)
        fig.tight_layout()
        _save(fig, "mc_fig_E_breakeven.png")

    # ── Generate dedicated MC-TEA PDF ─────────────────────────────────────────
    # Called directly here (not relying on run_experiment's post-hook call) for
    # two reasons:
    #   1. Avoids matplotlib backend conflicts — run_mc_tea already used Agg to
    #      save figures; calling generate_experiment_pdf here while Agg is active
    #      is safe. The parent call in run_experiment would switch backends again.
    #   2. Lets us pass extra_png_dirs to link the parent brine experiment's
    #      figures, producing one combined PDF per run rather than two orphans.
    # Close any residual figures and ensure clean state before PDF generation
    plt.close("all")

    try:
        # Find most recent run of the baseline (parent brine) experiment
        _extra_dirs = []
        _baseline_id = getattr(spec, "baseline_id", None)
        if _baseline_id:
            _parent_root = run_dir.parent.parent / _baseline_id
            if _parent_root.exists():
                _parent_runs = sorted(
                    [p for p in _parent_root.iterdir() if p.is_dir()],
                    key=lambda p: p.name
                )
                if _parent_runs:
                    _extra_dirs = [_parent_runs[-1]]
                    print(f"[mc_tea] Linking parent brine run: {_parent_runs[-1].name}")
        _pdf_path = generate_experiment_pdf(run_dir, extra_png_dirs=_extra_dirs)
        print(f"[mc_tea] Combined PDF written: {_pdf_path}")
    except Exception as _pdf_err:
        print(f"[mc_tea] Warning: PDF generation failed (non-fatal): {_pdf_err}")
        import traceback; traceback.print_exc()


print("Monte Carlo TEA engine loaded: run_mc_tea(run_dir, spec)")
print("Enable with: ExperimentSpec(..., run_monte_carlo=True, mc_config={...})")
print("  Legacy mode: omit npv_mode/enable_loss_model (defaults preserve original behavior)")
print("  MTFC mode:   mc_config={..., npv_mode: 'discounted', enable_loss_model: True}")

# ─────────────────────────────────────────────────────────────────────────────
# run_mc_tea_grid: single-stack NPV feasibility map over (N, A) grid
# Reads harm_sweep.parquet from an existing BRINE-* run; evaluates MC-TEA
# at each (N,A) point using the upgraded 7-axis engine.
# ─────────────────────────────────────────────────────────────────────────────
def run_mc_tea_grid(harm_sweep_path, spec, out_dir, n_samples=2000, seed=99):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    out_dir = _Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    cfg = dict(_MC_DEFAULTS)
    cfg.update(spec.mc_config or {})
    npv_mode = str(cfg.get("npv_mode", "discounted"))

    harm_df = _pd.read_parquet(harm_sweep_path)
    valid = harm_df[~harm_df.get("inv", _pd.Series(False, index=harm_df.index))]
    if len(valid) == 0:
        valid = harm_df
    grid_points = valid[["N_CELL_PAIRS","A_MEM_M2"]].drop_duplicates().values

    base_knobs = {k: f.default for k, f in DesignInputs.__dataclass_fields__.items()}
    base_knobs.update(spec.base_overrides or {})
    rng = _np.random.default_rng(seed)

    ep_mean = float(base_knobs.get("PLANT_ELEC_PRICE_USD_KWH", 0.22))
    ep_sfrac = float(cfg.get("elec_price_std_frac", 0.30))
    ep_sig = _np.sqrt(_np.log(1 + ep_sfrac**2))
    ep_mu  = _np.log(ep_mean) - 0.5 * ep_sig**2
    elec_prices      = rng.lognormal(ep_mu, ep_sig, n_samples)
    brine_disp_costs = rng.uniform(cfg.get("brine_disp_cost_lo",0.04), cfg.get("brine_disp_cost_hi",0.25), n_samples)
    capex_samples    = rng.triangular(cfg.get("capex_per_m2_min",30.), cfg.get("capex_per_m2_mode",50.), cfg.get("capex_per_m2_max",100.), n_samples)
    discount_rates   = rng.uniform(cfg.get("discount_rate_lo",0.03), cfg.get("discount_rate_hi",0.08), n_samples)
    capacity_factors = rng.uniform(cfg.get("capacity_factor_lo",0.80), cfg.get("capacity_factor_hi",0.95), n_samples)
    mem_lifetimes    = rng.uniform(cfg.get("membrane_lifetime_lo",5.), cfg.get("membrane_lifetime_hi",15.), n_samples)
    fouling_med = 0.5*(float(cfg.get("fouling_beta_lo",0.0))+float(cfg.get("fouling_beta_hi",0.8)))

    sw_mol = float(getattr(spec,"plot_seawater_mol_l",0.600) or 0.600)
    c_h_in = float(base_knobs.get("C_H_IN_MOL_L",1.15))
    harm_redux_req = float(cfg.get("harm_redux_req",90.0))

    rows = []
    print(f"[mc_grid] {len(grid_points)} points × {n_samples} samples ...")
    for idx,(N,A) in enumerate(grid_points):
        N=int(N); A=float(A)
        out = _mc_run_one(base_knobs, N, A, fouling_med)
        baseline = max(c_h_in-sw_mol, 0.0)
        worst = max(max(out["C_H_OUT"]-sw_mol,0.0), max(out["C_L_OUT"]-sw_mol,0.0))
        harm_redux = (1.-worst/baseline)*100. if baseline>0 else 0.
        p_dens = out["P_TOTAL_W"]/(N*A) if N*A>0 else 0.
        npvs = _np.array([_mc_tea_metrics(
            out["P_TOTAL_W"], float(elec_prices[j]), float(brine_disp_costs[j]),
            float(capex_samples[j]), float(discount_rates[j]), npv_mode,
            base_knobs, N, A,
            capacity_factor=float(capacity_factors[j]),
            membrane_lifetime_yr=float(mem_lifetimes[j]))["NPV_USD"]
            for j in range(n_samples)])
        rows.append(dict(
            N_CELL_PAIRS=N, A_MEM_M2=A, P_DENS_W_M2=p_dens,
            HARM_REDUX=harm_redux,
            NPV_P50_M=float(_np.median(npvs))/1e6,
            P_NPV_POS=float(_np.mean(npvs>0))*100.,
            FEASIBLE=(float(_np.median(npvs))>0) and (harm_redux>=harm_redux_req),
        ))
        if (idx+1)%10==0:
            print(f"[mc_grid]   {idx+1}/{len(grid_points)}")

    grid_df = _pd.DataFrame(rows)
    grid_df.to_parquet(out_dir/"mc_grid_table.parquet", index=False)

    # Feasibility heatmap
    N_vals=sorted(grid_df["N_CELL_PAIRS"].unique()); A_vals=sorted(grid_df["A_MEM_M2"].unique())
    Z_prob=_np.full((len(A_vals),len(N_vals)),_np.nan)
    Z_npv =_np.full((len(A_vals),len(N_vals)),_np.nan)
    for _,r in grid_df.iterrows():
        ni=N_vals.index(int(r["N_CELL_PAIRS"])); ai=A_vals.index(float(r["A_MEM_M2"]))
        Z_prob[ai,ni]=r["P_NPV_POS"]; Z_npv[ai,ni]=r["NPV_P50_M"]
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    for ax,Z,title,fmt in [(axes[0],Z_prob,"P(NPV>0) [%]",".0f"),(axes[1],Z_npv,"Median NPV [M$]",".1f")]:
        im=ax.pcolormesh(_np.arange(len(N_vals)+1)-.5,_np.arange(len(A_vals)+1)-.5,
                         Z,cmap="RdYlGn",shading="flat",vmin=(0 if fmt==".0f" else None))
        fig.colorbar(im,ax=ax,pad=0.02)
        for ai in range(len(A_vals)):
            for ni in range(len(N_vals)):
                if not _np.isnan(Z[ai,ni]):
                    ax.text(ni,ai,f"{Z[ai,ni]:{fmt}}",ha="center",va="center",fontsize=7)
                    sub=grid_df[(grid_df["N_CELL_PAIRS"]==N_vals[ni])&(grid_df["A_MEM_M2"]==A_vals[ai])]
                    if len(sub)>0 and sub.iloc[0]["FEASIBLE"]:
                        ax.add_patch(plt.Rectangle((ni-.5,ai-.5),1,1,fill=False,edgecolor="black",lw=2))
        ax.set_xticks(range(len(N_vals))); ax.set_xticklabels([str(n) for n in N_vals],fontsize=8)
        ax.set_yticks(range(len(A_vals))); ax.set_yticklabels([f"{a:.3f}" for a in A_vals],fontsize=8)
        ax.set_xlabel("N cell pairs",fontsize=9); ax.set_ylabel("A_mem (m²/pair)",fontsize=9)
        ax.set_title(title,fontsize=10,fontweight="bold")
    fig.suptitle(f"Single-Stack MC-TEA Grid\nFeasible = NPV_P50>0 AND harm_redux≥{harm_redux_req:.0f}% (bold)",
                 fontsize=10,fontweight="bold")
    fig.tight_layout()
    fig.savefig(out_dir/"mc_grid_fig_feasibility.png",dpi=100,bbox_inches="tight")
    plt.close(fig)
    print(f"[mc_grid] Complete. {grid_df['FEASIBLE'].sum()}/{len(grid_df)} feasible.")
    return grid_df

print("run_mc_tea_grid loaded")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cascade Optimization Sweep
#
# run_cascade_sweep(spec) evaluates all (S, N_per_stage, FOULING_BETA) triplets
# and quantifies two metrics:
#
#   CASCADE_GAIN  — extra power a cascade of S stages delivers compared to an
#   equivalent single stack with the same total cell pairs (S×N) and membrane
#   area (S×N×A), at the same feed conditions and flow rates.
#
#   FOULING_ADVANTAGE — how many percentage points less fouling penalty the
#   cascade incurs versus the single stack at each fouling severity level.
#
# Physical basis for the cascade advantage:
#   Each stage receives a fresh low-salinity inlet, resetting the Nernst
#   potential.  A single stack with the same total N sees a progressively
#   depleted gradient; later cell pairs contribute diminishing EMF.  Cascading
#   preserves the gradient at every stage inlet and recovers the power the
#   single stack leaves on the table.
#
#   Fouling amplifies this advantage because CDF fouling is inlet-heavy:
#   the first cell pairs of each stage face the highest local C_H and therefore
#   the highest divalent resistance penalty.  Fewer cell pairs per stage means
#   less cumulative time in the high-fouling regime across the stack.
#
# Comparison convention (important for fairness):
#   Single-stack equivalent: N_CELL_PAIRS = S × N_per_stage, same A_MEM_M2,
#   same inlet concentrations, same flow rates.  Both cascade and single stack
#   are evaluated in max_power mode (oracle impedance match).
#
# Flow-rate convention:
#   Fixed total flow rate regardless of stage count.  Q_H and Q_L are
#   properties of the brine source.  Each stage receives the full Q_H/Q_L;
#   salt depletion accumulates across stages through the outlet→inlet
#   concentration linkage.
#
# Clean-baseline caching:
#   The clean-condition (FOULING_BETA=0) power is the same for all beta values
#   at a given (S, N) geometry.  run_cascade_sweep computes it once per (S, N)
#   pair and reuses it across beta values, reducing total model calls by ~40%.
#
# Architecture:
#   Standalone function — does not use ExperimentSpec / run_experiment().
#   Calls model_1d_axial_series_current() directly per stage.
#   Saves cascade.parquet (tidy DF, one row per triplet) plus four
#   publication-quality PNG figures into runs/CASCADE-OPTIMIZATION/<timestamp>/.
# ─────────────────────────────────────────────────────────────────────────────

from __future__ import annotations
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


# ── Spec ─────────────────────────────────────────────────────────────────────

@dataclass
class CascadeSpec:
    """Configuration for a cascade vs single-stack optimization sweep.

    All physics parameters not listed here are taken from DesignInputs defaults
    overlaid with base_overrides — same pattern as ExperimentSpec.

    Attributes
    ----------
    experiment_id : str
        Used as the run directory name under runs/.
    description : str
        Human-readable summary written into the JSON envelope.
    base_overrides : dict
        Knob overrides applied to every run (feed conditions, membrane params,
        flow rates, etc.).  MODE and N_CELL_PAIRS are set internally; do not
        include them here.
    s_vals : list[int]
        Number of cascade stages to evaluate.  s=1 is a single stack with
        N=n_per_stage cell pairs (not the full equivalent — that is computed
        separately for every (s, n) pair).
    n_vals : list[int]
        Cell pairs per stage.  Total cell pairs for the equivalent single
        stack = s × n.
    fouling_beta_vals : list[float]
        FOULING_BETA values to sweep.  0.0 = clean baseline.
    a_mem_m2 : float
        Membrane area per cell pair [m²], fixed across all configurations.
        Uniform area is the defensible engineering constraint: a single
        membrane procurement order covers all stages.
    n_seg : int
        Axial segments per stage (higher = more spatial resolution).
    """
    experiment_id:     str           = "CASCADE-OPTIMIZATION"
    description:       str           = (
        "Cascade vs single-stack optimization sweep.  "
        "Evaluates CASCADE_GAIN_PCT and FOULING_ADVANTAGE_PP over "
        "(S, N_per_stage, FOULING_BETA) at fixed A_MEM_M2 and "
        "Carlsbad brine conditions (C_H=1.15 mol/L)."
    )
    base_overrides:    Dict[str, Any] = field(default_factory=lambda: {
        "C_H_IN_MOL_L":       1.15,    # Carlsbad Poseidon brine [mol/L]
        "C_L_IN_MOL_L":       0.02,    # low-salinity diluate [mol/L]
        "Q_H_M3_S":           5.0e-6,
        "Q_L_M3_S":           5.0e-6,
        "GAP_M":              2.0e-4,
        "DIVALENT_FRACTION":  0.20,    # typical seawater composition
        "FOULING_MODEL":      "none",  # phenomenological biofilm OFF; CDF via FOULING_BETA
        "FOULING_C_REF_MOL_L": 0.5,
        "FOULING_N_EXP":      0.5,
    })
    s_vals:            List[int]   = field(default_factory=lambda: [1, 2, 3, 4, 5, 6, 8, 10])
    n_vals:            List[int]   = field(default_factory=lambda: [3, 5, 10, 20, 40])
    fouling_beta_vals: List[float] = field(default_factory=lambda: [0.0, 0.2, 0.4, 0.6, 0.8])
    a_mem_m2:          float       = 0.05
    n_seg:             int         = 40
    fresh_dilute:      bool        = True
    p_overhead_per_stage_w: float  = 0.0    # per-stage fixed overhead [W] (electrodes + pumping)
    n_total_max:           int     = 0       # max total cell pairs (0 = unconstrained)
    n_total_exact:         int     = 0       # if >0, only keep geometries where S*N == this value   # True = replenish dilute feed each stage; False = serial pass-through


# ── Low-level helpers ─────────────────────────────────────────────────────────

def _run_single_stage(
    base_knobs:  dict,
    n_cell_pairs: int,
    a_mem_m2:    float,
    n_seg:       int,
    c_h_in:      float,
    c_l_in:      float,
) -> Dict[str, Any]:
    """Build knobs for one RED stage and call the series-current solver."""
    k = dict(base_knobs)
    k["N_CELL_PAIRS"] = int(n_cell_pairs)
    k["A_MEM_M2"]     = float(a_mem_m2)
    k["N_SEG"]        = int(n_seg)
    k["C_H_IN_MOL_L"] = float(c_h_in)
    k["C_L_IN_MOL_L"] = float(c_l_in)
    k["MODE"]         = "max_power"
    p = knobs_to_design_inputs(k)
    return model_1d_axial_series_current(p, mode="max_power")


def _outlet_concs(res: Dict[str, Any]) -> tuple[float, float]:
    """Return (C_H_out, C_L_out) from the last axial segment of a stage result."""
    seg = res["per_segment"]
    return float(seg["C_H_OUT_MOL_L"][-1]), float(seg["C_L_OUT_MOL_L"][-1])


# ── Cascade runner ────────────────────────────────────────────────────────────

def run_cascade(
    base_knobs:    dict,
    n_stages:      int,
    n_per_stage:   int,
    a_mem_m2:      float,
    n_seg:         int,
    p_clean_total: Optional[float] = None,
    fresh_dilute:  bool = True,
    p_overhead_per_stage_w: float = 0.0,
) -> Dict[str, Any]:
    """Run a cascade of n_stages RED stages and return aggregate metrics.

    Parameters
    ----------
    p_clean_total : float, optional
        Pre-computed clean-condition (FOULING_BETA=0) cascade power [W].
        When supplied the internal clean re-run is skipped, saving n_stages
        model calls.  Pass the P_CLEAN_CASCADE_W value from a prior call
        to this function with FOULING_BETA=0 for the same geometry.

    Returns
    -------
    dict
        P_CASCADE_W               total delivered power [W]
        P_DENS_CASCADE_W_M2       power density over total membrane area [W/m²]
        P_CLEAN_CASCADE_W         clean-condition cascade power [W]
        FOUL_PENALTY_CASCADE_FRAC (P_clean - P_fouled) / P_clean  [-]
        C_H_OUT_FINAL             outlet C_H of the last stage [mol/L]
        C_L_OUT_FINAL             outlet C_L of the last stage [mol/L]
        any_inversion             True if any stage flagged a concentration inversion
        converged                 True if all stages converged
        per_stage                 list of per-stage result dicts (full model schema)
    """
    c_h = float(base_knobs.get("C_H_IN_MOL_L", 1.15))
    c_l = float(base_knobs.get("C_L_IN_MOL_L", 0.02))

    per_stage: List[Dict[str, Any]] = []
    p_total  = 0.0
    any_inv  = False
    all_conv = True

    for _ in range(n_stages):
        res = _run_single_stage(base_knobs, n_per_stage, a_mem_m2, n_seg, c_h, c_l)
        per_stage.append(res)
        p_total  += float(res["P_TOTAL_W"])
        any_inv   = any_inv or bool(res["any_inversion"])
        all_conv  = all_conv and bool(res["converged"])
        c_h, c_l = _outlet_concs(res)
        if fresh_dilute:
            c_l = float(base_knobs.get("C_L_IN_MOL_L", 0.02))

    total_area = n_stages * n_per_stage * a_mem_m2  # m²

    # ── Per-stage parasitic losses (pumping + electrodes) ──────────────
    # Pumping: laminar slit flow, ΔP = 12·μ·L·v / gap²
    #   v = Q / (n_per_stage · width · gap)   [per-compartment velocity]
    #   L = width = √A_mem                    [square membrane assumption]
    #   Two streams (H + L) per stage, each traverses one channel pass.
    # Electrodes: ~0.1 V total loss per stage (Veerman et al. 2010)
    import math as _math
    _mu   = 1.0e-3       # Pa·s (water viscosity at ~25°C)
    _gap  = float(base_knobs.get("GAP_M", 2.0e-4))
    _Q_H  = float(base_knobs.get("Q_H_M3_S", 5.0e-6))
    _Q_L  = float(base_knobs.get("Q_L_M3_S", 5.0e-6))
    _width = _math.sqrt(a_mem_m2)             # membrane side length [m]
    _L     = _width                            # flow path length [m]

    # Per-compartment velocity for each stream
    _v_H = _Q_H / max(n_per_stage * _width * _gap, 1e-30)
    _v_L = _Q_L / max(n_per_stage * _width * _gap, 1e-30)

    # Pressure drop per stream (Hagen-Poiseuille for slit)
    _dP_H = 12.0 * _mu * _L * _v_H / (_gap ** 2)
    _dP_L = 12.0 * _mu * _L * _v_L / (_gap ** 2)

    # Pumping power per stage [W]
    _P_pump_per_stage = _dP_H * _Q_H + _dP_L * _Q_L

    # Electrode loss per stage [W]: V_electrode × I_stage
    # Use average current across stages; I ≈ V_oc / (2·R_int) at max power
    _V_electrode = 0.1   # V (Veerman et al. 2010, recirculating redox couple)
    _I_avg = sum(float(st.get("I_TOTAL_A", 0.0)) for st in per_stage) / max(n_stages, 1)
    _P_elec_per_stage = _V_electrode * _I_avg

    _P_parasitic_per_stage = _P_pump_per_stage + _P_elec_per_stage
    _P_parasitic_total = n_stages * _P_parasitic_per_stage

    # Apply parasitic losses (plus any user-specified fixed overhead)
    p_total = max(0.0, p_total - _P_parasitic_total - n_stages * p_overhead_per_stage_w)

    # Compute clean baseline if not supplied by caller
    if p_clean_total is None:
        clean_knobs = dict(base_knobs)
        clean_knobs["FOULING_BETA"] = 0.0
        p_clean_total = 0.0
        c_h_cl = float(clean_knobs.get("C_H_IN_MOL_L", 1.15))
        c_l_cl = float(clean_knobs.get("C_L_IN_MOL_L", 0.02))
        for _ in range(n_stages):
            res_cl = _run_single_stage(clean_knobs, n_per_stage, a_mem_m2,
                                       n_seg, c_h_cl, c_l_cl)
            p_clean_total += float(res_cl["P_TOTAL_W"])
            c_h_cl, c_l_cl = _outlet_concs(res_cl)
            if fresh_dilute:
                c_l_cl = float(clean_knobs.get("C_L_IN_MOL_L", 0.02))

    # Apply same parasitic model to clean baseline
    # Pumping is identical (same geometry, same flow rates)
    # Electrode loss: use average current from fouled run as approximation
    #   (clean current is slightly higher, so this slightly underestimates clean loss)
    p_clean_total = max(0.0, p_clean_total - _P_parasitic_total - n_stages * p_overhead_per_stage_w)

    foul_penalty_frac = (
        (p_clean_total - p_total) / max(p_clean_total, 1e-30)
        if p_clean_total > 0 else 0.0
    )

    return {
        "P_CASCADE_W":               p_total,
        "P_DENS_CASCADE_W_M2":       p_total / max(total_area, 1e-30),
        "P_CLEAN_CASCADE_W":         float(p_clean_total),
        "FOUL_PENALTY_CASCADE_FRAC": float(foul_penalty_frac),
        "C_H_OUT_FINAL":             float(c_h),
        "C_L_OUT_FINAL":             float(c_l),
        "any_inversion":             bool(any_inv),
        "converged":                 bool(all_conv),
        "per_stage":                 per_stage,
    }


# ── Single-stack equivalent runner ───────────────────────────────────────────

def run_single_equivalent(
    base_knobs:     dict,
    n_total:        int,
    a_mem_m2:       float,
    n_seg:          int,
    p_clean_single: Optional[float] = None,
) -> Dict[str, Any]:
    """Run the equivalent single stack (N_total = S×N, same A and feed).

    Parameters
    ----------
    p_clean_single : float, optional
        Pre-computed clean-condition single-stack power [W].  When supplied
        the internal clean re-run is skipped.
    """
    res      = _run_single_stage(
        base_knobs, n_total, a_mem_m2, n_seg,
        float(base_knobs.get("C_H_IN_MOL_L", 1.15)),
        float(base_knobs.get("C_L_IN_MOL_L", 0.02)),
    )
    p_single   = float(res["P_TOTAL_W"])
    total_area = n_total * a_mem_m2

    if p_clean_single is None:
        clean_knobs = dict(base_knobs)
        clean_knobs["FOULING_BETA"] = 0.0
        res_cl = _run_single_stage(
            clean_knobs, n_total, a_mem_m2, n_seg,
            float(clean_knobs.get("C_H_IN_MOL_L", 1.15)),
            float(clean_knobs.get("C_L_IN_MOL_L", 0.02)),
        )
        p_clean_single = float(res_cl["P_TOTAL_W"])

    foul_penalty_frac = (
        (p_clean_single - p_single) / max(p_clean_single, 1e-30)
        if p_clean_single > 0 else 0.0
    )

    return {
        "P_SINGLE_W":               p_single,
        "P_DENS_SINGLE_W_M2":       p_single / max(total_area, 1e-30),
        "P_CLEAN_SINGLE_W":         float(p_clean_single),
        "FOUL_PENALTY_SINGLE_FRAC": float(foul_penalty_frac),
        "any_inversion_single":     bool(res["any_inversion"]),
        "converged_single":         bool(res["converged"]),
    }


# ── Sweep driver ──────────────────────────────────────────────────────────────

def run_cascade_sweep(
    spec:       CascadeSpec,
    *,
    save:       bool = True,
    make_plots: bool = True,
    runs_root:  str  = "runs",
) -> pd.DataFrame:
    """Evaluate all (S, N_per_stage, FOULING_BETA) triplets defined in spec.

    Returns a tidy DataFrame (one row per triplet).  Saves cascade.parquet
    and four PNG figures into runs/<experiment_id>/<timestamp>/.

    Clean baselines are computed once per (S, N) geometry and reused across
    all beta values, reducing total model calls by ~40% versus naive re-runs.
    """
    base_knobs = make_base_knobs(spec.base_overrides)
    run_dir    = Path(runs_root) / spec.experiment_id / _now_tag()

    n_triplets = len(spec.s_vals) * len(spec.n_vals) * len(spec.fouling_beta_vals)
    n_geoms    = sum(1 for n in spec.n_vals for s in spec.s_vals
                       if spec.n_total_max <= 0 or s * n <= spec.n_total_max)
    print(
        f"[CASCADE] {n_triplets} triplets  "
        f"(S×{len(spec.s_vals)}, N×{len(spec.n_vals)}, β×{len(spec.fouling_beta_vals)})  "
        f"| {n_geoms} unique (S,N) geometries — clean baselines computed once each"
    )

    # ── Phase 1: pre-compute clean baselines for every (S, N) geometry ──────
    clean_baseline: Dict[tuple, Dict[str, float]] = {}
    clean_knobs = dict(base_knobs)
    clean_knobs["FOULING_BETA"] = 0.0

    print("[CASCADE] Phase 1/2: clean baselines ...")
    for n_per in spec.n_vals:
        for s in spec.s_vals:
            n_total = s * n_per
            if spec.n_total_max > 0 and n_total > spec.n_total_max:
                continue
            if spec.n_total_exact > 0 and n_total != spec.n_total_exact:
                continue

            # Clean cascade
            p_clean_casc = 0.0
            c_h = float(clean_knobs.get("C_H_IN_MOL_L", 1.15))
            c_l = float(clean_knobs.get("C_L_IN_MOL_L", 0.02))
            for _ in range(s):
                r = _run_single_stage(clean_knobs, n_per, spec.a_mem_m2,
                                      spec.n_seg, c_h, c_l)
                p_clean_casc += float(r["P_TOTAL_W"])
                c_h, c_l = _outlet_concs(r)
                if spec.fresh_dilute:
                    c_l = float(clean_knobs.get("C_L_IN_MOL_L", 0.02))

            # Clean single equivalent
            r_sing = _run_single_stage(
                clean_knobs, n_total, spec.a_mem_m2, spec.n_seg,
                float(clean_knobs.get("C_H_IN_MOL_L", 1.15)),
                float(clean_knobs.get("C_L_IN_MOL_L", 0.02)),
            )
            p_clean_sing = float(r_sing["P_TOTAL_W"])

            # Physics-based parasitic losses for clean baselines
            import math as _math
            _mu_b   = 1.0e-3
            _gap_b  = float(clean_knobs.get("GAP_M", 2.0e-4))
            _Q_H_b  = float(clean_knobs.get("Q_H_M3_S", 5.0e-6))
            _Q_L_b  = float(clean_knobs.get("Q_L_M3_S", 5.0e-6))
            _w_b    = _math.sqrt(spec.a_mem_m2)
            _v_H_b  = _Q_H_b / max(n_per * _w_b * _gap_b, 1e-30)
            _v_L_b  = _Q_L_b / max(n_per * _w_b * _gap_b, 1e-30)
            _dP_H_b = 12.0 * _mu_b * _w_b * _v_H_b / (_gap_b ** 2)
            _dP_L_b = 12.0 * _mu_b * _w_b * _v_L_b / (_gap_b ** 2)
            _P_pump_b = _dP_H_b * _Q_H_b + _dP_L_b * _Q_L_b
            # Electrode: use last stage's current as estimate
            _I_b = float(r.get("I_TOTAL_A", 0.12))
            _P_elec_b = 0.1 * _I_b
            _P_parasitic_b = s * (_P_pump_b + _P_elec_b + spec.p_overhead_per_stage_w)
            p_clean_casc = max(0.0, p_clean_casc - _P_parasitic_b)

            clean_baseline[(s, n_per)] = {
                "p_clean_cascade": p_clean_casc,
                "p_clean_single":  p_clean_sing,
            }

    print(f"[CASCADE] Phase 1/2 complete. {n_geoms} baselines cached.")

    # ── Phase 2: sweep all (S, N, beta) triplets ─────────────────────────────
    print("[CASCADE] Phase 2/2: full sweep ...")
    rows: List[Dict[str, Any]] = []
    run_idx = 0
    t0 = time.time()

    for beta in spec.fouling_beta_vals:
        beta_knobs = dict(base_knobs)
        beta_knobs["FOULING_BETA"] = float(beta)

        for n_per in spec.n_vals:
            for s in spec.s_vals:
                n_total = s * n_per
                if spec.n_total_max > 0 and n_total > spec.n_total_max:
                    continue
                if spec.n_total_exact > 0 and n_total != spec.n_total_exact:
                    continue
                cb = clean_baseline[(s, n_per)]

                # Cascade
                casc = run_cascade(
                    beta_knobs,
                    n_stages=s, n_per_stage=n_per,
                    a_mem_m2=spec.a_mem_m2, n_seg=spec.n_seg,
                    p_clean_total=cb["p_clean_cascade"],
                    fresh_dilute=spec.fresh_dilute,
                    p_overhead_per_stage_w=spec.p_overhead_per_stage_w,
                )

                # Equivalent single stack
                sing = run_single_equivalent(
                    beta_knobs,
                    n_total=n_total,
                    a_mem_m2=spec.a_mem_m2, n_seg=spec.n_seg,
                    p_clean_single=cb["p_clean_single"],
                )

                # Gain metrics
                p_casc = casc["P_CASCADE_W"]
                p_sing = sing["P_SINGLE_W"]
                gain_abs = p_casc / max(p_sing, 1e-30)
                gain_pct = (gain_abs - 1.0) * 100.0

                # Fouling advantage: positive = cascade suffers less penalty
                foul_adv_pp = (
                    sing["FOUL_PENALTY_SINGLE_FRAC"] - casc["FOUL_PENALTY_CASCADE_FRAC"]
                ) * 100.0

                row = {
                    # Sweep axes
                    "S":             int(s),
                    "N_PER_STAGE":   int(n_per),
                    "N_TOTAL":       int(n_total),
                    "FOULING_BETA":  float(beta),
                    "A_MEM_M2":      float(spec.a_mem_m2),
                    "FRESH_DILUTE":  bool(spec.fresh_dilute),

                    # Cascade outputs
                    "P_CASCADE_W":              casc["P_CASCADE_W"],
                    "P_DENS_CASCADE_W_M2":      casc["P_DENS_CASCADE_W_M2"],
                    "P_CLEAN_CASCADE_W":        casc["P_CLEAN_CASCADE_W"],
                    "FOUL_PENALTY_CASCADE_PCT": casc["FOUL_PENALTY_CASCADE_FRAC"] * 100.0,
                    "C_H_OUT_FINAL":            casc["C_H_OUT_FINAL"],
                    "C_L_OUT_FINAL":            casc["C_L_OUT_FINAL"],
                    "any_inversion":            casc["any_inversion"],
                    "converged":                casc["converged"],

                    # Single-stack outputs
                    "P_SINGLE_W":               sing["P_SINGLE_W"],
                    "P_DENS_SINGLE_W_M2":       sing["P_DENS_SINGLE_W_M2"],
                    "P_CLEAN_SINGLE_W":         sing["P_CLEAN_SINGLE_W"],
                    "FOUL_PENALTY_SINGLE_PCT":  sing["FOUL_PENALTY_SINGLE_FRAC"] * 100.0,
                    "any_inversion_single":     sing["any_inversion_single"],
                    "converged_single":         sing["converged_single"],

                    # Gain metrics
                    "CASCADE_GAIN_ABS":         float(gain_abs),
                    "CASCADE_GAIN_PCT":         float(gain_pct),
                    "FOULING_ADVANTAGE_PP":     float(foul_adv_pp),
                }
                rows.append(row)
                run_idx += 1

                if run_idx % 20 == 0 or run_idx == n_triplets:
                    elapsed = time.time() - t0
                    print(
                        f"  [{run_idx}/{n_triplets}]  "
                        f"S={s}, N={n_per}, β={beta:.1f}  "
                        f"gain={gain_pct:+.1f}%  "
                        f"foul_adv={foul_adv_pp:+.1f} pp  "
                        f"({elapsed:.1f}s)"
                    )

    df = pd.DataFrame(rows)

    if save:
        run_dir.mkdir(parents=True, exist_ok=True)
        parquet_path = run_dir / "cascade.parquet"
        df.to_parquet(parquet_path, index=False)
        print(f"[CASCADE] Saved {len(df)} rows → {parquet_path}")

    if make_plots:
        _plot_cascade_results(df, run_dir, spec)

    # ── Fouling-aware optimal-shift analysis ─────────────────────────────────
    if save and len(spec.fouling_beta_vals) > 1:
        opt_df = _compute_optimal_shift(df, run_dir, spec)
        if make_plots:
            _plot_optimal_shift(opt_df, df, run_dir, spec)

    # ── PDF report ───────────────────────────────────────────────────────────
    if save:
        _write_cascade_pdf(run_dir, spec)

    return df


# ── Figures ───────────────────────────────────────────────────────────────────

_PALETTE = {
    "gain_clean":  "#0072BD",
    "gain_fouled": "#D95319",
    "foul_adv":    "#77AC30",
}


def _plot_cascade_results(df: pd.DataFrame, run_dir: Path, spec: CascadeSpec) -> None:
    """Generate four publication-quality PNG figures for the cascade sweep."""

    run_dir.mkdir(parents=True, exist_ok=True)

    beta_clean  = 0.0
    beta_fouled = max(spec.fouling_beta_vals)

    df_clean  = df[df["FOULING_BETA"] == beta_clean].copy()
    df_fouled = df[df["FOULING_BETA"] == beta_fouled].copy()

    s_vals = sorted(df["S"].unique())
    n_vals = sorted(df["N_PER_STAGE"].unique())

    c_h    = spec.base_overrides.get("C_H_IN_MOL_L", 1.15)
    f_div  = spec.base_overrides.get("DIVALENT_FRACTION", 0.20)
    a_cm2  = spec.a_mem_m2 * 1e4

    # ── Figure A: CASCADE_GAIN_PCT heatmap, clean vs fouled ──────────────────

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
    fig.suptitle(
        f"Cascade Power Gain over Equivalent Single Stack  [%]\n"
        f"C_H = {c_h:.2f} mol/L,  A = {a_cm2:.0f} cm²/pair",
        fontsize=11,
    )

    for ax, dfsub, title in zip(
        axes,
        [df_clean, df_fouled],
        [f"Clean  (β = {beta_clean:.1f})",
         f"Fouled  (β = {beta_fouled:.1f},  f_div = {f_div:.0%})"],
    ):
        pivot = dfsub.pivot(index="N_PER_STAGE", columns="S",
                            values="CASCADE_GAIN_PCT")
        im = ax.imshow(
            pivot.values, aspect="auto", origin="lower",
            cmap="RdYlGn", vmin=-5, vmax=40,
        )
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        ax.set_xlabel("Number of stages  S")
        ax.set_ylabel("Cell pairs per stage  N")
        ax.set_title(title, fontsize=9)

        for ri, rn in enumerate(pivot.index):
            for ci, cs in enumerate(pivot.columns):
                val = pivot.loc[rn, cs]
                if np.isfinite(val):
                    txt_clr = "black" if abs(val) < 28 else "white"
                    ax.text(ci, ri, f"{val:.1f}", ha="center", va="center",
                            fontsize=7.5, color=txt_clr)

        plt.colorbar(im, ax=ax, label="Gain [%]", fraction=0.046, pad=0.04)

    _save_fig(fig, run_dir, "cascade_fig_A_gain_heatmap.png")

    # ── Figure B: Gain vs S, one line per N, clean & fouled overlaid ─────────

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True,
                             sharey=True)
    fig.suptitle("Cascade Power Gain vs Number of Stages", fontsize=11)

    for ax, dfsub, clr, lbl in zip(
        axes,
        [df_clean, df_fouled],
        [_PALETTE["gain_clean"], _PALETTE["gain_fouled"]],
        [f"Clean  (β = {beta_clean:.1f})",
         f"Fouled  (β = {beta_fouled:.1f})"],
    ):
        alphas = np.linspace(0.45, 1.0, len(n_vals))
        for n, alpha in zip(n_vals, alphas):
            sub = dfsub[dfsub["N_PER_STAGE"] == n].sort_values("S")
            ax.plot(sub["S"], sub["CASCADE_GAIN_PCT"],
                    marker="o", lw=1.8, label=f"N = {n}/stage",
                    color=clr, alpha=float(alpha))
        ax.axhline(0, color="gray", lw=0.8, ls="--")
        ax.set_xlabel("Number of stages  S")
        ax.set_ylabel("Cascade gain  [%]")
        ax.set_title(lbl, fontsize=9)
        ax.legend(fontsize=8, loc="upper left")
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    _save_fig(fig, run_dir, "cascade_fig_B_gain_vs_stages.png")

    # ── Figure C: Fouling advantage at optimal clean geometry ─────────────────

    best_row = df_clean.loc[df_clean["CASCADE_GAIN_PCT"].idxmax()]
    s_best   = int(best_row["S"])
    n_best   = int(best_row["N_PER_STAGE"])

    df_best = df[
        (df["S"] == s_best) & (df["N_PER_STAGE"] == n_best)
    ].sort_values("FOULING_BETA")

    fig, ax = plt.subplots(figsize=(6.5, 4.5), constrained_layout=True)
    bars = ax.bar(
        [f"β={b:.1f}" for b in df_best["FOULING_BETA"]],
        df_best["FOULING_ADVANTAGE_PP"],
        color=_PALETTE["foul_adv"], edgecolor="white", linewidth=0.5,
    )
    ax.axhline(0, color="gray", lw=0.8, ls="--")
    ax.set_xlabel("FOULING_BETA  β")
    ax.set_ylabel("Fouling penalty reduction  [pp]")
    ax.set_title(
        f"Cascade Fouling Advantage  (S={s_best}, N={n_best}/stage,  "
        f"N_total={s_best * n_best})\n"
        f"Positive = cascade suffers fewer pp of penalty than equivalent single stack",
        fontsize=9,
    )
    for bar, val in zip(bars, df_best["FOULING_ADVANTAGE_PP"]):
        va = "bottom" if val >= 0 else "top"
        offset = 0.05 if val >= 0 else -0.05
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + offset,
            f"{val:.1f} pp", ha="center", va=va, fontsize=9,
        )

    _save_fig(fig, run_dir, "cascade_fig_C_fouling_advantage.png")

    # ── Figure D: Per-stage power waterfall at optimal clean geometry ─────────

    clean_knobs = make_base_knobs(spec.base_overrides)
    clean_knobs["FOULING_BETA"] = 0.0
    casc_opt = run_cascade(
        clean_knobs, n_stages=s_best, n_per_stage=n_best,
        a_mem_m2=spec.a_mem_m2, n_seg=spec.n_seg,
        p_overhead_per_stage_w=spec.p_overhead_per_stage_w,
    )
    stage_powers = [float(st["P_TOTAL_W"]) for st in casc_opt["per_stage"]]
    stage_labels = [f"Stage {i + 1}" for i in range(s_best)]

    # Collect C_H inlet per stage for annotation
    c_h_inlets: List[float] = []
    c_h = float(clean_knobs.get("C_H_IN_MOL_L", 1.15))
    c_l = float(clean_knobs.get("C_L_IN_MOL_L", 0.02))
    for st in casc_opt["per_stage"]:
        c_h_inlets.append(c_h)
        c_h, c_l = _outlet_concs(st)

    fig, ax = plt.subplots(
        figsize=(max(5.5, 1.3 * s_best), 4.5), constrained_layout=True,
    )
    bar_objs = ax.bar(stage_labels, [p * 1e3 for p in stage_powers],
                      color=_PALETTE["gain_clean"], edgecolor="white", linewidth=0.5)
    ax.set_xlabel("Cascade stage")
    ax.set_ylabel("Stage power  [mW]")
    ax.set_title(
        f"Per-Stage Power  (S={s_best}, N={n_best}/stage, clean)\n"
        f"Cascade total = {sum(stage_powers)*1e3:.2f} mW    "
        f"Single-stack equivalent = {best_row['P_CLEAN_SINGLE_W']*1e3:.2f} mW    "
        f"(+{best_row['CASCADE_GAIN_PCT']:.1f}%)",
        fontsize=9,
    )
    for bar, c_h_in in zip(bar_objs, c_h_inlets):
        bar_h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar_h * 0.5,
            f"C_H = {c_h_in:.2f}",
            ha="center", va="center", fontsize=8.5, color="white",
        )

    _save_fig(fig, run_dir, "cascade_fig_D_stage_waterfall.png")

    print(f"[CASCADE] All figures saved → {run_dir}")


# ── Fouling-aware optimal-shift analysis ─────────────────────────────────────
#
# Novel contribution: quantify how the optimal cascade geometry (S, N) shifts
# when spatially non-uniform fouling is accounted for.  The "design penalty"
# measures how much power is left on the table by designing for clean conditions
# but operating under fouled conditions — versus choosing the fouling-aware
# optimal geometry from the start.
#
# This analysis is purely post-processing on the existing cascade.parquet
# DataFrame.  No additional model calls required.
# ─────────────────────────────────────────────────────────────────────────────

def _compute_optimal_shift(
    df: pd.DataFrame,
    run_dir: Path,
    spec: CascadeSpec,
) -> pd.DataFrame:
    """Find optimal (S, N) at each fouling severity and compute design penalty.

    For each FOULING_BETA level:
      1. Find (S*, N*) that maximises P_CASCADE_W at that beta.
      2. Look up what P_CASCADE_W the clean-optimal geometry delivers at that beta.
      3. Design penalty = (P_fouling_aware - P_clean_design) / P_fouling_aware × 100.

    Returns a summary DataFrame (one row per beta) saved as optimal_shift.parquet.
    """
    beta_vals = sorted(df["FOULING_BETA"].unique())

    # Clean-optimal geometry (beta = 0)
    df_clean = df[df["FOULING_BETA"] == 0.0]
    if df_clean.empty:
        df_clean = df[df["FOULING_BETA"] == min(beta_vals)]
    clean_best = df_clean.loc[df_clean["P_CASCADE_W"].idxmax()]
    s_clean_opt = int(clean_best["S"])
    n_clean_opt = int(clean_best["N_PER_STAGE"])

    # N_total budget inferred from the clean-optimal geometry.
    # All comparisons are restricted to rows that conserve this budget
    # (S × N_PER_STAGE == n_total_budget), so the fouling-aware optimum
    # never wins simply by using more membrane.
    n_total_budget = s_clean_opt * n_clean_opt

    rows = []
    for beta in beta_vals:
        dfb = df[df["FOULING_BETA"] == beta]
        if dfb.empty:
            continue

        # Restrict to N_total-conserving geometries only
        dfb_budget = dfb[dfb["S"] * dfb["N_PER_STAGE"] == n_total_budget]
        if dfb_budget.empty:
            dfb_budget = dfb  # fallback: no budget constraint possible

        # Fouling-aware optimal at this beta (within budget)
        best = dfb_budget.loc[dfb_budget["P_CASCADE_W"].idxmax()]
        s_opt = int(best["S"])
        n_opt = int(best["N_PER_STAGE"])
        p_opt = float(best["P_CASCADE_W"])

        # Power from the clean-optimal geometry at this beta
        clean_design_row = dfb[
            (dfb["S"] == s_clean_opt) & (dfb["N_PER_STAGE"] == n_clean_opt)
        ]
        if clean_design_row.empty:
            p_clean_design = p_opt  # fallback: no penalty computable
        else:
            p_clean_design = float(clean_design_row["P_CASCADE_W"].iloc[0])

        # Design penalty: % power lost by using clean-optimal instead of fouling-aware
        design_penalty_pct = (
            (p_opt - p_clean_design) / max(p_opt, 1e-30) * 100.0
        )

        rows.append({
            "FOULING_BETA":         float(beta),
            "S_CLEAN_OPT":          s_clean_opt,
            "N_CLEAN_OPT":          n_clean_opt,
            "S_FOULING_OPT":        s_opt,
            "N_FOULING_OPT":        n_opt,
            "P_FOULING_AWARE_W":    p_opt,
            "P_CLEAN_DESIGN_W":     p_clean_design,
            "DESIGN_PENALTY_PCT":   design_penalty_pct,
            "GEOMETRY_SHIFTED":     (s_opt != s_clean_opt) or (n_opt != n_clean_opt),
        })

    opt_df = pd.DataFrame(rows)

    # Save
    opt_path = run_dir / "optimal_shift.parquet"
    opt_df.to_parquet(opt_path, index=False)
    print(f"[CASCADE] Optimal-shift analysis saved → {opt_path}")

    # Console summary
    print(f"[CASCADE] Clean-optimal geometry: S={s_clean_opt}, N={n_clean_opt}/stage")
    for _, r in opt_df.iterrows():
        shifted = " ← SHIFTED" if r["GEOMETRY_SHIFTED"] else ""
        print(
            f"  β={r['FOULING_BETA']:.1f}  →  "
            f"S={int(r['S_FOULING_OPT'])}, N={int(r['N_FOULING_OPT'])}/stage  "
            f"penalty={r['DESIGN_PENALTY_PCT']:+.2f}%{shifted}"
        )

    return opt_df


def _plot_optimal_shift(
    opt_df: pd.DataFrame,
    sweep_df: pd.DataFrame,
    run_dir: Path,
    spec: CascadeSpec,
) -> None:
    """Two figures for the fouling-aware optimal-shift analysis.

    Figure E: Design penalty (%) vs FOULING_BETA — bar chart.
    Figure F: Optimal geometry trajectory — scatter showing how (S*, N*) moves
              as fouling increases, overlaid on the P_CASCADE heatmap at max beta.
    """
    c_h   = spec.base_overrides.get("C_H_IN_MOL_L", 1.15)
    f_div = spec.base_overrides.get("DIVALENT_FRACTION", 0.20)
    a_cm2 = spec.a_mem_m2 * 1e4

    # ── Figure E: Design penalty bar chart ───────────────────────────────────

    fig, ax = plt.subplots(figsize=(6.5, 4.5), constrained_layout=True)
    betas  = opt_df["FOULING_BETA"]
    penalty = opt_df["DESIGN_PENALTY_PCT"]

    bar_colors = ["#AAAAAA" if p == 0.0 else "#D95319" for p in penalty]
    bars = ax.bar(
        [f"β={b:.1f}" for b in betas], penalty,
        color=bar_colors, edgecolor="white", linewidth=0.5,
    )
    ax.axhline(0, color="gray", lw=0.8, ls="--")
    ax.set_xlabel("Fouling severity  β")
    ax.set_ylabel("Design penalty  [%]")
    ax.set_title(
        f"Cost of Ignoring Fouling in Cascade Design\n"
        f"C_H = {c_h:.2f} mol/L,  f_div = {f_div:.0%},  A = {a_cm2:.0f} cm²/pair\n"
        f"Penalty = power lost by using clean-optimal geometry under fouled conditions",
        fontsize=9,
    )
    for bar, val in zip(bars, penalty):
        if abs(val) > 0.01:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.05,
                f"{val:.2f}%", ha="center", va="bottom", fontsize=9,
            )

    # Annotate geometry shifts
    for i, (_, r) in enumerate(opt_df.iterrows()):
        if r["GEOMETRY_SHIFTED"]:
            ax.text(
                i, -0.3,
                f"S={int(r['S_FOULING_OPT'])},N={int(r['N_FOULING_OPT'])}",
                ha="center", va="top", fontsize=7, color="#D95319", fontweight="bold",
            )
        else:
            ax.text(
                i, -0.3,
                f"S={int(r['S_CLEAN_OPT'])},N={int(r['N_CLEAN_OPT'])}",
                ha="center", va="top", fontsize=7, color="#888888",
            )

    _save_fig(fig, run_dir, "cascade_fig_E_design_penalty.png")

    # ── Figure F: Geometry trajectory on heatmap ─────────────────────────────

    beta_max = opt_df["FOULING_BETA"].max()
    df_max = sweep_df[sweep_df["FOULING_BETA"] == beta_max].copy()

    if df_max.empty:
        return  # nothing to plot

    s_vals = sorted(df_max["S"].unique())
    n_vals = sorted(df_max["N_PER_STAGE"].unique())

    pivot = df_max.pivot(index="N_PER_STAGE", columns="S", values="P_CASCADE_W")
    pivot_mw = pivot * 1e3  # convert to mW for readability

    fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
    im = ax.imshow(
        pivot_mw.values, aspect="auto", origin="lower",
        cmap="viridis",
    )
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("Number of stages  S")
    ax.set_ylabel("Cell pairs per stage  N")
    plt.colorbar(im, ax=ax, label="Cascade power  [mW]", fraction=0.046, pad=0.04)

    # Annotate cell values
    for ri, rn in enumerate(pivot.index):
        for ci, cs in enumerate(pivot.columns):
            val = pivot_mw.loc[rn, cs]
            if np.isfinite(val):
                txt_clr = "white" if val < pivot_mw.values.mean() else "black"
                ax.text(ci, ri, f"{val:.1f}", ha="center", va="center",
                        fontsize=7, color=txt_clr)

    # Overlay optimal geometry trajectory
    for _, r in opt_df.iterrows():
        s_idx = s_vals.index(int(r["S_FOULING_OPT"])) if int(r["S_FOULING_OPT"]) in s_vals else None
        n_idx = n_vals.index(int(r["N_FOULING_OPT"])) if int(r["N_FOULING_OPT"]) in n_vals else None
        if s_idx is not None and n_idx is not None:
            marker = "D" if r["GEOMETRY_SHIFTED"] else "o"
            color  = "#D95319" if r["GEOMETRY_SHIFTED"] else "#FFFFFF"
            ax.plot(s_idx, n_idx, marker=marker, ms=12, mec="white", mew=2,
                    color=color, zorder=5)
            ax.annotate(
                f"β={r['FOULING_BETA']:.1f}",
                (s_idx, n_idx), textcoords="offset points",
                xytext=(8, 8), fontsize=7, color="white", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="white", lw=0.8),
            )

    # Mark clean optimal
    s0 = opt_df.iloc[0]
    s_idx0 = s_vals.index(int(s0["S_CLEAN_OPT"])) if int(s0["S_CLEAN_OPT"]) in s_vals else None
    n_idx0 = n_vals.index(int(s0["N_CLEAN_OPT"])) if int(s0["N_CLEAN_OPT"]) in n_vals else None
    if s_idx0 is not None and n_idx0 is not None:
        ax.plot(s_idx0, n_idx0, marker="*", ms=18, mec="white", mew=2,
                color="#77AC30", zorder=6, label="Clean optimal")
        ax.legend(fontsize=8, loc="upper right")

    ax.set_title(
        f"Cascade Power at β = {beta_max:.1f}  with Optimal Geometry Trajectory\n"
        f"C_H = {c_h:.2f} mol/L,  f_div = {f_div:.0%},  A = {a_cm2:.0f} cm²/pair\n"
        f"★ = clean optimal    ◆ = fouling-shifted optimal    ○ = unchanged",
        fontsize=9,
    )

    _save_fig(fig, run_dir, "cascade_fig_F_geometry_trajectory.png")


def _write_cascade_pdf(run_dir: Path, spec: CascadeSpec) -> None:
    """Write a PDF report for the cascade sweep, mirroring the experiment PDF format.

    Writes a run_bundle.json (metadata) then collects all *.png in run_dir
    into a multi-page PDF using reportlab.
    """
    import io
    run_dir.mkdir(parents=True, exist_ok=True)

    # Write metadata envelope
    meta = {
        "experiment_id":   spec.experiment_id,
        "run_tag":         run_dir.name,
        "run_datetime_iso": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "description":     spec.description,
        "base_overrides":  spec.base_overrides,
        "spec": {
            "s_vals":            spec.s_vals,
            "n_vals":            spec.n_vals,
            "fouling_beta_vals": spec.fouling_beta_vals,
            "a_mem_m2":          spec.a_mem_m2,
            "n_seg":             spec.n_seg,
            "fresh_dilute":      spec.fresh_dilute,
        },
    }
    meta_path = run_dir / "run_bundle.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, default=str)

    # Collect PNGs
    png_paths = sorted(run_dir.glob("cascade_fig_*.png"))
    if not png_paths:
        print("[CASCADE-PDF] No figures found — skipping PDF generation.")
        return

    try:
        from reportlab.lib.pagesizes import letter
        from reportlab.platypus import (
            SimpleDocTemplate, Paragraph, Spacer, PageBreak,
            Image as RLImage, Table, TableStyle,
        )
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib.units import inch
        from reportlab.lib import colors
        from reportlab.lib.utils import ImageReader
        from PIL import Image as PILImage
        PILImage.MAX_IMAGE_PIXELS = None
    except ImportError:
        print("[CASCADE-PDF] reportlab not available — skipping PDF.")
        return

    W, H = letter
    usable_w = W - 1.5 * inch

    styles = getSampleStyleSheet()
    s_title = ParagraphStyle("CTitle", parent=styles["Title"], fontSize=16,
                             spaceAfter=12)
    s_h1    = ParagraphStyle("CH1", parent=styles["Heading1"], fontSize=12,
                             spaceAfter=6)
    s_body  = ParagraphStyle("CBody", parent=styles["Normal"], fontSize=9,
                             spaceAfter=4)
    s_fig   = ParagraphStyle("CFigLbl", parent=styles["Normal"], fontSize=10,
                             textColor=colors.HexColor("#0072BD"),
                             fontName="Helvetica-Bold", spaceAfter=4)
    s_cap   = ParagraphStyle("CCap", parent=styles["Normal"], fontSize=8,
                             textColor=colors.gray, spaceAfter=6)

    pdf_name = f"{run_dir.name}_{spec.experiment_id}.pdf"
    pdf_path = run_dir / pdf_name
    doc = SimpleDocTemplate(
        str(pdf_path), pagesize=letter,
        leftMargin=0.75*inch, rightMargin=0.75*inch,
        topMargin=0.75*inch, bottomMargin=0.75*inch,
    )

    story = []

    # ── Cover page ───────────────────────────────────────────────────────────
    story.append(Paragraph("OsmoFlux — Cascade Optimization Report", s_title))
    story.append(Spacer(1, 0.15 * inch))
    story.append(Paragraph(f"<b>Experiment:</b> {spec.experiment_id}", s_body))
    story.append(Paragraph(f"<b>Run tag:</b> {run_dir.name}", s_body))
    story.append(Paragraph(
        f"<b>Generated:</b> {time.strftime('%Y-%m-%d %H:%M:%S')}", s_body))
    story.append(Spacer(1, 0.1 * inch))
    story.append(Paragraph(spec.description, s_body))
    story.append(Spacer(1, 0.15 * inch))

    # Parameters table
    story.append(Paragraph("Sweep Parameters", s_h1))
    param_data = [
        ["Parameter", "Value"],
        ["C_H_IN_MOL_L", f"{spec.base_overrides.get('C_H_IN_MOL_L', '—')}"],
        ["DIVALENT_FRACTION", f"{spec.base_overrides.get('DIVALENT_FRACTION', '—')}"],
        ["A_MEM_M2", f"{spec.a_mem_m2}"],
        ["Stages (S)", f"{spec.s_vals}"],
        ["Cell pairs/stage (N)", f"{spec.n_vals}"],
        ["FOULING_BETA", f"{spec.fouling_beta_vals}"],
        ["Fresh dilute", f"{spec.fresh_dilute}"],
        ["N_SEG", f"{spec.n_seg}"],
    ]
    tbl = Table(param_data, colWidths=[2.5*inch, 4.0*inch])
    tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#0072BD")),
        ("TEXTCOLOR",  (0, 0), (-1, 0), colors.white),
        ("FONTNAME",   (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE",   (0, 0), (-1, -1), 8),
        ("GRID",       (0, 0), (-1, -1), 0.5, colors.lightgrey),
        ("VALIGN",     (0, 0), (-1, -1), "MIDDLE"),
    ]))
    story.append(tbl)

    # ── Optimal shift summary table (if available) ───────────────────────────
    opt_path = run_dir / "optimal_shift.parquet"
    if opt_path.exists():
        story.append(Spacer(1, 0.2 * inch))
        story.append(Paragraph("Fouling-Aware Optimal Geometry Shift", s_h1))

        opt_df = pd.read_parquet(opt_path)
        shift_data = [["β", "Clean Opt", "Fouling Opt", "Penalty %", "Shifted?"]]
        for _, r in opt_df.iterrows():
            shift_data.append([
                f"{r['FOULING_BETA']:.1f}",
                f"S={int(r['S_CLEAN_OPT'])}, N={int(r['N_CLEAN_OPT'])}",
                f"S={int(r['S_FOULING_OPT'])}, N={int(r['N_FOULING_OPT'])}",
                f"{r['DESIGN_PENALTY_PCT']:.2f}%",
                "Yes" if r["GEOMETRY_SHIFTED"] else "—",
            ])
        stbl = Table(shift_data, colWidths=[0.8*inch, 1.5*inch, 1.5*inch, 1.2*inch, 1.0*inch])
        stbl.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#D95319")),
            ("TEXTCOLOR",  (0, 0), (-1, 0), colors.white),
            ("FONTNAME",   (0, 0), (-1, 0), "Helvetica-Bold"),
            ("FONTSIZE",   (0, 0), (-1, -1), 8),
            ("GRID",       (0, 0), (-1, -1), 0.5, colors.lightgrey),
            ("VALIGN",     (0, 0), (-1, -1), "MIDDLE"),
            ("ALIGN",      (0, 0), (-1, -1), "CENTER"),
        ]))
        story.append(stbl)

    # ── Figure pages ─────────────────────────────────────────────────────────
    for fig_idx, png_path in enumerate(png_paths, start=1):
        story.append(PageBreak())
        fig_label = f"FIG-{spec.experiment_id}-{fig_idx:03d}"
        story.append(Paragraph(fig_label, s_fig))

        # Derive caption from filename
        caption = png_path.stem.replace("cascade_fig_", "").replace("_", " ").title()
        story.append(Paragraph(caption, s_cap))

        with open(png_path, "rb") as pf:
            png_bytes = pf.read()
        ir = ImageReader(io.BytesIO(png_bytes))
        iw, ih = ir.getSize()
        scale = min(usable_w / iw, (H - 2.5 * inch) / ih)
        story.append(RLImage(io.BytesIO(png_bytes), width=iw * scale, height=ih * scale))

    doc.build(story)
    print(f"[CASCADE-PDF] Report saved: {pdf_path}")


def _save_fig(fig: plt.Figure, run_dir: Path, fname: str) -> None:
    path = run_dir / fname
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [fig] {fname}")


# ── Default spec (single-architecture, kept for backward compat) ──────────────

CASCADE_SPEC = CascadeSpec(
    p_overhead_per_stage_w=0.0,   # Physics-based pumping + electrode model now active; no extra fixed overhead
    n_total_max=200,              # Increased for pilot-scale exploration
)


# ── Fresh-dilute vs serial comparison ─────────────────────────────────────────

_CL_SCENARIOS = [0.02, 0.1, 0.5]   # treated WW, impaired water, ambient seawater

_ARCH_COLORS = {
    True:  "#0072BD",   # fresh-dilute (blue)
    False: "#D95319",   # serial pass-through (orange)
}
_ARCH_LABELS = {
    True:  "Fresh dilute feed",
    False: "Serial pass-through",
}


def _collect_stage_profiles(
    spec: CascadeSpec,
    beta: float = 0.0,
) -> Dict[int, List[Dict[str, float]]]:
    """Run cascade at each S and collect per-stage inlet/outlet C_H, C_L, power.

    Returns {s: [{'stage': 1, 'C_H_in': ..., 'C_H_out': ..., 'C_L_in': ...,
                   'C_L_out': ..., 'P_W': ...}, ...]} for each s in spec.s_vals.
    Uses the first (smallest) N_PER_STAGE value from spec.
    """
    base_knobs = make_base_knobs(spec.base_overrides)
    base_knobs["FOULING_BETA"] = float(beta)
    n_per = spec.n_vals[0]
    c_l_fresh = float(base_knobs.get("C_L_IN_MOL_L", 0.02))

    profiles: Dict[int, List[Dict[str, float]]] = {}

    for s in spec.s_vals:
        c_h = float(base_knobs.get("C_H_IN_MOL_L", 1.15))
        c_l = c_l_fresh
        stages = []
        for i in range(s):
            res = _run_single_stage(base_knobs, n_per, spec.a_mem_m2,
                                    spec.n_seg, c_h, c_l)
            c_h_out, c_l_out = _outlet_concs(res)
            stages.append({
                "stage":   i + 1,
                "C_H_in":  c_h,
                "C_H_out": c_h_out,
                "C_L_in":  c_l,
                "C_L_out": c_l_out,
                "P_W":     float(res["P_TOTAL_W"]),
            })
            c_h = c_h_out
            c_l = c_l_fresh if spec.fresh_dilute else c_l_out
        profiles[s] = stages

    return profiles


def _plot_architecture_comparison(
    sweep_results: Dict[tuple, pd.DataFrame],
    stage_profiles: Dict[tuple, Dict[int, list]],
    run_dir: Path,
) -> None:
    """Generate cross-architecture comparison figures.

    Parameters
    ----------
    sweep_results : {(c_l, fresh_dilute): DataFrame}
    stage_profiles : {(c_l, fresh_dilute): {s: [stage_dicts]}}
    """
    run_dir.mkdir(parents=True, exist_ok=True)
    c_h_label = "1.15"  # Carlsbad brine

    # ── Figure 1: Brine outlet C_H vs stage count ────────────────────────────
    # One panel per C_L, fresh vs serial overlaid

    fig, axes = plt.subplots(1, len(_CL_SCENARIOS), figsize=(4.5 * len(_CL_SCENARIOS), 4.5),
                             constrained_layout=True, sharey=True)
    if len(_CL_SCENARIOS) == 1:
        axes = [axes]
    fig.suptitle(
        f"Brine Outlet Concentration vs Cascade Stages\n"
        f"C_H = {c_h_label} mol/L (Carlsbad brine),  clean conditions",
        fontsize=11,
    )
    for ax, c_l in zip(axes, _CL_SCENARIOS):
        for fd in [True, False]:
            profs = stage_profiles.get((c_l, fd))
            if profs is None:
                continue
            s_vals = sorted(profs.keys())
            c_h_outs = [profs[s][-1]["C_H_out"] for s in s_vals]
            ax.plot(s_vals, c_h_outs, marker="o", lw=2,
                    color=_ARCH_COLORS[fd], label=_ARCH_LABELS[fd])
        # 40 ppt ≈ 0.685 mol/L NaCl — California discharge threshold
        ax.axhline(0.685, color="gray", ls="--", lw=0.8, label="40 ppt discharge limit")
        ax.set_xlabel("Number of stages  S")
        if ax == axes[0]:
            ax.set_ylabel("Brine outlet  C_H  [mol/L]")
        ax.set_title(f"C_L = {c_l} mol/L", fontsize=9)
        ax.legend(fontsize=7.5, loc="best")
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    _save_fig(fig, run_dir, "compare_fig1_brine_outlet.png")

    # ── Figure 2: Per-stage power waterfall, fresh vs serial ─────────────────
    # At C_L = 0.02 (best case), highest S, side-by-side bars

    c_l_show = 0.02
    s_max = max(stage_profiles.get((c_l_show, True), {}).keys(), default=1)
    fig, ax = plt.subplots(figsize=(max(6, 1.2 * s_max), 4.5), constrained_layout=True)
    x = np.arange(s_max)
    width = 0.38
    for offset, fd in [(-0.5, True), (0.5, False)]:
        profs = stage_profiles.get((c_l_show, fd), {}).get(s_max, [])
        powers_mw = [st["P_W"] * 1e3 for st in profs]
        ax.bar(x + offset * width, powers_mw, width, color=_ARCH_COLORS[fd],
               label=_ARCH_LABELS[fd], edgecolor="white", linewidth=0.5)
    ax.set_xlabel("Stage")
    ax.set_ylabel("Stage power  [mW]")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{i+1}" for i in range(s_max)])
    ax.set_title(
        f"Per-Stage Power at S = {s_max}  (C_L = {c_l_show} mol/L, clean)\n"
        f"Fresh-feed stages maintain gradient; serial stages degrade",
        fontsize=9,
    )
    ax.legend(fontsize=8)
    _save_fig(fig, run_dir, "compare_fig2_stage_waterfall.png")

    # ── Figure 3: Total cascade power vs stages ──────────────────────────────
    # Three C_L values × two architectures

    fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
    cl_styles = {0.02: "-", 0.1: "--", 0.5: ":"}
    for c_l in _CL_SCENARIOS:
        for fd in [True, False]:
            profs = stage_profiles.get((c_l, fd))
            if profs is None:
                continue
            s_vals = sorted(profs.keys())
            p_totals = [sum(st["P_W"] for st in profs[s]) * 1e3 for s in s_vals]
            ax.plot(s_vals, p_totals, marker="o", lw=1.8,
                    color=_ARCH_COLORS[fd], ls=cl_styles[c_l],
                    label=f"{_ARCH_LABELS[fd]}, C_L={c_l}")
    ax.set_xlabel("Number of stages  S")
    ax.set_ylabel("Total cascade power  [mW]")
    ax.set_title(
        f"Cascade Power Output  (C_H = {c_h_label} mol/L, clean)\n"
        f"Solid = C_L 0.02,  Dashed = C_L 0.1,  Dotted = C_L 0.5 mol/L",
        fontsize=9,
    )
    ax.legend(fontsize=7.5, ncol=2, loc="best")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    _save_fig(fig, run_dir, "compare_fig3_total_power.png")

    # ── Figure 4: Cascade gain (%) over single stack, fresh vs serial ────────
    # At C_L = 0.02, clean conditions, one line per N

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True, sharey=True)
    fig.suptitle(
        f"Cascade Gain over Equivalent Single Stack  [%]\n"
        f"C_H = {c_h_label} mol/L,  C_L = 0.02 mol/L,  clean",
        fontsize=11,
    )
    for ax, fd in zip(axes, [True, False]):
        df = sweep_results.get((0.02, fd))
        if df is None:
            continue
        df_clean = df[df["FOULING_BETA"] == 0.0]
        for n_per in sorted(df_clean["N_PER_STAGE"].unique()):
            sub = df_clean[df_clean["N_PER_STAGE"] == n_per].sort_values("S")
            ax.plot(sub["S"], sub["CASCADE_GAIN_PCT"],
                    marker="o", lw=1.8, label=f"N = {n_per}/stage")
        ax.axhline(0, color="gray", lw=0.8, ls="--")
        ax.set_xlabel("Number of stages  S")
        ax.set_ylabel("Cascade gain  [%]")
        ax.set_title(_ARCH_LABELS[fd], fontsize=9)
        ax.legend(fontsize=8, loc="upper left")
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    _save_fig(fig, run_dir, "compare_fig4_gain_vs_stages.png")

    # ── Figure 5: Fouling advantage — fresh vs serial at max beta ────────────
    # Bar chart: at each S, compare FOUL_PENALTY for fresh vs serial

    fig, ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
    c_l_foul = 0.02
    n_per_show = sorted(sweep_results.get((c_l_foul, True), pd.DataFrame()).get("N_PER_STAGE", pd.Series()).unique())[0] \
        if (c_l_foul, True) in sweep_results else 5

    for fd in [True, False]:
        df = sweep_results.get((c_l_foul, fd))
        if df is None:
            continue
        beta_max = df["FOULING_BETA"].max()
        sub = df[(df["FOULING_BETA"] == beta_max) & (df["N_PER_STAGE"] == n_per_show)].sort_values("S")
        ax.plot(sub["S"], sub["FOUL_PENALTY_CASCADE_PCT"],
                marker="s", lw=2, color=_ARCH_COLORS[fd], label=f"{_ARCH_LABELS[fd]} (β={beta_max:.1f})")
    ax.set_xlabel("Number of stages  S")
    ax.set_ylabel("Fouling penalty  [% of clean power]")
    ax.set_title(
        f"Fouling Penalty: Fresh-Feed vs Serial  "
        f"(N = {n_per_show}/stage, C_L = {c_l_foul})\n"
        f"Lower = more fouling-resilient",
        fontsize=9,
    )
    ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    _save_fig(fig, run_dir, "compare_fig5_fouling_resilience.png")

    print(f"[COMPARE] All comparison figures saved → {run_dir}")


# ── Architecture comparison runner (called from cell 20 experiment registry) ──

_CASCADE_BASE_OVERRIDES = {
    "C_H_IN_MOL_L":       1.15,    # Carlsbad brine
    "C_L_IN_MOL_L":       0.03,    # treated wastewater effluent (realistic) — overridden per scenario
    "Q_H_M3_S":           5.0e-6,
    "Q_L_M3_S":           5.0e-6,
    "GAP_M":              2.0e-4,
    "DIVALENT_FRACTION":  0.20,
    "FOULING_MODEL":      "none",
    "FOULING_C_REF_MOL_L": 0.5,
    "FOULING_N_EXP":      0.5,
}


def run_architecture_comparison(
    cl_scenarios: List[float] = None,
    base_overrides: dict = None,
    runs_root: str = "runs",
) -> Dict[tuple, pd.DataFrame]:
    """Run fresh-dilute vs serial cascade sweeps across C_L scenarios.

    Each (C_L, architecture) pair gets a full cascade sweep with make_plots=True,
    producing per-sweep figures (A-F) and PDF.  Then a cross-architecture
    comparison figure set is generated in the parent directory.

    Returns {(c_l, fresh_dilute): DataFrame}.
    """
    if cl_scenarios is None:
        cl_scenarios = list(_CL_SCENARIOS)
    if base_overrides is None:
        base_overrides = dict(_CASCADE_BASE_OVERRIDES)

    compare_dir = Path(runs_root) / "CASCADE-ARCHITECTURE-COMPARISON" / _now_tag()

    sweep_results: Dict[tuple, pd.DataFrame] = {}
    stage_profiles: Dict[tuple, Dict[int, list]] = {}

    for c_l in cl_scenarios:
        for fd in [True, False]:
            tag = "fresh" if fd else "serial"
            spec = CascadeSpec(
                experiment_id=f"CASCADE-CL{c_l}-{tag}",
                description=(
                    f"{'Fresh-dilute' if fd else 'Serial'} cascade, "
                    f"C_L={c_l} mol/L, Carlsbad brine C_H=1.15 mol/L."
                ),
                base_overrides={**base_overrides, "C_L_IN_MOL_L": c_l},
                fresh_dilute=fd,
            )
            # Full sweep with plots + PDF
            df = run_cascade_sweep(spec, save=True, make_plots=True,
                                   runs_root=str(compare_dir))
            sweep_results[(c_l, fd)] = df

            # Per-stage profiles (for brine outlet / waterfall figures)
            stage_profiles[(c_l, fd)] = _collect_stage_profiles(spec, beta=0.0)

    # Cross-architecture comparison figures
    _plot_architecture_comparison(sweep_results, stage_profiles, compare_dir)

    print(f"\n[COMPARE] Complete. {len(sweep_results)} sweep DataFrames collected.")
    print(f"[COMPARE] Output directory: {compare_dir}")
    return sweep_results


# ── Three-configuration comparison ────────────────────────────────────────────

_CONFIG_COLORS = {
    "single": "#7E2F8E",   # purple
    "serial": "#D95319",   # orange
    "fresh":  "#0072BD",   # blue
}
_CONFIG_LABELS = {
    "single": "Single stack",
    "serial": "Serial dilute cascade",
    "fresh":  "Fresh dilute cascade",
}


def run_three_config_comparison(
    n_total: int = 40,
    base_overrides: dict = None,
    s_vals: List[int] = None,
    fouling_beta_vals: List[float] = None,
    a_mem_m2: float = 0.01,
    q_l_vals: List[float] = None,
    runs_root: str = "runs",
) -> Dict[str, pd.DataFrame]:
    """Run single / serial / fresh-dilute configs at fixed total cell pairs.

    For each Q_L value and each configuration, sweep S values.
    S values where n_total is not divisible are skipped.

    Returns {"all": df_all} with CONFIG, Q_L_M3_S, S, etc. columns.
    """
    if base_overrides is None:
        base_overrides = dict(_CASCADE_BASE_OVERRIDES)
        base_overrides["C_L_IN_MOL_L"] = 0.02
    if fouling_beta_vals is None:
        fouling_beta_vals = [0.0, 0.2, 0.4, 0.6, 0.8]
    if q_l_vals is None:
        q_l_vals = [5.0e-6]

    if s_vals is None:
        s_vals = sorted({s for s in range(1, n_total + 1)
                         if n_total % s == 0 and n_total // s >= 2})
        if len(s_vals) > 12:
            s_vals = [s for s in s_vals if s <= n_total // 2]

    print(f"[3-CONFIG] N_total={n_total}, A_mem={a_mem_m2} m2, "
          f"total area={n_total * a_mem_m2:.2f} m2")
    print(f"[3-CONFIG] S values: {s_vals}")
    print(f"[3-CONFIG] Configs: single (S=1,N={n_total}) | serial | fresh")
    print(f"[3-CONFIG] beta values: {fouling_beta_vals}")
    print(f"[3-CONFIG] Q_L values: {[f'{q:.1e}' for q in q_l_vals]}")

    compare_dir = Path(runs_root) / "CASCADE-THREE-CONFIG" / _now_tag()
    compare_dir.mkdir(parents=True, exist_ok=True)

    all_n_vals = sorted({n_total // s for s in s_vals})
    all_config_dfs: List[pd.DataFrame] = []

    for q_l in q_l_vals:
        q_overrides = {**base_overrides, "Q_L_M3_S": q_l}

        # ── Config 1: Single stack (S=1 only) ──
        spec_single = CascadeSpec(
            experiment_id=f"CONFIG-single-QL{q_l:.0e}",
            description=f"Single stack, N={n_total}, Q_L={q_l:.1e}",
            base_overrides=q_overrides,
            s_vals=[1],
            n_vals=[n_total],
            fouling_beta_vals=fouling_beta_vals,
            a_mem_m2=a_mem_m2,
            fresh_dilute=True,
            n_total_max=0,
            n_total_exact=0,
        )
        print(f"\n[3-CONFIG] Config 1: Single stack, Q_L={q_l:.1e}")
        df_single = run_cascade_sweep(spec_single, save=True, make_plots=False,
                                       runs_root=str(compare_dir))
        df_single["CONFIG"] = "single"
        df_single["Q_L_M3_S"] = q_l

        # ── Config 2: Serial dilute cascade ──
        spec_serial = CascadeSpec(
            experiment_id=f"CONFIG-serial-QL{q_l:.0e}",
            description=f"Serial cascade, N_total={n_total}, Q_L={q_l:.1e}",
            base_overrides=q_overrides,
            s_vals=s_vals,
            n_vals=all_n_vals,
            fouling_beta_vals=fouling_beta_vals,
            a_mem_m2=a_mem_m2,
            fresh_dilute=False,
            n_total_max=n_total,
            n_total_exact=n_total,
        )
        print(f"\n[3-CONFIG] Config 2: Serial cascade, Q_L={q_l:.1e}")
        df_serial = run_cascade_sweep(spec_serial, save=True, make_plots=False,
                                       runs_root=str(compare_dir))
        df_serial["CONFIG"] = "serial"
        df_serial["Q_L_M3_S"] = q_l

        # ── Config 3: Fresh dilute cascade ──
        spec_fresh = CascadeSpec(
            experiment_id=f"CONFIG-fresh-QL{q_l:.0e}",
            description=f"Fresh-dilute cascade, N_total={n_total}, Q_L={q_l:.1e}",
            base_overrides=q_overrides,
            s_vals=s_vals,
            n_vals=all_n_vals,
            fouling_beta_vals=fouling_beta_vals,
            a_mem_m2=a_mem_m2,
            fresh_dilute=True,
            n_total_max=n_total,
            n_total_exact=n_total,
        )
        print(f"\n[3-CONFIG] Config 3: Fresh-dilute cascade, Q_L={q_l:.1e}")
        df_fresh = run_cascade_sweep(spec_fresh, save=True, make_plots=False,
                                      runs_root=str(compare_dir))
        df_fresh["CONFIG"] = "fresh"
        df_fresh["Q_L_M3_S"] = q_l

        all_config_dfs.extend([df_single, df_serial, df_fresh])
        print(f"[3-CONFIG] Q_L={q_l:.1e} complete: "
              f"{len(df_single)+len(df_serial)+len(df_fresh)} rows")

    # ── Combined DataFrame ──
    df_all = pd.concat(all_config_dfs, ignore_index=True)
    df_all.to_parquet(compare_dir / "three_config.parquet", index=False)
    print(f"\n[3-CONFIG] Combined: {len(df_all)} rows -> three_config.parquet")

    # ── Figures (at first Q_L for backward compat) ──
    df_first_ql = df_all[df_all["Q_L_M3_S"] == 5.0e-6]
    _plot_three_config(df_first_ql, compare_dir, n_total, a_mem_m2)

    # ── Flow-rate analysis (if multiple Q_L) ──
    if len(q_l_vals) > 1:
        _plot_flow_rate_analysis(df_all, compare_dir, n_total, a_mem_m2)

    # ── Cleaning cycle optimization (at first Q_L) ──
    if len(df_first_ql) > 0:
        _cleaning_cycle_analysis(df_first_ql, compare_dir, n_total)

    # ── PDF ──
    _write_three_config_pdf(compare_dir, n_total, df_all)

    print(f"\n[3-CONFIG] Complete. Output: {compare_dir}")
    return {"all": df_all}

def _plot_flow_rate_analysis(
    df: pd.DataFrame, out_dir: Path, n_total: int, a_mem_m2: float = 0.01,
) -> None:
    """Generate figures showing how Q_L affects optimal staging and power."""
    import matplotlib.pyplot as plt

    q_vals = sorted(df["Q_L_M3_S"].unique())
    betas = sorted(df["FOULING_BETA"].unique())

    # ── Figure I: Best net power vs Q_L for each config (at beta=0) ──
    fig, ax = plt.subplots(figsize=(8, 5))
    df0 = df[df["FOULING_BETA"] == 0.0]
    for cfg in ["single", "serial", "fresh"]:
        df_c = df0[df0["CONFIG"] == cfg]
        best_p = []
        for ql in q_vals:
            sub = df_c[df_c["Q_L_M3_S"] == ql]
            if len(sub) > 0:
                best_p.append((ql, sub["P_CASCADE_W"].max() * 1000))
        if best_p:
            qs, ps = zip(*best_p)
            ax.plot([q*1e6 for q in qs], ps, "o-", color=_CONFIG_COLORS[cfg],
                    label=_CONFIG_LABELS[cfg], linewidth=2, markersize=8)
    ax.set_xlabel("Dilute flow rate Q_L [mL/s]", fontsize=12)
    ax.set_ylabel("Best net power [mW]", fontsize=12)
    ax.set_title(f"Optimal net power vs dilute flow rate  |  N_total={n_total}, beta=0",
                 fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_dir / "three_fig_I_power_vs_flow.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [fig] three_fig_I_power_vs_flow.png")

    # ── Figure J: Optimal S* vs Q_L for fresh config at different beta ──
    fig, ax = plt.subplots(figsize=(8, 5))
    cmap = plt.cm.RdYlGn_r
    df_f = df[df["CONFIG"] == "fresh"]
    for i, beta in enumerate(betas):
        color = cmap(i / max(len(betas) - 1, 1))
        opt_s = []
        for ql in q_vals:
            sub = df_f[(df_f["Q_L_M3_S"] == ql) & (df_f["FOULING_BETA"] == beta)]
            if len(sub) > 0:
                best = sub.loc[sub["P_CASCADE_W"].idxmax()]
                opt_s.append((ql, int(best["S"])))
        if opt_s:
            qs, ss = zip(*opt_s)
            ax.plot([q*1e6 for q in qs], ss, "o-", color=color,
                    label=f"beta={beta:.1f}", linewidth=2, markersize=7)
    ax.set_xlabel("Dilute flow rate Q_L [mL/s]", fontsize=12)
    ax.set_ylabel("Optimal stages (S*)", fontsize=12)
    ax.set_title(f"Fresh cascade: optimal staging vs flow rate\n"
                 f"N_total={n_total}, A_mem={a_mem_m2} m2", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)
    fig.tight_layout()
    fig.savefig(out_dir / "three_fig_J_optS_vs_flow.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [fig] three_fig_J_optS_vs_flow.png")

    # ── Figure K: Net power vs S at different Q_L (fresh, beta=0) ──
    fig, ax = plt.subplots(figsize=(8, 5))
    df_f0 = df_f[df_f["FOULING_BETA"] == 0.0]
    cmap2 = plt.cm.viridis
    for i, ql in enumerate(q_vals):
        color = cmap2(i / max(len(q_vals) - 1, 1))
        sub = df_f0[df_f0["Q_L_M3_S"] == ql].sort_values("S")
        if len(sub) > 0:
            ax.plot(sub["S"], sub["P_CASCADE_W"] * 1000, "o-", color=color,
                    label=f"Q_L={ql*1e6:.0f} mL/s", linewidth=2, markersize=7)
    ax.set_xlabel("Number of stages (S)", fontsize=12)
    ax.set_ylabel("Net power [mW]", fontsize=12)
    ax.set_title(f"Fresh cascade: staging curves at different flow rates\n"
                 f"N_total={n_total}, beta=0", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_dir / "three_fig_K_staging_by_flow.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [fig] three_fig_K_staging_by_flow.png")

    # ── Figure L: Fouling penalty vs Q_L (fresh, at optimal S per Q_L) ──
    fig, ax = plt.subplots(figsize=(8, 5))
    for beta in [0.4, 0.8]:
        penalties = []
        for ql in q_vals:
            sub_clean = df_f[(df_f["Q_L_M3_S"] == ql) & (df_f["FOULING_BETA"] == 0.0)]
            sub_foul = df_f[(df_f["Q_L_M3_S"] == ql) & (df_f["FOULING_BETA"] == beta)]
            if len(sub_clean) > 0 and len(sub_foul) > 0:
                p_clean = sub_clean["P_CASCADE_W"].max()
                p_foul = sub_foul["P_CASCADE_W"].max()
                penalty = (1 - p_foul / p_clean) * 100 if p_clean > 0 else 0
                penalties.append((ql, penalty))
        if penalties:
            qs, ps = zip(*penalties)
            ax.plot([q*1e6 for q in qs], ps, "o-",
                    label=f"beta={beta:.1f}", linewidth=2, markersize=7)
    ax.set_xlabel("Dilute flow rate Q_L [mL/s]", fontsize=12)
    ax.set_ylabel("Fouling penalty [%]", fontsize=12)
    ax.set_title(f"Fouling penalty vs flow rate (fresh cascade, optimal S)\n"
                 f"N_total={n_total}", fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_dir / "three_fig_L_penalty_vs_flow.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [fig] three_fig_L_penalty_vs_flow.png")


def _plot_three_config(df: pd.DataFrame, out_dir: Path, n_total: int,
                       a_mem_m2: float = 0.01) -> None:
    """Generate comparison figures for three-config experiment."""
    import matplotlib.pyplot as plt

    # ── Figure A: Net power vs S for each config, at beta=0 (clean) ──
    fig, ax = plt.subplots(figsize=(8, 5))
    df0 = df[df["FOULING_BETA"] == 0.0]
    for cfg in ["single", "serial", "fresh"]:
        sub = df0[df0["CONFIG"] == cfg].sort_values("S")
        if len(sub) > 0:
            ax.plot(sub["S"], sub["P_CASCADE_W"] * 1000,
                    "o-", color=_CONFIG_COLORS[cfg], label=_CONFIG_LABELS[cfg],
                    linewidth=2, markersize=8)
    ax.set_xlabel("Number of stages (S)", fontsize=12)
    ax.set_ylabel("Net power [mW]", fontsize=12)
    ax.set_title(f"Net power vs staging  |  N_total={n_total}, clean (beta=0)\n"
                 f"Total membrane area = {n_total * a_mem_m2:.2f} m2", fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_dir / "three_fig_A_net_power_clean.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [fig] three_fig_A_net_power_clean.png")

    # ── Figure B: Net power vs S at multiple beta levels (fresh config only) ──
    fig, ax = plt.subplots(figsize=(8, 5))
    df_f = df[df["CONFIG"] == "fresh"]
    betas = sorted(df_f["FOULING_BETA"].unique())
    cmap = plt.cm.RdYlGn_r
    for i, beta in enumerate(betas):
        sub = df_f[df_f["FOULING_BETA"] == beta].sort_values("S")
        if len(sub) > 0:
            color = cmap(i / max(len(betas) - 1, 1))
            ax.plot(sub["S"], sub["P_CASCADE_W"] * 1000,
                    "o-", color=color, label=f"beta={beta:.1f}",
                    linewidth=2, markersize=7)
    ax.set_xlabel("Number of stages (S)", fontsize=12)
    ax.set_ylabel("Net power [mW]", fontsize=12)
    ax.set_title(f"Fresh-dilute cascade: net power vs staging under fouling\n"
                 f"N_total={n_total}, A_mem={a_mem_m2} m2", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_dir / "three_fig_B_fresh_fouling_sweep.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [fig] three_fig_B_fresh_fouling_sweep.png")

    # ── Figure C: Optimal S* vs beta for serial and fresh ──
    fig, ax = plt.subplots(figsize=(8, 5))
    for cfg in ["serial", "fresh"]:
        df_c = df[df["CONFIG"] == cfg]
        opt_s = []
        for beta in sorted(df_c["FOULING_BETA"].unique()):
            sub = df_c[df_c["FOULING_BETA"] == beta]
            if len(sub) > 0:
                best = sub.loc[sub["P_CASCADE_W"].idxmax()]
                opt_s.append((beta, int(best["S"])))
        if opt_s:
            bs, ss = zip(*opt_s)
            ax.plot(bs, ss, "o-", color=_CONFIG_COLORS[cfg],
                    label=_CONFIG_LABELS[cfg], linewidth=2, markersize=8)
    ax.axhline(1, color=_CONFIG_COLORS["single"], linestyle="--",
               label="Single stack (S=1)", linewidth=1.5)
    ax.set_xlabel("Fouling severity (beta)", fontsize=12)
    ax.set_ylabel("Optimal number of stages (S*)", fontsize=12)
    ax.set_title(f"Optimal staging vs fouling  |  N_total={n_total}", fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)
    fig.tight_layout()
    fig.savefig(out_dir / "three_fig_C_optimal_S_vs_beta.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [fig] three_fig_C_optimal_S_vs_beta.png")

    # ── Figure D: Best achievable net power vs beta for each config ──
    fig, ax = plt.subplots(figsize=(8, 5))
    for cfg in ["single", "serial", "fresh"]:
        df_c = df[df["CONFIG"] == cfg]
        best_power = []
        for beta in sorted(df_c["FOULING_BETA"].unique()):
            sub = df_c[df_c["FOULING_BETA"] == beta]
            if len(sub) > 0:
                best_power.append((beta, sub["P_CASCADE_W"].max() * 1000))
        if best_power:
            bs, ps = zip(*best_power)
            ax.plot(bs, ps, "o-", color=_CONFIG_COLORS[cfg],
                    label=_CONFIG_LABELS[cfg], linewidth=2, markersize=8)
    ax.set_xlabel("Fouling severity (beta)", fontsize=12)
    ax.set_ylabel("Best net power [mW]", fontsize=12)
    ax.set_title(f"Best achievable net power vs fouling  |  N_total={n_total}", fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_dir / "three_fig_D_best_power_vs_fouling.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [fig] three_fig_D_best_power_vs_fouling.png")

    # ── Figure E: All three configs at beta=0 ──
    fig, ax = plt.subplots(figsize=(8, 5))
    df_f0 = df[(df["CONFIG"] == "fresh") & (df["FOULING_BETA"] == 0.0)].sort_values("S")
    df_s0 = df[(df["CONFIG"] == "serial") & (df["FOULING_BETA"] == 0.0)].sort_values("S")
    single_p = df[(df["CONFIG"] == "single") & (df["FOULING_BETA"] == 0.0)]["P_CASCADE_W"].values
    if len(single_p) > 0:
        ax.axhline(single_p[0] * 1000, color=_CONFIG_COLORS["single"],
                   linestyle="--", label="Single stack", linewidth=2)
    if len(df_s0) > 0:
        ax.plot(df_s0["S"], df_s0["P_CASCADE_W"] * 1000,
                "s-", color=_CONFIG_COLORS["serial"], label="Serial cascade",
                linewidth=2, markersize=7)
    if len(df_f0) > 0:
        ax.plot(df_f0["S"], df_f0["P_CASCADE_W"] * 1000,
                "o-", color=_CONFIG_COLORS["fresh"], label="Fresh cascade",
                linewidth=2, markersize=8)
    ax.set_xlabel("Number of stages (S)", fontsize=12)
    ax.set_ylabel("Net power [mW]", fontsize=12)
    ax.set_title(f"All three configs  |  N_total={n_total}, beta=0", fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_dir / "three_fig_E_all_configs_clean.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [fig] three_fig_E_all_configs_clean.png")


def _write_three_config_pdf(out_dir: Path, n_total: int, df: pd.DataFrame) -> None:
    """Write a PDF report for the three-config comparison."""
    from reportlab.lib.pagesizes import letter
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, Image as RLImage,
        Table, TableStyle,
    )
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.lib import colors

    pdf_path = out_dir / f"{out_dir.name}_THREE-CONFIG.pdf"
    doc = SimpleDocTemplate(str(pdf_path), pagesize=letter,
                            leftMargin=0.75*inch, rightMargin=0.75*inch,
                            topMargin=0.75*inch, bottomMargin=0.75*inch)
    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph(
        "OsmoFlux: Three-Configuration Cascade Comparison", styles["Title"]))
    story.append(Spacer(1, 12))
    story.append(Paragraph(
        f"Fixed total membrane: N_total={n_total} cell pairs, "
        f"A_mem=0.01 m2, total area={n_total*0.01:.2f} m2<br/>"
        f"Parasitic model: Hagen-Poiseuille pumping (laminar slit, mu=1 mPa.s) "
        f"+ 0.1 V electrode loss per stage (Veerman 2010)<br/>"
        f"Fouling: CDF divalent model (Dlugolecki 2010), beta swept 0.0-0.8<br/>"
        f"Configurations compared:<br/>"
        f"&nbsp;&nbsp;1. <b>Single stack</b>: S=1, N={n_total} cell pairs<br/>"
        f"&nbsp;&nbsp;2. <b>Serial cascade</b>: S stages, dilute outlet feeds next inlet<br/>"
        f"&nbsp;&nbsp;3. <b>Fresh-dilute cascade</b>: S stages, fresh dilute per stage",
        styles["Normal"]))
    story.append(Spacer(1, 18))

    story.append(Paragraph("Optimal configuration at each fouling level:", styles["Heading2"]))
    story.append(Spacer(1, 6))

    table_data = [["beta", "Best Config", "S*", "N/stage", "Net Power (mW)", "vs Single"]]
    single_powers = {}
    for beta in sorted(df["FOULING_BETA"].unique()):
        sub_s = df[(df["CONFIG"] == "single") & (df["FOULING_BETA"] == beta)]
        if len(sub_s) > 0:
            single_powers[beta] = sub_s["P_CASCADE_W"].values[0]

    for beta in sorted(df["FOULING_BETA"].unique()):
        sub = df[df["FOULING_BETA"] == beta]
        if len(sub) > 0:
            best = sub.loc[sub["P_CASCADE_W"].idxmax()]
            sp = single_powers.get(beta, 0)
            gain = ((best["P_CASCADE_W"] - sp) / abs(sp) * 100) if sp != 0 else 0
            table_data.append([
                f"{beta:.1f}",
                str(best["CONFIG"]),
                str(int(best["S"])),
                str(int(best["N_PER_STAGE"])),
                f"{best['P_CASCADE_W']*1000:.1f}",
                f"+{gain:.0f}%" if gain > 0 else f"{gain:.0f}%",
            ])

    t = Table(table_data, colWidths=[0.5*inch, 1.0*inch, 0.5*inch, 0.7*inch, 1.0*inch, 0.8*inch])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#0072BD")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 9),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F0F8FF")]),
    ]))
    story.append(t)
    story.append(Spacer(1, 18))

    fig_files = sorted(out_dir.glob("three_fig_*.png"))
    for fp in fig_files:
        story.append(RLImage(str(fp), width=6.5*inch, height=4.0*inch))
        story.append(Spacer(1, 12))

    doc.build(story)
    print(f"[3-CONFIG PDF] {pdf_path}")



# ── Cleaning cycle optimization ───────────────────────────────────────────────

def _cleaning_cycle_analysis(
    df: pd.DataFrame,
    out_dir: Path,
    n_total: int,
    tau_days: float = 14.0,
    beta_min: float = 0.05,
    beta_max: float = 0.70,
    elec_price_usd_kwh: float = 0.10,
    brine_value_usd_kwh: float = 2.50,
    cleaning_costs_per_m2: List[float] = None,
    plant_area_m2: float = 1000.0,
    t_min_days: float = 1.0,
) -> None:
    """Cleaning cycle optimization using existing three-config sweep data.

    Model: beta(t) = beta_max - (beta_max - beta_min) * exp(-t / tau)
    At T_clean, operator resets beta -> beta_min at cost C_clean.
    Finds T_clean maximizing long-run average net revenue.

    Economics are computed at plant_area_m2 scale (default 1000 m2)
    to produce physically meaningful cleaning cost tradeoffs.
    Cleaning cost is per m2 of membrane area per event.

    Parameters from literature:
        tau ~ 7-30 days (Rijnaarts 2017, CEM scaling timescale)
        beta_min ~ 0.05-0.1 (residual after acid wash)
        beta_max ~ 0.6-0.8 (asymptotic fouling severity)
    """
    import matplotlib.pyplot as plt
    from scipy.interpolate import interp1d

    if cleaning_costs_per_m2 is None:
        cleaning_costs_per_m2 = [0.01, 0.02, 0.05, 0.10, 0.20, 0.50, 1.0]

    a_mem_m2 = 0.01
    total_lab_area = n_total * a_mem_m2  # m2 at lab scale
    scale_factor = plant_area_m2 / total_lab_area  # scale lab -> plant

    print(f"[CLEANING] Running cleaning cycle optimization ...")
    print(f"[CLEANING] Plant scale: {plant_area_m2:.0f} m2 "
          f"(scale factor: {scale_factor:.0f}x lab)")

    # ── Build P(beta) interpolator for each config (at its optimal S) ──
    config_interps = {}
    config_opt_S = {}

    for cfg in ["single", "serial", "fresh"]:
        df_c = df[df["CONFIG"] == cfg]
        if len(df_c) == 0:
            continue

        betas_sorted = sorted(df_c["FOULING_BETA"].unique())
        best_powers = []
        opt_s_map = {}
        for beta in betas_sorted:
            sub = df_c[df_c["FOULING_BETA"] == beta]
            best = sub.loc[sub["P_CASCADE_W"].idxmax()]
            best_powers.append(float(best["P_CASCADE_W"]))
            opt_s_map[beta] = int(best["S"])

        config_interps[cfg] = interp1d(
            betas_sorted, best_powers,
            kind="linear", fill_value="extrapolate",
        )
        config_opt_S[cfg] = opt_s_map

    # ── Beta trajectory ──
    t_days = np.linspace(0, 365, 2000)
    beta_t = beta_max - (beta_max - beta_min) * np.exp(-t_days / tau_days)

    # ── Figure F: Power degradation curves ──
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    ax1.plot(t_days, beta_t, "k-", linewidth=2)
    ax1.set_xlabel("Days since last cleaning", fontsize=11)
    ax1.set_ylabel("Fouling severity (beta)", fontsize=11)
    ax1.set_title(f"Fouling trajectory\ntau={tau_days}d, "
                  f"beta_min={beta_min}, beta_max={beta_max}", fontsize=10)
    ax1.axhline(beta_min, color="green", linestyle="--", alpha=0.5,
                label=f"beta_min={beta_min}")
    ax1.axhline(beta_max, color="red", linestyle="--", alpha=0.5,
                label=f"beta_max={beta_max}")
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    for cfg, interp_fn in config_interps.items():
        P_t = np.array([float(interp_fn(np.clip(b, 0, 0.8))) for b in beta_t])
        P_t_plant = P_t * scale_factor  # scale to plant
        ax2.plot(t_days, P_t_plant / 1000,  # kW
                 "-", color=_CONFIG_COLORS[cfg],
                 label=_CONFIG_LABELS[cfg], linewidth=2)
    ax2.set_xlabel("Days since last cleaning", fontsize=11)
    ax2.set_ylabel(f"Net power [kW] at {plant_area_m2:.0f} m2 scale", fontsize=11)
    ax2.set_title("Power degradation between cleanings", fontsize=10)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(out_dir / "three_fig_F_power_degradation.png", dpi=150,
                bbox_inches="tight")
    plt.close(fig)
    print(f"  [fig] three_fig_F_power_degradation.png")

    # ── Compute optimal cleaning interval ──
    # Revenue = integral(P_plant(t) dt) * elec_price  [$ over interval]
    # Cost = C_clean_per_m2 * plant_area_m2            [$ per event]
    # Avg rate = (Revenue - Cost) / T                  [$/day]
    # Constraint: T >= t_min_days

    dt = t_days[1] - t_days[0]
    hours_per_day = 24.0
    results_clean = []

    for cfg, interp_fn in config_interps.items():
        P_t_W = np.array([float(interp_fn(np.clip(b, 0, 0.8))) for b in beta_t])
        P_plant_W = P_t_W * scale_factor
        # Cumulative energy [kWh]
        cum_energy_kwh = np.cumsum(P_plant_W * dt * hours_per_day) / 1000.0
        cum_revenue = cum_energy_kwh * (elec_price_usd_kwh + brine_value_usd_kwh)

        # No-cleaning baseline: steady-state at beta_max
        P_at_max = float(interp_fn(np.clip(beta_max, 0, 0.8))) * scale_factor
        no_clean_rate = P_at_max * hours_per_day / 1000.0 * (elec_price_usd_kwh + brine_value_usd_kwh)  # $/day (elec + brine disposal value)

        for c_per_m2 in cleaning_costs_per_m2:
            c_event = c_per_m2 * plant_area_m2  # total cost per cleaning event

            # Only consider T >= t_min_days
            mask = t_days[1:] >= t_min_days
            T_arr = t_days[1:][mask]
            rev_arr = cum_revenue[1:][mask]

            if len(T_arr) == 0:
                continue

            avg_rate = (rev_arr - c_event) / T_arr

            best_idx = np.argmax(avg_rate)
            best_T = T_arr[best_idx]
            best_rate = avg_rate[best_idx]

            improvement = ((best_rate - no_clean_rate) / abs(no_clean_rate) * 100
                           if no_clean_rate != 0 else 0)

            results_clean.append({
                "config": cfg,
                "C_clean_per_m2": c_per_m2,
                "C_event_total": c_event,
                "T_opt_days": best_T,
                "avg_revenue_usd_day": best_rate,
                "no_clean_revenue_usd_day": no_clean_rate,
                "improvement_pct": improvement,
                "cleanings_per_year": 365.25 / best_T,
            })

    df_clean = pd.DataFrame(results_clean)
    df_clean.to_parquet(out_dir / "cleaning_optimization.parquet", index=False)

    # ── Figure G: Optimal cleaning interval and revenue ──
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    for cfg in ["single", "serial", "fresh"]:
        sub = df_clean[df_clean["config"] == cfg]
        if len(sub) > 0:
            ax1.plot(sub["C_clean_per_m2"], sub["T_opt_days"],
                     "o-", color=_CONFIG_COLORS[cfg], label=_CONFIG_LABELS[cfg],
                     linewidth=2, markersize=7)
            ax2.plot(sub["C_clean_per_m2"], sub["avg_revenue_usd_day"],
                     "o-", color=_CONFIG_COLORS[cfg], label=_CONFIG_LABELS[cfg],
                     linewidth=2, markersize=7)

    ax1.set_xlabel("Cleaning cost [$/m2/event]", fontsize=11)
    ax1.set_ylabel("Optimal cleaning interval [days]", fontsize=11)
    ax1.set_title(f"Optimal cleaning interval\n"
                  f"Plant: {plant_area_m2:.0f} m2, tau={tau_days}d", fontsize=10)
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    ax2.set_xlabel("Cleaning cost [$/m2/event]", fontsize=11)
    ax2.set_ylabel("Avg net revenue [$/day]", fontsize=11)
    ax2.set_title(f"Revenue at optimal cleaning schedule\n"
                  f"Elec: ${elec_price_usd_kwh}/kWh", fontsize=10)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(out_dir / "three_fig_G_cleaning_optimization.png", dpi=150,
                bbox_inches="tight")
    plt.close(fig)
    print(f"  [fig] three_fig_G_cleaning_optimization.png")

    # ── Figure H: Revenue improvement from optimal cleaning ──
    fig, ax = plt.subplots(figsize=(8, 5))
    for cfg in ["single", "serial", "fresh"]:
        sub = df_clean[df_clean["config"] == cfg]
        if len(sub) > 0:
            ax.plot(sub["C_clean_per_m2"], sub["improvement_pct"],
                    "o-", color=_CONFIG_COLORS[cfg], label=_CONFIG_LABELS[cfg],
                    linewidth=2, markersize=7)
    ax.set_xlabel("Cleaning cost [$/m2/event]", fontsize=11)
    ax.set_ylabel("Revenue improvement vs no cleaning [%]", fontsize=11)
    ax.set_title(f"Value of optimal maintenance scheduling\n"
                 f"tau={tau_days}d, beta: {beta_min} -> {beta_max}, "
                 f"plant: {plant_area_m2:.0f} m2", fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color="gray", linestyle="-", alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_dir / "three_fig_H_cleaning_improvement.png", dpi=150,
                bbox_inches="tight")
    plt.close(fig)
    print(f"  [fig] three_fig_H_cleaning_improvement.png")

    # ── Console summary ──
    print(f"\n[CLEANING] Plant: {plant_area_m2:.0f} m2, "
          f"tau={tau_days}d, beta={beta_min}->{beta_max}")
    print(f"[CLEANING] Effective value: ${elec_price_usd_kwh}+${brine_value_usd_kwh}=${elec_price_usd_kwh+brine_value_usd_kwh:.2f}/kWh (elec+brine)")
    print(f"{'Config':<8} {'$/m2':>6} {'C_event':>8} {'T_opt':>7} "
          f"{'Rev$/d':>8} {'vs none':>8} {'Clean/yr':>9}")
    print("-" * 62)
    for _, row in df_clean.iterrows():
        print(f"{row['config']:<8} ${row['C_clean_per_m2']:>4.1f}  "
              f"${row['C_event_total']:>6.0f}  "
              f"{row['T_opt_days']:>5.1f}d  "
              f"${row['avg_revenue_usd_day']:>6.1f}  "
              f"{row['improvement_pct']:>+6.1f}%  "
              f"{row['cleanings_per_year']:>7.1f}")
    print(f"[CLEANING] Saved -> cleaning_optimization.parquet")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# OSMOFLUX GRAND SWEEP  —  v12
# ══════════════════════════════════════════════════════════════════════════════
# Run AFTER cells 0–11.  Two sequential experiments:
#
#   Part 1  Single-stage sweep    N=1..1200, 5 sites, all β (48,000 rows)
#             Extended to N=1200 to capture compliance windows for Jebel Ali
#             and Hypersaline under heavy fouling (β=1.5 window ≈ N=990).
#             All β values swept (not just β=0) to characterize fouling-
#             dependent shift of compliance window and inversion boundary.
#   Part 2  Cascade sweep         8 N_total × 5 sites × 5 C_L × 8 β × 2 arch
#   Part 3  Figures               11 publication figures + 3 N_total curve figs
#   Part 4  Morris screening      r=50, 12 params, 5 sites × 3 base-β anchors
#   Part 5  MC-TEA                COMMENTED OUT — run separately
#   Part 6  Dashboard JSON
#   Part 7  Freeze Manifest
#
# ── FOULING CONFIGURATION ────────────────────────────────────────────────────
# ENABLE_BIOFILM = True   → combined CDF + biofilm (v12 default, recommended)
# ENABLE_BIOFILM = False  → CDF-only (v9 behavior, for regression comparison)
#
# When ENABLE_BIOFILM = True:
#   Total fouling per segment j:
#     foul_total(j) = CDF(j) + biofilm(j)
#
#   CDF (inlet-heavy, mechanistic):
#     foul_cdf(j) = β × f_div × (C_H,j / C_ref)^0.5
#     Represents divalent cation binding to CEM sulfonate groups.
#     Peak at stack inlet where C_H is highest.
#
#   Biofilm (outlet-heavy, phenomenological):
#     foul_bio(j) = β × BIOFILM_AMP × (exp(k·ξ) - 1) / (exp(k) - 1)
#     where ξ = j/N_SEG (normalised axial position, 0→1)
#           k = BIOFILM_K (shape parameter, default 4.0)
#     Represents biological/organic accumulation that grows with residence time.
#     Peak at stack outlet.
#
#   Both terms scale with the same β so that β means "overall fouling severity":
#     β = 0.0  → clean membranes (both terms zero)
#     β = 0.2  → mild fouling
#     β = 1.5  → severe fouling (Vermaas 2013 field regime)
#
#   BIOFILM_AMP = 1.0 (default) gives biofilm the same maximum amplitude as
#   CDF at reference conditions. Set BIOFILM_AMP < 1.0 to weight CDF higher,
#   or > 1.0 to weight biofilm higher.
#
# When ENABLE_BIOFILM = False:
#   FOULING_MODEL = "none" → biofilm term is zero, β drives CDF only.
#   Reproduces v9 exactly. Use this to confirm that changing ENABLE_BIOFILM
#   does not affect the β=0 baseline (it never should).
#
# Physical justification for SWRO brine:
#   SWRO concentrate is chlorinated and biologically inhibited, so biological
#   fouling is suppressed relative to natural water systems. CDF is therefore
#   the primary mechanism at SWRO sites. The biofilm term is included to test
#   robustness of the architecture-selection finding across the full spatial
#   fouling profile, not to represent a dominant mechanism at these sites.
# ─────────────────────────────────────────────────────────────────────────────

import hashlib, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

_T0   = time.time()
_TS   = time.strftime("%Y%m%d_%H%M%S")
_ROOT = Path("runs/GRAND-SWEEP") / _TS
(_ROOT / "figures").mkdir(parents=True, exist_ok=True)
FIG   = _ROOT / "figures"

# ─────────────────────────────────────────────────────────────────────────────
# FOULING CONFIGURATION — edit here only
# ─────────────────────────────────────────────────────────────────────────────

ENABLE_BIOFILM = True       # True = combined CDF + biofilm (v12)
                            # False = CDF only, reproduces v9 exactly

BIOFILM_AMP    = 1.0        # Biofilm maximum amplitude relative to β.
                            # At β=1.0, f_div=1.0, biofilm peaks at BIOFILM_AMP
                            # at the stack outlet. Default 1.0 = equal weight
                            # with CDF at reference conditions.

BIOFILM_K      = 4.0        # Shape parameter for exponential biofilm profile.
                            # Higher k → more back-loaded (sharper outlet peak).
                            # k=4 matches the legacy exp_outlet function.

# Derived: the FOULING_MODEL string passed to the physics engine
_FOULING_MODEL = "exp_outlet" if ENABLE_BIOFILM else "none"

# ─────────────────────────────────────────────────────────────────────────────
# BIOFILM SCALING PATCH
# ─────────────────────────────────────────────────────────────────────────────
# The legacy fouling_exp_outlet function has beta=0.40 hardcoded. We replace it
# with a version that accepts the FOULING_BETA parameter so β controls both
# mechanisms uniformly. This patch is applied before any model calls.
#
# The patched function computes:
#   foul_bio(j) = FOULING_BETA × BIOFILM_AMP × (exp(k·ξ)-1)/(exp(k)-1)
#
# When ENABLE_BIOFILM = False, FOULING_MODEL = "none" so this function is
# never called — the patch is harmless but present for completeness.

def _patched_exp_outlet_factory(amp, k):
    """Return a closure that reads FOULING_BETA from the DesignInputs p object.

    The returned function matches the signature expected by segment_resistance_ohm:
        phenom = f(x)   where x is the normalised axial position.

    However, segment_resistance_ohm passes only x; it cannot pass p.
    We therefore use a thread-local workaround: we store the current β in a
    module-level variable _CURRENT_BETA that is set once per model call in the
    grand sweep's _run_cascade wrapper, before calling run_cascade().
    """
    _k   = float(k)
    _amp = float(amp)
    _den = float(np.exp(_k) - 1.0)

    def _f(x: float) -> float:
        # Read current beta from the sweep context variable
        beta = float(_SWEEP_BETA)
        num  = float(np.exp(_k * float(x)) - 1.0)
        return beta * _amp * (num / max(_den, 1e-30))
    return _f

# Context variable: grand sweep sets this before each model call
_SWEEP_BETA = 0.0

# Apply patch to the global FOULING_FUNCS registry
_patched_biofilm = _patched_exp_outlet_factory(BIOFILM_AMP, BIOFILM_K)
if ENABLE_BIOFILM:
    FOULING_FUNCS["exp_outlet"] = _patched_biofilm
    print(f"  Biofilm patch applied: exp_outlet now scales with β "
          f"(AMP={BIOFILM_AMP}, k={BIOFILM_K})")
else:
    print(f"  Biofilm disabled (ENABLE_BIOFILM=False): FOULING_MODEL='none'")

# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────

_N_TOTALS        = [60, 120, 180, 240, 300, 360, 420, 480, 600]
_N_TOTAL_PRIMARY = 240
_A_MEM           = 0.05   # m²/pair
_Q_H             = 5.0e-6 # m³/s
_Q_L             = 5.0e-6 # m³/s
_GAP             = 2.0e-4 # m
_N_SEG           = 40

def _s_vals(n_total):
    """Valid S: exact divisors of n_total, N_per≥12, S>1."""
    return sorted({s for s in range(2, n_total + 1)
                   if n_total % s == 0 and n_total // s >= 12})

_BETA_NONZERO = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.5]
_BETA_ALL     = [0.0] + _BETA_NONZERO
_CL_VALS      = [0.02, 0.05, 0.10, 0.20, 0.50]

_SITES = [
    dict(tag="VALLEY-WATER", c_h=0.35, sw=0.30,  f_div=0.10,
         c_l_primary=0.02, brine_flow=50_000.,
         elec_price=0.22, disp_lo=0.04, disp_hi=0.10, mc_seed=500,
         label="Valley Water Brackish GW (0.35 mol/L)"),
    dict(tag="MONTEREY",     c_h=0.75, sw=0.600, f_div=0.15,
         c_l_primary=0.02, brine_flow=30_000.,
         elec_price=0.22, disp_lo=0.04, disp_hi=0.15, mc_seed=600,
         label="Monterey Bay Cal-Am MPWSP (0.75 mol/L)"),
    dict(tag="CARLSBAD",     c_h=1.15, sw=0.600, f_div=0.20,
         c_l_primary=0.02, brine_flow=190_000.,
         elec_price=0.22, disp_lo=0.04, disp_hi=0.25, mc_seed=42,
         label="Carlsbad Poseidon SWRO (1.15 mol/L)"),
    dict(tag="JEBEL-ALI",    c_h=1.80, sw=0.650, f_div=0.35,
         c_l_primary=0.02, brine_flow=500_000.,
         elec_price=0.08, disp_lo=0.08, disp_hi=0.20, mc_seed=99,
         label="Jebel Ali Arabian Gulf SWRO (1.80 mol/L)"),
    dict(tag="HYPERSALINE",  c_h=3.00, sw=0.600, f_div=0.35,
         c_l_primary=0.02, brine_flow=50_000.,
         elec_price=0.15, disp_lo=0.20, disp_hi=1.00, mc_seed=777,
         label="Hypersaline ZLD Brine (3.00 mol/L)"),
]
_MC_SITES   = {"VALLEY-WATER", "CARLSBAD", "JEBEL-ALI"}
_PILOT_AREA = 1_000  # m²

def _site_obj(tag):
    return next(s for s in _SITES if s["tag"] == tag)

print("=" * 70)
print(f"  OSMOFLUX GRAND SWEEP  v12  |  {_TS}")
print(f"  Output: {_ROOT}")
print(f"  Fouling model: {'CDF + biofilm (combined)' if ENABLE_BIOFILM else 'CDF only (v9 baseline)'}")
print("=" * 70)
print(f"\n  N_total values  : {_N_TOTALS}")
print(f"  Areas (m²)      : {[n*_A_MEM for n in _N_TOTALS]}")
print(f"  β values        : {_BETA_ALL}")
print(f"  C_L values      : {_CL_VALS}")
print(f"  Primary N_total : {_N_TOTAL_PRIMARY}")
print(f"  FOULING_MODEL   : {_FOULING_MODEL!r}")
print(f"  BIOFILM_AMP     : {BIOFILM_AMP}  (only used when ENABLE_BIOFILM=True)")

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _cl_physical(r):
    """True dilute outlet — NOT C_L_OUT_FINAL (reset value for fresh-dilute)."""
    try:
        arr = r["per_stage"][-1]["per_segment"]["C_L_OUT_MOL_L"]
        if arr is not None and len(arr) > 0:
            return float(arr[-1])
    except (KeyError, IndexError, TypeError):
        pass
    return float(r.get("C_L_OUT_FINAL", float("nan")))

def _hr(c_h_in, c_h_out, c_l_out, sw):
    base = max(c_h_in - sw, 0.0)
    if base == 0.0:
        return 0.0
    worst = max(max(c_h_out - sw, 0.0), max(c_l_out - sw, 0.0))
    return max(0.0, (1.0 - worst / base) * 100.0)

def _bk(site, c_l, beta):
    """Build base knobs for a model call, setting β context variable."""
    return {
        "C_H_IN_MOL_L": site["c_h"], "C_L_IN_MOL_L": c_l,
        "Q_H_M3_S": _Q_H, "Q_L_M3_S": _Q_L, "GAP_M": _GAP,
        "DIVALENT_FRACTION": site["f_div"], "FOULING_BETA": beta,
        "FOULING_MODEL": _FOULING_MODEL,
        "FOULING_C_REF_MOL_L": 0.5, "FOULING_N_EXP": 0.5,
    }

def _run_cascade(bk, n_stages, n_per_stage, fresh_dilute):
    """Thin wrapper: set _SWEEP_BETA context variable before calling run_cascade.

    This allows the patched fouling_exp_outlet closure to read the current β
    without needing access to the DesignInputs object directly.
    """
    global _SWEEP_BETA
    _SWEEP_BETA = float(bk.get("FOULING_BETA", 0.0))
    return run_cascade(bk, n_stages=n_stages, n_per_stage=n_per_stage,
                       a_mem_m2=_A_MEM, n_seg=_N_SEG,
                       fresh_dilute=fresh_dilute)

def _make_row(tag, site, c_l, beta, s, np_, arch, fresh, r, sw, n_total):
    clp = _cl_physical(r)
    cho = float(r["C_H_OUT_FINAL"])
    hr  = _hr(site["c_h"], cho, clp, sw)
    return {
        "SITE_TAG": tag, "C_H_IN_MOL_L": site["c_h"],
        "C_L_IN_MOL_L": c_l, "SW_MOL_L": sw,
        "DIVALENT_FRACTION": site["f_div"],
        "FOULING_BETA": beta, "S": s, "N_PER_STAGE": np_,
        "N_TOTAL": n_total, "A_MEM_M2": _A_MEM,
        "TOTAL_AREA_M2":       n_total * _A_MEM,
        "CONFIG": arch, "FRESH_DILUTE": fresh,
        "P_CASCADE_W":         float(r["P_CASCADE_W"]),
        "P_DENS_CASCADE_W_M2": float(r["P_DENS_CASCADE_W_M2"]),
        "P_CLEAN_CASCADE_W":   float(r["P_CLEAN_CASCADE_W"]),
        "C_H_OUT_FINAL":       cho,
        "C_L_OUT_FINAL":       float(r["C_L_OUT_FINAL"]),
        "C_L_OUT_PHYSICAL":    clp,
        "HARM_REDUX_PCT":      hr,
        "COMPLIANCE_FLAG":     bool(cho <= sw and clp <= sw),
        "ANY_INVERSION":       bool(r["any_inversion"]),
        "PHYS_COMPLY":         bool(cho <= sw and clp <= sw and not r["any_inversion"]),
    }

# ─────────────────────────────────────────────────────────────────────────────
# PART 1: SINGLE-STAGE COMPLIANCE SWEEP
# ─────────────────────────────────────────────────────────────────────────────
# Sweeps N = 1–200 at ALL β values (not just β=0) to characterize how fouling
# affects: (a) the unconstrained power optimum, (b) the compliance window
# width and position, (c) the inversion boundary, and (d) the power-compliance
# penalty as a function of fouling severity.
#
# Row count: 5 sites × 1200 N × 8 β = 48,000 rows.
# The β=0 rows reproduce v9/v10 clean baseline; fouled rows are new.
#
# The FOULING_MODEL and biofilm patch from the preamble apply here exactly as
# in Part 2: at β=0 both CDF and biofilm are zero regardless of model setting.
print(f"\n{'─'*70}")
print("  PART 1: Single-Stage Compliance Sweep (N=1..1200, 5 sites, all β)")
print(f"{'─'*70}")

ss_rows, _t1 = [], time.time()
_ss_total = len(_SITES) * 1200 * len(_BETA_ALL)
print(f"  Expected rows: {_ss_total:,}  "
      f"(5 sites × 1200 N × {len(_BETA_ALL)} β values)\n")

for site in _SITES:
    c_h, sw, tag = site["c_h"], site["sw"], site["tag"]

    for beta in _BETA_ALL:
        bk = _bk(site, site["c_l_primary"], beta)
        peak_hr, peak_N = 0.0, 0
        inv_N, first_c, last_c = None, None, None
        peak_P_unconstrained, peak_N_unconstrained = 0.0, 0

        for N in range(1, 1201):
            r   = _run_cascade(bk, n_stages=1, n_per_stage=N, fresh_dilute=True)
            clp = _cl_physical(r)
            cho = float(r["C_H_OUT_FINAL"])
            p   = float(r["P_DENS_CASCADE_W_M2"])
            hr  = _hr(c_h, cho, clp, sw)
            inv = bool(r["any_inversion"])
            pc  = bool(cho <= sw and clp <= sw and not inv)

            if inv_N is None and inv:   inv_N = N
            if pc and first_c is None:  first_c = N
            if pc:                      last_c = N
            if hr > peak_hr:            peak_hr, peak_N = hr, N
            # Track absolute peak power regardless of compliance
            if p > peak_P_unconstrained:
                peak_P_unconstrained = p
                peak_N_unconstrained = N

            ss_rows.append({
                "SITE_TAG":         tag,
                "C_H_IN_MOL_L":     c_h,
                "C_L_IN_MOL_L":     site["c_l_primary"],
                "SW_MOL_L":         sw,
                "FOULING_BETA":     beta,
                "FOULING_MODEL":    _FOULING_MODEL,
                "N":                N,
                "TOTAL_AREA_M2":    N * _A_MEM,
                "P_DENS_W_M2":      p,
                "C_H_OUT":          cho,
                "C_L_OUT_PHYSICAL": clp,
                "HARM_REDUX_PCT":   hr,
                "ANY_INVERSION":    inv,
                "PHYS_COMPLY":      pc,
            })

            # Once inversion is detected, all higher N will also invert.
            # Record this row then stop — no physical mechanism recovers
            # a stack from inversion by adding more cell pairs.
            if inv:
                break

        # Summary line per (site, beta)
        window_str = (f"N={first_c}–{last_c} ({last_c-first_c+1} pairs)"
                      if first_c else "none")
        print(f"  [{tag} β={beta:.1f}]  "
              f"unconstrained_peak={peak_P_unconstrained:.3f} W/m² @N={peak_N_unconstrained}  "
              f"window={window_str}  inv_N={inv_N}")

df_ss = pd.DataFrame(ss_rows)
df_ss.to_parquet(_ROOT / "single_stage_sweep.parquet", index=False)
print(f"\n  single_stage_sweep.parquet → {len(df_ss):,} rows"
      f"  ({time.time()-_t1:.0f}s)")

# ─────────────────────────────────────────────────────────────────────────────
# PART 2: CASCADE SWEEP
# ─────────────────────────────────────────────────────────────────────────────
# Parallelised across sites: each site is independent and runs in its own
# process.  Results are checkpointed per (N_total, site) block to disk so a
# crash loses at most one block rather than the entire run.
#
# Checkpoint files: _ROOT / "checkpoints" / f"{n_total}_{tag}.parquet"
# Final merge: all checkpoints concatenated → cascade_sweep.parquet
#
# Speed: typically 4–5× faster than serial on a quad-core machine.
# The _clean_cache is per-process (no shared state needed; each process
# recomputes β=0 for its own site independently).
# ─────────────────────────────────────────────────────────────────────────────

import multiprocessing as _mp

_CKPT_DIR = _ROOT / "checkpoints"
_CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n{'─'*70}")
print(f"  PART 2: Cascade Sweep  (N_total ∈ {_N_TOTALS})")
print(f"          Parallelised across {len(_SITES)} sites")
print(f"          Checkpointing per (N_total, site) block")
print(f"{'─'*70}")

_total_expected = sum(
    len(_s_vals(nt)) * len([c for c in _CL_VALS if c < s["c_h"]])
    * len(_BETA_ALL) * 2
    for nt in _N_TOTALS
    for s in _SITES
)
print(f"  Expected rows: {_total_expected:,}\n")

def _run_site_block(args):
    """Worker: compute all rows for one (n_total, site) block.

    Runs in a child process.  Has its own _SWEEP_BETA, _clean_cache,
    and FOULING_FUNCS patch — no shared state with other workers.
    Called via multiprocessing.Pool; returns the checkpoint path.
    """
    n_total, site, beta_all, beta_nonzero, cl_vals, a_mem, n_seg,         fouling_model, biofilm_amp, biofilm_k, enable_biofilm,         ckpt_path_str = args

    import numpy as _np

    # Re-apply biofilm patch in this child process
    _sweep_beta_local = [0.0]  # mutable container for closure

    def _patched_factory_local(amp, k):
        _k = float(k); _amp = float(amp)
        _den = float(_np.exp(_k) - 1.0)
        def _f(x):
            beta = _sweep_beta_local[0]
            num  = float(_np.exp(_k * float(x)) - 1.0)
            return beta * _amp * (num / max(_den, 1e-30))
        return _f

    if enable_biofilm:
        FOULING_FUNCS["exp_outlet"] = _patched_factory_local(biofilm_amp, biofilm_k)

    def _rc(bk, n_stages, n_per_stage, fresh_dilute):
        _sweep_beta_local[0] = float(bk.get("FOULING_BETA", 0.0))
        return run_cascade(bk, n_stages=n_stages, n_per_stage=n_per_stage,
                           a_mem_m2=a_mem, n_seg=n_seg,
                           fresh_dilute=fresh_dilute)

    tag, c_h, sw = site["tag"], site["c_h"], site["sw"]
    svs       = _s_vals(n_total)
    cl_valid  = [c for c in cl_vals if c < c_h]
    rows      = []
    cache     = {}

    for c_l in cl_valid:
        for s in svs:
            np_ = n_total // s
            for arch, fresh in [("fresh", True), ("serial", False)]:
                cache_key = (tag, c_l, s, np_, n_total, arch)
                if cache_key not in cache:
                    bk0 = {"C_H_IN_MOL_L": c_h, "C_L_IN_MOL_L": c_l,
                            "Q_H_M3_S": 5e-6, "Q_L_M3_S": 5e-6,
                            "GAP_M": 2e-4, "DIVALENT_FRACTION": site["f_div"],
                            "FOULING_BETA": 0.0, "FOULING_MODEL": fouling_model,
                            "FOULING_C_REF_MOL_L": 0.5, "FOULING_N_EXP": 0.5}
                    r0 = _rc(bk0, s, np_, fresh)
                    cache[cache_key] = float(r0["P_CASCADE_W"])
                    rows.append(_make_row(tag, site, c_l, 0.0, s,
                                         np_, arch, fresh, r0, sw, n_total))
                for beta in beta_nonzero:
                    bk = {"C_H_IN_MOL_L": c_h, "C_L_IN_MOL_L": c_l,
                           "Q_H_M3_S": 5e-6, "Q_L_M3_S": 5e-6,
                           "GAP_M": 2e-4, "DIVALENT_FRACTION": site["f_div"],
                           "FOULING_BETA": beta, "FOULING_MODEL": fouling_model,
                           "FOULING_C_REF_MOL_L": 0.5, "FOULING_N_EXP": 0.5}
                    r  = _rc(bk, s, np_, fresh)
                    row = _make_row(tag, site, c_l, beta, s,
                                    np_, arch, fresh, r, sw, n_total)
                    row["P_CLEAN_CASCADE_W"] = cache[cache_key]
                    rows.append(row)

    import pandas as _pd
    df_block = _pd.DataFrame(rows)
    ckpt = _pd.DataFrame(rows)
    ckpt.to_parquet(ckpt_path_str, index=False)
    return ckpt_path_str, len(rows)

# Build task list: one task per (n_total, site)
_tasks = []
for n_total in _N_TOTALS:
    for site in _SITES:
        ckpt_path = str(_CKPT_DIR / f"{n_total}_{site['tag']}.parquet")
        _tasks.append((
            n_total, site, _BETA_ALL, _BETA_NONZERO, _CL_VALS,
            _A_MEM, _N_SEG, _FOULING_MODEL, BIOFILM_AMP, BIOFILM_K,
            ENABLE_BIOFILM, ckpt_path
        ))

# Use min(5, cpu_count) workers — 5 sites run in parallel per N_total
_n_workers = min(len(_SITES), _mp.cpu_count())
print(f"  Using {_n_workers} parallel workers ({_mp.cpu_count()} CPUs available)")

# Jupyter / macOS compatibility: spawn requires __main__ guard which notebooks
# lack.  Fork avoids this; it is the default on Linux but must be forced on
# macOS + Jupyter: default start method is 'spawn', which requires the
# __main__ guard that notebooks lack.  Force 'fork' instead.
# On Linux, fork is already the default and this is a no-op.
try:
    if _mp.get_start_method(allow_none=True) != "fork":
        _mp.set_start_method("fork", force=True)
    print(f"  Multiprocessing start method: {_mp.get_start_method()}")
except RuntimeError as _e:
    print(f"  WARNING: {_e} — falling back to sequential (n_workers=1)")
    _n_workers = 1

_t2 = time.time()

_ckpt_paths = []
with _mp.Pool(processes=_n_workers) as pool:
    for i, (ckpt_path, n_rows) in enumerate(
            pool.imap_unordered(_run_site_block, _tasks), 1):
        _ckpt_paths.append(ckpt_path)
        pct     = i / len(_tasks) * 100
        elapsed = time.time() - _t2
        eta     = elapsed / i * (len(_tasks) - i)
        print(f"  [{i:>3}/{len(_tasks)}  {pct:>5.1f}%]  "
              f"{ckpt_path.split('/')[-1]:<35}  "
              f"{n_rows:>5} rows  {elapsed:.0f}s  ETA {eta:.0f}s",
              flush=True)

# Merge all checkpoints → final parquet
print(f"\n  Merging {len(_ckpt_paths)} checkpoint files...")
import pandas as _pd_merge
df_cas = _pd_merge.concat(
    [_pd_merge.read_parquet(p) for p in sorted(_ckpt_paths)],
    ignore_index=True
)
df_cas.to_parquet(_ROOT / "cascade_sweep.parquet", index=False)
print(f"  cascade_sweep.parquet → {len(df_cas):,} rows"
      f"  ({time.time()-_t2:.0f}s)")

# PART 3: FIGURES
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*70}")
print("  PART 3: Figures")
print(f"{'─'*70}")

cb_ss = df_ss[(df_ss["SITE_TAG"]=="CARLSBAD") & (df_ss["FOULING_BETA"]==0.0)].copy()

# ── Fig 1: Single-stage compliance ceiling ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Fig 1: Single-Stage Compliance Window — Carlsbad (C_H=1.15 mol/L, C_L=0.02)\n"
             "Compliance exists only in a narrow physically valid window (7 cell pairs)",
             fontsize=10, fontweight="bold")
ax = axes[0]
ax.plot(cb_ss["N"], cb_ss["P_DENS_W_M2"], color="#0072BD", lw=2)
pr = cb_ss.loc[cb_ss["HARM_REDUX_PCT"].idxmax()]
ax.axvline(pr["N"], color="green", ls="--", lw=1.5,
           label=f"Peak HR at N={int(pr['N'])} ({pr['HARM_REDUX_PCT']:.1f}%)")
inv_r = cb_ss[cb_ss["ANY_INVERSION"]]
if len(inv_r) > 0:
    ax.axvline(inv_r["N"].min(), color="red", ls=":", lw=1.5,
               label=f"First inversion N={inv_r['N'].min()}")
ax.set_xlabel("Cell pairs N"); ax.set_ylabel("Power density [W/m²]")
ax.set_title("Power density"); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax = axes[1]
ax.plot(cb_ss["N"], cb_ss["HARM_REDUX_PCT"], color="#7E2F8E", lw=2)
ax.axhline(100, color="green", ls=":", lw=1, label="Full compliance")
if len(inv_r) > 0:
    ax.axvline(inv_r["N"].min(), color="red", ls=":", lw=1.5)
ax.fill_between(cb_ss["N"], cb_ss["HARM_REDUX_PCT"],
                where=cb_ss["ANY_INVERSION"], alpha=0.15, color="red",
                label="Inversion zone")
pc_r = cb_ss[cb_ss["PHYS_COMPLY"]]
if len(pc_r) > 0:
    ax.axvspan(pc_r["N"].min(), pc_r["N"].max(), alpha=0.15, color="green",
               label=f"Compliance window N={pc_r['N'].min()}–{pc_r['N'].max()}")
ax.set_xlabel("Cell pairs N"); ax.set_ylabel("Harm Reduction [%]")
ax.set_title("Physically valid compliance window")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIG / "fig1_single_stage_compliance_ceiling.png", dpi=150, bbox_inches="tight")
plt.close(); print("  fig1", flush=True)

# ── Fig 2: Cascade vs single-stage matched area (primary N_total) ─────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"Fig 2: Cascade vs Single-Stage — Carlsbad, Matched Area "
             f"(N_total={_N_TOTAL_PRIMARY})", fontsize=11, fontweight="bold")
cb_v = df_prim_v[(df_prim_v["SITE_TAG"]=="CARLSBAD") &
                 (df_prim_v["C_L_IN_MOL_L"]==0.02) &
                 (df_prim_v["FOULING_BETA"]==0.0)]
for ai, col_y in enumerate(["P_DENS_CASCADE_W_M2", "HARM_REDUX_PCT"]):
    ax = axes[ai]
    for arch, col in [("fresh","#0072BD"),("serial","#D95319")]:
        sub = cb_v[cb_v["CONFIG"]==arch].sort_values("S")
        ax.plot(sub["TOTAL_AREA_M2"], sub[col_y], "o-", color=col,
                label=f"Cascade {arch}", lw=2, ms=6)
    ss_s = cb_ss.sort_values("N")
    y_ss = ss_s["P_DENS_W_M2"] if ai == 0 else ss_s["HARM_REDUX_PCT"]
    ax.plot(ss_s["TOTAL_AREA_M2"], y_ss, "k--", lw=1.5, label="Single stage", alpha=0.7)
    if ai == 1:
        ax.axhline(100, color="green", ls=":", lw=1, label="Full compliance")
    ax.set_xlabel("Total membrane area [m²]"); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
axes[0].set_ylabel("Power density [W/m²]")
axes[1].set_ylabel("Harm Reduction [%]")
plt.tight_layout()
fig.savefig(FIG / "fig2_cascade_vs_single_stage.png", dpi=150, bbox_inches="tight")
plt.close(); print("  fig2", flush=True)

# ── Fig 3: Power density vs S — all sites (primary N_total) ───────────────────
fig, axes = plt.subplots(1, len(_SITES), figsize=(18, 4))
fig.suptitle(f"Fig 3: Power Density vs S (N_total={_N_TOTAL_PRIMARY}, β=0, C_L=primary)",
             fontsize=11, fontweight="bold")
for ax, site in zip(axes, _SITES):
    sub = df_prim_v[(df_prim_v["SITE_TAG"]==site["tag"]) &
                    (df_prim_v["C_L_IN_MOL_L"]==site["c_l_primary"]) &
                    (df_prim_v["FOULING_BETA"]==0.0)]
    for arch, col in [("fresh","#0072BD"),("serial","#D95319")]:
        s = sub[sub["CONFIG"]==arch].sort_values("S")
        if len(s) > 0:
            ax.plot(s["S"], s["P_DENS_CASCADE_W_M2"], "o-", color=col,
                    label=arch.capitalize(), lw=2, ms=5)
    ax.set_title(f"C_H={site['c_h']}", fontsize=9)
    ax.set_xlabel("S"); ax.grid(True, alpha=0.3)
    if ax == axes[0]: ax.set_ylabel("Power density [W/m²]")
axes[-1].legend(fontsize=8)
plt.tight_layout()
fig.savefig(FIG / "fig3_power_vs_stages_all_sites.png", dpi=150, bbox_inches="tight")
plt.close(); print("  fig3", flush=True)

# ── Fig 4: Fresh advantage % vs S — all sites (primary N_total) ───────────────
fig, axes = plt.subplots(1, len(_SITES), figsize=(18, 4))
fig.suptitle(f"Fig 4: Fresh-Dilute Advantage Over Serial [%] (N_total={_N_TOTAL_PRIMARY})",
             fontsize=11, fontweight="bold")
for ax, site in zip(axes, _SITES):
    tag, c_l = site["tag"], site["c_l_primary"]
    for b, col in [(0.0,"#0d6efd"),(0.4,"#fd7e14"),(0.8,"#dc3545")]:
        sf = df_prim_v[(df_prim_v["SITE_TAG"]==tag)&(df_prim_v["CONFIG"]=="fresh")&
                       (df_prim_v["FOULING_BETA"]==b)&(df_prim_v["C_L_IN_MOL_L"]==c_l)]
        ss = df_prim_v[(df_prim_v["SITE_TAG"]==tag)&(df_prim_v["CONFIG"]=="serial")&
                       (df_prim_v["FOULING_BETA"]==b)&(df_prim_v["C_L_IN_MOL_L"]==c_l)]
        if len(sf)==0 or len(ss)==0: continue
        m = sf[["S","P_DENS_CASCADE_W_M2"]].merge(
            ss[["S","P_DENS_CASCADE_W_M2"]], on="S", suffixes=("_f","_s"))
        m = m[m["P_DENS_CASCADE_W_M2_s"] > 0]
        if len(m) > 0:
            ax.plot(m["S"], (m["P_DENS_CASCADE_W_M2_f"]/m["P_DENS_CASCADE_W_M2_s"]-1)*100,
                    "o-", color=col, label=f"β={b}", lw=2, ms=5)
    ax.axhline(0, color="gray", lw=0.8, ls="--")
    ax.set_title(f"C_H={site['c_h']}", fontsize=9)
    ax.set_xlabel("S"); ax.grid(True, alpha=0.3)
    if ax == axes[0]: ax.set_ylabel("Advantage [%]")
axes[-1].legend(fontsize=8)
plt.tight_layout()
fig.savefig(FIG / "fig4_fresh_advantage_all_sites.png", dpi=150, bbox_inches="tight")
plt.close(); print("  fig4", flush=True)

# ── Fig 5: Harm reduction — Carlsbad cascade vs single-stage ceiling ──────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"Fig 5: Harm Reduction — Carlsbad, C_L=0.02 (N_total={_N_TOTAL_PRIMARY})",
             fontsize=11, fontweight="bold")
for ax, beta in zip(axes, [0.0, 0.8]):
    for arch, col in [("fresh","#0072BD"),("serial","#D95319")]:
        sub = df_prim_v[(df_prim_v["SITE_TAG"]=="CARLSBAD")&
                        (df_prim_v["CONFIG"]==arch)&
                        (df_prim_v["FOULING_BETA"]==beta)&
                        (df_prim_v["C_L_IN_MOL_L"]==0.02)].sort_values("S")
        if len(sub) > 0:
            ax.plot(sub["S"], sub["HARM_REDUX_PCT"], "o-", color=col,
                    label=f"Cascade {arch}", lw=2, ms=6)
    ax.axhline(100, color="green", ls=":", lw=1, label="Full compliance")
    if beta == 0.0:
        pr = cb_ss.loc[cb_ss["HARM_REDUX_PCT"].idxmax()]
        ax.axhline(pr["HARM_REDUX_PCT"], color="black", ls="--", lw=1,
                   label=f"SS ceiling {pr['HARM_REDUX_PCT']:.0f}% (N={int(pr['N'])})")
    ax.set_xlabel("Stage count S"); ax.set_ylabel("Harm Reduction [%]")
    ax.set_title(f"β = {beta}"); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIG / "fig5_harm_reduction_with_ceiling.png", dpi=150, bbox_inches="tight")
plt.close(); print("  fig5", flush=True)

# ── Fig 6: Fouling paradox (primary N_total) ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"Fig 6: Fouling Paradox — Carlsbad, C_L=0.02 (N_total={_N_TOTAL_PRIMARY})",
             fontsize=11, fontweight="bold")
ax = axes[0]
for s in _s_vals(_N_TOTAL_PRIMARY):
    sub = df_prim_v[(df_prim_v["SITE_TAG"]=="CARLSBAD")&
                    (df_prim_v["CONFIG"]=="fresh")&
                    (df_prim_v["C_L_IN_MOL_L"]==0.02)&
                    (df_prim_v["S"]==s)].sort_values("FOULING_BETA")
    if len(sub) > 2:
        ax.plot(sub["FOULING_BETA"], sub["P_DENS_CASCADE_W_M2"],
                "o-", label=f"S={s}", lw=2, ms=5)
ax.set_xlabel("β"); ax.set_ylabel("Power density [W/m²]")
ax.set_title("Power vs fouling (fresh cascade)"); ax.legend(); ax.grid(True, alpha=0.3)
ax = axes[1]
betas = sorted(df_prim_v["FOULING_BETA"].unique())
opt_s = []
for b in betas:
    sub = df_prim_v[(df_prim_v["SITE_TAG"]=="CARLSBAD")&
                    (df_prim_v["CONFIG"]=="fresh")&
                    (df_prim_v["C_L_IN_MOL_L"]==0.02)&
                    (df_prim_v["FOULING_BETA"]==b)]
    opt_s.append(sub.loc[sub["P_DENS_CASCADE_W_M2"].idxmax(), "S"] if len(sub)>0 else np.nan)
ax.plot(betas, opt_s, "s-", color="#7E2F8E", lw=2, ms=8)
ax.set_xlabel("β"); ax.set_ylabel("Optimal S*")
ax.set_title("Optimal S* shifts with fouling"); ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIG / "fig6_fouling_paradox.png", dpi=150, bbox_inches="tight")
plt.close(); print("  fig6", flush=True)

# ── Fig 7: Harm reduction — all sites (primary N_total) ───────────────────────
fig, axes = plt.subplots(1, len(_SITES), figsize=(18, 4))
fig.suptitle(f"Fig 7: Harm Reduction vs S — All Sites (N_total={_N_TOTAL_PRIMARY}, β=0)",
             fontsize=11, fontweight="bold")
for ax, site in zip(axes, _SITES):
    tag = site["tag"]
    for arch, col in [("fresh","#0072BD"),("serial","#D95319")]:
        sub = df_prim_v[(df_prim_v["SITE_TAG"]==tag)&
                        (df_prim_v["CONFIG"]==arch)&
                        (df_prim_v["FOULING_BETA"]==0.0)&
                        (df_prim_v["C_L_IN_MOL_L"]==site["c_l_primary"])].sort_values("S")
        if len(sub) > 0:
            ax.plot(sub["S"], sub["HARM_REDUX_PCT"], "o-", color=col,
                    label=arch.capitalize(), lw=2, ms=5)
    ss_s = df_ss[(df_ss["SITE_TAG"]==tag)&(df_ss["C_L_IN_MOL_L"]==site["c_l_primary"])&
                 (df_ss["FOULING_BETA"]==0.0)]
    if len(ss_s) > 0:
        ax.axhline(ss_s["HARM_REDUX_PCT"].max(), color="black", ls="--", lw=1)
    ax.axhline(100, color="green", ls=":", lw=1)
    ax.set_title(f"C_H={site['c_h']}", fontsize=9)
    ax.set_xlabel("S"); ax.grid(True, alpha=0.3); ax.set_ylim(0, 105)
    if ax == axes[0]: ax.set_ylabel("Harm Reduction [%]")
axes[-1].legend(fontsize=8)
plt.tight_layout()
fig.savefig(FIG / "fig7_harm_reduction_all_sites.png", dpi=150, bbox_inches="tight")
plt.close(); print("  fig7", flush=True)

# ── Fig N1: Peak power density vs total area — N_total curve ─────────────────
# For each N_total, show the peak power density across all valid S values.
# This is the key new curve showing how architecture scales with membrane investment.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Fig N1: Peak Power Density vs Total Membrane Area\n"
             "All N_total values (3–24 m²), C_L=0.02, β=0 and β=0.8",
             fontsize=10, fontweight="bold")

for ax_i, (tag, c_h) in enumerate([("CARLSBAD",1.15),("JEBEL-ALI",1.80)]):
    ax = axes[ax_i]
    sub = df_valid[(df_valid["SITE_TAG"]==tag)&(df_valid["C_L_IN_MOL_L"]==0.02)]
    for arch, col in [("fresh","#0072BD"),("serial","#D95319")]:
        for beta, ls in [(0.0,"-"),(0.8,"--")]:
            grp = (sub[(sub["CONFIG"]==arch)&(sub["FOULING_BETA"]==beta)]
                   .groupby("N_TOTAL")["P_DENS_CASCADE_W_M2"].max()
                   .reset_index().sort_values("N_TOTAL"))
            grp["area"] = grp["N_TOTAL"] * _A_MEM
            ax.plot(grp["area"], grp["P_DENS_CASCADE_W_M2"],
                    ls, color=col, lw=2, marker="o", ms=6,
                    label=f"{arch.capitalize()} β={beta}")
    ax.set_xlabel("Total membrane area [m²]")
    ax.set_ylabel("Peak power density [W/m²]")
    ax.set_title(f"C_H={c_h} mol/L"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIG / "figN1_peak_power_vs_area.png", dpi=150, bbox_inches="tight")
plt.close(); print("  figN1", flush=True)

# ── Fig N2: Optimal S* vs N_total ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Fig N2: Optimal Stage Count S* and N_per_stage vs Total Area\n"
             "N_per_stage ≈ 12–15 across all N_total — architecture scales predictably",
             fontsize=10, fontweight="bold")
for ax_i, (tag, c_h) in enumerate([("CARLSBAD",1.15),("JEBEL-ALI",1.80)]):
    ax = axes[ax_i]
    sub = df_valid[(df_valid["SITE_TAG"]==tag)&
                   (df_valid["CONFIG"]=="fresh")&
                   (df_valid["FOULING_BETA"]==0.0)&
                   (df_valid["C_L_IN_MOL_L"]==0.02)]
    curve = []
    for nt, grp in sub.groupby("N_TOTAL"):
        best = grp.loc[grp["P_DENS_CASCADE_W_M2"].idxmax()]
        curve.append({"area": nt*_A_MEM, "S_star": best["S"],
                      "N_per": nt // best["S"]})
    df_c = pd.DataFrame(curve).sort_values("area")
    ax.plot(df_c["area"], df_c["S_star"], "o-", color="#0072BD", lw=2, ms=8,
            label="Optimal S*")
    ax2 = ax.twinx()
    ax2.plot(df_c["area"], df_c["N_per"], "s--", color="#D95319", lw=1.5, ms=6,
             label="N_per at S*")
    ax2.axhspan(12, 15, alpha=0.08, color="#D95319", label="N_per = 12–15 band")
    ax2.set_ylabel("N_per_stage at S*", color="#D95319", fontsize=9)
    ax.set_xlabel("Total membrane area [m²]")
    ax.set_ylabel("Optimal S*", color="#0072BD")
    ax.set_title(f"C_H={c_h} mol/L"); ax.grid(True, alpha=0.3)
    l1, b1 = ax.get_legend_handles_labels()
    l2, b2 = ax2.get_legend_handles_labels()
    ax.legend(l1+l2, b1+b2, fontsize=8)
plt.tight_layout()
fig.savefig(FIG / "figN2_optimal_stages_vs_area.png", dpi=150, bbox_inches="tight")
plt.close(); print("  figN2", flush=True)

# ── Fig N3: Compliance window N_total-invariance ───────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle("Fig N3: Single-Stage Compliance Window Is N_total-Independent\n"
             "Window N=154–160 is determined by single-pass physics, not cascade geometry",
             fontsize=10, fontweight="bold")
ax.plot(cb_ss["N"], cb_ss["HARM_REDUX_PCT"], color="#7E2F8E", lw=2.5,
        label="Single-stage HR% (Carlsbad)")
pc_r = cb_ss[cb_ss["PHYS_COMPLY"]]
if len(pc_r) > 0:
    ax.axvspan(pc_r["N"].min(), pc_r["N"].max(), alpha=0.20, color="green",
               label=f"Compliance window N={pc_r['N'].min()}–{pc_r['N'].max()}")
ax.axhline(100, color="green", ls=":", lw=1)
colors = plt.cm.tab10(np.linspace(0, 1, len(_N_TOTALS)))
for n, c in zip(_N_TOTALS, colors):
    ax.axvline(n, color=c, ls="--", lw=1, alpha=0.6, label=f"N_total={n}")
ax.set_xlabel("Single-stage cell pairs N")
ax.set_ylabel("Harm Reduction [%]")
ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIG / "figN3_compliance_invariance.png", dpi=150, bbox_inches="tight")
plt.close(); print("  figN3", flush=True)

print(f"\n  All figures saved → {FIG}", flush=True)

# ─────────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────────
# PART 4: MORRIS SENSITIVITY SCREENING
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*70}")
print("  PART 4: Morris Sensitivity Screening")
print(f"{'─'*70}")

# Morris uses the same _FOULING_MODEL and BIOFILM patch — fully consistent
# with the cascade sweep. The FOULING_BETA parameter is swept by Morris
# and the _SWEEP_BETA context variable is set via _run_cascade for each eval.

_MORRIS_R    = 100   # SE = σ/√100 ≈ σ/10
_MORRIS_P    = 4

# Base β anchors: ALL values used in cascade sweep (not a subset)
# This ensures Morris and cascade are evaluated at identical operating points.
_MORRIS_BETAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.5]  # matches _BETA_ALL

# Parameter ranges aligned with cascade sweep wherever possible.
# Ranges marked [CASCADE] are set to match the exact cascade sweep axes.
# Ranges marked [PHYSICAL] span the plausible engineering space.
_MORRIS_PARAMS = [
    # ── Concentration parameters ──────────────────────────────────────────────
    # C_H: scaled 0.5×–2.0× site anchor so Morris covers the full site range
    # and cross-site variation simultaneously.  [PHYSICAL]
    ("C_H_IN_MOL_L",      0.5,    2.0),

    # C_L: [CASCADE] matches _CL_VALS = [0.02, 0.05, 0.10, 0.20, 0.50]
    # Extended upper bound from 0.20 → 0.50 to match cascade sweep maximum.
    ("C_L_IN_MOL_L",      0.02,   0.50),

    # ── Geometry parameters ───────────────────────────────────────────────────
    # N_cell_pairs: [CASCADE] cascade N_per_stage ranges from 12 (max S, large
    # N_total) to 300 (S=2, N_total=600).  Upper bound set to 300 to cover the
    # full cascade range; lower bound 5 retained for sensitivity coverage.
    ("N_CELL_PAIRS",      5,      300),

    # A_mem: per-pair area fixed at 0.05 m² in cascade; Morris sweeps ±5× to
    # test sensitivity.  [PHYSICAL]
    ("A_MEM_M2",          0.005,  0.25),

    # ── Membrane properties ───────────────────────────────────────────────────
    # All three match published pilot ranges.  [PHYSICAL]
    ("ALPHA_PERM",        0.92,   0.99),
    ("ASR_CEM_OHM_M2",   2.0e-4, 6.0e-4),
    ("ASR_AEM_OHM_M2",   1.5e-4, 5.0e-4),

    # Channel gap: [PHYSICAL]
    ("GAP_M",             1.0e-4, 4.0e-4),

    # ── Fouling parameters ────────────────────────────────────────────────────
    # FOULING_BETA: [CASCADE] exact match to _BETA_ALL range [0.0, 1.5]
    ("FOULING_BETA",      0.0,    1.5),

    # DIVALENT_FRACTION: [CASCADE] site values span 0.10–0.35; lower bound 0
    # included so Morris can test the CDF=0 limit.
    ("DIVALENT_FRACTION", 0.0,    0.35),

    # Biofilm parameters — only active when ENABLE_BIOFILM=True.
    # BIOFILM_AMP [0, 2.0]: 0=CDF-only, 1.0=equal weight (v12 default), 2.0=double biofilm
    # BIOFILM_K   [2, 8]:   2=mild outlet-loading, 4=v12 default, 8=sharp outlet peak
    ("BIOFILM_AMP",       0.0,    2.0),
    ("BIOFILM_K",         2.0,    8.0),

    # ── Hydraulic parameters ──────────────────────────────────────────────────
    # [CASCADE] fixed at 5e-6 in sweep; Morris spans 1 decade for sensitivity.
    ("Q_H_M3_S",          5e-6,   5e-5),
    ("Q_L_M3_S",          5e-6,   5e-5),
]

def _morris_response(knobs_dict, site_c_h):
    """Single Morris evaluation: run one fresh-dilute stage, return P_DENS.

    Handles BIOFILM_AMP and BIOFILM_K by temporarily rebuilding the patched
    biofilm closure with the Morris-sampled values before each model call,
    then restoring the defaults.  This allows Morris to screen the biofilm
    shape parameters as well as the electrochemical and fouling parameters.
    """
    global _SWEEP_BETA, _patched_biofilm

    beta     = float(knobs_dict.get("FOULING_BETA", 0.0))
    bio_amp  = float(knobs_dict.get("BIOFILM_AMP",  BIOFILM_AMP))
    bio_k    = float(knobs_dict.get("BIOFILM_K",    BIOFILM_K))

    # Temporarily override the biofilm closure with Morris-sampled amplitude/shape
    if ENABLE_BIOFILM:
        _tmp_bio = _patched_exp_outlet_factory(bio_amp, bio_k)
        FOULING_FUNCS["exp_outlet"] = _tmp_bio

    bk = {
        "C_H_IN_MOL_L":      float(knobs_dict.get("C_H_IN_MOL_L", site_c_h)),
        "C_L_IN_MOL_L":      float(knobs_dict.get("C_L_IN_MOL_L",  0.02)),
        "N_CELL_PAIRS":       int(max(1, knobs_dict.get("N_CELL_PAIRS", 20))),
        "A_MEM_M2":           float(knobs_dict.get("A_MEM_M2", 0.05)),
        "ALPHA_PERM":         float(knobs_dict.get("ALPHA_PERM", 0.975)),
        "ASR_CEM_OHM_M2":     float(knobs_dict.get("ASR_CEM_OHM_M2", 3.6e-4)),
        "ASR_AEM_OHM_M2":     float(knobs_dict.get("ASR_AEM_OHM_M2", 2.35e-4)),
        "GAP_M":              float(knobs_dict.get("GAP_M", 2.0e-4)),
        "FOULING_BETA":       beta,
        "DIVALENT_FRACTION":  float(knobs_dict.get("DIVALENT_FRACTION", 0.20)),
        "Q_H_M3_S":           float(knobs_dict.get("Q_H_M3_S", 5e-6)),
        "Q_L_M3_S":           float(knobs_dict.get("Q_L_M3_S", 5e-6)),
        "FOULING_MODEL":      _FOULING_MODEL,
        "FOULING_C_REF_MOL_L": 0.5,
        "FOULING_N_EXP":      0.5,
    }
    _SWEEP_BETA = beta   # set context for biofilm patch

    r = run_cascade(bk, n_stages=1, n_per_stage=int(bk["N_CELL_PAIRS"]),
                    a_mem_m2=bk["A_MEM_M2"], n_seg=_N_SEG, fresh_dilute=True)

    # Restore canonical biofilm closure
    if ENABLE_BIOFILM:
        FOULING_FUNCS["exp_outlet"] = _patched_biofilm

    return float(r["P_DENS_CASCADE_W_M2"])

def _morris_trajectory(site, base_beta, rng):
    """One Morris trajectory: k+1 evaluations, return k elementary effects."""
    k   = len(_MORRIS_PARAMS)
    p   = _MORRIS_P
    # Build base point: grid-quantised random sample
    deltas = [1.0 / (2.0 * (p - 1))]  # step size
    delta  = deltas[0]

    # Per-parameter bounds (C_H is site-relative)
    bounds = []
    for name, lo, hi in _MORRIS_PARAMS:
        if name == "C_H_IN_MOL_L":
            bounds.append((lo * site["c_h"], hi * site["c_h"]))
        else:
            bounds.append((lo, hi))

    # Base point: random grid point
    base = {}
    for i, (name, _, _) in enumerate(_MORRIS_PARAMS):
        lo, hi = bounds[i]
        grid_pts = np.linspace(lo, hi, p)
        base[name] = float(rng.choice(grid_pts))

    # Override base β to the specified anchor
    base["FOULING_BETA"] = float(base_beta)

    # Evaluate base
    y0 = _morris_response(base, site["c_h"])

    # Random permutation of parameters
    perm = rng.permutation(k)
    effects = {}
    current = dict(base)

    for i in perm:
        name, _, _ = _MORRIS_PARAMS[i]
        lo, hi = bounds[i]
        r_range = hi - lo

        # Step direction: +δ if room, else -δ
        step = delta * r_range
        cand = current[name] + step
        if cand > hi:
            cand = current[name] - step
        current[name] = float(np.clip(cand, lo, hi))

        # For integer params, round
        if name == "N_CELL_PAIRS":
            current[name] = float(max(1, round(current[name])))

        y1 = _morris_response(current, site["c_h"])
        ee = (y1 - y0) / (step if abs(step) > 1e-15 else 1e-15)
        effects[name] = ee
        y0 = y1

    return effects

mor_rows = []
_t4 = time.time()
_total_morris = len(_SITES) * len(_MORRIS_BETAS) * _MORRIS_R

print(f"  r={_MORRIS_R}, p={_MORRIS_P}, k={len(_MORRIS_PARAMS)}, "
      f"evals/run={_MORRIS_R*(len(_MORRIS_PARAMS)+1)}")
print(f"  {len(_SITES)} sites × {len(_MORRIS_BETAS)} anchors × {_MORRIS_R} "
      f"trajectories = {_total_morris} total trajectory runs")
print(f"  Parameters: {[p[0] for p in _MORRIS_PARAMS]}")
print(f"  Note: BIOFILM_AMP/K screened only when ENABLE_BIOFILM=True")

for site in _SITES:
    for base_beta in _MORRIS_BETAS:
        rng = np.random.default_rng(seed=int(site["mc_seed"] * 1000 + base_beta * 100))
        all_ee = {name: [] for name, _, _ in _MORRIS_PARAMS}

        for _ in range(_MORRIS_R):
            ee = _morris_trajectory(site, base_beta, rng)
            for name, val in ee.items():
                all_ee[name].append(val)

        for name, vals in all_ee.items():
            arr = np.array(vals)
            mu_star = float(np.mean(np.abs(arr)))
            sigma   = float(np.std(arr))
            se      = sigma / np.sqrt(_MORRIS_R)
            mor_rows.append({
                "SITE_TAG":   site["tag"],
                "BASE_BETA":  base_beta,
                "parameter":  name,
                "mu":         float(np.mean(arr)),
                "mu_star":    mu_star,
                "sigma":      sigma,
                "se_mu_star": se,
                "n_effects":  _MORRIS_R,
            })
        print(f"  [{site['tag']}] base_β={base_beta}  done  ({time.time()-_t4:.0f}s)")

df_mor = pd.DataFrame(mor_rows)
df_mor.to_parquet(_ROOT / "morris_all.parquet", index=False)
print(f"\n  morris_all.parquet → {len(df_mor)} rows  ({time.time()-_t4:.0f}s)")

# PART 5: MONTE CARLO TEA  — COMMENTED OUT
# Uncomment and run separately after cascade sweep is finalized.
# ─────────────────────────────────────────────────────────────────────────────

# _MC_N = 5000
# mc_paths = {}
#
# for site in _SITES:
#     if site["tag"] not in _MC_SITES:
#         continue
#     tag, c_h, sw = site["tag"], site["c_h"], site["sw"]
#
#     df_prim_v_site = df_valid[df_valid["N_TOTAL"] == _N_TOTAL_PRIMARY]
#     sub_opt = df_prim_v_site[
#         (df_prim_v_site["SITE_TAG"]==tag) &
#         (df_prim_v_site["C_L_IN_MOL_L"]==site["c_l_primary"]) &
#         (df_prim_v_site["FOULING_BETA"]==0.4) &
#         (df_prim_v_site["CONFIG"]=="fresh")
#     ]
#     if len(sub_opt) == 0:
#         sub_opt = df_prim_v_site[
#             (df_prim_v_site["SITE_TAG"]==tag) &
#             (df_prim_v_site["C_L_IN_MOL_L"]==site["c_l_primary"]) &
#             (df_prim_v_site["FOULING_BETA"]==0.0) &
#             (df_prim_v_site["CONFIG"]=="fresh")
#         ]
#     best   = sub_opt.loc[sub_opt["P_DENS_CASCADE_W_M2"].idxmax()]
#     S_star = int(best["S"])
#
#     interp_df = df_cas[
#         (df_cas["N_TOTAL"]==_N_TOTAL_PRIMARY) &
#         (df_cas["SITE_TAG"]==tag) &
#         (df_cas["C_L_IN_MOL_L"]==site["c_l_primary"]) &
#         (df_cas["S"]==S_star) &
#         (df_cas["CONFIG"]=="fresh")
#     ].sort_values("FOULING_BETA")
#
#     beta_grid = interp_df["FOULING_BETA"].values
#     p_w_grid  = interp_df["P_CASCADE_W"].values
#     hr_grid   = interp_df["HARM_REDUX_PCT"].values
#
#     rng      = np.random.default_rng(site["mc_seed"])
#     ep_sigma = np.sqrt(np.log(1.09))
#     ep_mu    = np.log(site["elec_price"]) - 0.5 * ep_sigma**2
#     ep_s = rng.lognormal(ep_mu, ep_sigma, _MC_N)
#     fb_s = rng.uniform(0.0, 1.2, _MC_N)
#     bd_s = rng.uniform(site["disp_lo"], site["disp_hi"], _MC_N)
#     cx_s = rng.triangular(30.0, 50.0, 100.0, _MC_N)
#     dr_s = rng.uniform(0.03, 0.08, _MC_N)
#     cf_s = rng.uniform(0.80, 0.95, _MC_N)
#     ml_s = rng.uniform(5.0, 15.0, _MC_N)
#
#     p_w_s = np.interp(fb_s, beta_grid, p_w_grid)
#     hr_s  = np.interp(fb_s, beta_grid, hr_grid)
#
#     area_tot      = float(_PILOT_AREA)
#     n_par         = _PILOT_AREA / (_N_TOTAL_PRIMARY * _A_MEM)
#     brine_treated = n_par * _Q_H * 86_400
#
#     plant_capex = area_tot * cx_s
#     ann_rev     = p_w_s * n_par / 1000.0 * 8760.0 * cf_s * ep_s
#     ann_brine   = brine_treated * 365.0 * bd_s
#     ann_amort   = plant_capex / 20.0
#     ann_opex    = plant_capex * 0.03
#     ann_mem_rep = np.where(ml_s < 20.0, area_tot * cx_s / ml_s, 0.0)
#     net_ann     = ann_rev + ann_brine - ann_amort - ann_opex - ann_mem_rep
#     annuity     = (1.0 - (1.0 + dr_s)**(-20.0)) / np.maximum(dr_s, 1e-9)
#     npv         = net_ann * annuity - plant_capex
#
#     _mem_frac  = np.where(ml_s < 20.0, 1.0/ml_s, 0.0)
#     _acr       = 1.0/20.0 + 0.03 + _mem_frac
#     _num       = (ann_rev + ann_brine) * annuity
#     _den       = annuity * _acr + 1.0
#     be_capex   = np.where((_den>0)&((ann_rev+ann_brine)>0),
#                           _num/_den/area_tot, np.nan)
#
#     df_mc = pd.DataFrame({
#         "SITE_TAG":               tag,
#         "sample_idx":             np.arange(_MC_N),
#         "FOULING_BETA":           fb_s,
#         "ELEC_PRICE":             ep_s,
#         "CAPEX_PER_M2":           cx_s,
#         "BRINE_DISP_COST":        bd_s,
#         "DISCOUNT_RATE":          dr_s,
#         "CAPACITY_FACTOR":        cf_s,
#         "MEM_LIFETIME_YR":        ml_s,
#         "P_CASCADE_W":            p_w_s,
#         "NPV_USD":                npv,
#         "NET_ANNUAL_USD":         net_ann,
#         "ANN_REVENUE_USD":        ann_rev,
#         "ANN_BRINE_USD":          ann_brine,
#         "HARM_REDUX_PCT":         hr_s,
#         "S_STAR":                 S_star,
#         "N_TOTAL":                _N_TOTAL_PRIMARY,
#         "BRINE_TREATED_M3DAY":    float(brine_treated),
#         "PILOT_AREA_M2":          float(area_tot),
#         "BREAKEVEN_CAPEX_USD_M2": be_capex,
#     })
#     pnpv = float((df_mc["NPV_USD"]>0).mean()*100)
#     p50  = float(df_mc["NPV_USD"].median())
#     be50 = float(df_mc["BREAKEVEN_CAPEX_USD_M2"].median())
#     print(f"  [{tag}]  P(NPV>0)={pnpv:.1f}%  P50=${p50/1e3:.0f}k  BE=${be50:.0f}/m²")
#     mp = _ROOT / f"mc_tea_{tag}.parquet"
#     df_mc.to_parquet(mp, index=False)
#     mc_paths[tag] = (mp, df_mc, pnpv, p50, S_star)

print("  MC-TEA: SKIPPED (Part 5 commented out)")

# ─────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────
# PART 6 & 7: DASHBOARD JSON + FREEZE MANIFEST
# ─────────────────────────────────────────────────────────────────────────────

# Dashboard summary (key scalar results for dashboard.json)
_dashboard = {
    "experiment": "OSMOFLUX-GRAND-SWEEP",
    "version": "12",
    "timestamp": _TS,
    "fouling_config": {
        "ENABLE_BIOFILM": ENABLE_BIOFILM,
        "FOULING_MODEL":  _FOULING_MODEL,
        "BIOFILM_AMP":    BIOFILM_AMP,
        "BIOFILM_K":      BIOFILM_K,
        "note": ("Combined CDF + biofilm: β scales both mechanisms uniformly. "
                 if ENABLE_BIOFILM else
                 "CDF-only: v9 baseline behavior."),
    },
    "n_cascade_rows":     len(df_cas),
    "n_ss_rows":          len(df_ss),
    "n_morris_rows":      len(df_mor),
    "primary_n_total":    _N_TOTAL_PRIMARY,
    "beta_values":        _BETA_ALL,
    "n_total_values":     _N_TOTALS,
    "sites":              [s["tag"] for s in _SITES],
}
with open(_ROOT / "dashboard.json", "w") as f:
    json.dump(_dashboard, f, indent=2)

# Freeze manifest
def _sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

_files = {
    "cascade_sweep.parquet":      _ROOT / "cascade_sweep.parquet",
    "single_stage_sweep.parquet": _ROOT / "single_stage_sweep.parquet",
    "morris_all.parquet":         _ROOT / "morris_all.parquet",
    "dashboard.json":             _ROOT / "dashboard.json",
}

_manifest = {
    "experiment":   "OSMOFLUX-GRAND-SWEEP",
    "version":      "12",
    "timestamp":    _TS,
    "run_dir":      str(_ROOT),
    "fouling_config": _dashboard["fouling_config"],
    "params": {
        "N_TOTALS":         _N_TOTALS,
        "N_TOTAL_PRIMARY":  _N_TOTAL_PRIMARY,
        "A_MEM_M2":         _A_MEM,
        "N_SEG":            _N_SEG,
        "BETA_ALL":         _BETA_ALL,
        "CL_VALS":          _CL_VALS,
        "sites":            [s["tag"] for s in _SITES],
        "MORRIS_R":         _MORRIS_R,
        "MORRIS_P":         _MORRIS_P,
        "MORRIS_BASE_BETAS": _MORRIS_BETAS,
        "MC_STATUS":        "COMMENTED_OUT",
    },
    "notes": {
        "sweep":       "All N_total in one parquet; filter by N_TOTAL column",
        "primary":     "Paper results use N_TOTAL=240 slice",
        "phys_comply": "PHYS_COMPLY = COMPLIANCE_FLAG AND NOT ANY_INVERSION",
        "valid_subset": "df_valid = df_cas[~df_cas.ANY_INVERSION]",
        "morris":      "r=100; se_mu_star = σ/√n_effects; 95% CI = 1.96×se",
        "fouling":     ("v12: β scales both CDF and biofilm uniformly. "
                        "To reproduce v9 CDF-only, set ENABLE_BIOFILM=False." 
                        if ENABLE_BIOFILM else
                        "v12 run with ENABLE_BIOFILM=False: CDF-only, v9 baseline."),
    },
    "files": {
        name: {
            "path":   str(path),
            "sha256": _sha256(path),
        }
        for name, path in _files.items()
        if path.exists()
    },
}

with open(_ROOT / "GRAND-SWEEP-MANIFEST.json", "w") as f:
    json.dump(_manifest, f, indent=2)

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
_elapsed = time.time() - _T0
print(f"\n{'═'*70}")
print(f"  GRAND SWEEP v12 COMPLETE  |  {_elapsed/60:.1f} min total")
print(f"  Output: {_ROOT}")
print(f"  Fouling: {'CDF + biofilm (ENABLE_BIOFILM=True)' if ENABLE_BIOFILM else 'CDF only (ENABLE_BIOFILM=False)'}")
print(f"  Rows: cascade={len(df_cas):,}  single_stage={len(df_ss):,}  morris={len(df_mor)}")
print(f"\n  SHA-256 hashes:")
for name, path in _files.items():
    if path.exists():
        print(f"    {name}: {_sha256(path)[:16]}…")
print(f"{'═'*70}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GRAND SWEEP POST-PROCESSING — Figure Generator  (v7)
# ══════════════════════════════════════════════════════════════════════════════
#
# All 40 figures into one flat directory:
#   runs/GRAND-SWEEP/<RUN_TIMESTAMP>/figures/
#
# Prefix key:
#   paper_fig*   / paper_supp*  — Desalination journal (5 main + 4 supp)
#   genius_fig*                 — GENIUS Olympiad poster (6)
#   legacy_fig*  / legacy_figN* — 12 original paper figures (updated data)
#   legacy_figS*                — 13 storytelling figures (updated data)
#
# v7 changes vs v6 (all reviewer fixes applied):
#   paper_fig1  — shorter title/subtitle; thinner compliance rings; legend
#                 shrunk; serial annotation repositioned
#   paper_fig2  — x-axis label shortened; compliance vline opacity 0.35;
#                 Regime 2 marker enlarged; legend outside plot; serial dots
#                 alpha 0.65, fresh 0.55; dot-size legend removed from axes
#   paper_fig3  — left legend → lower right; shading alpha 0.18; fit label
#                 shortened; right legend more padding
#   paper_fig4  — single colorbar far right; title shortened; dash → "#444"
#   paper_fig5  — stats box smaller; legend removed (stats box is sufficient)
#   paper_supp1 — left legend outside upper right; annotations shifted
#   paper_supp2 — left legend ncol=3; right y-axis narrowed to data range ±1
#   genius_fig1 — "Inversion begins" moved inward; legend lower left;
#                 compliance label near right axis; footer → subtitle
#   genius_fig2 — title changed; ✓ checkmarks above bars
#   genius_fig3 — "✓ Compliant" moved inward; red axhspan below 100%;
#                 β range in subtitle
#   genius_fig4 — annotation only above Carlsbad; concentrations in x-labels
#   genius_fig5 — 587-case line added to stats box
#   paper_fig1  — legend handlelength+ncol reduced; SS diamond marker smaller;
#                 gray compliance ring for diamond thinned; "grey"→"gray" in hm
#   paper_fig2  — title→"Power–compliance design space"; "dot size" removed
#                 from title; gray curve lw reduced to 0.8; axhspan alpha→0.08;
#                 Regime 2 diamond gets white edge+larger size
#   paper_fig3  — "Full compliance" text y lowered; left-panel only β=0 and
#                 β=1.5 shading labels added; right legend label updated to
#                 "First inversion fit: 161+75.2β"
#   paper_fig4  — "grey"→"gray" in suptitle; serial y-axis tick labels hidden;
#                 site names matched to manuscript: Valley Water, Monterey,
#                 Carlsbad, Jebel Ali, Hypersaline
#   paper_fig5  — title→"Fresh-dilute advantage across matched valid cases";
#                 title fontsize 9.5; mean/median identified in stats box
#   paper_supp1 — right-panel annotation gets bbox white background;
#                 "Power peak N=14" annotation shifted right+down further
#   paper_supp2 — left legend moved outside right edge
#   genius_fig1 — power-optimum annotation shifted left to avoid compliance line
#   genius_fig2 — ✓ label y raised above x-axis ticks; footer fontsize→10.5
#   genius_fig3 — N_total=240 / 12 m² added to suptitle; non-compliant label
#                 shifted further below red line
#   genius_fig4 — "More salinity"→"Higher salinity" in subtitle
#   genius_fig5 — "Also 587/587 wins" line added explicitly; histogram
#                 title kept as-is (GENIUS OK)
#   genius_fig6 — No additional changes (Morris already corrected in v6)
# ══════════════════════════════════════════════════════════════════════════════

if True:

    import numpy as np
    import pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from matplotlib.lines import Line2D
    from matplotlib.colors import TwoSlopeNorm
    from pathlib import Path
    import warnings
    warnings.filterwarnings("ignore", category=UserWarning)

    # ── ① CONFIGURE ──────────────────────────────────────────────────────────
    RUN_TIMESTAMP = "20260531_183521"   # ← change to target run
    # ─────────────────────────────────────────────────────────────────────────

    _BASE    = Path("runs") / "GRAND-SWEEP" / RUN_TIMESTAMP
    _FIG_DIR = _BASE / "figures"
    _FIG_DIR.mkdir(parents=True, exist_ok=True)

    # ── ② LOAD ────────────────────────────────────────────────────────────────
    _df_cas = pd.read_parquet(_BASE / "cascade_sweep.parquet")
    _df_ss  = pd.read_parquet(_BASE / "single_stage_sweep.parquet")
    _df_mor = pd.read_parquet(_BASE / "morris_all.parquet")

    print(f"[FIGS v7] Loaded from {_BASE}")
    print(f"  cascade {len(_df_cas):,} | single_stage {len(_df_ss):,} | morris {len(_df_mor):,}")

    # ── ③ SHARED CONSTANTS ────────────────────────────────────────────────────
    _SITES = [
        {"tag": "VALLEY-WATER", "C_H": 0.35, "label": "Valley Water\nC_H=0.35"},
        {"tag": "MONTEREY",     "C_H": 0.75, "label": "Monterey\nC_H=0.75"},
        {"tag": "CARLSBAD",     "C_H": 1.15, "label": "Carlsbad\nC_H=1.15"},
        {"tag": "JEBEL-ALI",    "C_H": 1.80, "label": "Jebel Ali\nC_H=1.8"},
        {"tag": "HYPERSALINE",  "C_H": 3.00, "label": "Hypersaline\nC_H=3.0"},
    ]
    _SITE_TAGS = [s["tag"] for s in _SITES]
    _BETA_VALS = sorted(_df_cas["FOULING_BETA"].unique())
    _N_TOTALS  = sorted(_df_cas["N_TOTAL"].unique())
    _CL        = 0.02
    _BETA0     = 0.0
    _NT240     = 240
    _AMEM      = 0.05   # m² per cell pair

    _CF  = "#2166ac"   # fresh-dilute blue
    _CS  = "#d73027"   # serial red
    _CSS = "#555555"   # single-stage grey
    _CK  = "#1a9850"   # compliance green
    _CW  = "#f4a100"   # warning amber

    _BETA4 = [0.0, 0.4, 0.8, 1.5]
    _PAL4  = ["#2166ac", "#74add1", "#f46d43", "#a50026"]

    def _save(name):
        # Bold all axis labels, tick labels, and spine linewidths on every figure
        fig = plt.gcf()
        for ax in fig.get_axes():
            # Bold and slightly larger axis labels
            for lbl in (ax.xaxis.label, ax.yaxis.label):
                lbl.set_fontweight("bold")
                if lbl.get_fontsize() < 9:
                    lbl.set_fontsize(9)
            # Bold tick labels
            for tick in ax.get_xticklabels() + ax.get_yticklabels():
                tick.set_fontweight("bold")
            # Heavier spines
            for spine in ax.spines.values():
                spine.set_linewidth(1.2)
        plt.savefig(_FIG_DIR / name, dpi=200, bbox_inches="tight")
        plt.close()
        print(f"  [fig] {name}")

    def _best_fresh(df, site, ntot, beta, cl=_CL, comply=True):
        sub = df[(df.SITE_TAG==site) & (df.N_TOTAL==ntot) & (df.FOULING_BETA==beta)
                 & (df.C_L_IN_MOL_L==cl) & (df.FRESH_DILUTE==True)]
        sub = sub[sub.PHYS_COMPLY==True] if comply else sub[~sub.ANY_INVERSION]
        return sub.loc[sub.P_DENS_CASCADE_W_M2.idxmax()] if len(sub) else None

    def _best_serial(df, site, ntot, beta, cl=_CL):
        sub = df[(df.SITE_TAG==site) & (df.N_TOTAL==ntot) & (df.FOULING_BETA==beta)
                 & (df.C_L_IN_MOL_L==cl) & (df.FRESH_DILUTE==False) & (~df.ANY_INVERSION)]
        return sub.loc[sub.P_DENS_CASCADE_W_M2.idxmax()] if len(sub) else None

    # Pre-compute advantage distribution (reused multiple times)
    _advs = np.array([
        (_f.P_DENS_CASCADE_W_M2.max() / _s.P_DENS_CASCADE_W_M2.max() - 1) * 100
        for _, _g in _df_cas.groupby(["SITE_TAG","N_TOTAL","FOULING_BETA","C_L_IN_MOL_L"])
        for _f, _s in [(
            _g[(_g.FRESH_DILUTE==True)  & (~_g.ANY_INVERSION)],
            _g[(~_g.FRESH_DILUTE) & (~_g.ANY_INVERSION)]
        )]
        if len(_f) and len(_s)
    ])

    # Single-stage Carlsbad β=0 (reused many times)
    _ss0      = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD") & (_df_ss.FOULING_BETA==_BETA0)].sort_values("N")
    _win      = _ss0[_ss0.PHYS_COMPLY==True]
    _inv      = _ss0[_ss0.ANY_INVERSION==True]
    _n_win_lo = int(_win.N.min()) if len(_win) else 154
    _n_win_hi = int(_win.N.max()) if len(_win) else 160
    _n_inv    = int(_inv.N.min()) if len(_inv) else 161

    # ══════════════════════════════════════════════════════════════════════════
    print("\n[PAPER MAIN FIGURES]")
    # ══════════════════════════════════════════════════════════════════════════

    # ── PAPER FIG 1 ── Compliance frontier  (v6 fixes) ────────────────────────
    # Title shortened; subtitle line; legend smaller; green rings thinner;
    # serial annotation repositioned.

    _fp, _sp = [], []
    for _nt in _N_TOTALS:
        for _sub, _lst, _fresh in [
            (_df_cas[(_df_cas.SITE_TAG=="CARLSBAD") & (_df_cas.N_TOTAL==_nt)
                     & (_df_cas.FOULING_BETA==_BETA0) & (_df_cas.C_L_IN_MOL_L==_CL)
                     & (_df_cas.FRESH_DILUTE==True) & (~_df_cas.ANY_INVERSION)],
             _fp, True),
            (_df_cas[(_df_cas.SITE_TAG=="CARLSBAD") & (_df_cas.N_TOTAL==_nt)
                     & (_df_cas.FOULING_BETA==_BETA0) & (_df_cas.C_L_IN_MOL_L==_CL)
                     & (~_df_cas.FRESH_DILUTE) & (~_df_cas.ANY_INVERSION)],
             _sp, False)]:
            if len(_sub):
                _r = _sub.loc[_sub["P_DENS_CASCADE_W_M2"].idxmax()]
                _lst.append({"area": _nt*_AMEM, "P": _r.P_DENS_CASCADE_W_M2,
                              "HR": _r.HARM_REDUX_PCT, "comply": bool(_r.PHYS_COMPLY)})

    _dfr = pd.DataFrame(_fp);  _frC = _dfr[_dfr.comply]
    _dfs = pd.DataFrame(_sp);  _seC = _dfs[_dfs.comply]
    _ssC = _ss0[_ss0.PHYS_COMPLY==True]

    fig, (_a1, _a2) = plt.subplots(2, 1, figsize=(9, 7.5), sharex=True)
    fig.suptitle("Carlsbad power–compliance frontier", fontsize=12, fontweight="bold", y=0.98)
    _a1.set_title(r"C$_H$ = 1.15 mol/L,  C$_L$ = 0.02 mol/L,  β = 0"
                  "\nSingle-stage: continuous N sweep  ·  Cascades: tested N$_{total}$ values only",
                  fontsize=9, color="#444")

    _a1.plot(_ss0.TOTAL_AREA_M2, _ss0.P_DENS_W_M2, color=_CSS, lw=1.8, label="Single-stage", zorder=2)
    _a1.plot(_dfr.area, _dfr.P, "o-", color=_CF, lw=2.5, ms=7, label="Fresh-dilute cascade", zorder=3)
    _a1.plot(_dfs.area, _dfs.P, "s-", color=_CS, lw=2.5, ms=7, label="Serial cascade", zorder=3)
    # Compliance rings: thinner linewidth 1.0 to reduce clutter
    _a1.scatter(_frC.area, _frC.P, s=90, zorder=5, facecolors="none",
                edgecolors=_CK, linewidths=0.9, label="Compliant ✓")
    _a1.scatter(_ssC.TOTAL_AREA_M2, _ssC.P_DENS_W_M2, s=80, zorder=5,
                facecolors="none", edgecolors=_CK, linewidths=0.9, marker="D")
    _a1.scatter(_seC.area, _seC.P, s=90, zorder=5,
                facecolors="none", edgecolors=_CK, linewidths=0.9, marker="^")
    if len(_frC):
        _bfc = _frC.loc[_frC.P.idxmax()]
        _a1.annotate(f"Fresh-dilute best compliant\n{_bfc.P:.2f} W/m² @ {_bfc.area:.0f} m²",
                     xy=(_bfc.area, _bfc.P), xytext=(_bfc.area+1.8, _bfc.P+0.10),
                     arrowprops=dict(arrowstyle="->", color=_CF, lw=1.2),
                     fontsize=8, color=_CF, fontweight="bold")
    _a1.set_ylabel("Peak power density [W/m²]", fontsize=9)
    _a1.legend(fontsize=7, loc="upper right", framealpha=0.85,
               handlelength=1.2, ncol=2, columnspacing=0.8)
    _a1.grid(alpha=0.25)

    _a2.plot(_ss0.TOTAL_AREA_M2, _ss0.HARM_REDUX_PCT, color=_CSS, lw=1.8, label="Single-stage")
    _a2.plot(_dfr.area, _dfr.HR, "o-", color=_CF, lw=2.5, ms=7, label="Fresh-dilute cascade")
    _a2.plot(_dfs.area, _dfs.HR, "s-", color=_CS, lw=2.5, ms=7, label="Serial cascade")
    _a2.axhline(100, color=_CK, ls="--", lw=1.8, label="Full compliance (HR$_{worst}$=100%)")

    # Vertical compliance threshold labels — serial annotation now close to red line
    _annot_configs = [
        (_frC.area.min() if len(_frC) else None, _CF,  "Fresh-dilute\n{:.1f} m²",  30),
        (_ssC.TOTAL_AREA_M2.min() if len(_ssC) else None, _CSS, "Single-stage\n{:.1f} m²", 16),
        (_seC.area.min() if len(_seC) else None, _CS,  "Serial\n{:.1f} m²",         44),
    ]
    for _area, _col, _lab, _yp in _annot_configs:
        if _area is not None:
            _a2.axvline(_area, color=_col, ls=":", lw=1.5, alpha=0.8)
            _xoff = 0.2 if _col != _CS else -2.8
            _a2.text(_area + _xoff, _yp, _lab.format(_area), fontsize=9.5, color=_col,
                     ha="left" if _col != _CS else "right", fontweight="bold")

    _a2.set_xlabel("Total membrane area [m²]", fontsize=9)
    _a2.set_ylabel("HR$_{worst}$ [%]", fontsize=9)
    _a2.set_ylim(0, 108)
    _a2.legend(fontsize=7.5, loc="lower right", framealpha=0.85)
    _a2.grid(alpha=0.25)
    plt.tight_layout()
    _save("paper_fig1_compliance_frontier.png")

    # ── PAPER FIG 2 ── Design space scatter  (v6 fixes) ───────────────────────
    # x-axis label shortened; vline opacity 0.35; Regime 2 larger marker;
    # legend outside; serial alpha 0.65, fresh 0.55; dot-size legend removed.

    _ssA = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD") & (_df_ss.FOULING_BETA==_BETA0)].copy()
    _frA = _df_cas[(_df_cas.SITE_TAG=="CARLSBAD") & (_df_cas.FOULING_BETA==_BETA0)
                   & (_df_cas.C_L_IN_MOL_L==_CL) & (_df_cas.FRESH_DILUTE==True)
                   & (~_df_cas.ANY_INVERSION)].copy()
    _seA = _df_cas[(_df_cas.SITE_TAG=="CARLSBAD") & (_df_cas.FOULING_BETA==_BETA0)
                   & (_df_cas.C_L_IN_MOL_L==_CL) & (~_df_cas.FRESH_DILUTE)
                   & (~_df_cas.ANY_INVERSION)].copy()

    fig, ax = plt.subplots(figsize=(10, 6.5))
    ax.set_title(
        "Power–compliance design space — Carlsbad\n"
        r"β = 0,  C$_L$ = 0.02 mol/L",
        fontsize=10, fontweight="bold")

    ax.plot(_ssA.HARM_REDUX_PCT, _ssA.P_DENS_W_M2, color=_CSS, lw=0.8, zorder=1)
    ax.scatter(_ssA.HARM_REDUX_PCT, _ssA.P_DENS_W_M2,
               s=_ssA.TOTAL_AREA_M2*3.5, c=_CSS, alpha=0.28, zorder=2, label="Single-stage")
    # Serial: alpha 0.65 (was 0.5)
    ax.scatter(_seA.HARM_REDUX_PCT, _seA.P_DENS_CASCADE_W_M2,
               s=_seA.TOTAL_AREA_M2*4, c=_CS, alpha=0.65, zorder=3, marker="s", label="Serial cascade")
    # Fresh: alpha 0.55 (was 0.65)
    ax.scatter(_frA.HARM_REDUX_PCT, _frA.P_DENS_CASCADE_W_M2,
               s=_frA.TOTAL_AREA_M2*4, c=_CF, alpha=0.55, zorder=4, label="Fresh-dilute cascade")

    # Compliance vline: reduced opacity
    ax.axvline(100, color=_CK, ls="--", lw=1.8, alpha=0.35, zorder=5, label="Full compliance")
    ax.axhspan(1.5, 3.4, xmin=100/110, xmax=1.0, alpha=0.08, color=_CK, zorder=0)

    # Regime 1 annotation
    _pk = _ssA.loc[_ssA.P_DENS_W_M2.idxmax()]
    ax.annotate("Regime 1\nUnconstrained peak\nP=3.18 W/m²",
                xy=(_pk.HARM_REDUX_PCT, _pk.P_DENS_W_M2),
                xytext=(11, _pk.P_DENS_W_M2 - 0.6),
                arrowprops=dict(arrowstyle="->", color=_CSS, lw=1.1),
                fontsize=8, color=_CSS)

    # Regime 2: larger scatter marker to identify best compliant SS point
    _sw2 = _ssA[_ssA.PHYS_COMPLY==True]
    if len(_sw2):
        _sw2r = _sw2.loc[_sw2.P_DENS_W_M2.idxmax()]
        ax.scatter([_sw2r.HARM_REDUX_PCT], [_sw2r.P_DENS_W_M2],
                   s=300, color=_CSS, zorder=6, marker="D",
                   edgecolors="white", linewidths=2.0)
        ax.annotate("Regime 2\nBest compliant SS\nP=1.04 W/m²",
                    xy=(_sw2r.HARM_REDUX_PCT, _sw2r.P_DENS_W_M2),
                    xytext=(62, _sw2r.P_DENS_W_M2 + 0.38),
                    arrowprops=dict(arrowstyle="->", color=_CSS, lw=1.1),
                    fontsize=8, color=_CSS)

    # Regime 3
    _fw2 = _frA[_frA.PHYS_COMPLY==True]
    if len(_fw2):
        _fw2r = _fw2.loc[_fw2.P_DENS_CASCADE_W_M2.idxmax()]
        ax.annotate("Regime 3\nBest compliant\nfresh-dilute\nP=2.57 W/m²",
                    xy=(_fw2r.HARM_REDUX_PCT, _fw2r.P_DENS_CASCADE_W_M2),
                    xytext=(80, _fw2r.P_DENS_CASCADE_W_M2 + 0.30),
                    arrowprops=dict(arrowstyle="->", color=_CF, lw=1.1),
                    fontsize=8, color=_CF, fontweight="bold")

    # Shortened x-axis label; y label unchanged
    ax.set_xlabel("HR$_{worst}$ [%]", fontsize=10)
    ax.set_ylabel("Power density [W/m²]", fontsize=10)
    ax.set_xlim(-2, 110); ax.set_ylim(-0.1, 3.4)

    # Legend placed outside plot area to the right
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1.0),
              borderaxespad=0, framealpha=0.9)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    _save("paper_fig2_design_space.png")

    # ── PAPER FIG 3 ── Fouling window  (v6 fixes) ─────────────────────────────
    # Left legend → lower right; shading alpha 0.18; fit label shortened.

    fig, (_a1, _a2) = plt.subplots(1, 2, figsize=(12, 5.5))
    fig.suptitle(
        "Fouling shifts the single-stage compliance window — Carlsbad\n"
        r"Single-stage configurations only  ·  C$_H$ = 1.15 mol/L,  C$_L$ = 0.02 mol/L",
        fontsize=10, fontweight="bold")

    for _beta, _col in zip(_BETA4, _PAL4):
        _sub = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD") & (_df_ss.FOULING_BETA==_beta)].sort_values("N")
        if not len(_sub): continue
        _a1.plot(_sub.N, _sub.HARM_REDUX_PCT, color=_col, lw=2, label=f"β = {_beta:.1f}")
        _wn = _sub[_sub.PHYS_COMPLY==True]
        if len(_wn):
            _a1.axvspan(_wn.N.min(), _wn.N.max(), alpha=0.18, color=_col)   # darker: 0.12→0.18
            _a1.axvline(_wn.N.max()+1, color=_col, ls=":", lw=1, alpha=0.7)

    _a1.axhline(100, color=_CK, ls="--", lw=1.5, alpha=0.9)
    _a1.text(4, 99.0, "Full compliance (HR$_{worst}$ = 100%)", fontsize=8, color=_CK, va="top")

    # Add window-centre labels for β=0 and β=1.5 only to make the shift legible
    for _beta_lbl, _col_lbl in [(_BETA4[0], _PAL4[0]), (_BETA4[-1], _PAL4[-1])]:
        _sub_lbl = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD") & (_df_ss.FOULING_BETA==_beta_lbl)]
        _wn_lbl  = _sub_lbl[_sub_lbl.PHYS_COMPLY==True]
        if len(_wn_lbl):
            _mid = (_wn_lbl.N.min() + _wn_lbl.N.max()) / 2
            _a1.text(_mid, 91, f"β={_beta_lbl:.1f}", ha="center", fontsize=7.5,
                     color=_col_lbl, fontweight="bold")
    _a1.set_xlabel("Single-stage cell pairs N", fontsize=9)
    _a1.set_ylabel("HR$_{worst}$ [%]", fontsize=9)
    _a1.set_title("Compliance window (shaded) + inversion onset (dotted)", fontsize=9)
    _a1.legend(fontsize=8.5, loc="lower right")  # moved: was upper area
    _a1.grid(alpha=0.25); _a1.set_xlim(0, 310)

    _wo, _wc, _iv, _bab = [], [], [], []
    for _beta in _BETA_VALS:
        _sub = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD") & (_df_ss.FOULING_BETA==_beta)]
        _wn  = _sub[_sub.PHYS_COMPLY==True]
        _in  = _sub[_sub.ANY_INVERSION==True].sort_values("N")
        if len(_wn) and len(_in):
            _bab.append(_beta); _wo.append(_wn.N.min())
            _wc.append(_wn.N.max()); _iv.append(_in.N.min())

    _a2.plot(_bab, _wo, "o-",  color=_CK, lw=2, ms=7, label="Window open")
    _a2.plot(_bab, _wc, "s--", color=_CK, lw=2, ms=7, label="Window close")
    _a2.plot(_bab, _iv, "^-",  color=_CS, lw=2, ms=7, label="First inversion N")
    _co = np.polyfit(_bab, _iv, 1)
    _a2.plot([0, 1.5], np.polyval(_co, [0, 1.5]), color=_CS, ls=":", lw=1.5,
             label=f"First inversion fit: {_co[1]:.0f} + {_co[0]:.1f}β")
    _a2.set_xlabel("Fouling severity β", fontsize=9)
    _a2.set_ylabel("Cell pairs N", fontsize=9)
    _a2.set_title("Window position and inversion boundary vs β", fontsize=9)
    _a2.legend(fontsize=8.5, handlelength=1.8)  # extra pad so label fits
    _a2.grid(alpha=0.25)
    plt.tight_layout()
    _save("paper_fig3_fouling_window.png")

    # ── PAPER FIG 4 ── Compliance heatmap  (v6 fixes) ─────────────────────────
    # Single colorbar far right; shorter title; dash color "#444".

    _site_hm = ["Valley Water\n(0.35)", "Monterey\n(0.75)",
                "Carlsbad\n(1.15)", "Jebel Ali\n(1.80)", "Hypersaline\n(3.00)"]

    def _mam(fresh_flag):
        _m = np.full((len(_SITE_TAGS), len(_BETA_VALS)), np.nan)
        for _i, _t in enumerate(_SITE_TAGS):
            for _j, _b in enumerate(_BETA_VALS):
                _sub = _df_cas[(_df_cas.SITE_TAG==_t) & (_df_cas.FOULING_BETA==_b)
                               & (_df_cas.C_L_IN_MOL_L==_CL) & (_df_cas.FRESH_DILUTE==fresh_flag)
                               & (_df_cas.PHYS_COMPLY==True) & (~_df_cas.ANY_INVERSION)]
                if len(_sub): _m[_i, _j] = _sub.TOTAL_AREA_M2.min()
        return _m

    _fm = _mam(True); _sm = _mam(False)
    _vmin = np.nanmin([_fm, _sm]); _vmax = np.nanmax([_fm, _sm])
    _cmap_hm = plt.cm.viridis_r.copy(); _cmap_hm.set_bad(color="#cccccc")

    fig, (_a1, _a2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(
        "Minimum membrane area for full modeled compliance\n"
        r"C$_L$ = 0.02 mol/L  ·  cell values = m²  ·  gray = no compliant configuration up to 30 m²",
        fontsize=10, fontweight="bold")

    _im = None
    for _ax_idx, (_ax, _mat, _ttl) in enumerate([(_a1, _fm, "Fresh-dilute cascade"),
                                                  (_a2, _sm, "Serial cascade")]):
        _im = _ax.imshow(np.ma.masked_invalid(_mat), cmap=_cmap_hm,
                         vmin=_vmin, vmax=_vmax, aspect="auto", interpolation="nearest")
        _ax.set_title(_ttl, fontsize=10, fontweight="bold")
        _ax.set_xticks(range(len(_BETA_VALS)))
        _ax.set_xticklabels([str(b) for b in _BETA_VALS], fontsize=8)
        _ax.set_xlabel("Fouling severity β", fontsize=9)
        _ax.set_yticks(range(len(_SITE_TAGS)))
        # Hide duplicate y-axis labels on serial (right) panel to reduce width
        if _ax_idx == 0:
            _ax.set_yticklabels(_site_hm, fontsize=8)
        else:
            _ax.set_yticklabels([""] * len(_SITE_TAGS))
        for _i in range(_mat.shape[0]):
            for _j in range(_mat.shape[1]):
                _v = _mat[_i, _j]
                if not np.isnan(_v):
                    _tc = "white" if _v > (_vmax + _vmin) / 2 else "black"
                    _ax.text(_j, _i, f"{_v:.0f}", ha="center", va="center",
                             fontsize=9, fontweight="bold", color=_tc)
                else:
                    _ax.text(_j, _i, "—", ha="center", va="center",
                             fontsize=10, fontweight="bold", color="#444")

    # Single colorbar on the far right, outside both panels
    plt.colorbar(_im, ax=[_a1, _a2], shrink=0.82, pad=0.03,
                 label="Minimum compliance area [m²]")
    plt.tight_layout()
    _save("paper_fig4_compliance_heatmap.png")

    # ── PAPER FIG 5 ── Advantage distribution  (v6 fixes) ─────────────────────
    # Stats box smaller; legend removed.

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.set_title(
        f"Fresh-dilute advantage across matched valid cases  (n = {len(_advs):,})\n"
        r"Both architectures non-inverted  ·  C$_L$ = 0.02 mol/L",
        fontsize=9.5, fontweight="bold")
    _n, _bins, _ptch = ax.hist(_advs, bins=40, edgecolor="white", alpha=0.85, color=_CF)
    for _p, _lo in zip(_ptch, _bins[:-1]):
        if _lo > 100: _p.set_facecolor("#1a4f7a")
        elif _lo > 50: _p.set_facecolor(_CF)
        else: _p.set_facecolor("#74add1")
    ax.axvline(_advs.mean(),     color=_CS, ls="--", lw=2)
    ax.axvline(np.median(_advs), color=_CW, ls="--", lw=2)

    # Stats box; mean/median lines identified here since no separate legend
    ax.text(0.97, 0.96,
            f"Mean = {_advs.mean():.1f}%  (red dashed)\n"
            f"Median = {np.median(_advs):.1f}%  (amber dashed)\n"
            f"> 50%:  {((_advs>50).mean()*100):.0f}% of cases\n"
            f">100%:  {((_advs>100).mean()*100):.0f}% of cases",
            transform=ax.transAxes, ha="right", va="top", fontsize=8,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="#ccc"))

    ax.set_xlabel("Fresh-dilute power-density advantage over serial [%]", fontsize=10)
    ax.set_ylabel("Number of matched cases", fontsize=10)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    _save("paper_fig5_advantage_distribution.png")

    # ══════════════════════════════════════════════════════════════════════════
    print("\n[PAPER SUPPLEMENTARY]")
    # ══════════════════════════════════════════════════════════════════════════

    # ── SUPP 1 ── Regime curves  (v6 fixes) ────────────────────────────────────
    # Left legend outside upper right; annotations shifted further from curve.

    fig, (_a1, _a2) = plt.subplots(1, 2, figsize=(13, 5.5))
    fig.suptitle(
        "Single-stage regime curves — Carlsbad\n"
        r"C$_H$ = 1.15 mol/L,  C$_L$ = 0.02 mol/L;  four fouling severities",
        fontsize=10, fontweight="bold")

    for _beta, _col in zip(_BETA4, _PAL4):
        _sub = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD") & (_df_ss.FOULING_BETA==_beta)].sort_values("N")
        if not len(_sub): continue
        _a1.plot(_sub.N, _sub.P_DENS_W_M2, color=_col, lw=2, label=f"β = {_beta:.1f}")
        _a2.plot(_sub.N, _sub.HARM_REDUX_PCT, color=_col, lw=2, label=f"β = {_beta:.1f}")
        _wn = _sub[_sub.PHYS_COMPLY==True]
        if len(_wn): _a2.axvspan(_wn.N.min(), _wn.N.max(), alpha=0.10, color=_col)

    _a1.axhline(2.5719, color=_CF, ls="--", lw=1.8, label="Fresh-dilute cascade\nbest compliant (6 m²)")
    _a1.axhline(1.0435, color="#888", ls=":", lw=1.5, label="SS best compliant (β = 0)")

    _pk3 = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD") & (_df_ss.FOULING_BETA==0.0)]
    _pk3 = _pk3.loc[_pk3.P_DENS_W_M2.idxmax()]
    # Annotation shifted further right and down to clear curve
    _a1.annotate(f"Power peak N={int(_pk3.N)}\nP={_pk3.P_DENS_W_M2:.2f} W/m²",
                 xy=(_pk3.N, _pk3.P_DENS_W_M2),
                 xytext=(_pk3.N + 32, _pk3.P_DENS_W_M2 - 0.80),
                 arrowprops=dict(arrowstyle="->", lw=1.0), fontsize=8)
    _a2.axhline(100, color=_CK, ls="--", lw=1.8)
    _a2.annotate(f"Power peak N={int(_pk3.N)}\nHR$_{{worst}}$ only {_pk3.HARM_REDUX_PCT:.0f}%",
                 xy=(_pk3.N, _pk3.HARM_REDUX_PCT),
                 xytext=(_pk3.N + 28, _pk3.HARM_REDUX_PCT + 14),
                 arrowprops=dict(arrowstyle="->", lw=1.0), fontsize=8,
                 bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.85, edgecolor="none"))

    _a1.set_xlabel("Cell pairs N", fontsize=9); _a1.grid(alpha=0.25)
    _a1.set_ylabel("Power density [W/m²]", fontsize=9)
    _a1.set_title("Power density (tick = compliance window entry)", fontsize=9)
    # Legend outside upper right for left panel
    _a1.legend(fontsize=7.5, loc="upper right", bbox_to_anchor=(1.0, 1.0), framealpha=0.85)

    _a2.set_xlabel("Cell pairs N", fontsize=9); _a2.grid(alpha=0.25)
    _a2.set_ylabel("HR$_{worst}$ [%]", fontsize=9)
    _a2.set_title("HR$_{worst}$ (shading = compliance window)", fontsize=9)
    _a2.legend(fontsize=7.5, loc="lower right")
    plt.tight_layout()
    _save("paper_supp1_regime_curves.png")

    # ── SUPP 2 ── Fouling Paradox  (v6 fixes) ─────────────────────────────────
    # Left legend ncol=3; right y-axis narrowed.

    _cba = _df_cas[(_df_cas.SITE_TAG=="CARLSBAD") & (_df_cas.N_TOTAL==_NT240)
                   & (_df_cas.C_L_IN_MOL_L==_CL) & (_df_cas.FRESH_DILUTE==True)
                   & (_df_cas.PHYS_COMPLY==True)].copy()

    fig, (_a1, _a2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(
        "Fouling Paradox — Carlsbad\n"
        r"N$_{total}$ = 240 (12 m²),  C$_L$ = 0.02 mol/L,  fresh-dilute cascade",
        fontsize=10, fontweight="bold")

    _sv = sorted(_cba.S.unique())
    _cm6 = plt.cm.viridis(np.linspace(0.1, 0.9, len(_sv)))
    for _s, _col in zip(_sv, _cm6):
        _grp = _cba[_cba.S==_s].sort_values("FOULING_BETA")
        if len(_grp) >= 2:
            _a1.plot(_grp.FOULING_BETA, _grp.P_DENS_CASCADE_W_M2, "o-",
                     color=_col, lw=1.8, ms=5, label=f"S={_s}")
    _a1.set_xlabel("β"); _a1.set_ylabel("Power density [W/m²]")
    _a1.set_title("Power vs β for each S", fontsize=9)
    # Move legend outside right edge so it doesn't block low-S curves
    _a1.legend(fontsize=7, ncol=1, loc="upper left",
               bbox_to_anchor=(1.01, 1.0), borderaxespad=0, framealpha=0.85)
    _a1.grid(alpha=0.25)

    _sstar = [(b, int(_cba[_cba.FOULING_BETA==b].loc[
                          _cba[_cba.FOULING_BETA==b].P_DENS_CASCADE_W_M2.idxmax()].S))
              for b in _BETA_VALS if len(_cba[_cba.FOULING_BETA==b])]
    if _sstar:
        _bx, _sy = zip(*_sstar)
        _a2.plot(_bx, _sy, "s-", color="#7b2d8b", lw=2.5, ms=8)
        _unique_s = sorted(set(_sy))
        _a2.set_yticks(_unique_s)
        # Narrow y-axis to actual data range ±1
        _a2.set_ylim(min(_unique_s) - 1, max(_unique_s) + 1)
    _a2.set_xlabel("β"); _a2.set_ylabel("Optimal S*")
    _a2.set_title("Discrete S* shifts with fouling severity", fontsize=9)
    _a2.grid(alpha=0.25)
    plt.tight_layout()
    _save("paper_supp2_fouling_paradox.png")

    # ── SUPP 3 ── Morris sensitivity ───────────────────────────────────────────
    # No v6 changes needed beyond consistent axis labels.

    _mcb0 = (_df_mor[(_df_mor.SITE_TAG=="CARLSBAD") & (_df_mor.BASE_BETA==0.0)]
             .sort_values("mu_star", ascending=False).reset_index(drop=True))
    _params   = _mcb0["parameter"].tolist()
    _mu_stars = _mcb0["mu_star"].tolist()
    _cis      = (1.96 * _mcb0["se_mu_star"]).tolist()

    def _bc(p):
        if p in ("FOULING_BETA","BIOFILM_AMP","BIOFILM_K","DIVALENT_FRACTION"): return _CS
        return "#aaaaaa" if p in ("Q_L_M3_S","Q_H_M3_S") else _CF

    fig, ax = plt.subplots(figsize=(9.5, 6))
    ax.set_title(
        "Morris sensitivity ranking — Carlsbad, β = 0 anchor\n"
        "k = 14, r = 100  ·  μ* = mean absolute elementary effect on power density",
        fontsize=9.5, fontweight="bold")
    _y = np.arange(len(_params))[::-1]
    ax.barh(_y, _mu_stars, xerr=_cis, color=[_bc(p) for p in _params],
            edgecolor="white", height=0.7, capsize=3)
    _fb = _params.index("FOULING_BETA")
    ax.annotate(f"FOULING_BETA rank {_fb+1}/14\nμ* = {_mu_stars[_fb]:.3f}",
                xy=(_mu_stars[_fb], len(_params)-1-_fb),
                xytext=(_mu_stars[_fb]*12, len(_params)-1-_fb+2.2),
                arrowprops=dict(arrowstyle="->", color=_CS, lw=1.2),
                fontsize=10, color=_CS, fontweight="bold")
    ax.set_yticks(_y); ax.set_yticklabels(_params, fontsize=8)
    ax.set_xscale("log"); ax.set_xlabel("μ* [W/m²] — log scale"); ax.grid(axis="x", alpha=0.25)
    ax.legend(handles=[mpatches.Patch(color="#aaa", label="Flow rate"),
                        mpatches.Patch(color=_CF,   label="Design parameter"),
                        mpatches.Patch(color=_CS,   label="Fouling parameter")],
              fontsize=8, loc="lower right")
    plt.tight_layout()
    _save("paper_supp3_morris.png")

    # ── SUPP 4 ── Serial compliance degradation ────────────────────────────────
    # No v6 changes.

    _s7sites = [s for s in _SITES if s["tag"] in ("CARLSBAD","JEBEL-ALI","HYPERSALINE")]
    fig, _axes = plt.subplots(2, 3, figsize=(13, 8), sharey=False)
    fig.suptitle(
        "Serial cascade cannot maintain compliance — fresh-dilute can\n"
        r"N$_{total}$ = 240 (12 m²),  C$_L$ = 0.02 mol/L",
        fontsize=10, fontweight="bold")
    for _ci, _sd in enumerate(_s7sites):
        for _ri, _beta in enumerate([0.0, 0.8]):
            _ax = _axes[_ri][_ci]
            _sub = _df_cas[(_df_cas.SITE_TAG==_sd["tag"]) & (_df_cas.N_TOTAL==_NT240)
                           & (_df_cas.FOULING_BETA==_beta) & (_df_cas.C_L_IN_MOL_L==_CL)
                           & (~_df_cas.ANY_INVERSION)]
            for _fr2, _col, _lab in [(True,_CF,"Fresh-dilute"), (False,_CS,"Serial")]:
                _grp = _sub[_sub.FRESH_DILUTE==_fr2].sort_values("S")
                if len(_grp):
                    _ax.plot(_grp.S, _grp.HARM_REDUX_PCT, "o-", color=_col, lw=2.5, ms=5, label=_lab)
            _ax.axhline(100, color=_CK, ls="--", lw=1.5)
            _ax.axhspan(100, 106, alpha=0.07, color=_CK)
            _ax.set_ylim(55, 106)
            _ax.set_title(f"{_sd['label'].replace(chr(10),' ')}  β={_beta:.1f}", fontsize=9)
            if _ci==0: _ax.set_ylabel("HR$_{worst}$ [%]")
            if _ri==1: _ax.set_xlabel("Stage count S")
            if _ci==0 and _ri==0: _ax.legend(fontsize=8.5)
            _ax.grid(alpha=0.3)
    plt.tight_layout()
    _save("paper_supp4_serial_degradation.png")

    # ══════════════════════════════════════════════════════════════════════════
    print("\n[GENIUS POSTER FIGURES]")
    # ══════════════════════════════════════════════════════════════════════════

    # ── GENIUS FIG 1 ── "The Wrong Question"  (v6 fixes) ─────────────────────
    # "Inversion begins" moved inward; legend lower left; footer → subtitle;
    # compliance threshold label near right axis.

    _sst = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD") & (_df_ss.FOULING_BETA==_BETA0)
                  & (_df_ss.C_L_IN_MOL_L==_CL)].sort_values("N").copy()
    _pkr = _sst.loc[_sst.P_DENS_W_M2.idxmax()]
    _wnr = _sst[_sst.PHYS_COMPLY==True]
    _ivr = _sst[_sst.ANY_INVERSION==True].sort_values("N")

    fig, _ax1 = plt.subplots(figsize=(11, 6.5))
    fig.suptitle("The Wrong Question", fontsize=16, fontweight="bold", y=1.00)
    # Footer info promoted to subtitle
    _ax1.set_title(
        r"Carlsbad SWRO  ·  C$_H$ = 1.15 mol/L,  C$_L$ = 0.02 mol/L,  β = 0"
        "\nOptimizing single-stage RED for power alone cannot achieve compliance",
        fontsize=10, color="#444")

    _ax2 = _ax1.twinx()
    _ax1.plot(_sst.N, _sst.P_DENS_W_M2, color=_CF, lw=3, label="Power density", zorder=3)
    _ax2.plot(_sst.N, _sst.HARM_REDUX_PCT, color=_CS, lw=2.5, ls="--", label="HR$_{worst}$", zorder=2)

    if len(_wnr):
        _n0, _n1 = int(_wnr.N.min()), int(_wnr.N.max())
        _ax1.axvspan(_n0, _n1, alpha=0.20, color=_CK, zorder=0)
        _ax1.text((_n0+_n1)/2, _sst.P_DENS_W_M2.max()*0.45,
                  f"Compliance window\nN = {_n0}–{_n1}",
                  ha="center", va="center", fontsize=12, color=_CK,
                  fontweight="bold", zorder=4)

    if len(_ivr):
        _ni = int(_ivr.N.min())
        _ax1.axvline(_ni, color="#888", ls=":", lw=3.5)
        # Moved inward: was N+2, now placed at N-18 so it stays inside the plot
        _ax1.text(_ni - 18, _sst.P_DENS_W_M2.max()*0.72,
                  f"Inversion begins\nN = {_ni}", ha="right", fontsize=10, color="#555")

    _ax1.scatter([int(_pkr.N)], [_pkr.P_DENS_W_M2], s=200, color=_CF, zorder=6, marker="*")
    _ax1.annotate(
        f"Power optimum  N = {int(_pkr.N)}\nP = {_pkr.P_DENS_W_M2:.2f} W/m²\n"
        f"HR$_{{worst}}$ = {_pkr.HARM_REDUX_PCT:.0f}%  ✗",
        xy=(int(_pkr.N), _pkr.P_DENS_W_M2),
        # Shifted left so text sits clear of title/subtitle area
        xytext=(int(_pkr.N) - 10, _pkr.P_DENS_W_M2 + 0.30),
        arrowprops=dict(arrowstyle="->", color=_CF, lw=1.5),
        fontsize=11, color=_CF, fontweight="bold")

    # Compliance threshold: raised to 104 on the HR axis so it clears the red dashed line
    _ax2.axhline(100, color=_CK, ls="--", lw=3.0, zorder=5)
    _ax2.text(1.01, 104, "100% compliance\nthreshold", transform=_ax2.get_yaxis_transform(),
              fontsize=8.5, color=_CK, va="bottom")

    _ax1.set_xlabel("Single-stage cell pairs N", fontsize=13)
    _ax1.set_ylabel("Power density [W/m²]", fontsize=13, color=_CF)
    _ax2.set_ylabel("HR$_{worst}$ [%]", fontsize=13, color=_CS)
    _ax1.tick_params(axis="y", labelcolor=_CF)
    _ax2.tick_params(axis="y", labelcolor=_CS)
    _ax2.set_ylim(-5, 115)

    _h1, _l1 = _ax1.get_legend_handles_labels()
    _h2, _l2 = _ax2.get_legend_handles_labels()
    # Legend lower left — away from compliance window label
    _ax1.legend(_h1+_h2, _l1+_l2, fontsize=11, loc="lower left")
    _ax1.grid(alpha=0.25)
    plt.tight_layout()
    _save("genius_fig1_wrong_question.png")

    # ── GENIUS FIG 2 ── First design that actually complies  (v6 fixes) ───────
    # Title changed; ✓ checkmarks above bars.

    def _first_comply_cascade(fresh_flag):
        _sub = _df_cas[(_df_cas.SITE_TAG=="CARLSBAD") & (_df_cas.FOULING_BETA==_BETA0)
                       & (_df_cas.C_L_IN_MOL_L==_CL) & (_df_cas.FRESH_DILUTE==fresh_flag)
                       & (_df_cas.PHYS_COMPLY==True) & (~_df_cas.ANY_INVERSION)]
        _fa   = _sub.TOTAL_AREA_M2.min()
        _best = (_sub[_sub.TOTAL_AREA_M2==_fa]
                 .loc[_sub[_sub.TOTAL_AREA_M2==_fa].P_DENS_CASCADE_W_M2.idxmax()])
        return _fa, _best

    _ss_fc3 = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD") & (_df_ss.FOULING_BETA==_BETA0)
                     & (_df_ss.C_L_IN_MOL_L==_CL) & (_df_ss.PHYS_COMPLY==True)
                     & (~_df_ss.ANY_INVERSION)]
    _ss_fa3 = _ss_fc3.TOTAL_AREA_M2.min()
    _ss_b3  = (_ss_fc3[_ss_fc3.TOTAL_AREA_M2==_ss_fa3]
               .loc[_ss_fc3[_ss_fc3.TOTAL_AREA_M2==_ss_fa3].P_DENS_W_M2.idxmax()])
    _fr_fa3, _fr_b3 = _first_comply_cascade(True)
    _se_fa3, _se_b3 = _first_comply_cascade(False)

    _bdata = [
        {"label":"Single-stage",       "area":float(_ss_fa3), "P":float(_ss_b3.P_DENS_W_M2),
         "detail":f"N = {int(_ss_b3.N)}", "color":_CSS},
        {"label":"Fresh-dilute\ncascade","area":float(_fr_fa3), "P":float(_fr_b3.P_DENS_CASCADE_W_M2),
         "detail":f"S = {int(_fr_b3.S)}, N_per = {int(_fr_b3.N_PER_STAGE)}", "color":_CF},
        {"label":"Serial\ncascade",    "area":float(_se_fa3), "P":float(_se_b3.P_DENS_CASCADE_W_M2),
         "detail":f"S = {int(_se_b3.S)}, N_per = {int(_se_b3.N_PER_STAGE)}", "color":_CS},
    ]

    fig, ax = plt.subplots(figsize=(9, 6.5))
    # Changed title
    ax.set_title(
        "First Design That Actually Complies\n"
        r"Carlsbad  ·  C$_H$ = 1.15 mol/L,  C$_L$ = 0.02 mol/L,  β = 0"
        "\nSame membrane material — different architecture",
        fontsize=12, fontweight="bold")

    bars = ax.bar(range(3), [b["P"] for b in _bdata],
                  color=[b["color"] for b in _bdata],
                  width=0.55, edgecolor="white", linewidth=1.5)
    ax.set_xticks(range(3))
    ax.set_xticklabels([b["label"] for b in _bdata], fontsize=13)
    ax.set_ylabel("Power density at first full compliance [W/m²]", fontsize=11)

    for _i, _b in enumerate(_bdata):
        # Bar value + config detail above bar top
        ax.text(_i, _b["P"] + 0.04,
                f"{_b['P']:.3f} W/m²\n{_b['area']:.1f} m²\n{_b['detail']}",
                ha="center", va="bottom", fontsize=10, fontweight="bold")
        # ✓ checkmark raised: use 15% of bar height so it clears x-axis tick labels
        ax.text(_i, _b["P"] * 0.18, "✓ Complies",
                ha="center", va="bottom", fontsize=9, color=_CK, fontweight="bold")

    ax.set_ylim(0, max(b["P"] for b in _bdata) * 1.55)
    ax.grid(axis="y", alpha=0.25)
    ax.text(0.5, -0.14,
            "At the first area where each architecture achieves full compliance,\n"
            "fresh-dilute delivers the highest power at the smallest membrane investment.",
            transform=ax.transAxes, ha="center", fontsize=10.5, color="#444", style="italic")
    plt.tight_layout()
    _save("genius_fig2_three_regimes.png")

    # ── GENIUS FIG 3 ── Same membrane, different fate  (v6 fixes) ─────────────
    # "✓ Compliant" moved inward; red axhspan below 100%; β range in subtitle.

    _smfr, _smse = [], []
    for _beta in _BETA_VALS:
        for _sub2, _lst2, _fresh2 in [
            (_df_cas[(_df_cas.SITE_TAG=="CARLSBAD") & (_df_cas.N_TOTAL==_NT240)
                     & (_df_cas.FOULING_BETA==_beta) & (_df_cas.C_L_IN_MOL_L==_CL)
                     & (_df_cas.FRESH_DILUTE==True) & (~_df_cas.ANY_INVERSION)],
             _smfr, True),
            (_df_cas[(_df_cas.SITE_TAG=="CARLSBAD") & (_df_cas.N_TOTAL==_NT240)
                     & (_df_cas.FOULING_BETA==_beta) & (_df_cas.C_L_IN_MOL_L==_CL)
                     & (~_df_cas.FRESH_DILUTE) & (~_df_cas.ANY_INVERSION)],
             _smse, False)]:
            if len(_sub2):
                _r = _sub2.loc[_sub2.P_DENS_CASCADE_W_M2.idxmax()]
                _lst2.append({"beta":_beta, "P":_r.P_DENS_CASCADE_W_M2, "HR":_r.HARM_REDUX_PCT})

    _dfsmfr = pd.DataFrame(_smfr)
    _dfsmse = pd.DataFrame(_smse)

    fig, (_ax1, _ax2) = plt.subplots(1, 2, figsize=(13, 6.5))
    fig.suptitle(
        "Same Membrane Investment.  Different Discharge Fate.\n"
        r"N$_{total}$ = 240 (12 m² total membrane)",
        fontsize=14, fontweight="bold")
    # β range added to subtitle
    _ax1.set_title(
        r"Carlsbad SWRO  ·  12 m² total membrane  ·  C$_L$ = 0.02 mol/L  ·  β = 0.0 – 1.5",
        fontsize=9.5, color="#444")

    _ax1.plot(_dfsmfr.beta, _dfsmfr.P, "o-", color=_CF, lw=3, ms=9, label="Fresh-dilute cascade")
    _ax1.plot(_dfsmse.beta, _dfsmse.P, "s-", color=_CS, lw=3, ms=9, label="Serial cascade")
    _ax1.fill_between(_dfsmfr.beta, _dfsmfr.P, _dfsmse.P, alpha=0.12, color=_CF)
    _ax1.set_xlabel("Fouling severity β", fontsize=13)
    _ax1.set_ylabel("Power density [W/m²]", fontsize=13)
    _ax1.set_title("Power retained under fouling", fontsize=12, fontweight="bold")
    _ax1.legend(fontsize=11); _ax1.grid(alpha=0.25); _ax1.tick_params(labelsize=11)

    _ax2.plot(_dfsmfr.beta, _dfsmfr.HR, "o-", color=_CF, lw=3, ms=9, label="Fresh-dilute cascade")
    _ax2.plot(_dfsmse.beta, _dfsmse.HR, "s-", color=_CS, lw=3, ms=9, label="Serial cascade")
    _ax2.axhline(100, color=_CK, ls="--", lw=2.5, label="Full compliance required")
    # Light red shading below 100% to mark non-compliant zone
    _ax2.axhspan(_dfsmse.HR.min() - 1, 100, alpha=0.07, color=_CS, zorder=0)
    _ax2.set_xlabel("Fouling severity β", fontsize=13)
    _ax2.set_ylabel("HR$_{worst}$ [%]", fontsize=13)
    _ax2.set_title("Compliance outcome", fontsize=12, fontweight="bold")
    _ax2.set_ylim(_dfsmse.HR.min() - 2, 105)
    # "✓ Compliant" moved inward (x=1.1 → 0.85 of beta range)
    _bmax = float(_dfsmfr.beta.max())
    _ax2.text(_bmax * 0.82, 101.2, "✓ Compliant", fontsize=12, color=_CK, fontweight="bold")
    # Non-compliant label: pushed to lower portion, well below the red serial line
    _ax2.text(_bmax * 0.30, _dfsmse.HR.min() + 0.5,
              "✗ Non-compliant zone\n  (serial only)",
              fontsize=11, color=_CS, fontweight="bold")
    _ax2.legend(fontsize=11); _ax2.grid(alpha=0.25); _ax2.tick_params(labelsize=11)
    plt.tight_layout()
    _save("genius_fig3_same_membrane.png")

    # ── GENIUS FIG 4 ── Hidden brine ladder  (v6 fixes) ───────────────────────
    # Annotation only above Carlsbad; concentrations in x-tick labels.

    _ladder = []
    for _tag in _SITE_TAGS:
        _sub = _df_cas[(_df_cas.SITE_TAG==_tag) & (_df_cas.FOULING_BETA==_BETA0)
                       & (_df_cas.C_L_IN_MOL_L==_CL) & (_df_cas.FRESH_DILUTE==True)
                       & (_df_cas.PHYS_COMPLY==True) & (~_df_cas.ANY_INVERSION)]
        _ladder.append(_sub.TOTAL_AREA_M2.min() if len(_sub) else np.nan)

    # Concentrations added to x-tick labels
    _site_sh = [
        "Valley Water\n(brackish)\n0.35 mol/L",
        "Monterey\n(coastal)\n0.75 mol/L",
        "Carlsbad\n(SWRO)\n1.15 mol/L",
        "Jebel Ali\n(high-salinity)\n1.80 mol/L",
        "Hypersaline\n(ZLD)\n3.00 mol/L",
    ]
    _bcols = [_CK, _CK, _CW, _CS, "#8b0000"]

    fig, ax = plt.subplots(figsize=(12, 6.5))
    ax.set_title(
        "The Hidden Brine Ladder\n"
        "Higher salinity → more membrane needed to comply\n"
        r"Fresh-dilute cascade,  β = 0,  C$_L$ = 0.02 mol/L",
        fontsize=13, fontweight="bold")
    ax.bar(range(len(_SITE_TAGS)), _ladder, color=_bcols, edgecolor="white", linewidth=1.5, width=0.6)
    ax.set_xticks(range(len(_SITE_TAGS)))
    ax.set_xticklabels(_site_sh, fontsize=10)
    ax.set_ylabel("Minimum membrane area for full compliance [m²]", fontsize=11)

    for _i, _v in enumerate(_ladder):
        if not np.isnan(_v):
            ax.text(_i, _v + 0.4, f"{_v:.0f} m²", ha="center", va="bottom",
                    fontsize=12, fontweight="bold")

    # Annotation above Carlsbad only (index 2), pointing straight up
    _carl_v = _ladder[2]
    ax.annotate(
        "Carlsbad is the transition point:\narchitecture choice matters here.",
        xy=(2, _carl_v),
        xytext=(2, _carl_v + 6),
        arrowprops=dict(arrowstyle="->", color=_CW, lw=2),
        fontsize=10, color=_CW, fontweight="bold", ha="center")

    ax.set_ylim(0, max(v for v in _ladder if not np.isnan(v)) * 1.60)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    _save("genius_fig4_brine_ladder.png")

    # ── GENIUS FIG 5 ── Advantage summary  (v6 fixes) ─────────────────────────
    # 587 both-compliant subset line added to stats box.

    fig, ax = plt.subplots(figsize=(11, 6.5))
    ax.set_title(
        "Fresh-Dilute Wins — Every Single Time\n"
        "1,728 matched comparisons across five sites, five dilute concentrations,\n"
        "eight fouling levels, and nine membrane-investment levels",
        fontsize=13, fontweight="bold")
    _n2, _b2, _p2 = ax.hist(_advs, bins=35, edgecolor="white", alpha=0.85, color=_CF)
    for _p, _lo in zip(_p2, _b2[:-1]):
        if _lo > 100: _p.set_facecolor("#1a4f7a")
        elif _lo > 50: _p.set_facecolor(_CF)
        else: _p.set_facecolor("#74add1")
    ax.axvline(_advs.mean(),     color=_CS, ls="--", lw=2.5, label=f"Mean = {_advs.mean():.1f}%")
    ax.axvline(np.median(_advs), color=_CW, ls="--", lw=2.5, label=f"Median = {np.median(_advs):.1f}%")
    ax.set_xlabel("Fresh-dilute power-density advantage over serial [%]", fontsize=13)
    ax.set_ylabel("Number of matched comparisons", fontsize=13)
    ax.legend(fontsize=12); ax.grid(alpha=0.3); ax.tick_params(labelsize=12)
    # Stats box with explicit 587/587 line
    ax.text(0.97, 0.95,
            f"100% fresh-dilute wins\nAll {len(_advs):,} cases > 0%\n"
            f"Median +{np.median(_advs):.1f}%\n\n"
            "Also 587 / 587 wins\nin both-compliant subset\n"
            "(median +85.5%)",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=11, fontweight="bold", color=_CF,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.92, edgecolor=_CF))
    plt.tight_layout()
    _save("genius_fig5_advantage_summary.png")

    # ── GENIUS FIG 6 ── Robustness box  (v6 fixes) ────────────────────────────
    # Morris wording corrected; bottom text condensed and larger.

    _stats = [
        ("38,400",  "cascade configurations\nscreened"),
        ("10,811",  "single-stage\nconfigurations"),
        ("5",       "brine sites\n(0.35 – 3.00 mol/L)"),
        ("8",       "fouling severity\nlevels (β 0 – 1.5)"),
        ("9",       "membrane investment\nlevels (3 – 30 m²)"),
        # Corrected: describes the summary rows + screening parameters
        ("Morris\nscreening",  "k = 14 parameters\nr = 100 per anchor\n5 sites × 8 β anchors"),
    ]
    # Condensed, larger robustness text
    _robustness = (
        "Biofilm stress test added in v12.  Biofilm parameters ranked near the bottom\n"
        "(BIOFILM_AMP rank 12/14, BIOFILM_K rank 14/14).\n"
        "The architecture result is not driven by the biofilm assumption."
    )

    fig, ax = plt.subplots(figsize=(10, 6.5))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    fig.patch.set_facecolor("#f7f9fc")

    ax.text(0.5, 0.96, "What We Tested", ha="center", va="top",
            fontsize=18, fontweight="bold", color="#1a1a2e", transform=ax.transAxes)
    ax.text(0.5, 0.89, "OsmoFlux Grand Sweep v12  ·  38,400 cascade + 10,811 single-stage configs",
            ha="center", va="top", fontsize=10, color="#444", transform=ax.transAxes)

    _cols_g = [0.13, 0.50, 0.87]
    _rows_g = [0.70, 0.47]
    for _idx, (_num, _lbl) in enumerate(_stats):
        _cx = _cols_g[_idx % 3]; _cy = _rows_g[_idx // 3]
        _box = mpatches.FancyBboxPatch((_cx-0.14, _cy-0.12), 0.28, 0.19,
                                         boxstyle="round,pad=0.02",
                                         facecolor="white", edgecolor=_CF,
                                         linewidth=2, transform=ax.transAxes, zorder=2)
        ax.add_patch(_box)
        ax.text(_cx, _cy+0.055, _num, ha="center", va="center",
                fontsize=18, fontweight="bold", color=_CF, transform=ax.transAxes, zorder=3)
        ax.text(_cx, _cy-0.045, _lbl, ha="center", va="center",
                fontsize=8.5, color="#333", transform=ax.transAxes, zorder=3, multialignment="center")

    _rbox = mpatches.FancyBboxPatch((0.02, 0.03), 0.96, 0.20,
                                      boxstyle="round,pad=0.02",
                                      facecolor="#eaf4ea", edgecolor=_CK,
                                      linewidth=2, transform=ax.transAxes, zorder=2)
    ax.add_patch(_rbox)
    # Larger font for robustness text (was 9.5 → 11)
    ax.text(0.5, 0.13, _robustness, ha="center", va="center",
            fontsize=11, color="#1a3a1a", transform=ax.transAxes, zorder=3,
            multialignment="center")
    plt.tight_layout()
    _save("genius_fig6_robustness_box.png")

    # ══════════════════════════════════════════════════════════════════════════
    print("\n[LEGACY PAPER FIGURES  (prefix: legacy_fig / legacy_figN)]")
    # ══════════════════════════════════════════════════════════════════════════
    # These are regenerated from Grand Sweep v12 data but carry legacy filenames.
    # No visual polish changes applied here; they are reference figures only.

    # ── LEGACY FIG 1 ─────────────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(
        f"Single-Stage Compliance Window — Carlsbad (C_H=1.15 mol/L)\n"
        f"Compliance window: {_n_win_hi-_n_win_lo+1} cell pairs wide",
        fontsize=11, fontweight="bold")
    ax1.plot(_ss0.N, _ss0.P_DENS_W_M2, color=_CF, lw=2)
    ax1.axvline(_n_win_lo, color=_CK, ls="--", lw=1.5, label=f"Window open N={_n_win_lo}")
    ax1.axvline(_n_inv,    color=_CS, ls=":",  lw=1.5, label=f"First inversion N={_n_inv}")
    ax1.set_xlabel("Cell pairs N"); ax1.set_ylabel("Power density [W/m²]")
    ax1.set_title("Power density"); ax1.legend(fontsize=9); ax1.grid(alpha=0.25)
    ax2.plot(_ss0.N, _ss0.HARM_REDUX_PCT, color="#7b2d8b", lw=2)
    ax2.axhline(100, color=_CK, ls=":", lw=1.5, label="Full compliance")
    ax2.axvspan(_n_inv, _ss0.N.max()+5, alpha=0.12, color=_CS, label="Inversion zone")
    ax2.axvspan(_n_win_lo, _n_win_hi, alpha=0.18, color=_CK, label=f"Window N={_n_win_lo}–{_n_win_hi}")
    ax2.set_xlabel("Cell pairs N"); ax2.set_ylabel("Harm Reduction [%]")
    ax2.set_title("Compliance window"); ax2.legend(fontsize=9); ax2.grid(alpha=0.25)
    ax2.set_xlim(left=0)
    plt.tight_layout(); _save("legacy_fig1_single_stage_compliance_ceiling.png")

    # ── LEGACY FIG 2 ─────────────────────────────────────────────────────────
    _cas_cb = _df_cas[(_df_cas.SITE_TAG=="CARLSBAD") & (_df_cas.FOULING_BETA==0.0)
                      & (_df_cas.C_L_IN_MOL_L==_CL) & (_df_cas.N_TOTAL==_NT240)].copy()
    _f_b0   = _cas_cb[_cas_cb.FRESH_DILUTE==True][~_cas_cb[_cas_cb.FRESH_DILUTE==True].ANY_INVERSION]
    _s_b0   = _cas_cb[~_cas_cb.FRESH_DILUTE][~_cas_cb[~_cas_cb.FRESH_DILUTE].ANY_INVERSION]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f"Cascade vs Single-Stage — Carlsbad, N_total={_NT240}", fontsize=11, fontweight="bold")
    ax1.scatter([_NT240*_AMEM]*len(_f_b0), _f_b0.P_DENS_CASCADE_W_M2, color=_CF, s=40, label="Fresh")
    ax1.scatter([_NT240*_AMEM]*len(_s_b0), _s_b0.P_DENS_CASCADE_W_M2, color=_CS, s=40, label="Serial")
    ax1.plot(_ss0.TOTAL_AREA_M2, _ss0.P_DENS_W_M2, color=_CSS, ls="--", lw=1.5, label="Single stage")
    ax1.set_xlabel("Total area [m²]"); ax1.set_ylabel("Power density [W/m²]")
    ax1.legend(fontsize=9); ax1.grid(alpha=0.25)
    ax2.scatter([_NT240*_AMEM]*len(_f_b0), _f_b0.HARM_REDUX_PCT, color=_CF, s=40, label="Fresh")
    ax2.scatter([_NT240*_AMEM]*len(_s_b0), _s_b0.HARM_REDUX_PCT, color=_CS, s=40, label="Serial")
    ax2.plot(_ss0.TOTAL_AREA_M2, _ss0.HARM_REDUX_PCT, color=_CSS, ls="--", lw=1.5, label="Single stage")
    ax2.axhline(100, color=_CK, ls=":", lw=1.5, label="Full compliance")
    ax2.set_xlabel("Total area [m²]"); ax2.set_ylabel("Harm Reduction [%]")
    ax2.legend(fontsize=9); ax2.grid(alpha=0.25)
    plt.tight_layout(); _save("legacy_fig2_cascade_vs_single_stage.png")

    # ── LEGACY FIG 3 ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=False)
    fig.suptitle(f"Power Density vs S (N_total={_NT240}, β=0, C_L=0.02)", fontsize=11, fontweight="bold")
    for ax, sd in zip(axes, _SITES):
        sub = _df_cas[(_df_cas.SITE_TAG==sd["tag"]) & (_df_cas.N_TOTAL==_NT240)
                      & (_df_cas.FOULING_BETA==_BETA0) & (_df_cas.C_L_IN_MOL_L==_CL)
                      & (~_df_cas.ANY_INVERSION)]
        for fresh, col, lab in [(True,_CF,"Fresh"), (False,_CS,"Serial")]:
            grp = sub[sub.FRESH_DILUTE==fresh].sort_values("S")
            if len(grp): ax.plot(grp.S, grp.P_DENS_CASCADE_W_M2, "o-", color=col, lw=2, ms=5, label=lab)
        ax.set_title(sd["label"], fontsize=9); ax.set_xlabel("S"); ax.grid(alpha=0.25)
        if ax is axes[0]: ax.set_ylabel("Power density [W/m²]"); ax.legend(fontsize=8)
    plt.tight_layout(); _save("legacy_fig3_power_vs_stages_all_sites.png")

    # ── LEGACY FIG 4 ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    fig.suptitle(f"Fresh-Dilute Advantage Over Serial [%] (N_total={_NT240})", fontsize=11, fontweight="bold")
    for ax, sd in zip(axes, _SITES):
        for beta, col in zip([0.0,0.4,0.8], ["#2166ac","#f4a100","#d73027"]):
            sub = _df_cas[(_df_cas.SITE_TAG==sd["tag"]) & (_df_cas.N_TOTAL==_NT240)
                          & (_df_cas.FOULING_BETA==beta) & (_df_cas.C_L_IN_MOL_L==_CL)
                          & (~_df_cas.ANY_INVERSION)]
            pts = [(s, (sub[(sub.FRESH_DILUTE==True)&(sub.S==s)].P_DENS_CASCADE_W_M2.max() /
                        sub[(sub.FRESH_DILUTE==False)&(sub.S==s)].P_DENS_CASCADE_W_M2.max() - 1)*100)
                   for s in sorted(sub.S.unique())
                   if len(sub[(sub.FRESH_DILUTE==True)&(sub.S==s)]) and
                      len(sub[(sub.FRESH_DILUTE==False)&(sub.S==s)])]
            if pts:
                ss2, aa = zip(*pts)
                ax.plot(ss2, aa, "o-", color=col, lw=2, ms=5, label=f"β={beta}")
        ax.axhline(0, color="k", ls="--", lw=0.8, alpha=0.4)
        ax.set_title(sd["label"], fontsize=9); ax.set_xlabel("S"); ax.grid(alpha=0.25)
        if ax is axes[0]: ax.set_ylabel("Advantage [%]")
        if ax is axes[-1]: ax.legend(fontsize=8)
    plt.tight_layout(); _save("legacy_fig4_fresh_advantage_all_sites.png")

    # ── LEGACY FIG 5 ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"Harm Reduction — Carlsbad, C_L=0.02 (N_total={_NT240})", fontsize=11, fontweight="bold")
    for ax, beta in zip(axes, [0.0, 0.8]):
        sub = _df_cas[(_df_cas.SITE_TAG=="CARLSBAD") & (_df_cas.N_TOTAL==_NT240)
                      & (_df_cas.FOULING_BETA==beta) & (_df_cas.C_L_IN_MOL_L==_CL)
                      & (~_df_cas.ANY_INVERSION)]
        for fresh, col, lab in [(True,_CF,"Fresh"), (False,_CS,"Serial")]:
            grp = sub[sub.FRESH_DILUTE==fresh].sort_values("S")
            if len(grp): ax.plot(grp.S, grp.HARM_REDUX_PCT, "o-", color=col, lw=2, ms=5, label=lab)
        ax.axhline(100, color=_CK, ls=":", lw=1.5, label="Full compliance")
        ax.set_title(f"β = {beta:.1f}"); ax.set_xlabel("Stage count S")
        ax.set_ylabel("Harm Reduction [%]"); ax.legend(fontsize=9); ax.grid(alpha=0.25)
        ax.set_ylim(bottom=80)
    plt.tight_layout(); _save("legacy_fig5_harm_reduction_with_ceiling.png")

    # ── LEGACY FIG 6 ─────────────────────────────────────────────────────────
    _cb_all6 = _df_cas[(_df_cas.SITE_TAG=="CARLSBAD") & (_df_cas.N_TOTAL==_NT240)
                       & (_df_cas.C_L_IN_MOL_L==_CL) & (_df_cas.FRESH_DILUTE==True)
                       & (_df_cas.PHYS_COMPLY==True)].copy()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f"Fouling Paradox — Carlsbad, C_L=0.02 (N_total={_NT240})", fontsize=11, fontweight="bold")
    for s, col in zip(sorted(_cb_all6.S.unique()), plt.cm.viridis(np.linspace(0.1,0.9,len(sorted(_cb_all6.S.unique()))))):
        grp = _cb_all6[_cb_all6.S==s].sort_values("FOULING_BETA")
        if len(grp) >= 2: ax1.plot(grp.FOULING_BETA, grp.P_DENS_CASCADE_W_M2, "o-", color=col, lw=1.8, ms=5, label=f"S={s}")
    ax1.set_xlabel("β"); ax1.set_ylabel("Power density [W/m²]")
    ax1.set_title("Power vs fouling"); ax1.legend(fontsize=7, ncol=2); ax1.grid(alpha=0.25)
    _ss6 = [(b, int(_cb_all6[_cb_all6.FOULING_BETA==b].loc[_cb_all6[_cb_all6.FOULING_BETA==b].P_DENS_CASCADE_W_M2.idxmax()].S))
            for b in _BETA_VALS if len(_cb_all6[_cb_all6.FOULING_BETA==b])]
    if _ss6:
        _bx6, _sy6 = zip(*_ss6)
        ax2.plot(_bx6, _sy6, "s-", color="#7b2d8b", lw=2.5, ms=8)
        ax2.set_yticks(sorted(set(_sy6)))
    ax2.set_xlabel("β"); ax2.set_ylabel("Optimal S*")
    ax2.set_title("Optimal S* shift"); ax2.grid(alpha=0.25)
    plt.tight_layout(); _save("legacy_fig6_fouling_paradox.png")

    # ── LEGACY FIG 7 ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    fig.suptitle(f"Harm Reduction vs S — All Sites (N_total={_NT240}, β=0)", fontsize=11, fontweight="bold")
    for ax, sd in zip(axes, _SITES):
        sub = _df_cas[(_df_cas.SITE_TAG==sd["tag"]) & (_df_cas.N_TOTAL==_NT240)
                      & (_df_cas.FOULING_BETA==_BETA0) & (_df_cas.C_L_IN_MOL_L==_CL)
                      & (~_df_cas.ANY_INVERSION)]
        for fresh, col, lab in [(True,_CF,"Fresh"), (False,_CS,"Serial")]:
            grp = sub[sub.FRESH_DILUTE==fresh].sort_values("S")
            if len(grp): ax.plot(grp.S, grp.HARM_REDUX_PCT, "o-", color=col, lw=2, ms=5, label=lab)
        ax.axhline(100, color=_CK, ls=":", lw=1.2)
        ss_site = _df_ss[(_df_ss.SITE_TAG==sd["tag"]) & (_df_ss.FOULING_BETA==0.0) & (_df_ss.PHYS_COMPLY==True)]
        if len(ss_site): ax.axhline(ss_site.HARM_REDUX_PCT.max(), color="k", ls="--", lw=1, alpha=0.6)
        ax.set_title(sd["label"], fontsize=9); ax.set_xlabel("S"); ax.grid(alpha=0.25)
        if ax is axes[0]: ax.set_ylabel("Harm Reduction [%]"); ax.legend(fontsize=8)
    plt.tight_layout(); _save("legacy_fig7_harm_reduction_all_sites.png")

    # ── LEGACY FIG N1 ────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Peak Power Density vs Total Area (all N_total, β=0 and β=0.8)", fontsize=11, fontweight="bold")
    for ax, tag in zip(axes, ["CARLSBAD","JEBEL-ALI"]):
        for beta, ls in [(0.0,"-"), (0.8,"--")]:
            for fresh, col, lp in [(True,_CF,"Fresh"), (False,_CS,"Serial")]:
                pts = [(ntot*_AMEM, r.P_DENS_CASCADE_W_M2) for ntot in _N_TOTALS
                       for r in [(_best_fresh(_df_cas,tag,ntot,beta) if fresh
                                   else _best_serial(_df_cas,tag,ntot,beta))] if r is not None]
                if pts:
                    xs, ys = zip(*pts)
                    ax.plot(xs, ys, "o"+ls, color=col, lw=2, ms=6, label=f"{lp} β={beta}")
        ax.set_xlabel("Total area [m²]"); ax.set_ylabel("Peak power density [W/m²]")
        ax.set_title(f"C_H={'1.15' if tag=='CARLSBAD' else '1.80'} mol/L")
        ax.legend(fontsize=8); ax.grid(alpha=0.25)
    plt.tight_layout(); _save("legacy_figN1_peak_power_vs_area.png")

    # ── LEGACY FIG N2 ────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Optimal S* and N_per_stage vs Total Area", fontsize=11, fontweight="bold")
    for ax, tag in zip(axes, ["CARLSBAD","JEBEL-ALI"]):
        areas, sstars, npers = [], [], []
        for ntot in _N_TOTALS:
            r = _best_fresh(_df_cas, tag, ntot, 0.0)
            if r is not None:
                areas.append(ntot*_AMEM); sstars.append(int(r.S)); npers.append(int(r.N_PER_STAGE))
        ax2r = ax.twinx()
        ax.plot(areas, sstars, "o-", color=_CF, lw=2.5, ms=7, label="Optimal S*")
        ax2r.plot(areas, npers, "s--", color=_CS, lw=2, ms=7, label="N_per at S*")
        ax2r.axhspan(12, 15, alpha=0.10, color=_CS)
        ax.set_xlabel("Total area [m²]"); ax.set_ylabel("Optimal S*", color=_CF)
        ax2r.set_ylabel("N_per_stage at S*", color=_CS)
        ax.set_title(f"C_H={'1.15' if tag=='CARLSBAD' else '1.80'} mol/L")
        h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2r.get_legend_handles_labels()
        ax.legend(h1+h2, l1+l2, fontsize=8); ax.grid(alpha=0.25)
    plt.tight_layout(); _save("legacy_figN2_optimal_stages_vs_area.png")

    # ── LEGACY FIG N3 ────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.set_title("Single-Stage Compliance Window Is N_total-Independent", fontsize=10, fontweight="bold")
    ax.plot(_ss0.N, _ss0.HARM_REDUX_PCT, color="#7b2d8b", lw=2, label="Single-stage HR (Carlsbad)")
    ax.axhline(100, color=_CK, ls=":", lw=1.5)
    ax.axvspan(_n_win_lo, _n_win_hi, alpha=0.18, color=_CK, label=f"Compliance window N={_n_win_lo}–{_n_win_hi}")
    for ntot, col in zip(_N_TOTALS, plt.cm.tab10(np.linspace(0,1,len(_N_TOTALS)))):
        ax.axvline(ntot, color=col, ls="--", lw=1.0, alpha=0.8, label=f"N_total={ntot}")
    ax.set_xlabel("Single-stage cell pairs N"); ax.set_ylabel("Harm Reduction [%]")
    ax.legend(fontsize=7, ncol=2, loc="lower right"); ax.grid(alpha=0.25); ax.set_xlim(left=0)
    plt.tight_layout(); _save("legacy_figN3_compliance_invariance.png")

    # ── LEGACY FIG N4 ────────────────────────────────────────────────────────
    _mor_cb0 = (_df_mor[(_df_mor.SITE_TAG=="CARLSBAD") & (_df_mor.BASE_BETA==0.0)]
                .sort_values("mu_star", ascending=False).reset_index(drop=True))
    _params_n4 = _mor_cb0["parameter"].tolist()
    _mu_n4     = _mor_cb0["mu_star"].tolist()
    _cis_n4    = (1.96 * _mor_cb0["se_mu_star"]).tolist()
    def _bar_col(p):
        if p in ("FOULING_BETA","BIOFILM_AMP","BIOFILM_K","DIVALENT_FRACTION"): return _CS
        if p in ("Q_L_M3_S","Q_H_M3_S"): return "#aaaaaa"
        return _CF
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_title("Morris Sensitivity Ranking — Carlsbad, β=0 anchor (k=14, r=100)", fontsize=10, fontweight="bold")
    _yn4 = np.arange(len(_params_n4))[::-1]
    ax.barh(_yn4, _mu_n4, xerr=_cis_n4, color=[_bar_col(p) for p in _params_n4],
            edgecolor="white", height=0.7, capsize=3)
    _fb_idx = _params_n4.index("FOULING_BETA")
    ax.annotate(f"FOULING_BETA rank {_fb_idx+1}/{len(_params_n4)}\nμ*={_mu_n4[_fb_idx]:.3f}",
                xy=(_mu_n4[_fb_idx], len(_params_n4)-1-_fb_idx),
                xytext=(_mu_n4[_fb_idx]*3, len(_params_n4)-1-_fb_idx+1.5),
                arrowprops=dict(arrowstyle="->", color=_CS), fontsize=8, color=_CS)
    ax.set_yticks(_yn4); ax.set_yticklabels(_params_n4, fontsize=8); ax.set_xscale("log")
    ax.set_xlabel("μ* [W/m²] — log scale"); ax.grid(axis="x", alpha=0.25)
    ax.legend(handles=[mpatches.Patch(color="#aaaaaa", label="Flow rate"),
                        mpatches.Patch(color=_CF, label="Design parameter"),
                        mpatches.Patch(color=_CS, label="Fouling parameter")], fontsize=8, loc="lower right")
    plt.tight_layout(); _save("legacy_figN4_morris_sensitivity.png")

    # ── LEGACY FIG N5 ────────────────────────────────────────────────────────
    fig, (_a1, _a2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Fouling Shifts the Single-Stage Compliance Window — Carlsbad", fontsize=10, fontweight="bold")
    for beta, col in zip(_BETA4, _PAL4):
        sub = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD") & (_df_ss.FOULING_BETA==beta)].sort_values("N")
        if not len(sub): continue
        _a1.plot(sub.N, sub.HARM_REDUX_PCT, color=col, lw=2, label=f"β={beta:.1f}")
        win = sub[sub.PHYS_COMPLY==True]
        if len(win):
            _a1.axvspan(win.N.min(), win.N.max(), alpha=0.12, color=col)
            _a1.axvline(win.N.max()+1, color=col, ls=":", lw=1, alpha=0.7)
    _a1.axhline(100, color=_CK, ls="--", lw=1.2, alpha=0.8)
    _a1.set_xlabel("Cell pairs N"); _a1.set_ylabel("Harm Reduction [%]")
    _a1.legend(fontsize=9); _a1.grid(alpha=0.25); _a1.set_xlim(0, 310)
    _wo5, _wc5, _iv5, _ab5 = [], [], [], []
    for beta in _BETA_VALS:
        sub = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD") & (_df_ss.FOULING_BETA==beta)]
        win = sub[sub.PHYS_COMPLY==True]; inv = sub[sub.ANY_INVERSION==True].sort_values("N")
        if len(win) and len(inv):
            _ab5.append(beta); _wo5.append(win.N.min()); _wc5.append(win.N.max()); _iv5.append(inv.N.min())
    _a2.plot(_ab5, _wo5, "o-", color=_CK, lw=2, ms=7, label="Window open")
    _a2.plot(_ab5, _wc5, "s--", color=_CK, lw=2, ms=7, label="Window close")
    _a2.plot(_ab5, _iv5, "^-", color=_CS, lw=2, ms=7, label="First inversion N")
    _co5 = np.polyfit(_ab5, _iv5, 1)
    _a2.plot([0,1.5], np.polyval(_co5,[0,1.5]), color=_CS, ls=":", lw=1.5,
             label=f"Fit: {_co5[1]:.0f}+{_co5[0]:.1f}β")
    _a2.set_xlabel("β"); _a2.set_ylabel("Cell pairs N")
    _a2.legend(fontsize=8); _a2.grid(alpha=0.25)
    plt.tight_layout(); _save("legacy_figN5_compliance_window_vs_fouling.png")

    # ══════════════════════════════════════════════════════════════════════════
    print("\n[LEGACY STORYTELLING FIGURES  (prefix: legacy_figS)]")
    # ══════════════════════════════════════════════════════════════════════════

    # ── LEGACY FIG S1 ────────────────────────────────────────────────────────
    _unc_p  = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD") & (_df_ss.FOULING_BETA==0.0)]["P_DENS_W_M2"].max()
    _unc_n  = int(_df_ss[(_df_ss.SITE_TAG=="CARLSBAD") & (_df_ss.FOULING_BETA==0.0)].loc[
                  _df_ss[(_df_ss.SITE_TAG=="CARLSBAD") & (_df_ss.FOULING_BETA==0.0)]["P_DENS_W_M2"].idxmax(), "N"])
    _ss_p   = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD") & (_df_ss.FOULING_BETA==0.0)
                     & (_df_ss.PHYS_COMPLY==True)]["P_DENS_W_M2"].max()
    _casc_p = 2.5719
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.set_title("Three Operating Regimes — Carlsbad SWRO Brine (β=0)", fontsize=11, fontweight="bold")
    _rx = [0.7, 2.0, 3.3]
    _rp = [_unc_p, _ss_p, _casc_p]
    _rc = ["#d73027", "#f4a100", "#2166ac"]
    _rl = [f"Regime 1\nUnconstrained\nN={_unc_n}, P={_unc_p:.2f} W/m²",
           f"Regime 2\nCompliance-constrained SS\nN=154, P={_ss_p:.2f} W/m²",
           f"Regime 3\nFresh-dilute cascade\nN_total=120, S=8\nP={_casc_p:.2f} W/m²\n✓ Compliant"]
    bars = ax.bar(_rx, _rp, width=0.9, color=_rc, edgecolor="white", linewidth=1.5, zorder=3)
    for bar, lab, p in zip(bars, _rl, _rp):
        ax.text(bar.get_x()+bar.get_width()/2, p+0.04, lab, ha="center", va="bottom", fontsize=9)
    ax.annotate("", xy=(3.3, _casc_p), xytext=(3.3, _ss_p),
                arrowprops=dict(arrowstyle="<->", color="#555", lw=1.5))
    ax.text(3.65, (_casc_p+_ss_p)/2, f"+{(_casc_p/_ss_p-1)*100:.0f}%\nby cascade",
            va="center", ha="left", fontsize=9, color="#555")
    ax.set_xlim(0.1, 4.5); ax.set_ylim(0, _unc_p*1.55)
    ax.set_xticks([]); ax.set_ylabel("Power density [W/m²]", fontsize=11)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); _save("legacy_figS1_three_regime_hero.png")

    # ── LEGACY FIG S2 ────────────────────────────────────────────────────────
    _tags = [s["tag"] for s in _SITES]
    _site_labels_s2 = ["Valley Water\nC_H=0.35","Monterey\nC_H=0.75","Carlsbad\nC_H=1.15",
                       "Jebel Ali\nC_H=1.80","Hypersaline\nC_H=3.00"]
    _mab0, _mab15 = [], []
    for tag in _tags:
        for beta, lst in [(0.0,_mab0),(1.5,_mab15)]:
            found = False
            for ntot in _N_TOTALS:
                r = _best_fresh(_df_cas, tag, ntot, beta)
                if r is not None: lst.append(ntot*_AMEM); found=True; break
            if not found: lst.append(None)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.set_title("Minimum Membrane Investment for Full Compliance\nβ=0 vs β=1.5", fontsize=11, fontweight="bold")
    _ys2 = np.arange(len(_tags)); _bw = 0.35
    for i, (tag, a0, a15, lab) in enumerate(zip(_tags, _mab0, _mab15, _site_labels_s2)):
        if a0 is not None:
            ax.barh(_ys2[i]+_bw/2, a0, height=_bw, color=_CK, alpha=0.85, label="β=0 (clean)" if i==0 else "")
            ax.text(a0+0.3, _ys2[i]+_bw/2, f"{a0:.0f} m²", va="center", fontsize=9)
        if a15 is not None:
            ax.barh(_ys2[i]-_bw/2, a15, height=_bw, color=_CS, alpha=0.75, label="β=1.5 (severe)" if i==0 else "")
            ax.text(a15+0.3, _ys2[i]-_bw/2, f"{a15:.0f} m²", va="center", fontsize=9)
        elif a0 is not None:
            ax.text(a0*1.05, _ys2[i]-_bw/2, "infeasible at β=1.5", va="center", fontsize=8.5, color=_CS, style="italic")
    ax.set_yticks(_ys2); ax.set_yticklabels(_site_labels_s2, fontsize=10)
    ax.set_xlabel("Minimum compliance area [m²]"); ax.legend(fontsize=10); ax.grid(axis="x", alpha=0.3)
    ax.set_xlim(0, 35)
    plt.tight_layout(); _save("legacy_figS2_site_severity_ladder.png")

    # ── LEGACY FIG S3 ────────────────────────────────────────────────────────
    _cl_vals = [0.02, 0.05, 0.10, 0.20, 0.50]
    _cl_labs = ["0.02\n(treated\nwastewater)","0.05\n(groundwater\nmix)","0.10\n(brackish\nblend)",
                "0.20\n(dilute\nbrackish)","0.50\n(near-seawater)"]
    _cl_ma, _cl_bp = [], []
    for cl in _cl_vals:
        found = False
        for ntot in _N_TOTALS:
            r = _best_fresh(_df_cas, "CARLSBAD", ntot, 0.0, cl=cl)
            if r is not None: _cl_ma.append(ntot*_AMEM); _cl_bp.append(r.P_DENS_CASCADE_W_M2); found=True; break
        if not found: _cl_ma.append(None); _cl_bp.append(None)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Cost of Dilute Water Quality — Carlsbad (β=0)", fontsize=11, fontweight="bold")
    _xp = np.arange(len(_cl_vals))
    _vld = [(x,a,p) for x,a,p in zip(_xp,_cl_ma,_cl_bp) if a is not None]
    if _vld:
        _xv, _av, _pv = zip(*_vld)
        _cg = plt.cm.Blues_r(np.linspace(0.2, 0.8, len(_xv)))
        ax1.bar(_xv, _av, color=_cg, edgecolor="white", width=0.6)
        for x, a in zip(_xv, _av): ax1.text(x, a+0.3, f"{a:.0f} m²", ha="center", fontsize=10)
        ax2.bar(_xv, _pv, color=_cg, edgecolor="white", width=0.6)
        for x, p in zip(_xv, _pv): ax2.text(x, p+0.02, f"{p:.2f}", ha="center", fontsize=10)
    ax1.set_xticks(_xp); ax1.set_xticklabels(_cl_labs, fontsize=9)
    ax1.set_ylabel("Min compliance area [m²]"); ax1.set_xlabel("C_L [mol/L]")
    ax1.set_title("Min compliance area"); ax1.grid(axis="y", alpha=0.3); ax1.set_ylim(0, 35)
    ax2.set_xticks(_xp); ax2.set_xticklabels(_cl_labs, fontsize=9)
    ax2.set_ylabel("Best compliant P [W/m²]"); ax2.set_xlabel("C_L [mol/L]")
    ax2.set_title("Best compliant power density"); ax2.grid(axis="y", alpha=0.3)
    plt.tight_layout(); _save("legacy_figS3_dilute_water_quality_cost.png")

    # ── LEGACY FIG S4 ────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.set_title(f"Advantage Distribution — {len(_advs):,} matched cases", fontsize=11, fontweight="bold")
    _, bins_s4, patches_s4 = ax.hist(_advs, bins=40, color=_CF, edgecolor="white", alpha=0.85)
    for patch, left in zip(patches_s4, bins_s4[:-1]):
        if left > 100: patch.set_facecolor("#1a4f7a")
        elif left > 50: patch.set_facecolor(_CF)
        else: patch.set_facecolor("#74add1")
    ax.axvline(_advs.mean(), color=_CS, ls="--", lw=2, label=f"Mean={_advs.mean():.1f}%")
    ax.axvline(np.median(_advs), color=_CW, ls="--", lw=2, label=f"Median={np.median(_advs):.1f}%")
    ax.set_xlabel("Fresh-dilute advantage over serial [%]"); ax.set_ylabel("Count")
    ax.legend(fontsize=10); ax.grid(alpha=0.3)
    ax.text(0.97,0.95,f"n={len(_advs):,}\n>50%: {((_advs>50).mean()*100):.0f}%\n>100%: {((_advs>100).mean()*100):.0f}%",
            transform=ax.transAxes, ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
    plt.tight_layout(); _save("legacy_figS4_advantage_distribution.png")

    # ── LEGACY FIG S5 ────────────────────────────────────────────────────────
    _ht_matrix = np.full((len(_SITE_TAGS), len(_N_TOTALS)), np.nan)
    for i, tag in enumerate(_SITE_TAGS):
        for j, ntot in enumerate(_N_TOTALS):
            sub0  = _df_cas[(_df_cas.SITE_TAG==tag)&(_df_cas.N_TOTAL==ntot)&(_df_cas.FOULING_BETA==0.0)
                             &(_df_cas.C_L_IN_MOL_L==_CL)&(_df_cas.FRESH_DILUTE==True)&(~_df_cas.ANY_INVERSION)]
            sub15 = _df_cas[(_df_cas.SITE_TAG==tag)&(_df_cas.N_TOTAL==ntot)&(_df_cas.FOULING_BETA==1.5)
                             &(_df_cas.C_L_IN_MOL_L==_CL)&(_df_cas.FRESH_DILUTE==True)&(~_df_cas.ANY_INVERSION)]
            if len(sub0) and len(sub15):
                _ht_matrix[i,j] = sub15.P_DENS_CASCADE_W_M2.max() / sub0.P_DENS_CASCADE_W_M2.max() * 100
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.set_title("Fouling Resilience — P(β=1.5)/P(β=0) [%]", fontsize=11, fontweight="bold")
    _vmin_ht = np.nanmin(_ht_matrix); _vmax_ht = np.nanmax(_ht_matrix)
    _norm_ht = TwoSlopeNorm(vmin=_vmin_ht, vcenter=100, vmax=_vmax_ht)
    im_ht = ax.imshow(_ht_matrix, cmap="RdYlGn", norm=_norm_ht, aspect="auto")
    plt.colorbar(im_ht, ax=ax, label="P(β=1.5)/P(β=0) [%]", shrink=0.85)
    ax.set_xticks(range(len(_N_TOTALS)))
    ax.set_xticklabels([f"{ntot*_AMEM:.0f}" for ntot in _N_TOTALS], fontsize=9)
    ax.set_xlabel("Total area [m²]")
    ax.set_yticks(range(len(_SITE_TAGS)))
    ax.set_yticklabels([s["label"].replace("\n"," ") for s in _SITES], fontsize=9)
    for i in range(_ht_matrix.shape[0]):
        for j in range(_ht_matrix.shape[1]):
            v = _ht_matrix[i,j]
            if not np.isnan(v):
                ax.text(j,i,f"{v:.0f}%",ha="center",va="center",fontsize=8,fontweight="bold",
                        color="black" if 70<v<130 else "white")
            else: ax.text(j,i,"—",ha="center",va="center",fontsize=8,color="#aaa")
    plt.tight_layout(); _save("legacy_figS5_fouling_cost_heatmap.png")

    # ── LEGACY FIG S6 ────────────────────────────────────────────────────────
    _tech_labels = ["Wind\n(land-based)","Solar PV\n(utility-scale)","RED\n(fresh-dilute cascade)"]
    _annual_kwh  = [1.5*0.35*8760/1000, 6.0*0.18*8760/1000, 2.0*0.95*8760/1000]
    _cap_factor  = [0.35, 0.18, 0.95]
    _tech_cols   = ["#74add1","#f4a100","#2166ac"]
    _notes       = ["~1.5 W/m²\nCF≈0.35","~6 W/m² peak\nCF≈0.18","~2 W/m² continuous\nCF≈0.95"]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Annual Energy Yield per Membrane/Panel Area\nRED runs near-continuously", fontsize=11, fontweight="bold")
    for ax, vals, ylabel in [(ax1,_annual_kwh,"Annual energy yield [kWh/m²/yr]"),(ax2,_cap_factor,"Capacity factor")]:
        bars = ax.bar(range(3), vals, color=_tech_cols, edgecolor="white", width=0.55)
        for bar, v, note in zip(bars, vals, _notes):
            ax.text(bar.get_x()+bar.get_width()/2, v*1.03, f"{v:.1f}\n{note}", ha="center", va="bottom", fontsize=9)
        ax.set_xticks(range(3)); ax.set_xticklabels(_tech_labels, fontsize=10)
        ax.set_ylabel(ylabel); ax.grid(axis="y", alpha=0.3); ax.set_ylim(0, max(vals)*1.45)
    plt.tight_layout(); _save("legacy_figS6_energy_density_comparison.png")

    # ── LEGACY FIG S7 ────────────────────────────────────────────────────────
    _sites_s7 = [s for s in _SITES if s["tag"] in ("CARLSBAD","JEBEL-ALI","HYPERSALINE")]
    fig, axes_s7 = plt.subplots(2, 3, figsize=(14, 8), sharex=False)
    fig.suptitle("Serial Cascade Cannot Maintain Compliance — Fresh-Dilute Can", fontsize=11, fontweight="bold")
    for col_idx, sd in enumerate(_sites_s7):
        for row_idx, beta in enumerate([0.0, 0.8]):
            ax = axes_s7[row_idx][col_idx]
            sub = _df_cas[(_df_cas.SITE_TAG==sd["tag"])&(_df_cas.N_TOTAL==_NT240)
                          &(_df_cas.FOULING_BETA==beta)&(_df_cas.C_L_IN_MOL_L==_CL)&(~_df_cas.ANY_INVERSION)]
            for fresh, col, lab in [(True,_CF,"Fresh-dilute"),(False,_CS,"Serial")]:
                grp = sub[sub.FRESH_DILUTE==fresh].sort_values("S")
                if len(grp): ax.plot(grp.S, grp.HARM_REDUX_PCT, "o-", color=col, lw=2.5, ms=6, label=lab)
            ax.axhline(100, color=_CK, ls="--", lw=1.5); ax.axhspan(100, 105, alpha=0.07, color=_CK)
            ax.set_ylim(55, 106)
            ax.set_title(f"{sd['label'].replace(chr(10),' ')} β={beta}", fontsize=9)
            if col_idx==0: ax.set_ylabel("Harm Reduction [%]")
            if row_idx==1: ax.set_xlabel("Stage count S")
            if col_idx==0 and row_idx==0: ax.legend(fontsize=9)
            ax.grid(alpha=0.3)
    plt.tight_layout(); _save("legacy_figS7_serial_compliance_degradation.png")

    # ── LEGACY FIG S8 ────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.set_title("Power vs Compliance — Full Design Space, Carlsbad (β=0)", fontsize=10, fontweight="bold")
    _ss_s8 = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD")&(_df_ss.FOULING_BETA==0.0)].copy()
    _fr_s8 = _df_cas[(_df_cas.SITE_TAG=="CARLSBAD")&(_df_cas.FOULING_BETA==0.0)
                     &(_df_cas.C_L_IN_MOL_L==_CL)&(_df_cas.FRESH_DILUTE==True)&(~_df_cas.ANY_INVERSION)].copy()
    _se_s8 = _df_cas[(_df_cas.SITE_TAG=="CARLSBAD")&(_df_cas.FOULING_BETA==0.0)
                     &(_df_cas.C_L_IN_MOL_L==_CL)&(~_df_cas.FRESH_DILUTE)&(~_df_cas.ANY_INVERSION)].copy()
    ax.plot(_ss_s8.HARM_REDUX_PCT,_ss_s8.P_DENS_W_M2,color="#aaaaaa",lw=1.5,zorder=1)
    ax.scatter(_ss_s8.HARM_REDUX_PCT,_ss_s8.P_DENS_W_M2,s=_ss_s8.TOTAL_AREA_M2*4,c="#888888",alpha=0.45,zorder=2,label="Single-stage")
    ax.scatter(_se_s8.HARM_REDUX_PCT,_se_s8.P_DENS_CASCADE_W_M2,s=_se_s8.TOTAL_AREA_M2*5,c=_CS,alpha=0.55,zorder=3,marker="s",label="Serial cascade")
    ax.scatter(_fr_s8.HARM_REDUX_PCT,_fr_s8.P_DENS_CASCADE_W_M2,s=_fr_s8.TOTAL_AREA_M2*5,c=_CF,alpha=0.70,zorder=4,label="Fresh-dilute cascade")
    ax.axvline(100,color=_CK,ls="--",lw=2,zorder=5)
    ax.set_xlabel("Harm Reduction [%]",fontsize=11); ax.set_ylabel("Power density [W/m²]",fontsize=11)
    ax.legend(fontsize=9,loc="upper left"); ax.grid(alpha=0.25); ax.set_xlim(-2,108)
    plt.tight_layout(); _save("legacy_figS8_master_design_space.png")

    # ── LEGACY FIG S9 ────────────────────────────────────────────────────────
    fig, axes_s9 = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("Regime Curves — Single-Stage RED, Carlsbad", fontsize=11, fontweight="bold")
    for beta, col in zip(_BETA4, _PAL4):
        sub = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD")&(_df_ss.FOULING_BETA==beta)].sort_values("N")
        if not len(sub): continue
        win = sub[sub.PHYS_COMPLY==True]
        axes_s9[0].plot(sub.N,sub.P_DENS_W_M2,color=col,lw=2,label=f"β={beta:.1f}")
        axes_s9[1].plot(sub.N,sub.HARM_REDUX_PCT,color=col,lw=2,label=f"β={beta:.1f}")
        if len(win):
            n_e = win.N.min()
            axes_s9[0].scatter([n_e],[sub[sub.N==n_e].P_DENS_W_M2.values[0]],color=col,s=80,zorder=5,marker="|")
            axes_s9[1].axvspan(win.N.min(),win.N.max(),alpha=0.12,color=col)
    _pk_s9 = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD")&(_df_ss.FOULING_BETA==0.0)]
    _pk_s9 = _pk_s9.loc[_pk_s9.P_DENS_W_M2.idxmax()]
    axes_s9[0].axhline(2.5719,color=_CF,ls="--",lw=1.8,alpha=0.8,label="Fresh-dilute best compliant")
    axes_s9[0].axhline(1.0435,color="#888",ls=":",lw=1.5,alpha=0.8,label="SS best compliant β=0")
    axes_s9[0].set_xlabel("Cell pairs N"); axes_s9[0].set_ylabel("Power density [W/m²]")
    axes_s9[0].legend(fontsize=8); axes_s9[0].grid(alpha=0.25)
    axes_s9[1].axhline(100,color=_CK,ls="--",lw=1.8)
    axes_s9[1].set_xlabel("Cell pairs N"); axes_s9[1].set_ylabel("Harm Reduction [%]")
    axes_s9[1].legend(fontsize=8); axes_s9[1].grid(alpha=0.25)
    plt.tight_layout(); _save("legacy_figS9_regime_curves.png")

    # ── LEGACY FIG S10 ───────────────────────────────────────────────────────
    fig, axes_s10 = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
    fig.suptitle("Power Density Landscape Over (Area, Stage Count) — Fresh-Dilute Cascade, Carlsbad", fontsize=10, fontweight="bold")
    for ax, beta, title in zip(axes_s10,[0.0,1.5],["β=0.0 (clean)","β=1.5 (severe fouling)"]):
        sub = _df_cas[(_df_cas.SITE_TAG=="CARLSBAD")&(_df_cas.FOULING_BETA==beta)
                      &(_df_cas.C_L_IN_MOL_L==_CL)&(_df_cas.FRESH_DILUTE==True)&(~_df_cas.ANY_INVERSION)].copy()
        sub["area"] = sub.N_TOTAL*_AMEM
        sc = ax.scatter(sub.area,sub.S,c=sub.P_DENS_CASCADE_W_M2,cmap="plasma",vmin=0.7,vmax=2.95,s=120,marker="s",zorder=3)
        comply = sub[sub.PHYS_COMPLY==True]
        ax.scatter(comply.area,comply.S,c=comply.P_DENS_CASCADE_W_M2,cmap="plasma",vmin=0.7,vmax=2.95,
                   s=160,marker="s",edgecolors=_CK,linewidths=2.5,zorder=4)
        if len(comply):
            best = comply.loc[comply.P_DENS_CASCADE_W_M2.idxmax()]
            ax.scatter([best.area],[best.S],color="white",s=200,marker="*",zorder=6,edgecolors="black",linewidths=1)
        ax.set_xlabel("Total area [m²]",fontsize=10); ax.set_title(title,fontsize=10)
        ax.grid(alpha=0.2,color="white"); ax.set_facecolor("#1a1a2e")
    axes_s10[0].set_ylabel("Stage count S",fontsize=10)
    plt.colorbar(sc,ax=axes_s10,label="Power density [W/m²]",shrink=0.85,pad=0.02)
    plt.tight_layout(); _save("legacy_figS10_design_space_heatmap.png")

    # ── LEGACY FIG S11 ───────────────────────────────────────────────────────
    fig, axes_s11 = plt.subplots(2, 2, figsize=(13, 9), sharey=True, sharex=True)
    fig.suptitle("How Fouling Reshapes the Design Landscape — Fresh-Dilute Cascade, Carlsbad", fontsize=10, fontweight="bold")
    _af = axes_s11.flatten()
    for ax, beta in zip(_af, [0.0,0.4,0.8,1.5]):
        sub = _df_cas[(_df_cas.SITE_TAG=="CARLSBAD")&(_df_cas.FOULING_BETA==beta)
                      &(_df_cas.C_L_IN_MOL_L==_CL)&(_df_cas.FRESH_DILUTE==True)&(~_df_cas.ANY_INVERSION)].copy()
        sub["area"] = sub.N_TOTAL*_AMEM
        sc11 = ax.scatter(sub.area,sub.S,c=sub.P_DENS_CASCADE_W_M2,cmap="viridis",vmin=0.7,vmax=2.95,s=90,marker="s",zorder=2,alpha=0.85)
        comply = sub[sub.PHYS_COMPLY==True]
        ax.scatter(comply.area,comply.S,c=comply.P_DENS_CASCADE_W_M2,cmap="viridis",vmin=0.7,vmax=2.95,
                   s=120,marker="s",edgecolors=_CK,linewidths=2,zorder=3,alpha=0.95)
        for ntot in _N_TOTALS:
            sub_n = comply[comply.N_TOTAL==ntot]
            if len(sub_n):
                best_n = sub_n.loc[sub_n.P_DENS_CASCADE_W_M2.idxmax()]
                ax.scatter([best_n.area],[best_n.S],marker="*",color="white",s=150,zorder=5,edgecolors="#ffcc00",linewidths=1)
        ax.set_title(f"β = {beta:.1f}",fontsize=11,fontweight="bold")
        ax.grid(alpha=0.15,color="white"); ax.set_facecolor("#1a1a2e")
        if ax in axes_s11[:,0]: ax.set_ylabel("Stage count S",fontsize=9)
        if ax in axes_s11[1,:]: ax.set_xlabel("Total area [m²]",fontsize=9)
    plt.colorbar(sc11,ax=_af,label="Power density [W/m²]",shrink=0.7,pad=0.02)
    plt.tight_layout(); _save("legacy_figS11_fouling_design_landscape.png")

    # ── LEGACY FIG S12 ───────────────────────────────────────────────────────
    fig, axes_s12 = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
    fig.suptitle("All Three Architectures at Matched Membrane Investment — Carlsbad, β=0", fontsize=11, fontweight="bold")
    _ss_b0_s12 = _df_ss[(_df_ss.SITE_TAG=="CARLSBAD")&(_df_ss.FOULING_BETA==0.0)].sort_values("N").copy()
    _fr12, _se12 = [], []
    for ntot in _N_TOTALS:
        fr = _df_cas[(_df_cas.SITE_TAG=="CARLSBAD")&(_df_cas.N_TOTAL==ntot)&(_df_cas.FOULING_BETA==0.0)
                     &(_df_cas.C_L_IN_MOL_L==_CL)&(_df_cas.FRESH_DILUTE==True)&(~_df_cas.ANY_INVERSION)]
        se = _df_cas[(_df_cas.SITE_TAG=="CARLSBAD")&(_df_cas.N_TOTAL==ntot)&(_df_cas.FOULING_BETA==0.0)
                     &(_df_cas.C_L_IN_MOL_L==_CL)&(~_df_cas.FRESH_DILUTE)&(~_df_cas.ANY_INVERSION)]
        if len(fr):
            best=fr.loc[fr.P_DENS_CASCADE_W_M2.idxmax()]
            _fr12.append({"area":ntot*_AMEM,"P":best.P_DENS_CASCADE_W_M2,"HR":best.HARM_REDUX_PCT,"comply":best.PHYS_COMPLY})
        if len(se):
            best=se.loc[se.P_DENS_CASCADE_W_M2.idxmax()]
            _se12.append({"area":ntot*_AMEM,"P":best.P_DENS_CASCADE_W_M2,"HR":best.HARM_REDUX_PCT,"comply":best.PHYS_COMPLY})
    _df_fr12=pd.DataFrame(_fr12); _df_se12=pd.DataFrame(_se12)
    ax1_12=axes_s12[0]; ax2_12=axes_s12[1]
    ax1_12.plot(_ss_b0_s12.TOTAL_AREA_M2,_ss_b0_s12.P_DENS_W_M2,color=_CSS,lw=2,label="Single-stage",zorder=2)
    ax1_12.plot(_df_fr12.area,_df_fr12.P,"o-",color=_CF,lw=2.5,ms=8,label="Fresh-dilute",zorder=3)
    ax1_12.plot(_df_se12.area,_df_se12.P,"s-",color=_CS,lw=2.5,ms=8,label="Serial",zorder=3)
    _fr_c12=_df_fr12[_df_fr12.comply==True]; _se_c12=_df_se12[_df_se12.comply==True]
    ax1_12.scatter(_fr_c12.area,_fr_c12.P,color=_CF,s=120,zorder=4,edgecolors=_CK,linewidths=2.5,label="Compliant ✓")
    ax1_12.scatter(_se_c12.area,_se_c12.P,color=_CS,s=120,zorder=4,edgecolors=_CK,linewidths=2.5)
    ax1_12.set_ylabel("Peak power density [W/m²]",fontsize=10); ax1_12.legend(fontsize=9); ax1_12.grid(alpha=0.25)
    ax2_12.plot(_ss_b0_s12.TOTAL_AREA_M2,_ss_b0_s12.HARM_REDUX_PCT,color=_CSS,lw=2,label="Single-stage")
    ax2_12.plot(_df_fr12.area,_df_fr12.HR,"o-",color=_CF,lw=2.5,ms=8,label="Fresh-dilute")
    ax2_12.plot(_df_se12.area,_df_se12.HR,"s-",color=_CS,lw=2.5,ms=8,label="Serial")
    ax2_12.axhline(100,color=_CK,ls="--",lw=2,label="Full compliance")
    ax2_12.set_xlabel("Total area [m²]",fontsize=10); ax2_12.set_ylabel("Harm Reduction [%]",fontsize=10)
    ax2_12.legend(fontsize=9); ax2_12.grid(alpha=0.25); ax2_12.set_ylim(0,108)
    plt.tight_layout(); _save("legacy_figS12_three_arch_vs_area.png")

    # ══════════════════════════════════════════════════════════════════════════
    print(f"\n{'='*60}")
    print("[FIGS v7] COMPLETE FIGURE SUMMARY")
    print(f"{'='*60}")
    _all_figs = sorted(_FIG_DIR.glob("*.png"))
    _groups = {
        "Paper main    (paper_fig*)":   [f for f in _all_figs if f.name.startswith("paper_fig")],
        "Paper supp    (paper_supp*)":  [f for f in _all_figs if f.name.startswith("paper_supp")],
        "GENIUS poster (genius_fig*)":  [f for f in _all_figs if f.name.startswith("genius_fig")],
        "Legacy fig    (legacy_fig[0-9]*)": [f for f in _all_figs
                                             if f.name.startswith("legacy_fig")
                                             and not f.name.startswith("legacy_figN")
                                             and not f.name.startswith("legacy_figS")],
        "Legacy figN   (legacy_figN*)": [f for f in _all_figs if f.name.startswith("legacy_figN")],
        "Legacy figS   (legacy_figS*)": [f for f in _all_figs if f.name.startswith("legacy_figS")],
    }
    total = 0
    for group, figs in _groups.items():
        print(f"\n{group} — {len(figs)} figures:")
        for f in figs:
            print(f"  {f.name}")
        total += len(figs)
    print(f"\nTotal: {total} figures written to {_FIG_DIR}")